# Row-wise L2 norm and normalization

Embedding search L2-normalises every vector so that a dot product becomes a cosine similarity. Write a kernel that computes each row's L2 norm and the normalised row together.

Write `l2_normalize(x, eps)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor of shape `(M, D)`, $M, D \ge 1$. `D` may be larger than one block.
- `eps`: Python float, e.g. `1e-12`.

## Output
A tuple `(y, norms)`:
- `norms`: `float32` tensor of shape `(M,)`, $\lVert x_i \rVert_2 = \sqrt{\sum_j x_{ij}^2}$.
- `y`: `float32` tensor of shape `(M, D)`, $y_i = x_i / \max(\lVert x_i\rVert_2,\ \varepsilon)$. An all-zero row therefore maps to all zeros (the same rule as `torch.nn.functional.normalize`).

## Requirements
- One program per row. First loop over the row accumulating squares in a lane-wise `float32` accumulator, reduce it once, then loop again to scale and store.
- `BLOCK` is a `tl.constexpr` power of two; mask every load and store.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def l2_normalize(x, eps):
    """x: (M, D) float32 CUDA tensor. Return (x / max(||x_i||, eps), norms (M,))."""
    raise NotImplementedError


@triton.jit
def _l2_normalize_kernel(x_ptr, y_ptr, norm_ptr, D, stride_row, eps, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[3.0, 4.0], [1.0, 0.0]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[0.6000000238418579, 0.800000011920929], [1.0, 0.0]], "dtype": "float32"}, {"__nd__": [5.0, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0, 1.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[0.5, 0.5, 0.5, 0.5], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [2.0, 0.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[9.99999993922529e-09, 0.0, 0.0]], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[0.009999999776482582, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [9.99999993922529e-09], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-2.0]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[-1.0]], "dtype": "float32"}, {"__nd__": [2.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.8267565965652466, -3.07833194732666, 0.9580639600753784, 0.0696372240781784, 1.318250060081482, 0.38562923669815063, 1.827258586883545, 0.03174375742673874, -0.5162294507026672, 0.580484926700592, 0.4321068525314331, -0.3568393588066101, -0.2473038285970688, 0.7194406986236572, 0.7043160200119019, -0.4939342439174652, -0.36771371960639954, -1.8067903518676758, 1.679207444190979, -0.2242908775806427, 1.3372774124145508, 0.41746559739112854, 1.943962812423706, 1.5371099710464478, 0.31829842925071716, 1.4807634353637695, -0.9501235485076904, 1.2586181163787842, -1.4804235696792603, 0.34323635697364807, 1.0648764371871948, 0.22363214194774628, -0.36713749170303345, -0.8055600523948669, -0.3428000211715698, 1.051125168800354, 0.8908394575119019, -0.2621314525604248, -1.246004343032837, 0.6739972233772278, -1.449873924255371, -0.5308548808097839, -0.7348283529281616, 0.7432652115821838, 0.23594973981380463, 0.4618547260761261, 0.2724079191684723, -0.6779301166534424, 0.5354759693145752, 1.4124603271484375, -0.03677124157547951, 0.6335944533348083, -0.1259031891822815, 1.0285553932189941, 0.6666366457939148, 0.8758194446563721, 0.3484230041503906, 1.6400039196014404, -0.3611505925655365, -0.3341660797595978, -0.5918599963188171, 0.6109626889228821, -0.6224002838134766, -0.6445326209068298, 0.7274748086929321, 1.1620957851409912, 0.5743514895439148, -2.683103561401367, -0.9536324739456177, -1.0734894275665283, -1.2075111865997314, -0.407767653465271, 1.8175432682037354, -0.30739855766296387, -0.7354457974433899, 0.6988227367401123, 0.0691717267036438, 0.12068112939596176, -0.6866741180419922, -0.0015044669853523374, 0.4345371723175049, -1.911306619644165, -1.6102243661880493, 1.084560513496399, 0.843004584312439, -0.5066567063331604, -1.020203948020935, -0.1855466216802597, -1.1936522722244263, -0.34009698033332825, 1.6511003971099854, 0.1320718377828598, 1.1662884950637817, -0.22262053191661835, 0.3694155514240265, 1.3858305215835571, -1.3359375, -0.3474780023097992, -0.15689849853515625, 0.2819811701774597, 1.5536667108535767, 0.6855270266532898, -1.497666597366333, -0.27097204327583313, -0.8936434388160706, -0.9190470576286316, 2.024827718734741, -1.0629706382751465, 0.23639518022537231, -0.7152122259140015, 0.8609809279441833, 0.35370323061943054, 0.12742771208286285, -0.07616713643074036, 1.1377168893814087, -0.9316998720169067, -0.7269383072853088, -1.1166059970855713, 1.334546446800232, -0.29356637597084045, -1.037400722503662, -0.4190734028816223, 2.4996471405029297, -0.10747295618057251, 0.8500205874443054, 0.465181827545166, -0.01238328218460083, -0.692704975605011, -0.33429524302482605, -0.3544524013996124, -2.2092761993408203, 2.590132236480713, 0.924836277961731, 0.1162865087389946, -0.20334914326667786, -0.7896443605422974, 0.9496217370033264, -0.1868559867143631, 0.4792725443840027, -1.3874951601028442, 0.3784765303134918, 2.0003011226654053, -0.28962886333465576, 0.7999549508094788, 0.03533363714814186, 1.1051905155181885, 1.0674245357513428, 0.06164637207984924, -0.559585452079773, 0.28675416111946106, 0.04176902770996094, -0.6179032921791077, 0.4564788341522217, 0.4323228597640991, -0.8890265822410583, -0.11148789525032043, 2.9211394786834717, 0.8168345093727112, 0.09777931123971939, 2.511082410812378, -0.455403208732605, -2.476854085922241, -1.906941533088684, -1.5195109844207764, 0.2600550651550293, 0.9763562679290771, -2.1701903343200684, 1.117243766784668, -1.2277384996414185, 0.2564348578453064, 0.46391960978507996, 0.43232983350753784, 0.007547080051153898, 0.17099551856517792, -1.1398200988769531, 0.2782168984413147, 0.9796523451805115, 0.9262226223945618, 2.6128528118133545, 1.0832325220108032, 0.7264273762702942, 1.6635005474090576, 0.24113763868808746, -1.28987717628479, -2.2398598194122314, 1.2765709161758423, 0.568031370639801, 0.847296953201294, -0.3328748941421509, 0.34482941031455994, 1.882381558418274, 0.33483418822288513, -1.1155186891555786, -0.4314497113227844, 1.38511061668396, -1.089961051940918, -2.5814294815063477, -2.293955087661743, -1.3389389514923096, 0.7993935346603394, 0.0884632021188736, -0.7572771310806274, -1.2851365804672241, -0.11398930847644806, -0.15959563851356506, -0.5639269351959229, -0.24525414407253265, -0.7631818056106567, 1.8032219409942627, -2.366941452026367, 0.12218885868787766, -1.014405608177185, 0.706516444683075, -0.2351061999797821, 0.36602145433425903, 0.5315962433815002, 0.20532624423503876, 0.9910404682159424, -1.3914777040481567, 0.951698899269104, -0.21894244849681854, 0.38064438104629517, 1.5565333366394043, 1.1244068145751953, 1.9839062690734863, -2.25059175491333, -0.955881655216217, 0.9641676545143127, -1.1306781768798828, -0.18259280920028687, 1.7078758478164673, 0.8747000098228455, -0.9622686505317688, 0.45380643010139465, 0.6453005075454712, 0.06737638264894485, -0.10050634294748306, 0.1415574699640274, -1.5005136728286743, 0.12651242315769196, -0.9669110774993896, 0.39357107877731323, 0.4292169511318207, -0.8927932977676392, 0.6126711368560791, 0.3558124303817749, -0.04209628701210022, 0.725806713104248, -0.3127966821193695, -0.28415799140930176, 0.07778617739677429, -0.05922280251979828, 1.1421668529510498, -2.3739452362060547, -0.31047356128692627, -2.014955997467041, -0.5262386798858643, 0.23222823441028595, -0.0221461933106184, 0.9393318891525269, 0.2866276502609253, -2.776874542236328, -1.158342957496643, 1.5136624574661255, 1.5159204006195068, 0.052347078919410706, -1.6017876863479614, 2.1188039779663086, -2.1530354022979736, 0.2502431571483612, -0.1118730828166008, 0.17806975543498993, -1.2407174110412598, -0.4722427725791931, 1.0021493434906006, -0.5663898587226868, -1.4614338874816895, -0.0968598946928978, 0.006268835626542568, -0.22062526643276215, -0.8773841261863708, 0.4550834596157074, -0.2132079303264618, -1.2648979425430298, 0.13264484703540802, -0.11691341549158096, 0.05893535912036896, -1.793182373046875, -0.8577885627746582, 0.3729998469352722, 0.8514968752861023, 0.26848822832107544, 0.08492853492498398, 0.6574751138687134, 0.9197622537612915, 1.7089096307754517, 0.4161616563796997, -0.6178988218307495, 0.43328857421875, -1.994313359260559, 0.020228339359164238, -0.4773692488670349, -0.0515926256775856, 0.46636250615119934, 0.028592398390173912, -1.4722402095794678, -0.376668244600296, -0.8190875053405762, -0.03521423414349556, 0.34775832295417786, -0.9762750267982483, 0.723614513874054, 1.2932844161987305, -0.2788844108581543, 0.7214635610580444, -1.4895076751708984, -0.5475362539291382, 0.957765519618988, -1.7717266082763672, -0.5123825669288635, -0.9021143913269043, 1.1438186168670654, 1.1407461166381836, -0.6291700005531311, 0.11823820322751999, -2.6185953617095947, 0.8450719714164734, 0.34489306807518005, 1.32167649269104, -2.393613338470459, 0.3424515426158905, -0.35849234461784363, -2.6175827980041504, -0.6984831690788269, -0.08622463047504425, 0.7974618077278137, 0.1902010291814804, -0.5223810076713562, 0.43909767270088196, -0.11440622806549072, -0.6565077304840088, -1.0265802145004272, 0.14284253120422363, 0.15089504420757294, 1.835772156715393, -0.45719438791275024, 1.599266529083252, 0.3460620939731598, 1.173685908317566, -0.9101955890655518, -0.8784043192863464, -0.31216081976890564, 0.8927077651023865, 0.1732492744922638, 0.7507971525192261, 0.31595873832702637, 1.027280569076538, 0.19700369238853455, -0.5264894962310791, 0.5140479207038879, 0.09003371000289917, 1.748336672782898, -0.15972213447093964, 0.10902445018291473, -0.30643951892852783, 1.1561105251312256, 0.8746158480644226, 0.6510992050170898, -1.2466926574707031, -1.516545057296753, 0.2648726999759674, 0.7220378518104553, 1.4791980981826782, -1.795693278312683, -0.17030522227287292, -1.6904635429382324, -0.9607281684875488, -0.22110231220722198, 0.9630199074745178, -0.5266167521476746, -0.7181774973869324, -0.31983038783073425, 1.2576661109924316, 2.4159128665924072, -1.5420101881027222, -0.2734977900981903, -0.43029892444610596, 1.1688016653060913, 0.43566620349884033, 1.540729284286499, -1.0121275186538696, 0.013772513717412949, -0.45555511116981506, -1.096416711807251, -0.15803399682044983, -1.1844993829727173, 2.172621726989746, 0.4172911047935486, -1.0164598226547241, -0.6060484051704407, 1.7326364517211914, -0.478118360042572, 0.6941054463386536, 0.5836544036865234, 1.0927317142486572, 0.6347154378890991, -0.2101142555475235, -0.5914984345436096, -1.7757763862609863, -0.04765275865793228, 0.13870155811309814, -0.3601946234703064, -0.14363735914230347, -1.2139527797698975, 2.171170234680176, -0.5010127425193787, -1.035396695137024, -0.5970107913017273, -0.16847090423107147, 0.7418992519378662, -2.4273548126220703, 0.515755832195282, -0.615847647190094, -0.17923468351364136, 1.0174627304077148, 0.7666960954666138, -0.6781798005104065, -0.0927329957485199, 1.8740146160125732, 1.363098382949829, 0.8707621097564697, 0.011627346277236938, 0.9657787680625916, 0.4004381597042084, -0.5452161431312561, -0.997035801410675, 0.5270590782165527, 0.38650283217430115, 1.2115569114685059, -0.478994220495224, 1.4068458080291748, -0.3610649108886719, 1.2603949308395386, 0.5507586598396301, -0.5727424025535583, -0.9430860280990601, -0.11643341928720474, -0.26016443967819214, 0.9442658424377441, -0.6876183152198792, -1.474273443222046, -0.014948368072509766, 1.9265888929367065, 1.6422021389007568, -0.6180402040481567, -0.27046045660972595, 0.32261648774147034, -0.6595926880836487, -0.2415473908185959, -0.271843284368515, -1.0465047359466553, -0.6149859428405762, -0.4814890921115875, 0.23836323618888855, -1.1797868013381958, 0.10856445878744125, -0.9789382219314575, -0.8058544397354126, -1.1210262775421143, 4.015010356903076, 1.2599880695343018, 1.3482909202575684, 0.8161882758140564, -0.23215779662132263, 0.07068551331758499, 0.2922717332839966, 0.45044153928756714, 0.5978001952171326, 1.5296721458435059, -2.9571878910064697, -0.5523435473442078, 0.680171549320221, -0.3554946184158325, -0.1541263461112976, 1.4936680793762207, 0.194073885679245, 1.7850669622421265, -0.041467588394880295, 0.23957011103630066, -1.064625859260559, -1.9856423139572144, -0.41670241951942444, 0.06352199614048004, -0.3439176082611084, -1.346735954284668, 0.6585364937782288, 0.07284577935934067, 0.010964881628751755, -0.7770242094993591, 1.6502982378005981, 1.479138731956482, 0.30928856134414673, -0.11463054269552231, 0.25414302945137024, 1.697879672050476, 2.811840057373047, 0.013234156183898449, -0.42579028010368347, -1.8115838766098022, 2.7992970943450928, -1.103662133216858, 0.8742792010307312, 0.5016737580299377, 1.3359389305114746, -0.8596674203872681, -1.6201274394989014, 0.15047596395015717, 0.4127308428287506, 1.1513655185699463, -1.6848421096801758, -0.4526776671409607, -0.2618641257286072, -1.3526510000228882, -0.16133297979831696, 0.8022027015686035, -0.03215410187840462, -1.4381428956985474, 0.3487955331802368, 0.8612946271896362, 0.5389732718467712, 0.7925052046775818, -1.6518425941467285, 0.4636634886264801, 0.6031314134597778, -0.37379369139671326, -0.05242816358804703, -0.7700425386428833, 1.2389658689498901, -0.2961309850215912, -0.9423303604125977, 0.18934915959835052, 0.11445123702287674, 0.6718542575836182, -0.8322688341140747, -0.674513578414917, 0.3867115080356598, 2.2409212589263916, 0.47039511799812317, -1.3610966205596924, 0.1720748245716095, -0.7765471935272217, -1.8503470420837402, -0.6822056770324707, -0.5729925632476807, 0.42174381017684937, 1.3251672983169556, 1.1429766416549683, 0.813230037689209, -0.42983052134513855, 0.15966148674488068, -0.03665835037827492, 1.4138050079345703, -0.6498426198959351, -0.5474255681037903, 1.5014197826385498, 0.581462562084198, -0.4462331235408783, 0.1689763367176056, 0.1537400335073471, 0.1932879537343979, 0.8058782815933228, 0.3717736601829529, -0.1795397251844406, -0.5024283528327942, -1.7252230644226074, -0.43279680609703064, 1.1511259078979492, 0.23226463794708252, -0.6488568782806396, -0.23847316205501556, -0.8530846238136292, 0.29613813757896423, 2.0159146785736084, 1.8162868022918701, 1.2076218128204346, 0.789933979511261, -0.21979403495788574, -1.2355077266693115, 0.019639477133750916, -0.4406792223453522, 0.8423542380332947, 0.16933010518550873, -0.5659253597259521, 0.46401605010032654, 2.9363458156585693, -0.8966197371482849, 1.1501190662384033, 0.6092477440834045, -0.08420709520578384, -0.7451052069664001, 0.03494150936603546, 0.23494111001491547, 0.7859030961990356, 0.7207475900650024, 0.33564358949661255, -0.6681472659111023, 0.31849390268325806, -0.4522289037704468, -1.030527114868164, 0.9123314619064331, -0.6508362889289856, -1.1833113431930542, 0.09823982417583466, -1.9197437763214111, -0.05340863764286041, 0.46027740836143494, -0.4099349081516266, 0.24871130287647247, 2.167280673980713, 1.4355412721633911, 0.9425172209739685, 0.60926353931427, -0.5095142126083374, -0.1322103589773178, -0.27942389249801636, 0.033601392060518265, 0.12272394448518753, -0.8049627542495728, 1.6188915967941284, -1.0382230281829834, 0.3666912019252777, 1.0378625392913818, -0.026483602821826935, -0.6680334806442261, 0.033419422805309296, 1.9428584575653076, 0.6261038780212402, -0.2896069288253784, 0.5850505828857422, -0.19636918604373932, -1.8765511512756348, 0.2443511039018631, 1.0430858135223389, 1.1554746627807617, -0.06525295227766037, -1.412984848022461, 1.1992993354797363, -0.15496523678302765, 0.7376782298088074, 1.130954623222351, 0.3386093080043793, -0.06246346980333328, -0.6303871870040894, -0.4355660080909729, 0.02830701693892479, -0.46638649702072144, 0.3600184917449951, -1.9614372253417969, -1.0189257860183716, -0.03862045332789421, -0.7034804224967957, -0.5474863648414612, -1.1492044925689697, 0.6610785126686096, 1.6204195022583008, -1.0199918746948242, 1.1366065740585327, -1.4700734615325928, 0.12453320622444153, -2.0203676223754883, 0.46881750226020813, 0.0036097730044275522, 0.12210457772016525, 1.198567509651184, 0.5728311538696289, -0.04823875427246094, -0.8424931168556213, 0.02666337415575981, -0.7181218862533569, -0.44833970069885254, 1.4526641368865967, 0.8495866060256958, 0.2602483332157135, 0.6003328561782837, -1.4725618362426758, 0.0762244462966919, -1.7234350442886353, -0.6984401941299438, 0.3361152708530426, 0.34210526943206787, 0.6642372012138367, 0.11605605483055115, 0.008802109397947788, -0.04673934727907181, 1.3597087860107422, 1.2530686855316162, -1.5053163766860962, 0.07836554199457169, 1.2380720376968384, -0.31057482957839966, 0.16271592676639557, -0.6616724729537964, -1.2639461755752563, 0.06622907519340515, -0.48092004656791687, -0.2385380119085312, -1.529490351676941, -1.2471624612808228, -0.22655127942562103, -0.5447095632553101, 0.4433755874633789, -1.1642268896102905, 0.5917767286300659, 0.4475606381893158, -0.03319079056382179, 0.5150267481803894, -0.21642622351646423, -0.3321765065193176, -0.16966304183006287, -1.4113640785217285, -0.18801479041576385, 0.6438881754875183, -1.7600427865982056, 0.5280057787895203, -0.7939373850822449, -0.5173986554145813, -0.07322858273983002, 0.08427481353282928, 1.197696328163147, -0.30942636728286743, 1.5460668802261353, -0.1225753128528595, 0.3228130638599396, 0.5140435099601746, -0.8229801058769226, -1.271087884902954, -0.03993978351354599, 0.5501641631126404, 0.7179550528526306, -0.49595627188682556, 1.6254194974899292, -0.24865861237049103, -0.7929532527923584, 1.0531949996948242, 1.127305030822754, -1.0327833890914917, 0.7539494633674622, -1.326534390449524, -0.011146479286253452, 2.2225890159606934, 1.256201148033142, 0.6820923089981079, 0.6328900456428528, -0.8301624655723572, 0.33287736773490906, 0.4535401165485382, -1.0224860906600952, 0.21752507984638214, 0.5647282600402832, 0.23615916073322296, 0.4539389908313751, -0.19956548511981964, -1.0907214879989624, 0.06515006721019745, -1.8843028545379639, -0.6782270669937134, 1.1282567977905273, 1.822555422782898], [0.44775065779685974, -2.3638741970062256, 1.3792015314102173, 0.5875310897827148, -0.14244890213012695, 0.7963337302207947, 0.5948165655136108, -1.1406171321868896, -1.2444143295288086, 1.089666724205017, -2.237473964691162, 0.6528853178024292, 0.37071892619132996, 1.034990906715393, 1.7988975048065186, 1.3954088687896729, 1.6155163049697876, 0.8849202394485474, 0.5726985335350037, 1.1388957500457764, 0.060974378138780594, -0.8375439643859863, -0.005138245411217213, -1.5233927965164185, -0.4762767553329468, 2.1337692737579346, 0.5898863673210144, -0.104012630879879, -0.5286069512367249, -1.6987944841384888, -0.4690552055835724, -0.7093132138252258, 0.06291770190000534, 0.14164426922798157, -0.00807039812207222, -1.8494209051132202, 1.5060701370239258, 0.4952414035797119, 0.06110600382089615, -0.27262139320373535, 0.489059180021286, 1.2918702363967896, 0.6525207757949829, -1.1143360137939453, -1.2102105617523193, -2.320026159286499, 0.15964266657829285, 0.7433220744132996, 1.242090106010437, 0.15513943135738373, 0.8338120579719543, -0.8992630839347839, 0.5323162078857422, -0.11481258273124695, -0.6581799387931824, -1.6737364530563354, 1.156531572341919, -2.2498652935028076, 0.6252995729446411, -0.802219033241272, 0.605822741985321, -0.5549833178520203, 0.7501620650291443, -0.2201387733221054, -1.0477230548858643, -0.5206915736198425, -1.6154813766479492, 0.26982662081718445, 1.0361037254333496, 0.4142778813838959, 0.10155212134122849, 1.2841352224349976, -1.1704825162887573, 0.5507282614707947, 0.0051128738559782505, -0.4749050736427307, 0.5451962947845459, -1.1485017538070679, -0.13800767064094543, 0.3336654305458069, -1.168419599533081, -0.23796668648719788, 0.5225855708122253, 0.9484075903892517, -0.9456542134284973, 0.8959514498710632, -0.2982275187969208, 0.08971983194351196, -0.5518433451652527, 0.7771668434143066, -0.7082753777503967, 0.27944210171699524, 0.3755491077899933, 0.7516474723815918, -1.0428359508514404, 0.7702812552452087, 0.0294971764087677, 1.4479398727416992, -0.3476806581020355, -0.5720516443252563, -0.061392642557621, -0.31712234020233154, -0.17466117441654205, -1.2737936973571777, -2.087588310241699, -1.4747819900512695, -0.5981080532073975, 0.4211537539958954, -0.319654643535614, -0.968058705329895, -0.1642845720052719, 0.05345994979143143, 2.7524750232696533, 1.2700642347335815, 0.17909152805805206, -1.4630720615386963, -0.18444138765335083, -0.313263863325119, 0.5465628504753113, 2.1136679649353027, -0.07376551628112793, -1.9787038564682007, -0.12475525587797165, -1.6850863695144653, 0.8307342529296875, 1.0978639125823975, 0.3287368714809418, -1.4913262128829956, -0.18334804475307465, 1.208828330039978, -1.0075623989105225, -0.595819354057312, 0.2521924674510956, 0.49409934878349304, 0.07677619904279709, 0.3376830220222473, 0.2061121165752411, -0.5821773409843445, -0.08351884037256241, 0.28280600905418396, -1.0921801328659058, 0.1939898580312729, 0.0971963107585907, 0.46132692694664, 0.347756028175354, -1.0491350889205933, -1.258439064025879, 0.25008273124694824, -0.15061555802822113, 0.10054108500480652, -0.563538134098053, -1.2778362035751343, 0.2675737738609314, -0.1321062445640564, 0.6137542128562927, 1.0908143520355225, 1.0918139219284058, 0.9876492023468018, -0.9601809978485107, -2.1120848655700684, -0.6170183420181274, -1.2131603956222534, -1.323216199874878, 0.6497148275375366, -1.6165415048599243, 0.9054979681968689, 1.087734580039978, 2.6532206535339355, 1.0304796695709229, 2.088303565979004, -0.6487269401550293, -0.6892934441566467, 0.9701080918312073, -1.3215715885162354, 1.1325098276138306, -0.28140175342559814, -1.9719762802124023, 1.8421061038970947, -1.1265901327133179, 2.4894070625305176, -1.4824445247650146, -0.8530563712120056, 0.20653612911701202, 0.00630975840613246, -1.5863792896270752, 0.10066196322441101, 0.17944872379302979, 0.4475690722465515, -1.6133707761764526, -0.6118007898330688, -0.11296556890010834, 0.7982983589172363, 0.15912777185440063, -0.3530378043651581, -0.43053120374679565, 1.452958583831787, -0.8240624666213989, 0.2922510504722595, -2.1742522716522217, 1.6942808628082275, 0.4529268741607666, -1.76482093334198, 0.5994302034378052, 1.3820371627807617, 0.8122643828392029, 0.07000603526830673, 0.8404346108436584, -0.7480292320251465, -0.2588430643081665, -0.3622119724750519, 0.07093449681997299, -1.0481683015823364, 1.372219443321228, -0.12945406138896942, -1.0834840536117554, -0.2228236198425293, 0.3603581190109253, 0.04042252525687218, 1.0949475765228271, -0.3619973063468933, -0.7752352952957153, 0.4961172938346863, 1.2738149166107178, -0.9710970520973206, -1.5918551683425903, -0.6492876410484314, -0.08174177259206772, -1.0354732275009155, 0.15550249814987183, -0.8705102205276489, 0.8839381337165833, -1.195253610610962, -0.9227498769760132, 0.06125394254922867, 0.6640990376472473, 0.9097403287887573, -0.7009552717208862, 0.005886630155146122, 0.5257308483123779, 1.4118064641952515, -0.9054898023605347, -1.2608896493911743, 0.46770378947257996, -0.2671178877353668, 1.331496238708496, 0.008740151301026344, 0.3080451190471649, -0.8016861081123352, 0.26254361867904663, 1.2063270807266235, -0.26474371552467346, -1.2093778848648071, 0.8669277429580688, -0.3272186517715454, 0.03759843111038208, -0.7326170802116394, 0.9229820966720581, -0.7643222808837891, -0.5017606616020203, -0.012400643900036812, -0.2215316891670227, -2.2626779079437256, 0.4496886730194092, -1.186792016029358, 0.23659688234329224, -0.41109034419059753, -0.14485999941825867, -0.7568107843399048, -1.083967685699463, 0.1215178519487381, 0.19358517229557037, 0.9865535497665405, 0.5155532360076904, -0.7235081195831299, 1.1442936658859253, 2.1819169521331787, 0.693253219127655, -0.21459431946277618, -0.9912552237510681, 0.06385394930839539, -1.4611167907714844, 0.7682124972343445, -0.40619340538978577, -0.6010714173316956, 0.23076601326465607, -1.267837405204773, -1.767261266708374, -0.46547597646713257, 1.1878371238708496, -0.8773794770240784, 0.841112494468689, -1.8671338558197021, -0.09455468505620956, 0.48021113872528076, 1.75497305393219, -1.7903146743774414, -0.09582646936178207, -0.9374459981918335, 0.27377140522003174, -0.583678126335144, 0.07032898813486099, -0.7871711850166321, -0.5858622193336487, 0.36704233288764954, 1.4570567607879639, 0.17598600685596466, 0.08886750787496567, -0.49562785029411316, 1.5427303314208984, -1.9707074165344238, -2.021507978439331, -0.8112369179725647, 0.4069746434688568, 0.9122158288955688, 0.5340163707733154, 0.7175655364990234, 0.1304417848587036, -0.5012674331665039, -1.0909396409988403, 0.4139263331890106, -0.9783605337142944, 0.38580411672592163, 0.7065905332565308, -0.648687481880188, -0.47927507758140564, -0.5811485052108765, 0.6487062573432922, 0.12247626483440399, -0.6988786458969116, 0.23322932422161102, 1.4685178995132446, 0.24756722152233124, 0.09258831292390823, -0.4309394359588623, -0.9590957760810852, -0.028836945071816444, 0.03355853259563446, -0.3399026691913605, 1.1443601846694946, -0.2646050453186035, -0.19064830243587494, 0.16042467951774597, 0.49729830026626587, -1.8128713369369507, 0.23078995943069458, 1.2150660753250122, -1.137378454208374, 0.35712364315986633, -0.5346667170524597, -1.0136022567749023, 0.2333964705467224, -0.1293531209230423, -2.0938384532928467, -0.6124882102012634, 0.779683530330658, -0.6108347773551941, -0.7666376829147339, -0.3940533697605133, -0.46816059947013855, 0.7104989886283875, -0.8697062134742737, -1.1843339204788208, 0.40186214447021484, 0.9591366648674011, 0.6692366003990173, 0.8073505163192749, -0.6679377555847168, -1.4835524559020996, -0.7196649312973022, -0.04522299766540527, -1.7043129205703735, -0.3210396468639374, 0.5551077127456665, -0.7054114937782288, 1.9473063945770264, 0.7237274646759033, -1.2243596315383911, -1.1400084495544434, -0.7189697623252869, 0.6822137832641602, 0.44251781702041626, -0.5736066102981567, 0.14966906607151031, 0.772907555103302, 0.49959391355514526, -1.6073123216629028, -1.0937540531158447, -0.9238395690917969, -1.729121208190918, 0.831201434135437, -0.6552144289016724, -0.6666637063026428, -0.8465259671211243, 0.28037118911743164, 0.5423267483711243, 0.6492924690246582, 1.206636667251587, -0.9398483633995056, 1.5910676717758179, 2.436213731765747, -0.5414386987686157, -0.37358731031417847, 0.5808382630348206, 1.2108525037765503, -0.28188395500183105, -0.8142946362495422, 2.0458638668060303, -0.5110553503036499, -2.733222007751465, 0.5183192491531372, 0.9765257835388184, 1.6838244199752808, 1.7480032444000244, 3.0612001419067383, -0.5626429915428162, -0.9858656525611877, -1.4540878534317017, 1.6928385496139526, -0.4414542317390442, -1.6794966459274292, -2.0576424598693848, 0.11891292780637741, -0.4361839294433594, 0.3052470088005066, -0.862024188041687, -0.7415863275527954, 0.45976558327674866, -0.20840628445148468, 0.5679107308387756, -0.1355060636997223, -0.2929134964942932, 0.3522234261035919, -0.3338576555252075, 0.16878901422023773, 0.4375035762786865, -0.517559826374054, -0.1254808008670807, 0.0939362645149231, 0.6600291132926941, 0.22660307586193085, 0.23603983223438263, 0.4526379108428955, -0.3192278742790222, 0.34548914432525635, -0.7427550554275513, -1.8258566856384277, 0.3977758586406708, -1.9598420858383179, -0.16466903686523438, 1.5149199962615967, 0.7077186703681946, -1.024963140487671, 0.7204521298408508, 0.22499941289424896, -0.8434364199638367, -0.1098717525601387, -0.5424540638923645, -0.32217487692832947, 0.3195338249206543, -1.5536657571792603, 0.8300284743309021, 1.9890038967132568, -0.16686052083969116, -0.9768466949462891, -0.4141215682029724, 1.2105835676193237, -0.5079039931297302, 0.4667705297470093, 2.263789176940918, -0.7231724262237549, 0.09705383330583572, -0.5700691938400269, -0.36967751383781433, 0.1170865073800087, -0.7418762445449829, 0.4523126184940338, 0.17620611190795898, 1.1612439155578613, 0.6839698553085327, 0.3830125331878662, 1.4962236881256104, -0.18894369900226593, 0.6344724893569946, -0.6307652592658997, 1.4336544275283813, -0.7074540257453918, 0.9859297871589661, -0.6929429173469543, 0.24380528926849365, -1.0253357887268066, -0.7271009683609009, -0.9044414758682251, 0.47686535120010376, 0.5680517554283142, 0.7994596362113953, 0.44900667667388916, -0.14832571148872375, 0.7238885164260864, 1.610827088356018, -0.30399173498153687, -0.6673916578292847, 0.6379453539848328, 2.110591173171997, 0.0024049936328083277, -0.23644722998142242, -0.9114506244659424, -0.6265468597412109, -0.8414252996444702, -0.8643729090690613, 1.1450847387313843, 1.3597193956375122, 0.20086640119552612, 0.2067112922668457, 0.9218200445175171, 0.13083302974700928, 0.2233761101961136, 0.7104780077934265, 0.07873336970806122, 1.3994570970535278, 3.3433470726013184, 0.12473997473716736, 0.4054220914840698, -0.6959803700447083, -0.3173444867134094, 0.2171364277601242, 0.5479338765144348, -0.05660434439778328, -1.0545189380645752, 0.5163818001747131, -0.39629068970680237, 0.5369794964790344, -0.08610078692436218, 0.005029276013374329, 0.055874478071928024, -1.9752676486968994, 1.363325834274292, -0.7710289359092712, 0.49171385169029236, 0.683422863483429, -0.4276729226112366, -1.2165101766586304, 0.7069383263587952, -0.8167572617530823, 0.05048482492566109, 0.5852422714233398, -0.3454992175102234, -1.5791666507720947, -0.5497377514839172, 0.6179208159446716, -0.6875657439231873, 0.9896419644355774, -1.4107714891433716, -0.846373438835144, -1.6374831199645996, -0.8460994362831116, 1.6032077074050903, 0.6579160094261169, -0.18076545000076294, -0.5411076545715332, 1.217711329460144, -0.04887506365776062, -0.9169337749481201, 1.6083085536956787, -1.2434449195861816, 0.25046828389167786, 0.7000219821929932, 1.0181289911270142, 2.0188143253326416, 0.8900254368782043, -0.6391757726669312, 1.195995569229126, 0.35024526715278625, -1.1676905155181885, -1.0283946990966797, -0.7655251026153564, -2.111659049987793, 0.46637052297592163, 0.5945751667022705, 0.5197609066963196, -0.7922329306602478, -0.02396240457892418, 1.2159638404846191, 0.40419289469718933, -0.11806651949882507, -0.7136350870132446, -0.018587974831461906, 1.2941796779632568, 0.04682822525501251, 2.306591272354126, 2.4369754791259766, 0.734538197517395, -1.600783109664917, -0.23558059334754944, 1.291766881942749, -0.5365603566169739, -0.6670299768447876, 0.04185165837407112, 2.6933677196502686, -0.4537692964076996, -1.9255388975143433, 0.3028927445411682, 0.4130943715572357, 0.7574179172515869, 0.0892002284526825, 0.35836976766586304, -0.4533739387989044, 0.9728851318359375, -0.012046006508171558, 0.5285264253616333, -0.7250410914421082, 1.4016215801239014, 0.2560257613658905, -0.417578786611557, 0.05601584166288376, -1.0895644426345825, -0.5916461944580078, 0.8337103128433228, 0.6148226857185364, 0.8461495637893677, 0.7177664041519165, -0.268921822309494, 0.11134298145771027, 1.8966385126113892, -0.10079926997423172, -1.527734398841858, -0.011885493993759155, -0.4173445701599121, -0.10999348759651184, 0.009748228825628757, -1.0405703783035278, 0.4064355492591858, -0.1833367943763733, 0.5692316293716431, -0.7196189761161804, -1.0147780179977417, -0.4598208963871002, 1.6190237998962402, -1.8392267227172852, 1.0560879707336426, -0.09007644653320312, 1.1082093715667725, -0.43494337797164917, 0.34126073122024536, 0.3171912729740143, 2.0729658603668213, -0.17337548732757568, 0.18565769493579865, -0.8161389231681824, 0.24527406692504883, -0.7978491187095642, -0.17011240124702454, 0.32510000467300415, 0.5010623335838318, -0.9618199467658997, -0.9415929317474365, 1.108639121055603, -0.5814531445503235, 0.12439379096031189, -0.3385330140590668, -1.6272461414337158, -0.16101756691932678, -0.1767140030860901, 1.7098890542984009, -1.9512141942977905, -0.5820896029472351, 0.5601062178611755, -0.5887768864631653, -2.2030770778656006, 0.5634773969650269, 0.11301595717668533, -0.9000091552734375, -1.0478037595748901, -0.7392122745513916, 0.13294117152690887, -0.23677732050418854, -0.7163217663764954, 0.27896517515182495, 0.9772326350212097, -0.663250207901001, 1.4899704456329346, -0.035980165004730225, 0.528157114982605, 0.3477920591831207, 0.4249107837677002, 0.16110846400260925, -2.3220739364624023, -1.021207571029663, 0.7946293950080872, -1.1193976402282715, 0.02737055905163288, -1.6342350244522095, -0.6981988549232483, 1.4370665550231934, 0.15244001150131226, -0.9525130391120911, -0.7268542051315308, -1.7801212072372437, 0.3558832108974457, -0.3465251922607422, 0.6166972517967224, 0.6755572557449341, -0.4097643792629242, -1.0668455362319946, -1.3211506605148315, -0.5260226130485535, -0.11231470108032227, 1.2370514869689941, -1.0799612998962402, 0.31379827857017517, -1.1745085716247559, -0.7127522230148315, 0.6289868354797363, 0.3720860779285431, -0.6709147691726685, -0.7879875302314758, 0.9521892666816711, -0.3231487274169922, -0.4494050443172455, -1.1935861110687256, 2.3457138538360596, 1.6887389421463013, 0.3824992775917053, 2.082157850265503, 0.36646926403045654, -0.4537978172302246, 0.4309796094894409, 1.30819571018219, -1.2055360078811646, 1.142181158065796, 1.0984951257705688, 1.417262077331543, 0.4132705330848694, 1.0435433387756348, -1.7792726755142212, -0.11308328062295914, 0.9729127883911133, 0.5013329386711121, 0.22494368255138397, -1.2804738283157349, -0.20711612701416016, -1.7590967416763306, -0.27783218026161194, 0.2951955199241638, -0.5500257611274719, 0.4456641972064972, -0.02577967382967472, 0.8578847646713257, -0.3791158199310303, -1.6262537240982056, -1.2840887308120728, -0.7869983315467834, -0.026459597051143646, 0.07307395339012146, -0.7598562836647034, 0.09142988920211792, -0.6729117631912231, 0.6250157356262207, -1.3393405675888062, 1.1499557495117188, 0.27494069933891296, -0.3953324854373932, -1.0692309141159058, 0.5591391324996948, -0.5808485746383667, 0.6656216382980347, 1.8508687019348145, 0.20305289328098297, 0.27762892842292786], [0.5845356583595276, -0.47953420877456665, 1.9907095432281494, -0.1443684697151184, 0.7674277424812317, 0.42349815368652344, -0.48408952355384827, 1.4619698524475098, 0.4862474799156189, -0.5275807976722717, -0.5163500308990479, -1.778374433517456, -0.9224792122840881, 0.9791021943092346, 0.9944040179252625, -0.7681300640106201, 0.7822250127792358, 1.136113166809082, 0.9392929673194885, 1.414772629737854, -1.7190144062042236, 0.3739514946937561, -0.7750612497329712, 0.8448584079742432, -0.565291166305542, 0.2718915343284607, 1.0407123565673828, -1.39875328540802, -0.48576152324676514, -0.5406025648117065, 0.11544476449489594, -1.4176619052886963, 0.9189150333404541, 2.048274517059326, -0.1538051813840866, 0.6333823800086975, -2.5327701568603516, -1.0917696952819824, 0.7542370557785034, 0.3814716935157776, 0.7481553554534912, 1.407186508178711, 0.171371728181839, -0.17000891268253326, -0.39622893929481506, 0.16784140467643738, -2.0492119789123535, 1.528622031211853, -2.1127326488494873, -1.5375896692276, 0.10520374774932861, 0.9682373404502869, 0.575650155544281, 1.5183576345443726, -0.4139949083328247, -0.623506486415863, 0.5676472187042236, -1.1050031185150146, 0.22072859108448029, 0.522184431552887, 0.3052465617656708, -2.0223019123077393, -0.22747084498405457, -2.0348386764526367, 0.8553742170333862, -0.43800088763237, 0.9893076419830322, 1.3115365505218506, -1.3475371599197388, 0.9075148701667786, 0.3933284282684326, 1.4877996444702148, -1.2619246244430542, -1.273620367050171, -0.7876749038696289, -1.8410521745681763, 0.12286150455474854, -0.7244023084640503, 0.18965278565883636, 0.33337754011154175, 0.8286624550819397, -1.3311611413955688, 0.7191262245178223, 0.7118955850601196, -1.004725694656372, -0.6327580213546753, -1.4686652421951294, 1.1996169090270996, 1.315615177154541, -0.8690211176872253, -0.13080786168575287, 0.37573978304862976, 0.6708853244781494, -0.09925352782011032, 1.6177996397018433, -0.9634950160980225, 0.8266293406486511, 0.06467942148447037, 0.2613316774368286, -0.7632781863212585, -1.7530707120895386, -0.5468441843986511, 0.4415927827358246, -0.8607746958732605, -1.716618537902832, -0.19865894317626953, -0.05292048305273056, -1.028702735900879, -0.1465432345867157, 0.37996768951416016, 0.17681942880153656, 0.8824663162231445, 1.1482384204864502, -0.06461270898580551, -0.8348608016967773, -1.767059087753296, 0.9657735228538513, 0.3118235766887665, 0.9715597033500671, -0.8493519425392151, 0.09210927039384842, -0.8413611054420471, -0.9279946088790894, -1.4833877086639404, -0.2612299919128418, 0.43986770510673523, 0.7383082509040833, 0.20193950831890106, -0.31630104780197144, 1.968917965888977, 1.5019629001617432, 0.9787485003471375, 0.11384434252977371, 1.2301392555236816, -0.7912428975105286, -1.3030693531036377, 0.5896487236022949, 0.13553768396377563, 1.0098938941955566, -0.8163918256759644, 0.2579323947429657, -1.0703152418136597, -0.37620338797569275, 0.43027976155281067, -0.10019189864397049, -0.9706488847732544, -1.4790925979614258, -0.41844281554222107, -0.9517046213150024, -1.4696910381317139, 0.05265706777572632, -0.9404695630073547, 1.2350375652313232, 2.0330770015716553, 1.0818620920181274, 2.752122163772583, -1.2449170351028442, -1.7410012483596802, -0.1728101670742035, -1.7858127355575562, 0.11355924606323242, 0.8494930267333984, 0.7487286329269409, 0.9426158666610718, -0.23440130054950714, -0.09531242400407791, 0.21618403494358063, -0.08522100001573563, 0.8985418081283569, -0.43267858028411865, -0.07155722379684448, -0.7200949788093567, 0.9296237230300903, 0.09417017549276352, 0.10390852391719818, -0.5140706300735474, -1.7980337142944336, -0.4822388291358948, -2.1904726028442383, -0.09932276606559753, -1.261834740638733, 1.2356501817703247, -0.6708986163139343, -0.10174262523651123, -0.5266040563583374, -1.172978401184082, -1.5715320110321045, -2.5436511039733887, -0.6865460872650146, -2.064882755279541, 1.076335072517395, -1.330354928970337, -0.6464306712150574, -0.07188676297664642, -0.19745315611362457, 1.1169219017028809, -0.11215995252132416, -0.5300148725509644, -0.1622435301542282, 1.4855096340179443, 0.7327160239219666, -1.9899672269821167, -0.23498967289924622, -0.9473857283592224, -0.020927507430315018, 0.5866513848304749, -0.029792485758662224, 0.5628293752670288, 1.6385867595672607, -1.7721197605133057, -0.07820016145706177, 0.021199028939008713, 0.4136437177658081, 0.06362367421388626, 0.8861387968063354, -0.04937543347477913, 0.40480339527130127, -1.571861982345581, 0.16625536978244781, -1.2300441265106201, 0.5494053363800049, 1.7282483577728271, -0.07905030995607376, 0.4106321632862091, 0.7222981452941895, 1.3559833765029907, 0.768801212310791, 1.2178843021392822, -0.5317310690879822, -0.07661865651607513, -1.4510732889175415, 2.1998379230499268, -1.1725199222564697, 0.9152979254722595, 0.16637073457241058, 0.12496829032897949, -1.1589643955230713, -0.031032394617795944, 0.09331639111042023, 0.4173242151737213, -0.3257514536380768, 0.2501222491264343, -0.9436382055282593, 1.2132713794708252, -0.6346766948699951, 3.5988948345184326, -0.2956105172634125, -0.624888002872467, 0.20428843796253204, -0.10813352465629578, 1.1098604202270508, 1.244063138961792, -0.585770845413208, -0.6278762817382812, -0.7041975855827332, 0.5142398476600647, 1.022289514541626, 0.7094058394432068, -0.09020742774009705, 0.09084340929985046, 0.7156215906143188, 1.296935796737671, 0.02314324490725994, 1.230141282081604, -0.27136144042015076, 0.259357750415802, 1.1857556104660034, -0.7939895391464233, -1.6506966352462769, -1.0421545505523682, -0.08499765396118164, -0.9691756963729858, -0.06947799772024155, 1.4153538942337036, -0.2993451952934265, 1.3701659440994263, 0.9142482876777649, -1.0023345947265625, 1.0129270553588867, 0.9107785224914551, 1.0256211757659912, 0.26100021600723267, 0.32308927178382874, -0.49880358576774597, -0.8167302012443542, 0.0036268457770347595, 0.7639923095703125, -0.5356144309043884, 1.6818517446517944, -0.3731782138347626, 0.3950251638889313, -1.2600624561309814, 1.6222283840179443, 0.80512535572052, 1.3765612840652466, -1.1071611642837524, 1.530419945716858, 0.5725051164627075, -0.5069452524185181, -0.28882384300231934, -1.381693959236145, -0.06499097496271133, 1.0040756464004517, -0.7374705672264099, 0.49513909220695496, -1.0822627544403076, -2.5364019870758057, -0.07052753120660782, 1.5793399810791016, 1.3015578985214233, -0.565034806728363, 1.3569674491882324, 1.1336069107055664, 0.46679577231407166, -0.2644661068916321, 1.0664005279541016, -0.01182188093662262, -0.3163752555847168, 1.1203705072402954, -0.6191259026527405, 0.7445191740989685, 1.545314908027649, -0.6190828680992126, 0.23230156302452087, 0.5879582762718201, -1.701650619506836, -0.4457305669784546, -0.507330596446991, -0.4976319372653961, -0.8317630290985107, 1.3160392045974731, -0.8737195134162903, -0.07963448017835617, 0.6166697144508362, 1.580049991607666, -1.5761690139770508, -1.2838739156723022, -1.0395601987838745, -0.2910478413105011, 0.9444125890731812, -0.12275513261556625, 0.16092249751091003, 1.541927456855774, -0.30543938279151917, 0.1861838698387146, 1.9907615184783936, 0.8012474179267883, -0.5865586400032043, -0.6244691610336304, 1.139201283454895, -1.1139960289001465, -0.856836199760437, 0.8296802043914795, 1.3650490045547485, 1.7008588314056396, -0.9467962980270386, 0.5489357113838196, -3.285169839859009, -0.8107802271842957, 2.9718384742736816, 1.6045339107513428, 0.7611344456672668, -0.8743405938148499, -1.2008424997329712, 1.1632174253463745, -0.09881898760795593, -0.7509102821350098, 0.7894387245178223, 0.7471385598182678, -0.635106086730957, -0.3735889494419098, 2.2796781063079834, -0.9129741787910461, 1.083204984664917, -0.6337116956710815, -0.7871981263160706, -0.9754806756973267, -0.05158914253115654, 0.04852529987692833, 0.809025764465332, -0.6730905175209045, 0.47825825214385986, -0.49277395009994507, -0.6633763909339905, 0.7016656994819641, 0.5173653960227966, -1.4533313512802124, 0.21645213663578033, -1.7677258253097534, -1.0109204053878784, 1.6707602739334106, 1.3657597303390503, -0.14584539830684662, 0.6444857120513916, 1.3078677654266357, 1.763980507850647, 1.7523466348648071, -0.7336893677711487, -0.020208382979035378, 0.13416379690170288, -2.400268077850342, -0.7348417043685913, 0.41042622923851013, -0.20520509779453278, -0.5946008563041687, -1.1660566329956055, -0.5725105404853821, -0.022488372400403023, -0.3139510750770569, 1.139778971672058, 0.8154771327972412, -1.2871806621551514, 0.935413658618927, 0.029044657945632935, 0.7627830505371094, -0.6575496196746826, 0.8966464400291443, -0.7136605978012085, 1.0195720195770264, -0.029828747734427452, -0.2585071921348572, -1.3179148435592651, -1.610634446144104, 0.8638087511062622, -1.1188827753067017, -1.1740583181381226, 0.3959372937679291, 0.01975967548787594, 0.5803695917129517, -1.3112907409667969, -2.1222569942474365, -0.907675564289093, 0.994873583316803, -0.035811420530080795, 0.43461766839027405, 0.5176786780357361, 0.16470591723918915, 0.1666765809059143, -0.20844659209251404, 0.21658925712108612, 0.3222002387046814, -0.6505431532859802, 0.5969361662864685, -0.0018493353854864836, -0.08371639996767044, -0.15730921924114227, 0.8785064220428467, -0.14439719915390015, 1.1890110969543457, -1.0004775524139404, 0.5745886564254761, -1.53922700881958, 0.052565086632966995, -0.5011278986930847, -0.4562002718448639, -1.0339288711547852, -0.0259527787566185, 0.5409347414970398, -1.8305275440216064, -0.13030052185058594, 0.18041835725307465, 0.6039357781410217, 0.3523498773574829, -1.0781608819961548, 0.9867830872535706, -0.8685605525970459, 0.48879310488700867, 0.18667560815811157, -0.5836264491081238, -0.9093582034111023, -0.21926407516002655, 1.422776460647583, 0.6303192377090454, -0.5812960267066956, 0.7173916101455688, 1.6504762172698975, 2.1431210041046143, -1.7132384777069092, 0.7185304760932922, 0.823879063129425, -0.8061841726303101, -1.8597822189331055, 1.255383014678955, -0.46310415863990784, -1.6140966415405273, -1.0675679445266724, 0.7273188233375549, 2.1745762825012207, 0.9268898963928223, -0.8571982979774475, 1.2645622491836548, -1.8664213418960571, 0.991232693195343, 0.9234642386436462, 1.7908982038497925, 0.9805452227592468, 1.1414623260498047, 0.12191624939441681, -0.3240090012550354, 0.04354578256607056, 1.93315851688385, -0.550883412361145, -0.037222567945718765, -0.4416966438293457, 0.6917722225189209, 0.5008761882781982, 2.0546228885650635, 2.1952993869781494, 1.156290054321289, 0.9191031455993652, -0.10520371049642563, -1.4597537517547607, 0.19501107931137085, 0.07118874788284302, -1.1001379489898682, 0.24863523244857788, 1.0036053657531738, -0.8585929870605469, 0.9148151874542236, 0.4423814117908478, -0.5117728114128113, 0.14303277432918549, 0.7860252857208252, -0.1498887687921524, -0.030315907672047615, 1.350305199623108, 0.5907137393951416, 0.569054126739502, -0.4718348979949951, 0.7372592687606812, 1.5341087579727173, -0.9118852615356445, -1.6219547986984253, -0.7734887599945068, -0.10126695036888123, -0.9776982665061951, -1.6687580347061157, 1.6795696020126343, -0.393322616815567, -0.7923671007156372, -2.8604817390441895, 0.936558723449707, -1.5346144437789917, 1.3454325199127197, 0.591635525226593, 0.24091248214244843, -0.19254951179027557, 0.9868427515029907, -0.5479680895805359, 0.39543041586875916, -0.4786953032016754, -0.29399123787879944, 1.6032747030258179, 1.1433967351913452, -0.9178977608680725, -0.8471530079841614, 0.06149466708302498, -0.1118304431438446, 0.5185171365737915, -0.3615606725215912, -1.0309433937072754, 0.851961612701416, -0.20432867109775543, 0.7114765644073486, -0.8878845572471619, 1.5430611371994019, 0.29891982674598694, 0.28276005387306213, 0.40333032608032227, -1.5712931156158447, 0.7296010851860046, -0.5055705308914185, -1.1180262565612793, 0.40833428502082825, 1.7598012685775757, -0.6914130449295044, 1.583095908164978, -2.219372034072876, 0.32272377610206604, -0.586661159992218, 0.2713642418384552, 2.183260440826416, 0.7003292441368103, -1.4934561252593994, 1.4965006113052368, -1.0562076568603516, -0.5603970885276794, -0.9016335010528564, 0.14851228892803192, -0.09834994375705719, 1.6732375621795654, 0.3683803081512451, 0.7229626774787903, -0.1361512988805771, 1.16121506690979, -1.0349698066711426, -1.4425190687179565, 0.9271638989448547, 0.7888785600662231, 0.07937249541282654, 2.3122246265411377, -0.8768450617790222, 1.4769903421401978, 1.5820516347885132, -1.222352147102356, 0.5989021062850952, 0.8334861397743225, -0.0976409912109375, -0.08775646239519119, -0.014554084278643131, -0.8357809782028198, -1.355550765991211, 0.836452305316925, 0.2721535861492157, 0.07723639905452728, 0.579251766204834, -0.7757176160812378, -0.6126006841659546, 0.8605212569236755, 0.04074988141655922, 0.5374335050582886, 1.1388882398605347, 1.809524655342102, -0.8793338537216187, -0.5395975112915039, -1.4108197689056396, -0.6620339155197144, 0.5528425574302673, -0.5059802532196045, 2.2444210052490234, -0.3978612720966339, -1.7708632946014404, -0.8718477487564087, 2.1454694271087646, -0.8689576387405396, -0.03906780481338501, -0.7955455183982849, -1.0794081687927246, 1.2072350978851318, -1.172958493232727, -1.7807588577270508, -0.7218365669250488, -0.6380513310432434, 0.4752994179725647, 0.1462768316268921, 1.283409833908081, -0.17837683856487274, 0.47201213240623474, 0.5365804433822632, 1.066602349281311, -1.201523780822754, -0.3040740489959717, -2.727889060974121, -0.7730317115783691, -1.202517032623291, -0.8539469242095947, 1.5942704677581787, 1.0622845888137817, -1.500462532043457, 1.8387550115585327, 1.3147168159484863, 0.20845629274845123, -0.08046787232160568, -0.11641066521406174, -0.01629859209060669, 0.6772767901420593, -0.25819215178489685, 1.0194425582885742, -0.318491131067276, -0.683420717716217, -0.13177984952926636, 0.3409929573535919, 0.431687593460083, -0.01008804701268673, -1.5234442949295044, -1.3603088855743408, -1.726844310760498, -0.27782195806503296, 0.6866108775138855, 0.029187176376581192, -0.04730188474059105, -0.4849826991558075, -0.3437245786190033, -0.6742150783538818, 1.0719468593597412, -1.2458324432373047, -0.8883633017539978, 0.4877975285053253, -0.36500245332717896, 0.6668940186500549, -0.4613325893878937, 0.433394193649292, 0.23466132581233978, -1.2864577770233154, 0.567180335521698, 1.2767521142959595, -0.6633318662643433, -0.12437562644481659, 0.6506073474884033, 1.556775689125061, 0.37440454959869385, 2.3541128635406494, -0.38598835468292236, 1.9523229598999023, -0.7258800864219666, -0.002690602093935013, -0.8916632533073425, 1.0145825147628784, 0.659573495388031, -1.7216839790344238, -2.6078238487243652, 2.1841535568237305, -0.27108293771743774, -1.6511963605880737, -0.8813006281852722, -0.44433027505874634, 1.6046336889266968, 0.4644690752029419, 0.06601184606552124, -0.9317823648452759, 0.6883710026741028, -1.7415903806686401, 1.144287109375, 0.9930378794670105, -0.453390896320343, -1.4888906478881836, -0.6049903035163879, -0.10647507011890411, 2.027841091156006, 1.7215056419372559, -0.6296871304512024, -0.7486513257026672, -1.356175184249878, 0.3941717743873596, -0.5316721796989441, 0.7191656827926636, 0.08262159675359726, -0.5294016599655151, -0.9907830357551575, -1.5858862400054932, 0.5984381437301636, -1.3104066848754883, 0.7201768159866333, -1.64700448513031, -2.5358524322509766, 0.02395775355398655, -0.5860015749931335, 0.9295604228973389, 0.010813710279762745, -0.06349556148052216, 0.8780414462089539, -0.6453883051872253, 0.6536540389060974, 2.472222089767456, 0.24479377269744873, -0.9034851789474487, -0.015531892888247967, -0.7818406820297241, -0.31370842456817627, 1.6924299001693726, -0.9609875679016113, -1.204599142074585, 0.5447084903717041, -0.37794801592826843, -0.13998286426067352, -0.594426155090332, 0.2733118534088135, 1.50230872631073], [0.670592188835144, -0.38513582944869995, -0.31957197189331055, -0.875079333782196, -0.847316324710846, 0.9746069312095642, 0.5749966502189636, 0.5024157166481018, 0.5392088890075684, -0.5846816301345825, 0.031657684594392776, 0.5614797472953796, 1.6452656984329224, 0.30445927381515503, 0.11982614547014236, -0.7380986213684082, 1.0774646997451782, -0.19779923558235168, 0.30563023686408997, 0.44599154591560364, -0.7206485867500305, 0.3464657962322235, 1.1809710264205933, -0.8222395777702332, -0.45830804109573364, -0.7547037601470947, 0.15880165994167328, 2.1413846015930176, 0.3843885362148285, 1.2962630987167358, 0.3023602366447449, -0.3575131595134735, 0.07538828998804092, 0.29555466771125793, -0.12986090779304504, -0.003353912616148591, 0.14980366826057434, -0.5631812214851379, 1.5751917362213135, 0.49498364329338074, -0.058811865746974945, -0.5065656900405884, 0.4309263229370117, -0.7066366076469421, -1.1770740747451782, -0.3090123236179352, 1.2964657545089722, -0.1418582797050476, 0.23225247859954834, -0.8956320881843567, 0.5661347508430481, -0.4865369498729706, -0.511075496673584, -0.7154276967048645, -0.07986099272966385, -0.9928259253501892, -0.3538915514945984, -0.1436949223279953, -0.9037287831306458, -1.8856627941131592, 0.47778812050819397, -1.1913214921951294, -1.892270803451538, 0.04745529964566231, -0.006603903137147427, 1.13968026638031, 1.3104697465896606, -2.1441762447357178, -0.5284736752510071, 0.3066878318786621, -1.2943168878555298, 1.1669408082962036, 0.37954047322273254, -0.6293964385986328, -0.4834604859352112, 0.3515823483467102, 2.129608392715454, 1.9432408809661865, -1.3812071084976196, 0.6383273005485535, -1.4173336029052734, -0.09218928217887878, -0.9433489441871643, -0.2484969198703766, -0.396664559841156, 0.3988482356071472, -0.706125795841217, 0.7280862331390381, -0.08103047311306, -0.8706178069114685, 1.47853684425354, 0.3652712404727936, -0.1714733988046646, 0.12621116638183594, 0.7336657643318176, -0.09621353447437286, -0.5619644522666931, -0.11492392420768738, -0.5088387727737427, -0.49661770462989807, 1.632995843887329, 0.8093201518058777, 1.3539255857467651, -0.4667532742023468, -1.7064491510391235, -0.3444546163082123, 0.5519288182258606, -0.45559433102607727, 0.21698547899723053, -0.913348376750946, 0.5274308323860168, -0.394631952047348, 0.037970032542943954, 0.6969019770622253, -0.6944564580917358, -1.0224967002868652, -0.7737802267074585, 0.6035006642341614, -0.299558162689209, 1.0796200037002563, 0.5548640489578247, 1.1254547834396362, 0.20874322950839996, 1.2175579071044922, 0.7573316097259521, -1.876133680343628, 1.440887451171875, 0.8938486576080322, -0.3660537898540497, -1.155191421508789, -1.798447847366333, 0.18354737758636475, -0.8066619038581848, -1.1091262102127075, -0.9357006549835205, -1.576414942741394, 0.9762282371520996, -0.8237019181251526, 0.02091001532971859, 0.19762404263019562, 0.4081558585166931, -0.11663070321083069, 0.8591071963310242, -0.5541098713874817, 2.1532175540924072, -1.532599925994873, 0.7313835024833679, 2.215183973312378, -0.9130692481994629, 1.4549368619918823, 2.0712265968322754, -0.4084274470806122, 1.5671172142028809, -0.5218011736869812, 0.3462474048137665, -0.1757536232471466, -0.9212508797645569, -1.043845534324646, -0.526829719543457, -1.4456478357315063, 0.2231065183877945, 0.17478282749652863, 1.141573429107666, 0.456428587436676, -0.11709321290254593, -0.27314916253089905, 2.4991750717163086, -1.4960846900939941, -1.2774426937103271, 2.282726287841797, 1.1189085245132446, 0.23594892024993896, 0.5894771814346313, 1.0485419034957886, -1.4076787233352661, -0.25485068559646606, 0.7726362347602844, 0.7224931716918945, 1.1691408157348633, 0.3078297972679138, -0.28784725069999695, -0.5318975448608398, 0.6907681822776794, 0.05628291144967079, 0.5510700941085815, 1.478314757347107, 1.3145594596862793, -1.1726621389389038, 1.2443701028823853, -0.8766912817955017, -1.543776273727417, 2.200939416885376, 0.37940287590026855, -0.49714869260787964, -0.31552183628082275, 1.0073809623718262, 2.1104602813720703, 1.4230893850326538, -0.11112675070762634, -1.0409939289093018, -0.001203016610816121, 1.129710078239441, 0.6202628016471863, -1.1969120502471924, 0.24144165217876434, 0.7318564653396606, -0.805108368396759, 1.0019632577896118, 1.582713007926941, 1.0353236198425293, -0.7265402674674988, -0.1536531150341034, 0.05182371661067009, -0.22529321908950806, 0.982387363910675, 0.21111813187599182, 0.1246730163693428, 0.44574475288391113, -0.31912723183631897, -1.2119642496109009, 0.2347366362810135, -0.21435193717479706, 0.7813396453857422, 0.34579339623451233, -0.14367328584194183, 1.1928367614746094, 0.09818992763757706, -1.1575629711151123, 1.6559675931930542, 1.2634918689727783, 0.08761391043663025, -1.4510771036148071, -0.8860579133033752, 1.0716596841812134, 0.028137633576989174, -0.2434956580400467, 0.5563176274299622, -0.4711095094680786, -0.39861080050468445, 1.3055106401443481, 0.6234378814697266, 0.8665753602981567, 0.700188398361206, 1.1519279479980469, 0.9405567646026611, -0.6735557913780212, -0.07582785934209824, 1.7491793632507324, 0.2932051718235016, 1.834443211555481, -0.18177743256092072, -1.2512664794921875, 0.21072666347026825, -0.3754580020904541, -0.5602225065231323, -0.6167373657226562, -1.8843475580215454, 0.6530359983444214, -0.03136279061436653, -0.7967686653137207, -1.4885914325714111, 0.7634350657463074, 0.13435785472393036, 0.7781624794006348, 0.48215818405151367, 0.8519366979598999, 0.07404649257659912, -1.3356850147247314, 0.26682692766189575, 0.9855099320411682, -1.296012043952942, -0.3413166105747223, 1.5221645832061768, -1.1231878995895386, 1.4199503660202026, -0.8311486840248108, -1.5307753086090088, -0.43582668900489807, -0.5390875935554504, 0.9837601780891418, -2.0158393383026123, -0.8933808207511902, -0.8928428888320923, -1.040183186531067, 0.0982741042971611, -0.5509833097457886, -0.02419998124241829, -0.2781130373477936, 1.0353460311889648, -0.6710696816444397, -0.005604251753538847, 0.8030185699462891, 0.47562986612319946, -0.014380965381860733, 0.7883139848709106, 0.40568554401397705, 0.43593260645866394, 0.08011019229888916, 0.9818692207336426, 1.0752722024917603, -0.5457916259765625, 0.5916450619697571, 0.4759906232357025, 0.7301297783851624, -0.641105055809021, -1.102533221244812, -0.12449263036251068, 1.1538383960723877, -0.9087536334991455, 1.0525891780853271, 0.11746714264154434, 0.2706475853919983, 0.28484269976615906, -0.24141748249530792, -0.13271914422512054, 0.7197964191436768, 1.6055978536605835, 0.2839817404747009, 0.5303655862808228, -0.7180814743041992, -0.021144304424524307, 1.146529197692871, 1.1907480955123901, -0.1656370609998703, 0.05249098688364029, -0.32167211174964905, -0.738832950592041, 0.5961370468139648, -0.3180397152900696, 1.5743436813354492, -2.0556983947753906, -1.6951607465744019, -1.2028557062149048, 0.07662172615528107, -0.146636500954628, -0.7620878219604492, -3.1222455501556396, 0.1841154843568802, -1.597212553024292, 1.3810186386108398, 0.13915307819843292, -0.7245992422103882, -0.2580240070819855, 1.5833004713058472, -0.3421348035335541, -1.126213550567627, -2.541135311126709, -0.17468638718128204, -0.7649868130683899, 1.1228318214416504, 0.13979274034500122, 1.0573475360870361, -0.42901015281677246, -0.8447040319442749, 0.9338836073875427, -0.16577909886837006, -0.7278337478637695, -0.10085957497358322, 0.9959483742713928, -0.6334193348884583, 1.9286868572235107, 0.5039256811141968, -1.1833857297897339, -1.0785667896270752, -0.05851342901587486, -0.33880558609962463, -0.3716844618320465, -0.1375434547662735, 0.2619905471801758, -0.2468193918466568, -0.527576744556427, -1.8690606355667114, 1.6232225894927979, 0.12453728914260864, 0.5595053434371948, 0.1810324490070343, 1.2648844718933105, 0.03163999691605568, 1.235419750213623, -0.5535674691200256, -0.2271808385848999, 0.5351879000663757, 0.5886643528938293, 1.1869899034500122, -0.5632916688919067, 0.4661467969417572, -0.5125950574874878, -1.4004507064819336, 0.9766579866409302, -0.06664743274450302, -0.9703311920166016, -0.7462954521179199, 0.5181816816329956, -0.0014444690896198153, -0.11370739340782166, 0.6491490602493286, -0.03760072961449623, -1.7823238372802734, 0.8162660002708435, -0.48441991209983826, -0.37375837564468384, -2.385294198989868, -3.2273173332214355, 0.02751808799803257, -0.4780197739601135, -0.020401762798428535, 0.2589188516139984, 0.8656225204467773, 1.2980546951293945, 1.1890771389007568, -0.989672839641571, -0.7857160568237305, 0.10865115374326706, 1.3904250860214233, 1.107441782951355, -0.5129866003990173, -1.1783367395401, 0.28601107001304626, -0.1834953874349594, 0.6715542078018188, 0.7930706143379211, 0.3298071026802063, -0.8641093373298645, -1.1875593662261963, -0.26251354813575745, 0.4852810502052307, -0.051355645060539246, 0.7085462212562561, -0.38545748591423035, 1.0957671403884888, 0.35975632071495056, -0.2941959798336029, -0.6967887878417969, -0.11443357914686203, 0.10572060942649841, 1.579756736755371, 0.11814261972904205, 0.6852779388427734, -2.000821113586426, 0.9996795654296875, -0.04589291289448738, -0.8770105838775635, -2.208137273788452, -0.8078765869140625, 1.6290111541748047, 0.8458405137062073, 0.8128373026847839, -0.44599950313568115, 0.6332911849021912, 0.6331822276115417, 0.3564397394657135, -0.892342209815979, -0.5762864947319031, -0.28289178013801575, 3.011108636856079, -1.9321037530899048, 1.6460450887680054, 1.3967775106430054, 0.236491858959198, 0.8253347277641296, 0.3794839084148407, 0.5690209269523621, -0.563953697681427, -1.9325520992279053, 2.6811554431915283, 1.7503963708877563, -1.369516372680664, 1.1405307054519653, -0.3857475817203522, 0.4741288125514984, -1.0114266872406006, -1.043641209602356, 1.8648779392242432, -0.4447840750217438, -0.07559643685817719, 0.7080802917480469, 1.4803683757781982, -0.46838152408599854, 0.6247944235801697, 0.7703527808189392, -0.57734215259552, -0.5617460012435913, 0.6217780709266663, -0.10497324913740158, 1.5759494304656982, -1.7711920738220215, 0.5303853750228882, -0.13811102509498596, -1.3695333003997803, -0.09742662310600281, -1.4032105207443237, -0.3211364448070526, 1.094102144241333, -0.6879416108131409, 0.6158252954483032, 0.3094520568847656, -0.47789737582206726, -0.2638340890407562, 0.2240997552871704, 0.33311566710472107, -0.48600471019744873, -0.14191566407680511, 0.7813335061073303, -0.9752448201179504, -1.1553008556365967, 0.32638242840766907, 0.47686344385147095, -2.2728540897369385, 0.12018007040023804, 1.3902944326400757, 0.8392533659934998, -0.3396320343017578, 0.3919253349304199, -1.6062719821929932, 0.64543217420578, 0.3116759955883026, 0.8775521516799927, -0.13265734910964966, 0.6806203722953796, -0.411650151014328, 1.0926340818405151, 0.47353801131248474, -1.7504557371139526, -1.6173672676086426, 0.5539897084236145, 0.9953787922859192, 1.2866928577423096, -0.4789280295372009, 0.23771756887435913, -0.27767080068588257, -0.6226606369018555, -0.20658916234970093, 1.2598106861114502, -0.4114130735397339, 1.8119463920593262, -1.2566196918487549, -0.2879292070865631, 0.5663073658943176, 0.5569887757301331, 0.17345650494098663, -1.6111243963241577, -0.22246143221855164, 1.4503120183944702, 0.7000195384025574, 0.5694031119346619, 0.4766360819339752, -0.04239661246538162, 0.012685083784162998, -0.5275203585624695, 1.4056950807571411, 0.4866303503513336, 0.4326239824295044, 0.20698727667331696, -0.19562777876853943, -1.3914309740066528, 0.02237658016383648, -0.6336767673492432, -0.5144887566566467, 0.23699544370174408, 1.3470268249511719, -0.9787402153015137, 0.04928493872284889, 1.631370186805725, -1.193405270576477, 0.5677766799926758, -0.4007743000984192, 0.7799684405326843, 1.616447925567627, -0.024477552622556686, 1.0238162279129028, 0.3621923327445984, 1.9488199949264526, -3.3399746417999268, -0.8455861210823059, -0.05638084560632706, 0.1589854210615158, 1.7204774618148804, 1.1853101253509521, -0.3112499713897705, -1.3700900077819824, 1.2188414335250854, 0.3303041160106659, -0.20382626354694366, 0.9202942848205566, 1.848158836364746, 0.41692787408828735, 1.4871234893798828, -1.7202353477478027, 0.9320449233055115, 2.414832592010498, 1.2954665422439575, -0.5093578696250916, -1.0406347513198853, 0.45720425248146057, -0.7671383619308472, -0.44213104248046875, -1.6665103435516357, -0.9215645790100098, 0.6254644393920898, -0.7052180767059326, -0.7772582769393921, -1.2769404649734497, -0.10492055863142014, 0.3873146176338196, 0.13723616302013397, -2.7970821857452393, 0.027563273906707764, -0.9207392334938049, -0.08566804975271225, -1.0794901847839355, 1.1453388929367065, 1.9179109334945679, 0.9326950311660767, 1.761918306350708, 0.26127946376800537, -0.24864281713962555, -2.0101752281188965, -0.6207100749015808, -0.4208184480667114, 1.5834262371063232, -0.9288801550865173, 0.09875825047492981, 0.6658575534820557, 0.0859876275062561, 0.011546040885150433, 1.0809513330459595, -0.8358385562896729, 0.05659894645214081, 0.2591283619403839, 1.6306430101394653, 1.4024982452392578, 0.6008769869804382, 0.8878783583641052, -2.0648579597473145, 0.33045223355293274, -0.5000439882278442, -1.500081181526184, 0.6831008195877075, -0.3887232542037964, -0.6708144545555115, -1.7225342988967896, 0.1714450716972351, 1.2317801713943481, 0.16445274651050568, 0.7347275018692017, 0.9919075965881348, -0.01724531501531601, -0.3793828785419464, -0.6185859441757202, -0.3025955557823181, -0.5542683005332947, 0.05989008769392967, -0.49325141310691833, 1.0890051126480103, -0.8860631585121155, 1.5911555290222168, -0.4705512821674347, -0.4747038781642914, 0.6590140461921692, -0.7367799878120422, -1.8841816186904907, 0.8209676146507263, 1.4160737991333008, -1.938378095626831, 0.5093221068382263, -0.569536566734314, -0.23229719698429108, 1.5544512271881104, -1.369061827659607, -0.5317641496658325, 1.2354071140289307, 0.22484079003334045, 0.4413471221923828, -0.6742023229598999, -0.8112847208976746, -0.785825252532959, 0.20555956661701202, -0.4537810981273651, 0.40907350182533264, 2.59980845451355, -1.7888997793197632, 0.2422710359096527, -0.1180923730134964, -1.0840787887573242, 0.15928182005882263, -0.9156166315078735, -1.3819080591201782, -0.028554799035191536, 1.5379384756088257, -0.24582508206367493, 0.3851791024208069, -0.6992952823638916, -0.7817243933677673, -1.4972554445266724, -0.059409212321043015, 0.4250675439834595, -1.777699589729309, -1.9579143524169922, 0.4531559646129608, -0.7447763085365295, -1.6518431901931763, 0.5826365351676941, -0.2951030433177948, -0.4867437481880188, -0.09370922297239304, -0.7336824536323547, 2.745920419692993, -2.226959705352783, -0.5866004228591919, 0.48793405294418335, 0.3137880265712738, -1.2662593126296997, -0.4110107123851776, -0.16999858617782593, 1.290571689605713, 1.556011438369751, 1.5849684476852417, 0.13838472962379456, 1.3345582485198975, 0.5468093752861023, -0.2463541477918625, -0.32002758979797363, 0.6958820819854736, 0.3322922885417938, -0.5266872644424438, -0.9256709218025208, -0.9486210942268372, 1.405737042427063, -0.23992471396923065, 0.13063748180866241, 1.4843467473983765, 1.5558395385742188, 0.8981403112411499, 0.4587079584598541, -0.42972531914711, 0.07748709619045258, 0.2704310119152069, 0.7145557403564453, -0.5819495916366577, 0.8303293585777283, 1.0462090969085693, 0.3853539824485779, -1.3571178913116455, -1.8281184434890747, 0.7008045315742493, -1.7133018970489502, 0.20744507014751434, 2.8444812297821045, -0.11855773627758026, -1.33207368850708, 0.8889098167419434, -0.435440331697464, -0.07847224175930023, -1.7012027502059937, -0.6408867835998535, 1.3614927530288696, -0.26268109679222107, -0.7053789496421814, -0.04842748865485191, 0.17742416262626648, 0.9426542520523071, 0.6518017649650574, 0.9328856468200684, -1.2943865060806274, -0.36182886362075806, -1.943619728088379, 1.4896384477615356, -0.6219466924667358], [-0.40396174788475037, 1.7685517072677612, 0.8425514698028564, -0.33773884177207947, -0.2761881351470947, -0.4343853294849396, 0.8248178958892822, 2.9658212661743164, -1.4055925607681274, 1.4303765296936035, 1.2244794368743896, -0.4800889194011688, 0.3037025034427643, 0.2598588168621063, -0.7201319336891174, -0.6195021867752075, 1.4418379068374634, 0.4616117477416992, -0.37267744541168213, -0.8581954836845398, -0.0770857036113739, 0.07321120798587799, 0.8537617921829224, 1.5761383771896362, 0.07398912310600281, -0.8560740947723389, 0.4106401801109314, 0.1971994936466217, -0.5124701857566833, 0.7550622820854187, -0.051956869661808014, 0.38026028871536255, 0.04067220538854599, 0.49934640526771545, 0.6330521702766418, -0.9322757720947266, 0.9478749632835388, 0.0716804713010788, -1.6132972240447998, -0.28422942757606506, 1.2745307683944702, 1.4768208265304565, -0.5716171860694885, 0.3365396559238434, -1.2202868461608887, 0.5172891020774841, -2.1656277179718018, -1.6408379077911377, 0.4639829695224762, 0.5447462201118469, -0.5931980013847351, 0.5376080274581909, 1.6074397563934326, -0.09301795065402985, 0.6407800912857056, -0.7543603181838989, -1.253487467765808, 1.8142074346542358, 0.1887386292219162, -0.5805987119674683, -1.6722253561019897, 0.6504896879196167, 0.9255449175834656, 1.434251070022583, -0.1685163676738739, 0.9599881172180176, 0.08707691729068756, -0.8555300831794739, -0.002667626366019249, -0.5425073504447937, -0.8488016128540039, -0.5903496742248535, 0.9022741913795471, 0.09364020079374313, -0.7060945630073547, -0.11410555243492126, -0.11042560636997223, -0.0818646103143692, 0.2158806025981903, -0.7060686349868774, 1.4481055736541748, -1.0889886617660522, -0.5501996278762817, -1.3312499523162842, -0.3530964255332947, -0.4890892803668976, 0.25881659984588623, -1.425775408744812, 0.26799502968788147, -1.6199705600738525, 1.3016432523727417, -0.4848363697528839, 1.2718786001205444, 1.5822222232818604, -1.08576238155365, 2.0808932781219482, 0.10946901142597198, -2.155057430267334, 0.29593396186828613, 0.2622908353805542, 1.79541015625, -1.84953773021698, -3.044786214828491, -0.06674325466156006, -0.48611366748809814, -0.9601022601127625, -0.0019695337396115065, 1.804583191871643, -0.0393218994140625, 0.9361198544502258, -0.7203637957572937, -1.3351048231124878, -0.4554721713066101, 1.8759253025054932, 1.3790500164031982, 0.8107775449752808, -0.31521645188331604, 0.38181591033935547, 0.6397772431373596, -1.5155906677246094, -0.2744789719581604, 1.9721800088882446, -1.3584022521972656, 0.24767793715000153, -0.4792095422744751, -0.3352903127670288, 1.0020923614501953, -0.8031667470932007, 0.7993660569190979, -0.022409742698073387, 1.1291911602020264, -0.8507716059684753, 1.2112410068511963, -0.008794370107352734, -0.5353925824165344, -0.7362657189369202, -2.0344650745391846, 0.9948235154151917, -0.7834998369216919, 1.2209917306900024, 2.367502212524414, -1.6574941873550415, 0.2831256687641144, -0.7382742762565613, 0.08700570464134216, 0.4373442828655243, -1.29562509059906, 1.0287195444107056, -0.210819810628891, 1.641721487045288, 0.0011391419684514403, 0.18426434695720673, 0.01449121069163084, 2.857189416885376, -0.49896106123924255, -2.4209437370300293, 0.9694898724555969, 0.08414993435144424, 0.8710277080535889, 1.2083603143692017, 1.5448030233383179, 0.2502979636192322, 0.46934473514556885, 1.1432973146438599, 1.4255105257034302, -1.3913297653198242, 1.9412189722061157, -1.1249171495437622, 0.23306843638420105, 1.1932936906814575, -0.50236576795578, -0.5102028846740723, 1.5570075511932373, 1.5249093770980835, -1.4352879524230957, -0.042129356414079666, 0.41578951478004456, 1.491700530052185, 0.4436294436454773, 0.9671694040298462, 0.47837838530540466, 0.839093804359436, 2.699920892715454, 0.7026450634002686, 0.5371375679969788, -1.4853466749191284, -1.0223852396011353, -1.380225658416748, -0.027858933433890343, -0.30637744069099426, -0.7328572869300842, 0.7126734256744385, -1.5537550449371338, 0.2064899057149887, -0.16118159890174866, -0.8273063898086548, -0.9070391654968262, -0.5007280111312866, 0.8029313087463379, 0.9428603649139404, -1.5353673696517944, 0.06861548125743866, -0.6893009543418884, 0.9700625538825989, 1.249950885772705, 0.8972249627113342, 0.5533168315887451, 0.7020283341407776, -1.7658389806747437, 0.4738142490386963, 0.3226320445537567, -0.16137893497943878, -0.7304857969284058, -1.2131046056747437, -0.8251623511314392, -0.6091238260269165, -2.650817394256592, -0.09608355164527893, 0.22428765892982483, 1.6288514137268066, 1.6613738536834717, 0.5181747078895569, 0.10168439894914627, 0.06068892031908035, 0.9243081212043762, 1.0894582271575928, -1.540497899055481, 0.8203302025794983, -0.443296879529953, 0.026012416929006577, -0.32769861817359924, 0.4076686501502991, -0.9879329800605774, 0.07042787969112396, -1.4709097146987915, 1.0538818836212158, 0.8822095394134521, -0.3523580729961395, -0.07620307058095932, -0.59536212682724, 0.7409249544143677, 0.5893867015838623, -0.39792323112487793, -0.14772173762321472, 0.2972710132598877, -0.7242751717567444, 0.3284463882446289, 0.5573767423629761, 2.498046398162842, 0.4048769474029541, -0.6212624311447144, 1.3280513286590576, 0.40516191720962524, -0.35747095942497253, 0.9199165105819702, -0.48740196228027344, 1.519829273223877, 0.8489220142364502, -0.23085792362689972, 0.059109702706336975, 2.1642282009124756, -0.183842271566391, -0.2990058660507202, 0.800944983959198, 0.2789308726787567, -0.07866953313350677, 0.7682797312736511, -1.9072436094284058, 0.19040286540985107, 1.5187855958938599, 1.444676160812378, -0.6666692495346069, 0.0976346954703331, 0.8901374936103821, -0.17571868002414703, 0.601898193359375, 0.5684327483177185, -1.6900744438171387, 0.8207048773765564, 0.12045550346374512, -0.05868373066186905, -0.22641028463840485, -2.114586591720581, -0.04448578879237175, 1.0339571237564087, 1.3984943628311157, 0.2350405603647232, -0.10707439482212067, -0.3265059292316437, 1.3615119457244873, 2.1674764156341553, 0.10878574848175049, -2.5172324180603027, 0.3216055631637573, 1.6222150325775146, -1.2764140367507935, -0.7487925291061401, -1.4930503368377686, 0.2715160548686981, 2.381467819213867, 1.090132713317871, -1.5634765625, -1.279088020324707, 0.5858190059661865, -0.2501157820224762, -1.6739835739135742, 0.3591102659702301, -0.8359280228614807, 0.4886559247970581, 0.5909976363182068, 0.33104369044303894, -1.0452159643173218, -0.8304569125175476, -0.16881175339221954, -0.5934788584709167, 1.0522581338882446, 1.1718295812606812, 0.2023177593946457, 0.9073431491851807, 1.5246953964233398, -0.9389344453811646, -0.3865225911140442, 1.9431071281433105, -0.16058410704135895, -0.5235275030136108, 0.06046416237950325, 0.49862632155418396, -1.4721148014068604, 0.03436671569943428, 0.9909331202507019, -0.4086025357246399, -0.47618621587753296, 0.9608758091926575, -0.7984333634376526, -1.2547075748443604, -0.17979766428470612, 0.6872251629829407, -0.3144865334033966, -2.6788430213928223, -0.7755390405654907, 0.36003515124320984, 1.6751784086227417, -0.12297989428043365, 2.477046489715576, -1.0433647632598877, -0.5776107311248779, 0.07212820649147034, 0.7706814408302307, -0.32573050260543823, 0.6396445631980896, -0.3071846663951874, 0.47520267963409424, 0.5519726276397705, 0.15930256247520447, 0.20601820945739746, -0.25180619955062866, -0.6979271769523621, 2.9675660133361816, -0.8800317645072937, 1.4161474704742432, 1.084635615348816, 0.06258807331323624, -0.005322332959622145, -0.6994736194610596, 0.25158339738845825, -1.0825525522232056, 0.048628710210323334, 2.0636279582977295, 1.2447062730789185, -1.3224797248840332, 0.21143372356891632, -0.338952898979187, -1.317351222038269, 1.1914833784103394, -1.7460272312164307, 0.5432092547416687, 1.9281970262527466, 1.6585434675216675, -1.1650097370147705, -0.1696958690881729, -0.10597384721040726, 0.009176526218652725, 1.004343032836914, 0.7364439368247986, 1.883129358291626, -0.5166074633598328, 0.525843620300293, 1.3586188554763794, 0.6222531795501709, -1.7283685207366943, 0.24647754430770874, 0.33459338545799255, 0.17694547772407532, -0.813716471195221, 0.8606323599815369, 0.19609688222408295, -0.6951507329940796, 0.6068761348724365, 0.005317931063473225, -0.42928963899612427, -0.5742053389549255, -0.97018963098526, 0.11804354935884476, -0.4934537708759308, 1.1308116912841797, -0.02724168635904789, 0.8051120042800903, -0.747714638710022, -0.4309273362159729, 0.0915476381778717, 0.7692331075668335, 0.9444231390953064, 0.16875523328781128, 0.2735322415828705, 0.6145146489143372, 0.8206976056098938, 1.0712908506393433, -1.6286091804504395, 0.9048022627830505, -0.03265930339694023, 0.1358170509338379, -0.7183337211608887, 0.8129758238792419, 1.2557382583618164, 1.2324249744415283, -0.5643959045410156, -1.490092158317566, 1.3490031957626343, -0.37253084778785706, -1.6615538597106934, 1.0007284879684448, -2.0749824047088623, 1.050241470336914, -0.46829962730407715, 1.9131025075912476, 0.09083381295204163, -0.5659886002540588, 0.14229871332645416, -0.8069026470184326, -0.5561097264289856, 0.4525800943374634, 0.07061924040317535, -1.184828758239746, -0.38904044032096863, 1.831741213798523, -1.301836609840393, 0.5979900360107422, 2.5432162284851074, -0.7658623456954956, 0.7995527982711792, 0.08297143131494522, -0.924187958240509, 0.8326864242553711, 0.7220970988273621, -0.006532507017254829, -0.7371259927749634, 1.0021880865097046, 0.6168938875198364, -0.26255562901496887, -0.8588775992393494, -0.2712271213531494, -0.47417470812797546, -0.5665594339370728, 1.2627906799316406, -0.09822102636098862, 1.4506536722183228, 1.1974811553955078, 1.0030888319015503, 0.5387202501296997, -0.05260651931166649, -0.04327453300356865, 2.0274457931518555, 1.1472090482711792, -0.3281440734863281, 0.7404688000679016, -0.5024834275245667, 0.006288661155849695, -0.9539056420326233, -0.9153535962104797, -1.0307247638702393, 0.37582868337631226, 1.1723538637161255, 0.02076609618961811, 0.3980984687805176, -0.24842536449432373, -0.5006099343299866, 0.3080090284347534, -1.156599760055542, 1.0786287784576416, 0.797358512878418, 0.3862922787666321, 0.026214275509119034, -0.5187779664993286, 1.624704122543335, -0.19201888144016266, -0.5654698610305786, -0.9846094250679016, -0.7030495405197144, -0.1829180121421814, -0.34061750769615173, 0.6730765700340271, -0.43075212836265564, -0.8024366497993469, 0.544248640537262, -0.7410117983818054, 1.816135287284851, -1.2790627479553223, -0.1297074258327484, -0.7912268042564392, 0.3372630774974823, 1.3587348461151123, -0.4485931098461151, -1.5733566284179688, 0.31231144070625305, 0.08465145528316498, 0.6620534658432007, -0.806540310382843, -0.6335685849189758, -0.7507585287094116, -0.5436558127403259, -0.08105389028787613, 1.3859360218048096, 0.4595661759376526, -0.36174413561820984, -1.4391441345214844, 1.4555625915527344, 0.6743341088294983, 1.4155501127243042, -2.76557993888855, 2.406003952026367, -0.9668239951133728, -1.0953117609024048, -0.7090142965316772, -0.5345151424407959, 1.38316810131073, -2.3014845848083496, 0.60566645860672, 1.3253849744796753, 1.1543794870376587, 0.6937899589538574, 0.9759087562561035, -0.3696712553501129, -0.7063874006271362, -0.6422846913337708, -1.7723320722579956, 0.9852070212364197, -0.04352862015366554, 0.3690893054008484, 1.5166877508163452, 1.4003154039382935, 0.5798710584640503, -0.8045787215232849, -1.7750385999679565, 0.8914026021957397, -1.6803473234176636, -0.2875310480594635, -0.12944947183132172, -0.7539559602737427, 0.11502167582511902, 0.9209204316139221, 1.462192177772522, -0.44932612776756287, -1.3749850988388062, -0.22583240270614624, -1.0868841409683228, -1.012609839439392, 0.2343895435333252, 1.0849181413650513, 0.14142581820487976, 0.25158390402793884, 1.137803077697754, 0.6237659454345703, -1.1762542724609375, -0.6020542979240417, -0.4297182857990265, 1.4333062171936035, -1.356695532798767, -0.2201797366142273, -2.8132996559143066, -0.5694693922996521, -0.9687359929084778, -1.0570579767227173, 0.5842776894569397, -0.9264652729034424, 1.3030503988265991, 0.10166919231414795, -0.33322688937187195, -1.136864185333252, 1.6642982959747314, 0.06775465607643127, 1.7431777715682983, 2.4472405910491943, -0.0071297804825007915, 0.7400540113449097, 1.3805993795394897, 0.03191603347659111, -1.6291273832321167, 1.0324780941009521, -0.5313614010810852, -1.1609939336776733, -0.6078178286552429, 1.4257051944732666, -0.9977236390113831, -0.570035994052887, -1.0842159986495972, 1.236306071281433, 0.7357566952705383, 1.679677963256836, 1.02485191822052, 0.5820419788360596, 1.5608524084091187, 1.1494715213775635, 0.3672601580619812, 0.8349547982215881, 2.095437526702881, 0.7937461137771606, -1.6052318811416626, -2.0729002952575684, -0.8813530802726746, 1.1696738004684448, 0.5851617455482483, 0.8589641451835632, -0.02111407369375229, -0.09902986884117126, -0.46104222536087036, -0.12341226637363434, -0.07457125186920166, -0.16472427546977997, -1.165820837020874, -1.4455435276031494, 1.2783756256103516, 0.11787553876638412, 0.38269278407096863, -0.6524879336357117, -0.03742975741624832, 0.42617204785346985, -0.4319530129432678, -0.97300785779953, 0.4710332155227661, 0.5176971554756165, -1.3778643608093262, -1.0892540216445923, 1.1320103406906128, -1.0374737977981567, -1.0606310367584229, -1.0327003002166748, 0.06078599765896797, -0.2834794223308563, 0.6635103225708008, -1.2750194072723389, -0.7974661588668823, -0.12704196572303772, -0.1945815235376358, -1.14289128780365, -0.515985369682312, -0.958632230758667, -0.3734370172023773, -1.337563395500183, 0.9325788021087646, -0.37493470311164856, -0.5037890672683716, -2.8114564418792725, -0.8684823513031006, 0.23336386680603027, 0.7744171619415283, -1.3207452297210693, 1.7027392387390137, 2.1528923511505127, -0.8747212290763855, 1.028992772102356, -1.0543649196624756, 0.30564799904823303, 0.3081786036491394, 0.9366925954818726, 0.2923131287097931, -0.3059340715408325, 0.09755175560712814, 0.6604947447776794, -0.4920262098312378, 0.6940356492996216, -0.6780015230178833, 0.24865855276584625, 0.9805629849433899, -0.599010705947876, 0.3318909704685211, -2.6006758213043213, -0.5510551929473877, 2.1091127395629883, -1.0211163759231567, -1.1007779836654663, 0.8256857991218567, -0.44566619396209717, -0.7165394425392151, -0.5580030083656311, 0.9380759000778198, -0.41745898127555847, -1.8738220930099487, 1.190698504447937, 1.2136366367340088, 0.6127249598503113, -0.19988799095153809, -1.2270900011062622, 1.84199059009552, -2.186030149459839, 1.1879411935806274, 1.0110450983047485, 0.6889472603797913, 0.5024582743644714, 0.967692494392395, 0.060258232057094574, 0.9552602767944336, 0.7526636123657227, 0.11329799145460129, 0.14301598072052002, 0.08062425255775452, 1.4896584749221802, -1.0286037921905518, -0.2420467585325241, 1.39024817943573, -0.7929385900497437, 0.33115077018737793, -0.026223575696349144, -1.5746082067489624, 0.15314048528671265, 0.8761118054389954, -0.19885961711406708, 0.9706213474273682, 0.5160332322120667, 0.34038814902305603, -1.6420249938964844, 0.5721516609191895, 0.3268831670284271, 0.723476231098175, 0.637150764465332, -0.9152162075042725, -0.5645313262939453, -1.2481272220611572, 0.260157972574234, -0.7928549647331238, 1.3375167846679688, 0.9375148415565491, -1.2213608026504517, 0.5833132863044739, -0.5938892364501953, -2.042079210281372, 0.504184901714325, -1.3502929210662842, -0.4763583838939667, 0.6148995757102966, 0.5689529180526733, 2.6950786113739014, 0.5329428315162659, -0.15651758015155792, -0.869663655757904, -2.1979026794433594, -0.565466582775116, -0.14996235072612762, 1.8366305828094482, -0.3998720943927765, -1.6735179424285889, 0.19253495335578918, 0.0036213870625942945, 0.6053935289382935, 0.039994243532419205, -0.8968195915222168, -1.633188247680664, -0.4565337002277374, -0.3079160749912262]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[0.06504808366298676, -0.10961481928825378, 0.03411523252725601, 0.002479677787050605, 0.04694092273712158, 0.013731682673096657, 0.06506595760583878, 0.0011303479550406337, -0.018382161855697632, 0.020670203492045403, 0.015386681072413921, -0.012706518173217773, -0.008806120604276657, 0.025618212297558784, 0.025079645216464996, -0.017588263377547264, -0.01309373788535595, -0.0643371120095253, 0.05979407578706741, -0.007986663840711117, 0.047618456184864044, 0.014865328557789326, 0.0692216232419014, 0.05473419651389122, 0.011334133334457874, 0.05272778123617172, -0.03383248299360275, 0.044817518442869186, -0.05271567776799202, 0.012222135439515114, 0.03791866451501846, 0.007963207550346851, -0.013073218986392021, -0.028684793040156364, -0.012206598185002804, 0.037429001182317734, 0.03172146528959274, -0.009334110654890537, -0.04436836019158363, 0.02400003746151924, -0.05162785202264786, -0.01890295185148716, -0.026166142895817757, 0.02646656706929207, 0.008401818573474884, 0.016445958986878395, 0.009700040332973003, -0.02414008229970932, 0.01906750164926052, 0.05029560998082161, -0.0013093691086396575, 0.02256135456264019, -0.0044832248240709305, 0.03662532567977905, 0.023737937211990356, 0.031186625361442566, 0.01240682415664196, 0.05839809775352478, -0.012860034592449665, -0.011899156495928764, -0.021075252443552017, 0.02175547182559967, -0.022162746638059616, -0.022950846701860428, 0.025904294103384018, 0.04138049855828285, 0.020451800897717476, -0.0955413207411766, -0.033957432955503464, -0.038225360214710236, -0.04299767315387726, -0.01451999880373478, 0.06472001224756241, -0.010946004651486874, -0.02618812955915928, 0.024884037673473358, 0.0024631021078675985, 0.004297275096178055, -0.024451442062854767, -5.357182817533612e-05, 0.01547322142869234, -0.06805878132581711, -0.0573376901447773, 0.03861958533525467, 0.030018137767910957, -0.01804129034280777, -0.036327943205833435, -0.006607038900256157, -0.04250418022274971, -0.012110346928238869, 0.05879322439432144, 0.004702881444245577, 0.04152979701757431, -0.007927185855805874, 0.01315433718264103, 0.04934735968708992, -0.04757074639201164, -0.012373174540698528, -0.005586921703070402, 0.010040929540991783, 0.055323757231235504, 0.02441059611737728, -0.053329676389694214, -0.009648910723626614, -0.03182131052017212, -0.032725896686315536, 0.07210110127925873, -0.03785080090165138, 0.00841767992824316, -0.0254676416516304, 0.03065824881196022, 0.01259484514594078, 0.004537511151283979, -0.0027121983002871275, 0.04051240533590317, -0.033176444470882416, -0.025885190814733505, -0.03976067528128624, 0.047521211206912994, -0.010453461669385433, -0.03694029524922371, -0.014922579750418663, 0.08900871127843857, -0.003826951840892434, 0.03026796691119671, 0.01656443253159523, -0.00044095024350099266, -0.02466619201004505, -0.011903755366802216, -0.012621521949768066, -0.07866903394460678, 0.09223075211048126, 0.03293204307556152, 0.004140789154917002, -0.007240959908813238, -0.028118059039115906, 0.03381461650133133, -0.006653663236647844, 0.01706618070602417, -0.049406636506319046, 0.01347698550671339, 0.07122774422168732, -0.010313252918422222, 0.02848520502448082, 0.001258178148418665, 0.03935418650507927, 0.0380093976855278, 0.0021951354574412107, -0.019926004111766815, 0.010210888460278511, 0.0014873328618705273, -0.02200261689722538, 0.01625453121960163, 0.01539437286555767, -0.031656913459300995, -0.003969918005168438, 0.10401742905378342, 0.029086260125041008, 0.0034817757550626993, 0.08941590785980225, -0.016216229647397995, -0.08819708228111267, -0.06790334731340408, -0.05410752445459366, 0.009260173887014389, 0.03476659208536148, -0.07727724313735962, 0.03978338837623596, -0.04371793940663338, 0.009131263010203838, 0.016519486904144287, 0.015394621528685093, 0.0002687402884475887, 0.006088895723223686, -0.04058729484677315, 0.00990688893944025, 0.03488396108150482, 0.03298140689730644, 0.09303979575634003, 0.038572296500205994, 0.025866996496915817, 0.059234775602817535, 0.008586551994085312, -0.04593060538172722, -0.07975807040929794, 0.045456789433956146, 0.020226750522851944, 0.03017098270356655, -0.011853178963065147, 0.012278861366212368, 0.06702880561351776, 0.011922947131097317, -0.039721958339214325, -0.015363281592726707, 0.049321725964546204, -0.03881188854575157, -0.09192086011171341, -0.08168432116508484, -0.047677621245384216, 0.028465213254094124, 0.0031500428449362516, -0.026965510100126266, -0.045761801302433014, -0.004058989696204662, -0.0056829629465937614, -0.020080598071217537, -0.008733134716749191, -0.027175767347216606, 0.06421004980802536, -0.0842832624912262, 0.004350963048636913, -0.03612147271633148, 0.025157999247312546, -0.008371781557798386, 0.013033478520810604, 0.01892935112118721, 0.007311361841857433, 0.035289473831653595, -0.049548447132110596, 0.03388858214020729, -0.0077962144277989864, 0.013554179109632969, 0.05542583391070366, 0.04003845155239105, 0.07064394652843475, -0.08014021813869476, -0.034037522971630096, 0.03433257341384888, -0.04026176407933235, -0.006501858122646809, 0.06081491708755493, 0.031146764755249023, -0.034264951944351196, 0.01615937054157257, 0.02297819033265114, 0.0023991726338863373, -0.0035788812674582005, 0.005040650721639395, -0.053431056439876556, 0.0045049190521240234, -0.034430261701345444, 0.014014479704201221, 0.015283776447176933, -0.03179103881120682, 0.021816305816173553, 0.012669950723648071, -0.0014989860355854034, 0.025844896212220192, -0.01113822404295206, -0.010118442587554455, 0.0027698499616235495, -0.0021088358480483294, 0.040670860558748245, -0.08453265577554703, -0.01105550117790699, -0.07174958288669586, -0.018738575279712677, 0.008269301615655422, -0.000788592966273427, 0.03344820812344551, 0.010206383652985096, -0.09888036549091339, -0.041246868669986725, 0.053899265825748444, 0.05397966876626015, 0.0018640015041455626, -0.05703727528452873, 0.0754474550485611, -0.07666638493537903, 0.008910786360502243, -0.003983634058386087, 0.006340798921883106, -0.04418009892106056, -0.01681586168706417, 0.03568504378199577, -0.020168298855423927, -0.052039485424757004, -0.003449036506935954, 0.00022322390577755868, -0.007856137119233608, -0.03124234266579151, 0.016204843297600746, -0.007592016831040382, -0.04504113271832466, 0.00472328532487154, -0.0041631124913692474, 0.0020986003801226616, -0.06385255604982376, -0.030544573441147804, 0.013281969353556633, 0.030320536345243454, 0.009560465812683105, 0.003024178557097912, 0.023411709815263748, 0.03275136277079582, 0.060851726680994034, 0.014818896539509296, -0.022002456709742546, 0.015428761020302773, -0.07101453095674515, 0.000720301060937345, -0.0169984083622694, -0.0018371365731582046, 0.016606474295258522, 0.001018132665194571, -0.05242428183555603, -0.01341259479522705, -0.029166486114263535, -0.0012539264280349016, 0.012383155524730682, -0.034763701260089874, 0.025766834616661072, 0.046051930636167526, -0.009930658154189587, 0.025690242648124695, -0.053039148449897766, -0.01949695125222206, 0.0341046042740345, -0.0630885437130928, -0.01824517920613289, -0.03212295100092888, 0.04072967916727066, 0.04062027111649513, -0.022403806447982788, 0.004210286308079958, -0.0932442843914032, 0.030091755092144012, 0.012281128205358982, 0.04706292971968651, -0.08523300290107727, 0.012194189243018627, -0.012765378691256046, -0.09320822358131409, -0.02487194538116455, -0.003070330712944269, 0.028396427631378174, 0.006772775202989578, -0.018601208925247192, 0.015635613352060318, -0.00407383544370532, -0.02337726205587387, -0.03655499219894409, 0.005086409859359264, 0.0053731477819383144, 0.06536911427974701, -0.01628001034259796, 0.0569474995136261, 0.01232275553047657, 0.041793208569288254, -0.03241071105003357, -0.03127866983413696, -0.01111558172851801, 0.031787995249032974, 0.006169148720800877, 0.026734769344329834, 0.011250820010900497, 0.036579929292201996, 0.0070150080136954784, -0.01874750666320324, 0.018304480239748955, 0.0032059664372354746, 0.06225566565990448, -0.005687467288225889, 0.0038821983616799116, -0.010911854915320873, 0.041167374700307846, 0.03114376775920391, 0.023184673860669136, -0.044392868876457214, -0.05400191247463226, 0.009431722573935986, 0.0257106926292181, 0.05267203971743584, -0.06394196301698685, -0.00606431532651186, -0.060194890946149826, -0.03421010076999664, -0.007873124442994595, 0.03429170325398445, -0.01875203847885132, -0.025573231279850006, -0.011388683691620827, 0.04478361830115318, 0.08602705597877502, -0.05490868538618088, -0.00973884854465723, -0.015322303399443626, 0.04161928594112396, 0.015513424761593342, 0.05486307665705681, -0.036040354520082474, 0.0004904186935164034, -0.01622163876891136, -0.03904176503419876, -0.005627355072647333, -0.042178258299827576, 0.07736382633447647, 0.014859114773571491, -0.03619461879134178, -0.02158048190176487, 0.061696603894233704, -0.017025083303451538, 0.024716060608625412, 0.020783064886927605, 0.03891054913401604, 0.02260127104818821, -0.007481855805963278, -0.021062377840280533, -0.06323274970054626, -0.0016968437703326344, 0.004938955884426832, -0.012825993821024895, -0.0051147122867405415, -0.043227050453424454, 0.07731214165687561, -0.017840318381786346, -0.03686893358826637, -0.02125866524875164, -0.005998997949063778, 0.0264179278165102, -0.0864344909787178, 0.018365297466516495, -0.021929416805505753, -0.00638228002935648, 0.036230333149433136, 0.02730090543627739, -0.0241489727050066, -0.0033020838163793087, 0.06673087179660797, 0.048537902534008026, 0.03100654110312462, 0.0004140324890613556, 0.03438994288444519, 0.014259006828069687, -0.01941433548927307, -0.03550295904278755, 0.018767789006233215, 0.013762789778411388, 0.04314173758029938, -0.01705627143383026, 0.050095684826374054, -0.012856983579695225, 0.044880785048007965, 0.019611695781350136, -0.020394504070281982, -0.03358189016580582, -0.004146020859479904, -0.009264068678021431, 0.03362390026450157, -0.024485064670443535, -0.05249668285250664, -0.000532289152033627, 0.06860296428203583, 0.05847637355327606, -0.022007491439580917, -0.009630694054067135, 0.01148789282888174, -0.023487113416194916, -0.008601143024861813, -0.009679934941232204, -0.03726447373628616, -0.02189873345196247, -0.01714511029422283, 0.008487760089337826, -0.04201045259833336, 0.003865818725898862, -0.03485853224992752, -0.028695276007056236, -0.03991807624697685, 0.1429685354232788, 0.04486629739403725, 0.048010632395744324, 0.02906324900686741, -0.008266793563961983, 0.002517005894333124, 0.010407361201941967, 0.01603955216705799, 0.021286774426698685, 0.05446934700012207, -0.10530105978250504, -0.019668130204081535, 0.024219896644353867, -0.012658634223043919, -0.005488209426403046, 0.053187295794487, 0.006910682190209627, 0.06356357783079147, -0.0014765990199521184, 0.008530735038220882, -0.03790974244475365, -0.07070576399564743, -0.014838152565062046, 0.002261923626065254, -0.012246393598616123, -0.047955263406038284, 0.023449502885341644, 0.002593929646536708, 0.0003904431068804115, -0.027668675407767296, 0.058764662593603134, 0.05266992747783661, 0.011013304814696312, -0.00408182293176651, 0.009049654938280582, 0.060458965599536896, 0.10012543946504593, 0.0004712485824711621, -0.015161757357418537, -0.06450780481100082, 0.09967879951000214, -0.03929976373910904, 0.03113177977502346, 0.01786385476589203, 0.04757079482078552, -0.030611475929617882, -0.0576903261244297, 0.005358225200325251, 0.014696730300784111, 0.04099841043353081, -0.05999471992254257, -0.016119178384542465, -0.009324591606855392, -0.04816588759422302, -0.005744826979935169, 0.028565242886543274, -0.0011449597077444196, -0.051210127770900726, 0.012420089915394783, 0.03066941909492016, 0.019192036241292953, 0.028219930827617645, -0.05881965532898903, 0.016510365530848503, 0.021476611495018005, -0.013310236856341362, -0.0018668888369575143, -0.02742006815969944, 0.04411773011088371, -0.010544783435761929, -0.033554982393980026, 0.006742441561073065, 0.004075438249856234, 0.023923728615045547, -0.029635854065418243, -0.0240184236317873, 0.013770220801234245, 0.07979586720466614, 0.016750069335103035, -0.048466622829437256, 0.006127328146249056, -0.02765168994665146, -0.06588809937238693, -0.02429232746362686, -0.020403411239385605, 0.015017668716609478, 0.047187235206365585, 0.04069969430565834, 0.02895791083574295, -0.015305624343454838, 0.005685307551175356, -0.0013053492875769734, 0.05034349113702774, -0.02313992753624916, -0.01949300989508629, 0.053463321179151535, 0.02070501632988453, -0.015889696776866913, 0.006016995757818222, 0.005474453791975975, 0.006882695946842432, 0.028696125373244286, 0.013238306157290936, -0.00639314204454422, -0.017890725284814835, -0.061432622373104095, -0.015411249361932278, 0.04098987951874733, 0.008270598016679287, -0.023104827851057053, -0.008491674438118935, -0.03037707321345806, 0.010545037686824799, 0.07178372144699097, 0.06467527151107788, 0.04300161451101303, 0.028128372505307198, -0.007826537825167179, -0.043994590640068054, 0.0006993325077928603, -0.015691930428147316, 0.02999497950077057, 0.006029592826962471, -0.020151758566498756, 0.01652291975915432, 0.10455889999866486, -0.03192729502916336, 0.04095402732491493, 0.021694405004382133, -0.002998489188030362, -0.026532087475061417, 0.0012442150618880987, 0.008365903049707413, 0.02798483893275261, 0.02566474862396717, 0.011951768770813942, -0.023791728541254997, 0.011341093108057976, -0.016103196889162064, -0.036695536226034164, 0.03248676285147667, -0.023175310343503952, -0.04213595390319824, 0.0034981737844645977, -0.06835921853780746, -0.0019018020248040557, 0.016389792785048485, -0.014597171917557716, 0.008856238797307014, 0.0771736353635788, 0.051117487251758575, 0.03356163576245308, 0.0216949675232172, -0.01814304292201996, -0.004707814194262028, -0.00994986854493618, 0.0011964954901486635, 0.004370016977190971, -0.028663525357842445, 0.05764631927013397, -0.03696957603096962, 0.013057327829301357, 0.03695673868060112, -0.0009430416394025087, -0.02378767728805542, 0.0011900159297510982, 0.06918229907751083, 0.022294627502560616, -0.01031247153878212, 0.020832780748605728, -0.006992414128035307, -0.06682119518518448, 0.008700978942215443, 0.03714273124933243, 0.041144732385873795, -0.002323560416698456, -0.0503142848610878, 0.04270526394248009, -0.00551808113232255, 0.026267623528838158, 0.040271610021591187, 0.01205737330019474, -0.0022242311388254166, -0.022447148337960243, -0.01550985686480999, 0.0010079707717522979, -0.01660732924938202, 0.012819722294807434, -0.06984385848045349, -0.036282431334257126, -0.001375216874293983, -0.025049889460206032, -0.019495174288749695, -0.04092146083712578, 0.02354002185165882, 0.05770072340965271, -0.03632039204239845, 0.04047286882996559, -0.052347127348184586, 0.004434442147612572, -0.0719422847032547, 0.01669389382004738, 0.00012853863881900907, 0.004347962327301502, 0.042679205536842346, 0.020397664979100227, -0.001717710169032216, -0.029999924823641777, 0.0009494430269114673, -0.025571251288056374, -0.015964709222316742, 0.05172720551490784, 0.03025251440703869, 0.009267055429518223, 0.021376958116889, -0.0524357333779335, 0.0027142390608787537, -0.06136895343661308, -0.024870414286851883, 0.011968564242124557, 0.012181859463453293, 0.023652497678995132, 0.004132583271712065, 0.0003134300059173256, -0.0016643184935674071, 0.04841720312833786, 0.044619910418987274, -0.05360207334160805, 0.0027904801536351442, 0.044085901230573654, -0.011059107258915901, 0.005794071592390537, -0.02356117218732834, -0.045007240027189255, 0.0023583187721669674, -0.017124846577644348, -0.008493983186781406, -0.054462872445583344, -0.04440959915518761, -0.00806715339422226, -0.01939629577100277, 0.015787944197654724, -0.04145638644695282, 0.02107228711247444, 0.015936966985464096, -0.0011818746570497751, 0.018339335918426514, -0.00770661560818553, -0.011828310787677765, -0.006041448097676039, -0.05025657266378403, -0.006694926880300045, 0.022927898913621902, -0.06267250329256058, 0.018801499158143997, -0.02827092818915844, -0.018423795700073242, -0.0026075607165694237, 0.0030009006150066853, 0.042648181319236755, -0.011018211953341961, 0.05505313724279404, -0.004364724270999432, 0.01149489264935255, 0.01830432377755642, -0.029305096715688705, -0.04526154696941376, -0.0014221961610019207, 0.019590526819229126, 0.025565309450030327, -0.017660263925790787, 0.0578787662088871, -0.008854363113641739, -0.02823588438332081, 0.0375027060508728, 0.04014165326952934, -0.03677587956190109, 0.02684701792895794, -0.04723591357469559, -0.0003969095123466104, 0.07914308458566666, 0.04473145306110382, 0.024288291111588478, 0.022536272183060646, -0.02956084907054901, 0.011853267438709736, 0.016149887815117836, -0.03640920668840408, 0.007745744194835424, 0.020109131932258606, 0.00840927567332983, 0.016164090484380722, -0.007106229662895203, -0.03883896768093109, 0.002319896826520562, -0.06709721684455872, -0.024150656536221504, 0.04017554596066475, 0.06489848345518112], [0.016662413254380226, -0.08796826750040054, 0.051325052976608276, 0.02186414785683155, -0.0053010364063084126, 0.029634445905685425, 0.022135265171527863, -0.04244647175073624, -0.04630913585424423, 0.04055042192339897, -0.08326446264982224, 0.024296212941408157, 0.01379578560590744, 0.038515739142894745, 0.0669434517621994, 0.051928188651800156, 0.060119178146123886, 0.032931067049503326, 0.021312175318598747, 0.042382411658763885, 0.002269076183438301, -0.031168026849627495, -0.0001912126172101125, -0.05669093132019043, -0.017723971977829933, 0.07940524071455002, 0.021951796486973763, -0.0038706844206899405, -0.0196713674813509, -0.06321825832128525, -0.01745523326098919, -0.026396099478006363, 0.0023413943126797676, 0.00527109345421195, -0.00030032856739126146, -0.06882361322641373, 0.056046292185783386, 0.018429715186357498, 0.002273974474519491, -0.010145223699510098, 0.018199652433395386, 0.04807514324784279, 0.02428264729678631, -0.041468456387519836, -0.04503629356622696, -0.08633652329444885, 0.005940878298133612, 0.02766169048845768, 0.04622264578938484, 0.005773297045379877, 0.031029148027300835, -0.033464815467596054, 0.019809402525424957, -0.004272589460015297, -0.024493245407938957, -0.062285758554935455, 0.04303870350122452, -0.08372558653354645, 0.023269647732377052, -0.0298534594476223, 0.022544845938682556, -0.020652927458286285, 0.027916232123970985, -0.008192156441509724, -0.03898954764008522, -0.019376808777451515, -0.06011787801980972, 0.010041220113635063, 0.03855714946985245, 0.015416771173477173, 0.0037791200447827578, 0.047787293791770935, -0.04355786740779877, 0.020494582131505013, 0.00019026844529435039, -0.017672928050160408, 0.02028871700167656, -0.04273988679051399, -0.005135762505233288, 0.012416891753673553, -0.04348110035061836, -0.008855597116053104, 0.019447291269898415, 0.035293661057949066, -0.03519119694828987, 0.033341579139232635, -0.011098119430243969, 0.003338797949254513, -0.02053607814013958, 0.028921175748109818, -0.02635747753083706, 0.0103990463539958, 0.013975534588098526, 0.027971507981419563, -0.03880767896771431, 0.02866493910551071, 0.0010976961348205805, 0.053883057087659836, -0.012938449159264565, -0.021288100630044937, -0.0022846413776278496, -0.011801264248788357, -0.006499771028757095, -0.04740244895219803, -0.07768667489290237, -0.054881948977708817, -0.02225775457918644, 0.015672648325562477, -0.011895500123500824, -0.036024950444698334, -0.006113620474934578, 0.0019894372671842575, 0.10242950916290283, 0.04726366326212883, 0.006664640735834837, -0.054446179419755936, -0.006863728165626526, -0.01165767665952444, 0.020339570939540863, 0.07865719497203827, -0.002745080506429076, -0.07363469153642654, -0.004642592277377844, -0.06270813196897507, 0.030914612114429474, 0.040855467319488525, 0.012233482673764229, -0.05549761652946472, -0.006823041010648012, 0.0449848547577858, -0.037495024502277374, -0.022172583267092705, 0.009384989738464355, 0.018387215211987495, 0.002857118844985962, 0.012566400691866875, 0.007670173887163401, -0.02166491560637951, -0.0031080367043614388, 0.01052423007786274, -0.04064395651221275, 0.007219061255455017, 0.0036170247476547956, 0.01716763526201248, 0.012941254302859306, -0.039042092859745026, -0.046831049025058746, 0.00930647924542427, -0.005604946985840797, 0.0037414957769215107, -0.020971283316612244, -0.047552887350320816, 0.009957383386790752, -0.004916149191558361, 0.022840004414319992, 0.04059312865138054, 0.04063032567501068, 0.03675398230552673, -0.03573179244995117, -0.07859828323125839, -0.02296147495508194, -0.045146066695451736, -0.04924163967370987, 0.02417822740972042, -0.060157328844070435, 0.033696841448545456, 0.04047852009534836, 0.09873589128255844, 0.03834785893559456, 0.07771329581737518, -0.02414146438241005, -0.02565109170973301, 0.036101214587688446, -0.04918043687939644, 0.0421447679400444, -0.01047197263687849, -0.07338433712720871, 0.06855139881372452, -0.041924476623535156, 0.09263979643583298, -0.055167097598314285, -0.0317452996969223, 0.007685952819883823, 0.00023480883101001382, -0.059034883975982666, 0.0037459940649569035, 0.006677933037281036, 0.016655657440423965, -0.06003933399915695, -0.022767310962080956, -0.004203855525702238, 0.02970755659043789, 0.005921717267483473, -0.013137808069586754, -0.016021616756916046, 0.054069820791482925, -0.030666330829262733, 0.010875714011490345, -0.08091175556182861, 0.0630502924323082, 0.016855040565133095, -0.06567534059286118, 0.022306956350803375, 0.05143057927489281, 0.030227283015847206, 0.002605176530778408, 0.03127559646964073, -0.027836861088871956, -0.009632482193410397, -0.013479211367666721, 0.002639728132635355, -0.03900611773133278, 0.05106522515416145, -0.004817451816052198, -0.040320344269275665, -0.008292068727314472, 0.013410222716629505, 0.0015042676823213696, 0.04074694216251373, -0.013471223413944244, -0.028849296271800995, 0.01846231147646904, 0.0474032387137413, -0.03613801673054695, -0.05923866108059883, -0.024162331596016884, -0.0030419055838137865, -0.038533687591552734, 0.005786807741969824, -0.03239481896162033, 0.032894521951675415, -0.04447969049215317, -0.03433884680271149, 0.002279479755088687, 0.024713516235351562, 0.033854711800813675, -0.02608506940305233, 0.0002190626983065158, 0.01956433802843094, 0.05253840237855911, -0.03369653597474098, -0.046922244131565094, 0.017404941841959953, -0.009940418414771557, 0.04954976961016655, 0.00032525250571779907, 0.011463467963039875, -0.029833626002073288, 0.009770193137228489, 0.04489177465438843, -0.00985206663608551, -0.04500530660152435, 0.032261501997709274, -0.01217698398977518, 0.0013991730520501733, -0.027263319119811058, 0.03434748575091362, -0.02844318374991417, -0.018672320991754532, -0.00046147260582074523, -0.008243991993367672, -0.08420239388942719, 0.01673453487455845, -0.04416480287909508, 0.0088046221062541, -0.015298151411116123, -0.005390762351453304, -0.02816365472972393, -0.040338341146707535, 0.004522116854786873, 0.0072040013037621975, 0.03671320900321007, 0.019185591489076614, -0.026924341917037964, 0.04258328676223755, 0.08119698613882065, 0.02579844743013382, -0.007985827513039112, -0.03688817471265793, 0.0023762353230267763, -0.05437341704964638, 0.028587952256202698, -0.015115919522941113, -0.02236803248524666, 0.008587634190917015, -0.047180794179439545, -0.06576615571975708, -0.017322037369012833, 0.04420369490981102, -0.03265044838190079, 0.031300824135541916, -0.06948277354240417, -0.003518720157444477, 0.017870385199785233, 0.06530886888504028, -0.06662405282258987, -0.003566047875210643, -0.03488574177026749, 0.010188019834458828, -0.02172076515853405, 0.0026171947829425335, -0.029293473809957504, -0.021802041679620743, 0.01365896686911583, 0.05422232672572136, 0.006549072917550802, 0.0033070798963308334, -0.01844409666955471, 0.057410549372434616, -0.0733371153473854, -0.07522758841514587, -0.030189046636223793, 0.01514499168843031, 0.03394683450460434, 0.019872670993208885, 0.02670319750905037, 0.0048542083241045475, -0.01865396648645401, -0.04059779271483421, 0.01540368888527155, -0.03640831634402275, 0.014357159845530987, 0.026294777169823647, -0.024139996618032455, -0.017835551872849464, -0.021626628935337067, 0.024140695109963417, 0.004557782784104347, -0.026007790118455887, 0.008679303340613842, 0.05464883893728256, 0.009212867356836796, 0.0034455445129424334, -0.016036808490753174, -0.035691406577825546, -0.001073126564733684, 0.0012488338397815824, -0.012649002484977245, 0.042585764080286026, -0.009846906177699566, -0.0070947096683084965, 0.005969979800283909, 0.018506260588765144, -0.0674634724855423, 0.008588525466620922, 0.045216985046863556, -0.0423259474337101, 0.013289856724441051, -0.019896874204277992, -0.037719789892435074, 0.008685522712767124, -0.004813695326447487, -0.07791926711797714, -0.022792890667915344, 0.0290148314088583, -0.022731361910700798, -0.028529347851872444, -0.01466414425522089, -0.017421942204236984, 0.026440225541591644, -0.03236490115523338, -0.044073328375816345, 0.014954737387597561, 0.03569293022155762, 0.024904703721404076, 0.030044419690966606, -0.024856368079781532, -0.05520832911133766, -0.026781322434544563, -0.0016829106025397778, -0.06342361867427826, -0.011947041377425194, 0.020657556131482124, -0.026250900700688362, 0.07246628403663635, 0.026932504028081894, -0.045562829822301865, -0.04242381826043129, -0.026755454018712044, 0.025387631729245186, 0.016467681154608727, -0.021345967426896095, 0.005569724831730127, 0.02876267209649086, 0.018591688945889473, -0.05981387943029404, -0.04070252552628517, -0.034379396587610245, -0.06434682756662369, 0.030931998044252396, -0.02438288740813732, -0.024808956310153008, -0.03150228038430214, 0.0104336217045784, 0.02018193155527115, 0.02416251040995121, 0.04490329325199127, -0.034975141286849976, 0.05920935794711113, 0.09066028147935867, -0.020148882642388344, -0.013902529142796993, 0.021615082398056984, 0.0450601801276207, -0.010489916428923607, -0.030302835628390312, 0.07613395899534225, -0.019018210470676422, -0.10171303153038025, 0.019288524985313416, 0.03634003922343254, 0.06266117095947266, 0.06504949182271957, 0.11391828209161758, -0.020937971770763397, -0.03668760880827904, -0.054111845791339874, 0.06299661844968796, -0.016428101807832718, -0.06250011920928955, -0.07657228410243988, 0.0044251782819628716, -0.016231974586844444, 0.011359340511262417, -0.032079022377729416, -0.02759709767997265, 0.01710953190922737, -0.0077555482275784016, 0.02113400213420391, -0.005042668431997299, -0.010900366120040417, 0.013107501901686192, -0.012424045242369175, 0.006281246896833181, 0.01628108322620392, -0.019260264933109283, -0.004669592250138521, 0.0034957067109644413, 0.024562058970332146, 0.008432716131210327, 0.008783891797065735, 0.016844285652041435, -0.011879618279635906, 0.012856895104050636, -0.02764059044420719, -0.06794670224189758, 0.01480267196893692, -0.07293277978897095, -0.006127927917987108, 0.05637562647461891, 0.026336759328842163, -0.03814256936311722, 0.026810618117451668, 0.008373037911951542, -0.03138730674982071, -0.004088723566383123, -0.020186668261885643, -0.011989287100732327, 0.01189100369811058, -0.0578174963593483, 0.030888348817825317, 0.07401799410581589, -0.0062094805762171745, -0.03635198250412941, -0.015410954132676125, 0.04505017399787903, -0.018900936469435692, 0.017370212823152542, 0.08424374461174011, -0.026911849156022072, 0.003611722495406866, -0.021214326843619347, -0.013757031410932541, 0.004357210360467434, -0.027607886120676994, 0.016832180321216583, 0.006557263899594545, 0.043214067816734314, 0.025452980771660805, 0.014253275468945503, 0.055679868906736374, -0.007031275425106287, 0.023611005395650864, -0.023473046720027924, 0.05335144326090813, -0.02632691152393818, 0.03668999671936035, -0.025786900892853737, 0.009072872810065746, -0.03815643489360809, -0.02705804444849491, -0.03365752473473549, 0.01774587668478489, 0.021139251068234444, 0.029750771820545197, 0.016709154471755028, -0.005519733764231205, 0.02693849802017212, 0.059944674372673035, -0.011312627233564854, -0.024836046621203423, 0.023740243166685104, 0.07854269444942474, 8.949846960604191e-05, -0.0087990527972579, -0.0339183583855629, -0.023316064849495888, -0.03131246566772461, -0.03216642886400223, 0.04261272773146629, 0.05060005560517311, 0.007474962156265974, 0.0076924716122448444, 0.034304242581129074, 0.004868768155574799, 0.008312629535794258, 0.026439445093274117, 0.0029299522284418344, 0.05207883566617966, 0.1244179829955101, 0.004642023704946041, 0.015087216161191463, -0.025899935513734818, -0.011809530667960644, 0.008080428466200829, 0.020390592515468597, -0.00210645142942667, -0.039242446422576904, 0.019216425716876984, -0.01474740356206894, 0.019982939586043358, -0.003204120323061943, 0.00018715746409725398, 0.002079290570691228, -0.07350682467222214, 0.0507342629134655, -0.028692763298749924, 0.018298443406820297, 0.02543262578547001, -0.015915248543024063, -0.04527072235941887, 0.026307720690965652, -0.030394477769732475, 0.0018787221051752567, 0.021778972819447517, -0.012857269495725632, -0.058766476809978485, -0.020457720384001732, 0.022995058447122574, -0.025586796924471855, 0.036828141659498215, -0.052499886602163315, -0.03149660304188728, -0.060936640948057175, -0.03148640692234039, 0.0596611313521862, 0.024483423680067062, -0.006726933177560568, -0.02013656497001648, 0.045315422117710114, -0.0018188171088695526, -0.03412240743637085, 0.05985095351934433, -0.046273063868284225, 0.009320826269686222, 0.02605033852159977, 0.03788824379444122, 0.07512734830379486, 0.03312104940414429, -0.02378603257238865, 0.04450730234384537, 0.01303388737142086, -0.043453969061374664, -0.038270268589258194, -0.02848794497549534, -0.07858243584632874, 0.01735532656311989, 0.02212628349661827, 0.019342174753546715, -0.029481839388608932, -0.0008917273371480405, 0.04525039345026016, 0.01504147332161665, -0.004393680486828089, -0.02655693143606186, -0.000691725465003401, 0.048161085695028305, 0.0017426469130441546, 0.08583656698465347, 0.09068863093852997, 0.02733481116592884, -0.059570904821157455, -0.00876680202782154, 0.04807129502296448, -0.019967343658208847, -0.02482258714735508, 0.0015574508579447865, 0.100229911506176, -0.016886388882994652, -0.071656234562397, 0.01127172913402319, 0.015372728928923607, 0.028186248615384102, 0.0033194615971297026, 0.013336229138076305, -0.016871675848960876, 0.03620455786585808, -0.00044827527017332613, 0.019668370485305786, -0.02698138915002346, 0.052159383893013, 0.009527640417218208, -0.01553960982710123, 0.0020845511462539434, -0.040546614676713943, -0.022017285227775574, 0.031025363132357597, 0.022879766300320625, 0.03148827329277992, 0.026710672304034233, -0.010007549077272415, 0.0041434732265770435, 0.07058075070381165, -0.0037511037662625313, -0.05685249716043472, -0.00044230202911421657, -0.015530894510447979, -0.004093253519386053, 0.00036276670289225876, -0.03872337192296982, 0.015124930068850517, -0.0068226223811507225, 0.021183159202337265, -0.02677961252629757, -0.03776354342699051, -0.01711159199476242, 0.060249704867601395, -0.06844425201416016, 0.039300836622714996, -0.003352068830281496, 0.04124046117067337, -0.016185808926820755, 0.012699540704488754, 0.011803829111158848, 0.07714252173900604, -0.006451926194131374, 0.006908991374075413, -0.030371468514204025, 0.009127531200647354, -0.029690837487578392, -0.006330494768917561, 0.012098141945898533, 0.01864633336663246, -0.03579278290271759, -0.03504006192088127, 0.04125645384192467, -0.021637964993715286, 0.004629140719771385, -0.012598032131791115, -0.06055568903684616, -0.005992043297737837, -0.006576164159923792, 0.06363112479448318, -0.07261170446872711, -0.021661650389432907, 0.020843569189310074, -0.02191050723195076, -0.08198443055152893, 0.020969022065401077, 0.004205730743706226, -0.03349258005619049, -0.03899255022406578, -0.027508750557899475, 0.004947219509631395, -0.008811336942017078, -0.02665691263973713, 0.010381298139691353, 0.03636634349822998, -0.024681927636265755, 0.05544716492295265, -0.0013389515224844217, 0.01965462788939476, 0.012942594476044178, 0.015812460333108902, 0.00599542586132884, -0.08641273528337479, -0.03800281137228012, 0.02957102097570896, -0.041656818240880966, 0.0010185569990426302, -0.06081577017903328, -0.02598249353468418, 0.0534784197807312, 0.005672841798514128, -0.03544643893837929, -0.027048861607909203, -0.06624472141265869, 0.013243695721030235, -0.012895449995994568, 0.022949524223804474, 0.02513991855084896, -0.015248808078467846, -0.03970116376876831, -0.049164772033691406, -0.019575195387005806, -0.004179634619504213, 0.04603514075279236, -0.04018924757838249, 0.01167756412178278, -0.04370769113302231, -0.026524076238274574, 0.0234068650752306, 0.01384666282683611, -0.02496715448796749, -0.02932385355234146, 0.03543439134955406, -0.012025527656078339, -0.016723979264497757, -0.04441763833165169, 0.08729245513677597, 0.06284405291080475, 0.014234174974262714, 0.07748459279537201, 0.01363764051347971, -0.016887450590729713, 0.016038302332162857, 0.048682671040296555, -0.04486233368515968, 0.04250467196106911, 0.040878959000110626, 0.05274142324924469, 0.015379284508526325, 0.03883400559425354, -0.06621314585208893, -0.004208236001431942, 0.036205586045980453, 0.018656402826309204, 0.008370964787900448, -0.04765104129910469, -0.007707536686211824, -0.06546232104301453, -0.010339135304093361, 0.010985287837684155, -0.020468438044190407, 0.016584768891334534, -0.0009593545109964907, 0.031924981623888016, -0.014108264818787575, -0.06051875650882721, -0.04778556525707245, -0.02928704209625721, -0.0009846568573266268, 0.0027193448040634394, -0.028276987373828888, 0.003402435453608632, -0.025041470304131508, 0.023259086534380913, -0.049841683357954025, 0.04279399290680885, 0.010231533087790012, -0.014711745083332062, -0.03978993371129036, 0.020807581022381783, -0.021615466102957726, 0.024770177900791168, 0.06887748837471008, 0.0075563290156424046, 0.010331572033464909], [0.02048245072364807, -0.016803143545985222, 0.06975556164979935, -0.005058750975877047, 0.026891091838479042, 0.014839609153568745, -0.01696276292204857, 0.05122822895646095, 0.01703837886452675, -0.01848672144114971, -0.018093189224600792, -0.06231522187590599, -0.032324180006980896, 0.03430828079581261, 0.03484446555376053, -0.02691570110619068, 0.027409596368670464, 0.03981003165245056, 0.032913342118263245, 0.049574412405490875, -0.06023521348834038, 0.013103466480970383, -0.027158573269844055, 0.02960430458188057, -0.01980811357498169, 0.009527228772640228, 0.036467134952545166, -0.049013085663318634, -0.01702135056257248, -0.01894301176071167, 0.004045248031616211, -0.049675654619932175, 0.03219928964972496, 0.07177266478538513, -0.005389418452978134, 0.022194067016243935, -0.08874966204166412, -0.0382562130689621, 0.026428882032632828, 0.013366978615522385, 0.026215776801109314, 0.04930859059095383, 0.006004959810525179, -0.005957205779850483, -0.013884080573916435, 0.005881255492568016, -0.07180551439523697, 0.05356375873088837, -0.07403131574392319, -0.05387799069285393, 0.003686397336423397, 0.03392757102847099, 0.02017109841108322, 0.053204089403152466, -0.014506610110402107, -0.021848011761903763, 0.019890671595931053, -0.038719918578863144, 0.0077344514429569244, 0.018297629430890083, 0.010696007870137691, -0.07086257636547089, -0.007970703765749931, -0.07130187004804611, 0.029972784221172333, -0.015347792766988277, 0.03466588631272316, 0.04595696181058884, -0.0472184456884861, 0.03179982304573059, 0.013782445341348648, 0.05213331803679466, -0.044218532741069794, -0.044628359377384186, -0.027600562199950218, -0.0645114853978157, 0.004305134993046522, -0.02538345567882061, 0.006645537912845612, 0.011681732721626759, 0.029036788269877434, -0.04664462059736252, 0.02519857883453369, 0.0249452143907547, -0.0352061428129673, -0.022172190248966217, -0.05146284028887749, 0.042035236954689026, 0.04609988257288933, -0.030450979247689247, -0.004583579488098621, 0.013166128657758236, 0.023508191108703613, -0.0034778984263539314, 0.056688591837882996, -0.0337613970041275, 0.028965547680854797, 0.0022664025891572237, 0.009157205931842327, -0.026745688170194626, -0.06142856553196907, -0.01916172169148922, 0.015473654493689537, -0.030162019655108452, -0.06015126034617424, -0.006961118895560503, -0.0018543628975749016, -0.036046311259269714, -0.0051349555142223835, 0.013314276933670044, 0.00619585020467639, 0.03092210553586483, 0.04023490846157074, -0.002264064969494939, -0.02925398200750351, -0.061918724328279495, 0.03384123742580414, 0.010926470160484314, 0.03404398635029793, -0.02976175956428051, 0.0032275596167892218, -0.029481757432222366, -0.0325174443423748, -0.05197872221469879, -0.009153642691671848, 0.01541320700198412, 0.025870727375149727, 0.007076071575284004, -0.011083362624049187, 0.06899197399616241, 0.052629608660936356, 0.034295886754989624, 0.00398916844278574, 0.043104756623506546, -0.0277255866676569, -0.045660268515348434, 0.020661616697907448, 0.004749315325170755, 0.03538724035024643, -0.028606818988919258, 0.009038093499839306, -0.03750443458557129, -0.013182373717427254, 0.015077239833772182, -0.003510779235512018, -0.034012071788311005, -0.05182822048664093, -0.014662466943264008, -0.03334825485944748, -0.051498785614967346, 0.0018451326759532094, -0.03295457363128662, 0.04327639564871788, 0.0712401419878006, 0.037909042090177536, 0.09643587470054626, -0.04362257942557335, -0.061005644500255585, -0.006055363453924656, -0.06257586181163788, 0.003979178611189127, 0.02976670302450657, 0.02623586356639862, 0.03302977979183197, -0.00821355078369379, -0.0033397998195141554, 0.007575207855552435, -0.002986190840601921, 0.03148540109395981, -0.015161296352744102, -0.0025074046570807695, -0.025232525542378426, 0.032574526965618134, 0.0032997748348861933, 0.003641011891886592, -0.01801331900060177, -0.06300409138202667, -0.016897914931178093, -0.07675536721944809, -0.0034803245216608047, -0.04421538487076759, 0.043297864496707916, -0.02350865676999092, -0.0035651177167892456, -0.018452497199177742, -0.041101809591054916, -0.05506734922528267, -0.0891309380531311, -0.024056952446699142, -0.07235462963581085, 0.037715375423431396, -0.04661637172102928, -0.02265128679573536, -0.002518951892852783, -0.006918867584317923, 0.03913755714893341, -0.003930146340280771, -0.018572013825178146, -0.0056851026602089405, 0.05205307528376579, 0.02567477338016033, -0.06972955167293549, -0.008234167471528053, -0.03319691866636276, -0.0007333114044740796, 0.020556587725877762, -0.0010439450852572918, 0.019721850752830505, 0.05741698294878006, -0.06209605187177658, -0.002740176860243082, 0.0007428256794810295, 0.014494303613901138, 0.0022294085938483477, 0.0310507919639349, -0.001730142394080758, 0.01418453361839056, -0.055078908801078796, 0.005825679749250412, -0.04310142248868942, 0.019251465797424316, 0.06055877357721329, -0.0027699663769453764, 0.014388777315616608, 0.025309724733233452, 0.047514405101537704, 0.02693921886384487, 0.04267533868551254, -0.018632149323821068, -0.00268475990742445, -0.0508464090526104, 0.0770835354924202, -0.04108574613928795, 0.03207254409790039, 0.0058297221548855305, 0.00437895767390728, -0.04061075299978256, -0.0010873922146856785, 0.0032698577269911766, 0.014623270370066166, -0.011414510197937489, 0.008764421567320824, -0.03306560218334198, 0.04251369833946228, -0.02223942056298256, 0.12610726058483124, -0.0103583550080657, -0.021896420046687126, 0.007158379536122084, -0.0037890581879764795, 0.03889011964201927, 0.0435926578938961, -0.02052573300898075, -0.022001130506396294, -0.02467547170817852, 0.01801924780011177, 0.0358215868473053, 0.02485797181725502, -0.0031609178986400366, 0.003183203050866723, 0.025075774639844894, 0.04544534534215927, 0.0008109520422294736, 0.04310482740402222, -0.009508654475212097, 0.009088038466870785, 0.04154952988028526, -0.02782183140516281, -0.05784131959080696, -0.0365176685154438, -0.0029783647041767836, -0.03396045044064522, -0.0024345472920686007, 0.049594782292842865, -0.010489220730960369, 0.04801137000322342, 0.032035764306783676, -0.035122357308864594, 0.03549352288246155, 0.031914182007312775, 0.0359383299946785, 0.009145591408014297, 0.011321226134896278, -0.017478352412581444, -0.028618676587939262, 0.00012708667782135308, 0.026770710945129395, -0.018768224865198135, 0.058933012187480927, -0.01307637058198452, 0.013841899111866951, -0.04415328428149223, 0.056843776255846024, 0.028212036937475204, 0.048235464841127396, -0.03879553824663162, 0.053626757115125656, 0.020060895010828972, -0.017763640731573105, -0.010120546445250511, -0.04841531813144684, -0.0022773195523768663, 0.035183362662792206, -0.025841375812888145, 0.01734994724392891, -0.03792308270931244, -0.08887692540884018, -0.0024713235907256603, 0.05534094199538231, 0.0456073060631752, -0.019799131900072098, 0.04754888638854027, 0.03972221165895462, 0.016356781125068665, -0.009267037734389305, 0.037367261946201324, -0.00041424520895816386, -0.011085962876677513, 0.03925840184092522, -0.02169451303780079, 0.02608836255967617, 0.054148685187101364, -0.021693004295229912, 0.008139974437654018, 0.020602382719516754, -0.05962677672505379, -0.015618644654750824, -0.01777714304625988, -0.017437297850847244, -0.029145434498786926, 0.046114739030599594, -0.030615612864494324, -0.0027904361486434937, 0.021608445793390274, 0.055365823209285736, -0.055229831486940384, -0.044987648725509644, -0.036426763981580734, -0.01019847672432661, 0.03309273719787598, -0.004301407374441624, 0.005638813134282827, 0.05402998626232147, -0.010702764615416527, 0.006523985415697098, 0.069757379591465, 0.02807615138590336, -0.020553337410092354, -0.021881744265556335, 0.039918240159749985, -0.03903503343462944, -0.030024012550711632, 0.02907245233654976, 0.04783206805586815, 0.05959903076291084, -0.03317626565694809, 0.01923500932753086, -0.11511415988206863, -0.028410185128450394, 0.10413485765457153, 0.056223753839731216, 0.02667056955397129, -0.030637376010417938, -0.042078182101249695, 0.04075977951288223, -0.0034626717679202557, -0.026312310248613358, 0.027662368491292, 0.026180146262049675, -0.022254467010498047, -0.013090762309730053, 0.07988117635250092, -0.03199111670255661, 0.03795609995722771, -0.02220560796558857, -0.027583856135606766, -0.0341813787817955, -0.0018077120184898376, 0.0017003532266244292, 0.028348708525300026, -0.02358546294271946, 0.016758432611823082, -0.017267070710659027, -0.02324507385492325, 0.0245867520570755, 0.0181287694722414, -0.05092553049325943, 0.007584602106362581, -0.06194208562374115, -0.03542320802807808, 0.05854436010122299, 0.047856975346803665, -0.005110503174364567, 0.022583134472370148, 0.045828405767679214, 0.061810847371816635, 0.061403192579746246, -0.0257088802754879, -0.0007081129006110132, 0.004701173398643732, -0.08410671353340149, -0.02574925869703293, 0.014381561428308487, -0.007190499920397997, -0.020835142582654953, -0.04085926711559296, -0.020061085000634193, -0.0007880049524828792, -0.011001018807291985, 0.03993848338723183, 0.02857476845383644, -0.045103520154953, 0.03277741000056267, 0.0010177408112213016, 0.02672833763062954, -0.023040901869535446, 0.03141898661851883, -0.025007059797644615, 0.03572636470198631, -0.0010452157584950328, -0.00905823428183794, -0.04618046060204506, -0.05643751844763756, 0.030268333852291107, -0.03920627012848854, -0.04113965108990669, 0.01387386117130518, 0.0006923899054527283, 0.020336469635367393, -0.04594834893941879, -0.0743650570511818, -0.03180545195937157, 0.034860920161008835, -0.0012548519298434258, 0.015229242853820324, 0.01813974604010582, 0.005771385971456766, 0.005840439349412918, -0.007304083555936813, 0.0075894067995250225, 0.011290074326097965, -0.022795390337705612, 0.020916972309350967, -6.480173033196479e-05, -0.0029334689024835825, -0.005512201692909002, 0.03078334964811802, -0.005059757269918919, 0.041663605719804764, -0.03505728393793106, 0.02013390325009823, -0.05393536388874054, 0.0018419096013531089, -0.017559798434376717, -0.015985509380698204, -0.036229439079761505, -0.0009093996486626565, 0.018954651430249214, -0.06414269655942917, -0.004565801937133074, 0.006321958731859922, 0.0211622416973114, 0.012346534058451653, -0.03777934983372688, 0.034577421844005585, -0.03043484129011631, 0.01712757907807827, 0.006541216280311346, -0.020450592041015625, -0.031864412128925323, -0.00768313417211175, 0.04985487088561058, 0.02208673395216465, -0.02036893367767334, 0.025137796998023987, 0.05783359706401825, 0.07509613782167435, -0.06003282219171524, 0.02517770417034626, 0.028869176283478737, -0.028249137103557587, -0.06516779214143753, 0.043989311903715134, -0.016227424144744873, -0.05655883625149727, -0.037408169358968735, 0.025485651567578316, 0.07619835436344147, 0.032478731125593185, -0.030036700889468193, 0.044310957193374634, -0.06540042906999588, 0.03473334014415741, 0.032358694821596146, 0.0627540573477745, 0.03435884416103363, 0.03999746963381767, 0.004272012505680323, -0.011353453621268272, 0.001525868196040392, 0.06773894280195236, -0.019303258508443832, -0.0013042993377894163, -0.015477294102311134, 0.024240080267190933, 0.01755097694694996, 0.07199511677026749, 0.07692450284957886, 0.040517039597034454, 0.03220587968826294, -0.003686395939439535, -0.051150575280189514, 0.006833295803517103, 0.002494493033736944, -0.03854944184422493, 0.008712315931916237, 0.03516688570380211, -0.030085571110248566, 0.032055627554655075, 0.015501288697123528, -0.017932800576090813, 0.005011947359889746, 0.027542758733034134, -0.0052521852776408195, -0.0010622860863804817, 0.047315437346696854, 0.02069893479347229, 0.01993997022509575, -0.01653335429728031, 0.025833971798419952, 0.05375601723790169, -0.03195296227931976, -0.05683419108390808, -0.027103472501039505, -0.0035484498366713524, -0.03425908461213112, -0.05847420170903206, 0.05885304510593414, -0.013782241381704807, -0.027764979749917984, -0.10023285448551178, 0.0328175351023674, -0.05377373471856117, 0.04714469611644745, 0.020731234923005104, 0.00844170618802309, -0.006747040897607803, 0.03457951545715332, -0.01920110359787941, 0.013856099918484688, -0.01677374728024006, -0.010301615111529827, 0.056179627776145935, 0.04006525129079819, -0.03216364234685898, -0.029684707522392273, 0.0021548070944845676, -0.003918600268661976, 0.01816912554204464, -0.012669284828007221, -0.0361248254776001, 0.029853204265236855, -0.007159789092838764, 0.024930531159043312, -0.03111196495592594, 0.05406971275806427, 0.010474315844476223, 0.009908067993819714, 0.014132916927337646, -0.055058978497982025, 0.025565624237060547, -0.017715470865368843, -0.03917625546455383, 0.014308257959783077, 0.0616644062101841, -0.02422749437391758, 0.05547255277633667, -0.07776802033185959, 0.011308418586850166, -0.02055693045258522, 0.009508752264082432, 0.07650265097618103, 0.024539923295378685, -0.05233152583241463, 0.05243820697069168, -0.03701009973883629, -0.019636623561382294, -0.03159373626112938, 0.005203952547162771, -0.003446236252784729, 0.05863116681575775, 0.012908249162137508, 0.025333011522889137, -0.004770816769450903, 0.040689617395401, -0.036265913397073746, -0.050546664744615555, 0.03248833492398262, 0.027642739936709404, 0.0027812558691948652, 0.08102162182331085, -0.030725134536623955, 0.051754556596279144, 0.05543596297502518, -0.042831894010305405, 0.020985860377550125, 0.029205814003944397, -0.0034213941544294357, -0.003075034823268652, -0.0005099831614643335, -0.029286226257681847, -0.04749924689531326, 0.029309749603271484, 0.009536411613225937, 0.0027064059395343065, 0.020297301933169365, -0.02718157321214676, -0.021465865895152092, 0.03015313856303692, 0.0014278982998803258, 0.018831966444849968, 0.03990727290511131, 0.06340674310922623, -0.030812343582510948, -0.01890779472887516, -0.04943590238690376, -0.02319803275167942, 0.01937190815806389, -0.017729826271533966, 0.0786457508802414, -0.01394127868115902, -0.06205202639102936, -0.030550025403499603, 0.07517842948436737, -0.030448755249381065, -0.0013689574552699924, -0.02787635289132595, -0.03782305866479874, 0.04230218380689621, -0.041101112961769104, -0.062398772686719894, -0.02529355138540268, -0.022357670590281487, 0.01665475405752659, 0.0051256208680570126, 0.04497138783335686, -0.006250422913581133, 0.016539564356207848, 0.01880207471549511, 0.037374332547187805, -0.04210205376148224, -0.01065492257475853, -0.09558673948049545, -0.027087457478046417, -0.04213685914874077, -0.029922770336270332, 0.0558641143143177, 0.03722303733229637, -0.05257703363895416, 0.06443098932504654, 0.046068400144577026, 0.007304423488676548, -0.0028196386992931366, -0.004079093690961599, -0.0005711116245947778, 0.023732151836156845, -0.009047195315361023, 0.035721827298402786, -0.011160104535520077, -0.02394743822515011, -0.004617638420313597, 0.011948580853641033, 0.01512657105922699, -0.0003534907300490886, -0.053382325917482376, -0.047665972262620926, -0.0605095773935318, -0.00973503477871418, 0.024059223011136055, 0.0010227347956970334, -0.0016574840992689133, -0.016994060948491096, -0.012044298462569714, -0.02362486720085144, 0.03756161034107208, -0.043654654175043106, -0.031128739938139915, 0.017092693597078323, -0.012789887376129627, 0.023368334397673607, -0.016165347769856453, 0.01518637128174305, 0.008222661912441254, -0.04507818818092346, 0.019874311983585358, 0.04473809897899628, -0.023243514820933342, -0.004358190577477217, 0.022797640413045883, 0.054550278931856155, 0.013119341805577278, 0.08248940855264664, -0.013525244779884815, 0.0684104710817337, -0.02543523907661438, -9.428017801837996e-05, -0.031244371086359024, 0.0355515293776989, 0.023111818358302116, -0.060328755527734756, -0.09137958288192749, 0.07653394341468811, -0.009498895145952702, -0.05785883218050003, -0.030881259590387344, -0.015569577924907207, 0.056227248162031174, 0.016275253146886826, 0.002313091419637203, -0.0326501689851284, 0.02412089891731739, -0.06102628633379936, 0.040096450597047806, 0.03479659557342529, -0.0158870667219162, -0.05217155069112778, -0.021199192851781845, -0.0037309450563043356, 0.07105667144060135, 0.060322508215904236, -0.022064583376049995, -0.02623315528035164, -0.04752112552523613, 0.013811996206641197, -0.01863008551299572, 0.02519996277987957, 0.002895106328651309, -0.018550526350736618, -0.034717582166194916, -0.0555703267455101, 0.020969603210687637, -0.04591737315058708, 0.025235392153263092, -0.05771194398403168, -0.08885766565799713, 0.0008394928881898522, -0.020533818751573563, 0.03257231041789055, 0.0003789183683693409, -0.0022249193862080574, 0.030767057090997696, -0.02261476218700409, 0.022904397919774055, 0.0866280272603035, 0.008577709086239338, -0.03165861964225769, -0.0005442461115308106, -0.027396127581596375, -0.010992515832185745, 0.05930367484688759, -0.03367353230714798, -0.042209818959236145, 0.019086886197328568, -0.013243506662547588, -0.004905076697468758, -0.02082902006804943, 0.009576997719705105, 0.05264172703027725], [0.024479812011122704, -0.014059294015169144, -0.01166590116918087, -0.03194456920027733, -0.030931085348129272, 0.03557780012488365, 0.020990120247006416, 0.018340568989515305, 0.01968369632959366, -0.021343667060136795, 0.0011556564131751657, 0.020496688783168793, 0.0600600428879261, 0.011114215478301048, 0.004374225623905659, -0.026944119483232498, 0.03933259844779968, -0.007220615167170763, 0.01115696132183075, 0.016280818730592728, -0.026307109743356705, 0.012647653929889202, 0.04311107471585274, -0.030015666037797928, -0.016730429604649544, -0.027550285682082176, 0.005797017831355333, 0.07817074656486511, 0.014032014645636082, 0.04731978476047516, 0.011037589982151985, -0.013050935231149197, 0.0027520321309566498, 0.010789155028760433, -0.004740542266517878, -0.00012243380479048938, 0.005468548275530338, -0.020558800548315048, 0.057502008974552155, 0.01806926354765892, -0.0021469134371727705, -0.018492063507437706, 0.01573086529970169, -0.025795605033636093, -0.04296881705522537, -0.011280423030257225, 0.047327183187007904, -0.005178503692150116, 0.008478322997689247, -0.03269484266638756, 0.020666617900133133, -0.01776091940701008, -0.018656693398952484, -0.026116522029042244, -0.0029153069481253624, -0.036242879927158356, -0.012918729335069656, -0.005245549604296684, -0.032990410923957825, -0.06883568316698074, 0.01744154468178749, -0.043488916009664536, -0.06907690316438675, 0.00173234473913908, -0.00024107395438477397, 0.04160376265645027, 0.04783839359879494, -0.07827265560626984, -0.01929180882871151, 0.01119556836783886, -0.04724873602390289, 0.0425989031791687, 0.013855037279427052, -0.022975971922278404, -0.017648613080382347, 0.012834432534873486, 0.0777408629655838, 0.07093755900859833, -0.050420645624399185, 0.0233019907027483, -0.05173943564295769, -0.0033653483260422945, -0.03443673625588417, -0.009071322157979012, -0.014480147510766983, 0.014559862203896046, -0.025776958093047142, 0.026578618213534355, -0.0029579985421150923, -0.031781699508428574, 0.053973644971847534, 0.013334142044186592, -0.006259596440941095, 0.004607309587299824, 0.02678229846060276, -0.0035122528206557035, -0.020514382049441338, -0.00419527105987072, -0.018575040623545647, -0.01812891475856304, 0.05961213633418083, 0.02954404428601265, 0.04942473769187927, -0.017038719728589058, -0.06229352951049805, -0.012574235908687115, 0.020148033276200294, -0.016631364822387695, 0.007921004667878151, -0.03334157168865204, 0.019253740087151527, -0.014405948109924793, 0.0013860872713848948, 0.025440245866775513, -0.025350971147418022, -0.03732600435614586, -0.028246667236089706, 0.022030651569366455, -0.010935300961136818, 0.0394112765789032, 0.020255183801054955, 0.04108446463942528, 0.007620123215019703, 0.0444466695189476, 0.02764621563255787, -0.06848782300949097, 0.052599262446165085, 0.03262973949313164, -0.013362708501517773, -0.04216999560594559, -0.06565192341804504, 0.006700354628264904, -0.029447006061673164, -0.04048839583992958, -0.034157536923885345, -0.05754666030406952, 0.035636983811855316, -0.030069047585129738, 0.0007633152417838573, 0.0072142197750508785, 0.014899634756147861, -0.004257577005773783, 0.03136150911450386, -0.020227652043104172, 0.07860270887613297, -0.0559472031891346, 0.026698986068367958, 0.08086477965116501, -0.033331383019685745, 0.05311213433742523, 0.07560964673757553, -0.014909549616277218, 0.057207249104976654, -0.019048230722546577, 0.012639680877327919, -0.006415845360606909, -0.03363005071878433, -0.03810533881187439, -0.019231796264648438, -0.052773039788007736, 0.008144451305270195, 0.006380406673997641, 0.04167287424206734, 0.01666181907057762, -0.004274460487067699, -0.009971247054636478, 0.0912318080663681, -0.054614223539829254, -0.04663274809718132, 0.08333039283752441, 0.04084549471735954, 0.008613260462880135, 0.021518727764487267, 0.038276780396699905, -0.051386985927820206, -0.009303265251219273, 0.02820490673184395, 0.026374446228146553, 0.04267921298742294, 0.011237255297601223, -0.01050779689103365, -0.019416796043515205, 0.02521633170545101, 0.0020545946899801493, 0.020116686820983887, 0.05396553874015808, 0.0479876883327961, -0.04280775785446167, 0.04542544111609459, -0.032003410160541534, -0.05635519325733185, 0.0803447812795639, 0.013850013725459576, -0.01814829744398594, -0.011518051847815514, 0.03677420690655708, 0.0770418643951416, 0.051949549466371536, -0.004056656267493963, -0.03800124302506447, -4.391584297991358e-05, 0.04123980551958084, 0.022642549127340317, -0.043692998588085175, 0.008813771419227123, 0.02671625092625618, -0.02939029410481453, 0.03657643496990204, 0.057776570320129395, 0.03779424726963043, -0.026522183790802956, -0.005609071347862482, 0.001891812775284052, -0.008224276825785637, 0.035861823707818985, 0.007706818636506796, 0.004551159683614969, 0.016271809116005898, -0.011649665422737598, -0.04424247518181801, 0.008569006808102131, -0.007824867963790894, 0.02852262184023857, 0.012623107992112637, -0.00524475984275341, 0.0435442291200161, 0.0035844005178660154, -0.042256567627191544, 0.0604507140815258, 0.046123478561639786, 0.003198325401172042, -0.05297123268246651, -0.03234533965587616, 0.039120689034461975, 0.0010271577630192041, -0.008888752199709415, 0.020308246836066246, -0.01719774305820465, -0.0145511943846941, 0.04765736311674118, 0.02275845594704151, 0.0316341333091259, 0.02556021511554718, 0.042050860822200775, 0.034334804862737656, -0.024587998166680336, -0.002768078353255987, 0.06385339051485062, 0.01070338673889637, 0.06696592271327972, -0.006635743193328381, -0.04567719250917435, 0.007692527957260609, -0.013706007041037083, -0.020450793206691742, -0.02251385524868965, -0.06878767162561417, 0.02383892796933651, -0.0011448913719505072, -0.029085855931043625, -0.05434068664908409, 0.02786901965737343, 0.004904702305793762, 0.028406640514731407, 0.017601072788238525, 0.031099751591682434, 0.0027030501514673233, -0.04875887185335159, 0.009740455076098442, 0.03597581014037132, -0.04731062054634094, -0.012459684163331985, 0.05556626617908478, -0.04100171476602554, 0.05183495953679085, -0.0303408894687891, -0.0558805987238884, -0.015909751877188683, -0.019679266959428787, 0.035911936312913895, -0.07358774542808533, -0.03261265903711319, -0.03259302303195, -0.03797164559364319, 0.003587473416700959, -0.020113518461585045, -0.0008834147010929883, -0.010152451694011688, 0.037795066833496094, -0.024497242644429207, -0.00020458191283978522, 0.029314005747437477, 0.01736275851726532, -0.0005249737878330052, 0.028777219355106354, 0.014809456653892994, 0.015913618728518486, 0.0029244041070342064, 0.035842906683683395, 0.03925256058573723, -0.019923996180295944, 0.021597865968942642, 0.017375927418470383, 0.026653219014406204, -0.023403391242027283, -0.04024771973490715, -0.0045445747673511505, 0.04212060198187828, -0.03317384049296379, 0.038424525409936905, 0.0042881108820438385, 0.009879927150905132, 0.010398116894066334, -0.008812889456748962, -0.004844881594181061, 0.0262760017067194, 0.058611977845430374, 0.010366687551140785, 0.019360871985554695, -0.026213398203253746, -0.0007718679262325168, 0.041853781789541245, 0.04346798360347748, -0.006046542432159185, 0.001916171284392476, -0.011742565780878067, -0.02697092480957508, 0.02176184393465519, -0.011609965935349464, 0.057471051812171936, -0.07504279166460037, -0.061881449073553085, -0.04390996694564819, 0.002797058317810297, -0.005352931562811136, -0.027819838374853134, -0.11397685110569, 0.006721092853695154, -0.05830587446689606, 0.050413765013217926, 0.005079750902950764, -0.02645132690668106, -0.009419106878340244, 0.05779801681637764, -0.012489551678299904, -0.04111216589808464, -0.09276355803012848, -0.006376886274665594, -0.02792566642165184, 0.04098871350288391, 0.00510310148820281, 0.03859822824597359, -0.01566091552376747, -0.030835725367069244, 0.03409120440483093, -0.006051727570593357, -0.02656940184533596, -0.003681855509057641, 0.036356862634420395, -0.02312282659113407, 0.07040626555681229, 0.018395690247416496, -0.04319922253489494, -0.03937283158302307, -0.0021360190585255623, -0.012368019670248032, -0.013568255119025707, -0.005020991899073124, 0.009563904255628586, -0.00901008490473032, -0.019259067252278328, -0.06822962313890457, 0.059255365282297134, 0.004546205047518015, 0.0204246137291193, 0.006608547642827034, 0.0461743138730526, 0.0011550107738003135, 0.04509871080517769, -0.020207852125167847, -0.008293183520436287, 0.019536910578608513, 0.021489055827260017, 0.04333079233765602, -0.020562831312417984, 0.0170165803283453, -0.018712162971496582, -0.05112312734127045, 0.03565267473459244, -0.0024329491425305605, -0.035421714186668396, -0.027243342250585556, 0.01891610212624073, -5.273001079331152e-05, -0.004150861874222755, 0.023697035387158394, -0.0013726059114560485, -0.06506332010030746, 0.029797600582242012, -0.017683636397123337, -0.013643963262438774, -0.08707461506128311, -0.11781246960163116, 0.00100454140920192, -0.017450001090765, -0.0007447616080753505, 0.009451773017644882, 0.03159935027360916, 0.047385185956954956, 0.04340698570013046, -0.03612777590751648, -0.02868238277733326, 0.00396628538146615, 0.05075714364647865, 0.04042690619826317, -0.01872645691037178, -0.04301491007208824, 0.010440767742693424, -0.006698456592857838, 0.024514930322766304, 0.028950858861207962, 0.012039531953632832, -0.03154411166906357, -0.04335157945752144, -0.009582996368408203, 0.017715072259306908, -0.0018747259164229035, 0.02586531639099121, -0.014071036130189896, 0.04000072553753853, 0.013132820837199688, -0.010739555582404137, -0.02543611265718937, -0.00417737103998661, 0.0038593062199652195, 0.05766865238547325, 0.004312769044190645, 0.02501591295003891, -0.07303950935602188, 0.03649307042360306, -0.0016753101954236627, -0.03201506659388542, -0.08060754090547562, -0.029491348192095757, 0.05946667492389679, 0.030877212062478065, 0.02967243641614914, -0.016281109303236008, 0.023118147626519203, 0.023114170879125595, 0.01301174983382225, -0.03257474675774574, -0.021037204191088676, -0.010326898656785488, 0.10991982370615005, -0.07053099572658539, 0.060088492929935455, 0.05098903924226761, 0.008633080869913101, 0.03012865222990513, 0.013852972537279129, 0.020771976560354233, -0.020586999133229256, -0.07054736465215683, 0.0978749543428421, 0.06389781087636948, -0.04999387636780739, 0.04163480922579765, -0.014081626199185848, 0.017307963222265244, -0.036921896040439606, -0.03809788078069687, 0.06807693839073181, -0.016236739233136177, -0.00275963032618165, 0.025848306715488434, 0.05404050275683403, -0.017098158597946167, 0.022807976230978966, 0.028121549636125565, -0.021075742319226265, -0.02050640806555748, 0.02269786410033703, -0.0038320240564644337, 0.057529669255018234, -0.06465695798397064, 0.019361594691872597, -0.0050417110323905945, -0.04999449476599693, -0.003556536277756095, -0.05122387409210205, -0.01172301173210144, 0.039939943701028824, -0.025113148614764214, 0.022480560466647148, 0.011296475306153297, -0.017445532605051994, -0.009631202556192875, 0.008180709555745125, 0.01216031052172184, -0.01774149015545845, -0.005180598236620426, 0.028522398322820663, -0.03560108691453934, -0.04217398911714554, 0.011914514936506748, 0.017407789826393127, -0.08297000825405121, 0.004387145396322012, 0.05075237527489662, 0.030636750161647797, -0.012398188933730125, 0.014307143166661263, -0.058636587113142014, 0.023561351001262665, 0.011377659626305103, 0.03203483670949936, -0.004842625930905342, 0.02484588883817196, -0.01502719335258007, 0.03988635540008545, 0.0172863956540823, -0.06389997899532318, -0.05904161557555199, 0.020223265513777733, 0.03633607178926468, 0.0469704233109951, -0.01748315617442131, 0.008677824400365353, -0.010136308148503304, -0.022730082273483276, -0.0075414893217384815, 0.04598909616470337, -0.015018538571894169, 0.0661446824669838, -0.045872610062360764, -0.010510789230465889, 0.02067291922867298, 0.02033274620771408, 0.006331989541649818, -0.05881372094154358, -0.008120902813971043, 0.05294330418109894, 0.025554051622748375, 0.020785927772521973, 0.017399489879608154, -0.0015476784901693463, 0.0004630660405382514, -0.019257009029388428, 0.0513145737349987, 0.01776432804763317, 0.015792837366461754, 0.007556022610515356, -0.007141346577554941, -0.05079386383295059, 0.0008168518543243408, -0.02313222363591194, -0.01878129318356514, 0.008651463314890862, 0.04917290061712265, -0.035728685557842255, 0.00179913523606956, 0.059552788734436035, -0.04356498271226883, 0.020726555958390236, -0.014630173332989216, 0.028472566977143288, 0.05900805816054344, -0.0008935473742894828, 0.03737417235970497, 0.013221747241914272, 0.07114122062921524, -0.1219249963760376, -0.030867924913764, -0.0020581698045134544, 0.005803726147860289, 0.06280563026666641, 0.04326947033405304, -0.011362108401954174, -0.05001481622457504, 0.04449352249503136, 0.012057675048708916, -0.007440630346536636, 0.03359512984752655, 0.0674666091799736, 0.0152198551222682, 0.05428709834814072, -0.06279679387807846, 0.034024082124233246, 0.08815290033817291, 0.047290705144405365, -0.018593991175293922, -0.03798813000321388, 0.01669013500213623, -0.028004208579659462, -0.016139890998601913, -0.0608355738222599, -0.033641502261161804, 0.022832434624433517, -0.025743821635842323, -0.028373632580041885, -0.04661441594362259, -0.003830100642517209, 0.01413882989436388, 0.0050097741186618805, -0.10210683941841125, 0.0010061908978968859, -0.033611372113227844, -0.003127292264252901, -0.039406538009643555, 0.041810330003499985, 0.0700128972530365, 0.03404781594872475, 0.06431841850280762, 0.009537946432828903, -0.009076648391783237, -0.0733809769153595, -0.022658877074718475, -0.015361879952251911, 0.057802606374025345, -0.033908553421497345, 0.003605147125199437, 0.024306975305080414, 0.003138958476483822, 0.00042148554348386824, 0.03945987671613693, -0.030512092635035515, 0.0020661314483731985, 0.009459421038627625, 0.059526246041059494, 0.05119787156581879, 0.021934874355793, 0.03241179510951042, -0.07537715882062912, 0.012063082307577133, -0.01825398951768875, -0.05476011708378792, 0.024936437606811523, -0.0141902519389987, -0.024487925693392754, -0.06288071721792221, 0.006258562672883272, 0.044965848326683044, 0.00600330950692296, 0.026821056380867958, 0.0362093560397625, -0.0006295362254604697, -0.013849283568561077, -0.02258133701980114, -0.011046180501580238, -0.020233435556292534, 0.002186273690313101, -0.01800602860748768, 0.039753880351781845, -0.03234552964568138, 0.05808476358652115, -0.017177365720272064, -0.017328955233097076, 0.02405715547502041, -0.026895983144640923, -0.06878161430358887, 0.02996923215687275, 0.051693446934223175, -0.07076004147529602, 0.01859268546104431, -0.02079080045223236, -0.008479955606162548, 0.056744880974292755, -0.04997728392481804, -0.019411927089095116, 0.04509824886918068, 0.008207760751247406, 0.016111275181174278, -0.02461159974336624, -0.02961575984954834, -0.028686368837952614, 0.007503904402256012, -0.016565173864364624, 0.014933133497834206, 0.09490540623664856, -0.06530337035655975, 0.008844047784805298, -0.00431093480437994, -0.039574045687913895, 0.005814545787870884, -0.03342437371611595, -0.05044623464345932, -0.0010423862840980291, 0.056142088025808334, -0.008973787538707256, 0.014060873538255692, -0.02552761137485504, -0.028536668047308922, -0.05465696379542351, -0.0021687194239348173, 0.015516991727054119, -0.06489451229572296, -0.07147321105003357, 0.016542352735996246, -0.027187885716557503, -0.060300152748823166, 0.02126901224255562, -0.010772667825222015, -0.017768466845154762, -0.003420833498239517, -0.02678290754556656, 0.10023918747901917, -0.08129464834928513, -0.021413711830973625, 0.017811918631196022, 0.011454759165644646, -0.046224500983953476, -0.015003850683569908, -0.006205759011209011, 0.04711202159523964, 0.05680183693766594, 0.057858906686306, 0.005051702260971069, 0.048717740923166275, 0.019961150363087654, -0.008993101306259632, -0.011682532727718353, 0.025403013452887535, 0.0121302530169487, -0.019226595759391785, -0.033791400492191315, -0.03462919220328331, 0.05131610482931137, -0.008758395910263062, 0.004768890794366598, 0.05418573319911957, 0.05679555982351303, 0.032786402851343155, 0.016745027154684067, -0.015687022358179092, 0.0028286485467106104, 0.009872021153569221, 0.026084691286087036, -0.021243935450911522, 0.0303109809756279, 0.038191620260477066, 0.01406725775450468, -0.04954127222299576, -0.06673503667116165, 0.025582706555724144, -0.06254369020462036, 0.007572734262794256, 0.10383712500333786, -0.004327922593802214, -0.04862704128026962, 0.032449446618556976, -0.015895647928118706, -0.0028646111022681, -0.06210201233625412, -0.023395422846078873, 0.04970097541809082, -0.009589112363755703, -0.02574969455599785, -0.0017678342992439866, 0.006476827897131443, 0.03441137447953224, 0.023793872445821762, 0.034054774791002274, -0.04725128039717674, -0.013208478689193726, -0.07095138728618622, 0.05437890440225601, -0.022704020142555237], [-0.014207390137016773, 0.06220020726323128, 0.029632650315761566, -0.011878321878612041, -0.009713574312627316, -0.01527739129960537, 0.029008958488702774, 0.1043083444237709, -0.04943488538265228, 0.0503065399825573, 0.043065108358860016, -0.01688479259610176, 0.010681258514523506, 0.009139270521700382, -0.025327138602733612, -0.0217879768460989, 0.05070963874459267, 0.016234949231147766, -0.013107117265462875, -0.0301828533411026, -0.0027111147064715624, 0.0025748482439666986, 0.03002692013978958, 0.0554330050945282, 0.0026022077072411776, -0.030108243227005005, 0.014442271552979946, 0.006935533136129379, -0.018023647367954254, 0.02655564434826374, -0.0018273302121087909, 0.013373807072639465, 0.0014304470969364047, 0.017562082037329674, 0.02226453274488449, -0.032788265496492386, 0.03333689272403717, 0.0025210119783878326, -0.05673988536000252, -0.009996388107538223, 0.04482542350888252, 0.05193999037146568, -0.020103855058550835, 0.011836146004498005, -0.042917657643556595, 0.01819312945008278, -0.07616542279720306, -0.0577084943652153, 0.016318345442414284, 0.019158799201250076, -0.02086285501718521, 0.01890774816274643, 0.056533876806497574, -0.0032714540138840675, 0.022536324337124825, -0.026530956849455833, -0.04408532753586769, 0.06380592286586761, 0.006637963466346264, -0.020419737324118614, -0.05881239473819733, 0.022877812385559082, 0.032551541924476624, 0.050442807376384735, -0.0059267436154186726, 0.033762913197278976, 0.003062507137656212, -0.030089110136032104, -9.382078860653564e-05, -0.019080057740211487, -0.02985246852040291, -0.02076268009841442, 0.031733106821775436, 0.0032933386974036694, -0.024833442643284798, -0.004013108089566231, -0.0038836838211864233, -0.002879189560189843, 0.007592550478875637, -0.024832531809806824, 0.050930071622133255, -0.03829988092184067, -0.019350595772266388, -0.04682024568319321, -0.012418449856340885, -0.017201337963342667, 0.009102615527808666, -0.05014471709728241, 0.009425422176718712, -0.056974586099386215, 0.045778971165418625, -0.017051761969923973, 0.044732145965099335, 0.055646974593400955, -0.038186412304639816, 0.07318530231714249, 0.0038500402588397264, -0.07579366862773895, 0.010408038273453712, 0.009224805049598217, 0.06314482539892197, -0.06504849344491959, -0.10708554834127426, -0.002347369445487857, -0.017096685245633125, -0.03376692906022072, -6.926877540536225e-05, 0.06346743553876877, -0.001382956630550325, 0.032923463732004166, -0.025335293263196945, -0.04695582017302513, -0.01601901836693287, 0.06597655266523361, 0.048501379787921906, 0.028515158221125603, -0.011086205951869488, 0.013428517617285252, 0.022501053288578987, -0.05330353230237961, -0.009653463028371334, 0.0693618431687355, -0.04777519404888153, 0.008710866793990135, -0.01685386523604393, -0.011792206205427647, 0.03524372726678848, -0.02824748493731022, 0.02811381407082081, -0.000788153731264174, 0.03971380740404129, -0.029921755194664, 0.04259951412677765, -0.00030929921194911003, -0.0188298299908638, -0.025894567370414734, -0.07155241817235947, 0.034988079220056534, -0.02755579724907875, 0.04294244572520256, 0.08326537907123566, -0.058294299989938736, 0.009957568719983101, -0.025965208187699318, 0.003060002578422427, 0.015381458215415478, -0.045567311346530914, 0.03618020936846733, -0.0074145616963505745, 0.05773957073688507, 4.006378003396094e-05, 0.0064806025475263596, 0.0005096578970551491, 0.10048774629831314, -0.01754852943122387, -0.08514492213726044, 0.0340970903635025, 0.002959564793854952, 0.030634164810180664, 0.04249819740653038, 0.05433093383908272, 0.00880301371216774, 0.016506919637322426, 0.040209922939538956, 0.050135400146245956, -0.04893326014280319, 0.06827294081449509, -0.03956349194049835, 0.008197048678994179, 0.04196830466389656, -0.01766827329993248, -0.017943905666470528, 0.0547601692378521, 0.05363127216696739, -0.05047927424311638, -0.0014816952170804143, 0.014623374678194523, 0.052463311702013016, 0.015602508559823036, 0.0340154804289341, 0.0168246328830719, 0.029511043801903725, 0.09495659172534943, 0.024712122976779938, 0.01889120228588581, -0.05223984643816948, -0.03595742955803871, -0.048542726784944534, -0.0009798025712370872, -0.010775336995720863, -0.025774691253900528, 0.025064822286367416, -0.05464577674865723, 0.007262278348207474, -0.005668778903782368, -0.02909647859632969, -0.031900692731142044, -0.017610672861337662, 0.028239203616976738, 0.033160530030727386, -0.05399908125400543, 0.0024132158141583204, -0.024242809042334557, 0.034117233008146286, 0.043960943818092346, 0.031555525958538055, 0.019460229203104973, 0.024690432474017143, -0.06210479885339737, 0.016664111986756325, 0.011347013525664806, -0.005675719119608402, -0.025691285729408264, -0.04266505688428879, -0.029021073132753372, -0.021422969177365303, -0.09322961419820786, -0.0033792718313634396, 0.007888227701187134, 0.057286929339170456, 0.0584307461977005, 0.01822427660226822, 0.003576254239305854, 0.002134437672793865, 0.03250804543495178, 0.03831639513373375, -0.05417952314019203, 0.02885112538933754, -0.01559081207960844, 0.0009148602839559317, -0.011525205336511135, 0.014337762258946896, -0.03474573791027069, 0.0024769583251327276, -0.05173209682106972, 0.03706517070531845, 0.031027430668473244, -0.012392481788992882, -0.0026800725609064102, -0.020938968285918236, 0.026058431714773178, 0.020728811621665955, -0.013995015062391758, -0.005195393692702055, 0.010455062612891197, -0.02547285705804825, 0.011551504954695702, 0.01960301585495472, 0.08785663545131683, 0.014239577576518059, -0.02184988558292389, 0.046707749366760254, 0.01424960047006607, -0.012572302483022213, 0.03235359117388725, -0.01714199408888817, 0.05345260351896286, 0.02985670417547226, -0.008119304664433002, 0.002078896388411522, 0.07611620426177979, -0.0064657581970095634, -0.010516077280044556, 0.02816934511065483, 0.009810036979615688, -0.0027668182738125324, 0.02702050469815731, -0.06707801669836044, 0.00669649476185441, 0.053415898233652115, 0.050809457898139954, -0.02344684861600399, 0.003433825680986047, 0.03130625933408737, -0.006180050317198038, 0.021168842911720276, 0.019991857931017876, -0.05944015085697174, 0.02886430360376835, 0.004236436448991299, -0.0020639149006456137, -0.007962880656123161, -0.07437030225992203, -0.0015645712846890092, 0.03636441379785538, 0.049185238778591156, 0.008266408927738667, -0.003765821224078536, -0.011483258567750454, 0.047884561121463776, 0.07623044401407242, 0.0038260098081082106, -0.08853141218423843, 0.011310911737382412, 0.05705352500081062, -0.04489165544509888, -0.026335136964917183, -0.052510786801576614, 0.009549257345497608, 0.08375655114650726, 0.038340117782354355, -0.05498768389225006, -0.04498570039868355, 0.02060333453118801, -0.008796606212854385, -0.058874230831861496, 0.012629957869648933, -0.029399704188108444, 0.01718609593808651, 0.020785467699170113, 0.011642851866781712, -0.03676038980484009, -0.02920728363096714, -0.005937132518738508, -0.020872732624411583, 0.03700806200504303, 0.04121340811252594, 0.007115543354302645, 0.03191138431429863, 0.0536237470805645, -0.033022452145814896, -0.013594052754342556, 0.06833934783935547, -0.005647765006870031, -0.018412534147500992, 0.0021265328396111727, 0.01753675565123558, -0.05177447944879532, 0.0012086820788681507, 0.034851253032684326, -0.01437060721218586, -0.01674753427505493, 0.033794134855270386, -0.028081011027097702, -0.044128239154815674, -0.0063235084526240826, 0.02416980266571045, -0.011060534976422787, -0.0942152738571167, -0.02727581560611725, 0.012662486173212528, 0.058916255831718445, -0.0043252198956906796, 0.08711806684732437, -0.03669528290629387, -0.020314648747444153, 0.002536759013310075, 0.027104971930384636, -0.011455986648797989, 0.022496387362480164, -0.010803726501762867, 0.016712943091988564, 0.019412953406572342, 0.005602693185210228, 0.007245688699185848, -0.008856059052050114, -0.024546194821596146, 0.10436970740556717, -0.030950838699936867, 0.049806103110313416, 0.03814678266644478, 0.0022012311965227127, -0.0001871871791081503, -0.02460058405995369, 0.008848222903907299, -0.038073521107435226, 0.0017102784477174282, 0.0725780799984932, 0.0437764897942543, -0.04651179537177086, 0.0074361530132591724, -0.011921020224690437, -0.04633142426609993, 0.04190463572740555, -0.061408016830682755, 0.019104743376374245, 0.06781495362520218, 0.0583312027156353, -0.04097355157136917, -0.0059682270511984825, -0.003727114759385586, 0.0003227396809961647, 0.0353228822350502, 0.025900835171341896, 0.0662299171090126, -0.018169155344367027, 0.01849399320781231, 0.047782812267541885, 0.021884730085730553, -0.060786958783864975, 0.008668649010360241, 0.011767695657908916, 0.006223196629434824, -0.028618520125746727, 0.030268559232354164, 0.006896754261106253, -0.024448547512292862, 0.021343916654586792, 0.0001870323612820357, -0.015098175965249538, -0.020194880664348602, -0.03412170335650444, 0.004151607863605022, -0.01735483668744564, 0.03977080434560776, -0.0009580938494764268, 0.028315899893641472, -0.026297226548194885, -0.015155773609876633, 0.003219743026420474, 0.027054034173488617, 0.033215492963790894, 0.0059351446107029915, 0.009620166383683681, 0.021612564101815224, 0.028864048421382904, 0.037677448242902756, -0.057278409600257874, 0.03182201832532883, -0.001148632145486772, 0.004776704125106335, -0.025263896211981773, 0.028592471033334732, 0.04416448622941971, 0.04334455728530884, -0.019849881529808044, -0.05240674689412117, 0.047444626688957214, -0.013101961463689804, -0.058437079191207886, 0.03519575670361519, -0.07297741621732712, 0.03693713620305061, -0.016470162197947502, 0.06728407740592957, 0.0031946375966072083, -0.019905896857380867, 0.005004665348678827, -0.02837887778878212, -0.019558455795049667, 0.015917304903268814, 0.0024836882948875427, -0.04167059063911438, -0.01368260569870472, 0.0644225925207138, -0.045785773545503616, 0.021031390875577927, 0.08944526314735413, -0.026935484260320663, 0.028120381757616997, 0.0029181167483329773, -0.03250381723046303, 0.029285695403814316, 0.025396253913640976, -0.00022974917374085635, -0.025924822315573692, 0.035247091203927994, 0.021696243435144424, -0.009234117344021797, -0.030206842347979546, -0.009539094753563404, -0.01667678914964199, -0.019925972446799278, 0.0444125235080719, -0.0034544470254331827, 0.05101969093084335, 0.042115576565265656, 0.035278771072626114, 0.01894686557352543, -0.0018501784652471542, -0.001521971309557557, 0.07130555063486099, 0.04034750163555145, -0.011540872044861317, 0.026042388752102852, -0.01767241209745407, 0.0002211730752605945, -0.0335489921271801, -0.03219311311841011, -0.03625073283910751, 0.013217946514487267, 0.041231848299503326, 0.0007303464808501303, 0.014001177623867989, -0.008737154304981232, -0.017606521025300026, 0.010832719504833221, -0.04067777097225189, 0.037935521453619, 0.02804320864379406, 0.01358595211058855, 0.0009219596977345645, -0.01824549213051796, 0.05714106559753418, -0.006753330584615469, -0.01988765224814415, -0.03462884947657585, -0.024726349860429764, -0.006433251779526472, -0.011979565024375916, 0.02367219515144825, -0.015149611979722977, -0.02822180651128292, 0.01914129965007305, -0.026061486452817917, 0.06387373059988022, -0.044984813779592514, -0.0045618279837071896, -0.027827555313706398, 0.011861588805913925, 0.0477868914604187, -0.01577708125114441, -0.05533516779541969, 0.010984036140143871, 0.002977203344926238, 0.023284511640667915, -0.02836613357067108, -0.022282693535089493, -0.026404280215501785, -0.01912044920027256, -0.0028506764210760593, 0.048743560910224915, 0.01616300642490387, -0.012722590938210487, -0.05061489716172218, 0.05119233578443527, 0.02371642366051674, 0.04978509247303009, -0.09726582467556, 0.08461949229240417, -0.03400333225727081, -0.03852226585149765, -0.0249361302703619, -0.01879897154867649, 0.04864621162414551, -0.0809435322880745, 0.021301371976733208, 0.04661397263407707, 0.04059968516230583, 0.02440068870782852, 0.03432284668087959, -0.013001388870179653, -0.024843741208314896, -0.022589240223169327, -0.06233316287398338, 0.034649867564439774, -0.0015309075824916363, 0.012980921193957329, 0.05334211513400078, 0.04924928396940231, 0.02039414457976818, -0.028297144919633865, -0.06242835149168968, 0.0313507504761219, -0.05909804627299309, -0.010112506337463856, -0.004552755504846573, -0.026516735553741455, 0.004045328125357628, 0.032388899475336075, 0.051425501704216, -0.015802862122654915, -0.04835841432213783, -0.007942556403577328, -0.03822586312890053, -0.03561362624168396, 0.008243512362241745, 0.038156721740961075, 0.00497396569699049, 0.008848240599036217, 0.040016692131757736, 0.021937934681773186, -0.04136902466416359, -0.02117433212697506, -0.01511325128376484, 0.050409577786922455, -0.047715168446302414, -0.007743751630187035, -0.09894413501024246, -0.020028317347168922, -0.03407057747244835, -0.03717687353491783, 0.020549125969409943, -0.03258391097187996, 0.04582846164703369, 0.003575719427317381, -0.01171963568776846, -0.03998367115855217, 0.058533601462841034, 0.002382940612733364, 0.06130780279636383, 0.08606978505849838, -0.0002507553726900369, 0.026027800515294075, 0.04855586960911751, 0.0011224913178011775, -0.05729663372039795, 0.036312397569417953, -0.018688052892684937, -0.040832314640283585, -0.021377036347985268, 0.05014224722981453, -0.03509007766842842, -0.020048243924975395, -0.03813202679157257, 0.04348105564713478, 0.025876663625240326, 0.05907450616359711, 0.03604418411850929, 0.020470496267080307, 0.054895393550395966, 0.0404270701110363, 0.012916590087115765, 0.029365474358201027, 0.07369682937860489, 0.027916159480810165, -0.056456226855516434, -0.07290419191122055, -0.03099730983376503, 0.041137587279081345, 0.020580219104886055, 0.030209887772798538, -0.0007425848743878305, -0.003482894040644169, -0.016214918345212936, -0.004340426530689001, -0.0026226812042295933, -0.005793375428766012, -0.04100207984447479, -0.05083996430039406, 0.04496064782142639, 0.0041456990875303745, 0.013459357433021069, -0.022948089987039566, -0.0013164096744731069, 0.014988529495894909, -0.015191846527159214, -0.03422081843018532, 0.016566302627325058, 0.018207479268312454, -0.04845967888832092, -0.038309212774038315, 0.03981295973062515, -0.03648809716105461, -0.03730253875255585, -0.03632021322846413, 0.0021378519013524055, -0.009970010258257389, 0.02333574928343296, -0.04484260827302933, -0.028046993538737297, -0.004468083381652832, -0.006843458861112595, -0.04019564390182495, -0.01814727671444416, -0.033715225756168365, -0.013133831322193146, -0.04704228788614273, 0.032798923552036285, -0.013186505064368248, -0.017718330025672913, -0.09887930750846863, -0.030544644221663475, 0.008207439444959164, 0.02723635733127594, -0.046450790017843246, 0.059885572642087936, 0.07571752369403839, -0.03076406568288803, 0.036189816892147064, -0.03708215802907944, 0.01074968185275793, 0.010838683694601059, 0.03294360637664795, 0.01028069294989109, -0.010759742930531502, 0.0034309085458517075, 0.02322969026863575, -0.01730462908744812, 0.024409329518675804, -0.023845406249165535, 0.008745355531573296, 0.03448653593659401, -0.021067289635539055, 0.011672651395201683, -0.09146612882614136, -0.019380686804652214, 0.07417778670787811, -0.035912804305553436, -0.038714513182640076, 0.02903948351740837, -0.01567414216697216, -0.02520078979432583, -0.019625041633844376, 0.03299225866794586, -0.014682089909911156, -0.06590258330106735, 0.041877031326293945, 0.04268376901745796, 0.02154962159693241, -0.007030087988823652, -0.04315692558884621, 0.06478305906057358, -0.0768829807639122, 0.041780054569244385, 0.03555859625339508, 0.024230370298027992, 0.017671527341008186, 0.0340338796377182, 0.0021192904096096754, 0.03359663486480713, 0.02647128328680992, 0.00398470601066947, 0.0050298916175961494, 0.002835565945133567, 0.05239149183034897, -0.03617613762617111, -0.008512817323207855, 0.0488952212035656, -0.02788775973021984, 0.011646618135273457, -0.0009222867665812373, -0.05537918582558632, 0.00538597209379077, 0.030812973156571388, -0.00699392007663846, 0.03413688763976097, 0.018148960545659065, 0.011971497908234596, -0.057750243693590164, 0.020122652873396873, 0.011496526189148426, 0.0254447590559721, 0.022408679127693176, -0.032188281416893005, -0.019854644313454628, -0.04389680549502373, 0.009149791672825813, -0.027884818613529205, 0.04704064875841141, 0.03297252580523491, -0.0429554283618927, 0.020515209063887596, -0.020887166261672974, -0.0718202069401741, 0.017732251435518265, -0.047489989548921585, -0.01675358973443508, 0.021626101806759834, 0.020010152831673622, 0.09478628635406494, 0.018743673339486122, -0.005504744593054056, -0.03058619052171707, -0.0773005411028862, -0.019887536764144897, -0.005274196621030569, 0.06459455192089081, -0.014063556678593159, -0.05885785445570946, 0.006771480664610863, 0.0001273646776098758, 0.02129177376627922, 0.0014066030271351337, -0.03154126927256584, -0.05743945389986038, -0.016056353226304054, -0.010829450562596321]], "dtype": "float32"}, {"__nd__": [28.083173751831055, 26.871896743774414, 28.53836441040039, 27.3936824798584, 28.433212280273438], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.042006254196167, 12.69905948638916, -3.7219669818878174, -7.529580116271973, -28.423019409179688, 6.284220218658447, -14.314746856689453, -5.252018451690674, -10.702752113342285, -0.1829805076122284, -4.620168685913086, -4.555231094360352, -9.501836776733398, 6.270668983459473, -13.591132164001465, -6.038661479949951, 11.146846771240234, -12.549779891967773, 20.70351791381836, -4.823592662811279, 13.068181991577148, 18.030160903930664, -1.7642042636871338, 3.1688919067382812, 14.055402755737305, 5.634268760681152, 26.098785400390625, 9.070474624633789, -2.0452418327331543, 11.814239501953125, 1.5126687288284302, 19.43738555908203, -14.645088195800781, 17.469669342041016, 21.53154182434082, 16.933670043945312, 0.9220840930938721, -21.116674423217773, -8.583001136779785, 0.025183765217661858, 14.986557960510254, 2.1220879554748535, -6.039311408996582, 3.5084404945373535, 14.493850708007812, -3.9665980339050293, 1.1161253452301025, 3.0075466632843018, 5.30825662612915, 9.333151817321777, 12.582539558410645, -15.784936904907227, 5.682192325592041, -0.8180428147315979, -8.881811141967773, 2.57987117767334, 0.5995199680328369, -2.320847272872925, 5.634647369384766, -2.847959041595459, 3.0791430473327637, 1.1567267179489136, 0.5842412710189819, -0.41246485710144043, -3.5440688133239746, 12.889114379882812, 6.385922908782959, 3.506603240966797, 8.476215362548828, -6.3963775634765625, 14.737863540649414, -16.607995986938477, -3.4706175327301025, -17.16248321533203, -5.8801069259643555, 3.3868160247802734, 0.2672603130340576, 20.021116256713867, 2.1899936199188232, 4.30126953125, -0.936491847038269, -10.531210899353027, 7.700690746307373, 14.054221153259277, -12.153366088867188, -19.747390747070312, -17.98077392578125, -3.6141457557678223, 2.450131416320801, 6.489483833312988, -23.294382095336914, -16.27594757080078, 1.0219204425811768, 3.9876224994659424, -4.457340717315674, -6.3639397621154785, 4.417640686035156, -7.302294731140137, -17.515504837036133, 13.552918434143066, 8.942005157470703, -3.8545031547546387, -1.3661160469055176, -8.614312171936035, 16.20233917236328, 4.761343479156494, 9.95370101928711, -2.3942575454711914, -10.655045509338379, 22.592243194580078, 4.182609558105469, 10.548182487487793, 14.015488624572754, 19.453161239624023, 5.539669513702393, -15.028961181640625, 5.879568099975586, -5.1524200439453125, -2.1814496517181396, 10.995336532592773, 3.1117124557495117, -2.6042962074279785, -0.8980088233947754, 5.170881748199463, 14.32858657836914, 13.417505264282227, -13.818999290466309, 5.695342540740967, 3.684563159942627, 7.687098979949951, 7.721402645111084, 6.768171310424805, 10.616059303283691, 2.7103326320648193, 12.534830093383789, 20.14004135131836, -3.1679177284240723, -11.188533782958984, -2.635906934738159, 0.3972465991973877, -2.2601702213287354, -8.300776481628418, -5.762572288513184, 7.747332572937012, 9.884317398071289, -8.265298843383789, -1.3903424739837646, -13.828256607055664, -6.800962448120117, 1.850569248199463, -3.867741823196411, -0.7863800525665283, -13.596209526062012, -1.9490714073181152, -16.504352569580078, 0.0013156912755221128, -0.48929110169410706, -8.853621482849121, -2.0696141719818115, 2.4117791652679443, 4.510173320770264, 21.00470542907715, 16.32552719116211, -3.8879590034484863, 18.823402404785156, 2.2525742053985596, 25.673831939697266, 4.4538984298706055, -4.58484411239624, 6.145851135253906, 1.1002213954925537, 14.997705459594727, -2.2995409965515137, -3.4255526065826416, -22.8281307220459, -2.3015835285186768, -6.119527339935303, -6.017045974731445, 11.50114917755127, 8.57467269897461, 0.8856529593467712, -1.1615487337112427, -7.795909881591797, -10.011056900024414, 5.8905229568481445, 3.0240345001220703, -10.494348526000977, -0.4469051659107208, -2.979006290435791, -0.2930304706096649, 16.472026824951172, -7.212688446044922, 13.764322280883789, 10.418819427490234, 0.7901712656021118, 9.427765846252441, 4.722661018371582, -1.9732184410095215, -17.72980499267578, -13.0427885055542, -9.815289497375488, -3.635204315185547, -0.5407268404960632, 13.901105880737305, 1.0294313430786133, 18.254474639892578, 12.857513427734375, 11.826736450195312, 0.21419501304626465, 1.9807332754135132, 8.140007019042969, -1.8161911964416504, 2.1933717727661133, -15.42188835144043, 7.403578758239746, 5.308229446411133, -2.990969181060791, 15.689000129699707, 17.82485580444336, 6.248628616333008, -5.118285179138184, 6.407539367675781, 4.476619720458984, 10.649866104125977, -2.4995081424713135, 7.0809197425842285, 8.625362396240234, 6.688117027282715, 5.646783828735352, 1.5071001052856445, -3.431077480316162, -9.720873832702637, -20.380290985107422, -19.753644943237305, -6.790555953979492, -4.384692668914795, 13.319633483886719, 6.341583251953125, -7.096686840057373, -3.902041435241699, -6.512468338012695, -7.444596290588379, 20.060503005981445, -7.16858434677124, 4.33945369720459, -2.4662485122680664, -11.333152770996094, -1.635562777519226, -15.896650314331055, -1.835476040840149, 8.544268608093262, -10.173726081848145, 10.595321655273438, -7.143797397613525, -5.260708332061768, -1.062617540359497, 11.57955551147461, -9.01656723022461, 4.305975914001465, -10.134848594665527, 7.707201957702637, 21.8526668548584, -18.171276092529297, -2.9999778270721436, -6.832070827484131, -5.742455005645752, -17.589275360107422, -0.12406285107135773, -8.919201850891113, -6.4774274826049805, 3.3785481452941895, -3.922832489013672, -7.034513473510742, 18.081533432006836, 1.3291717767715454, -3.0977988243103027, -0.5328712463378906, -0.14736060798168182, -1.9554541110992432, 5.095195293426514, -9.078402519226074, 13.310688972473145, 0.8309221863746643, 11.267696380615234, -3.164982795715332, 17.558238983154297, 5.610620498657227, 1.521633267402649, 16.881858825683594, 1.688403844833374, 8.87631607055664, -2.2852184772491455, 29.19468879699707, 0.8377578854560852, -3.8364734649658203, 3.6613800525665283, 11.404678344726562, -7.440340995788574, 10.835790634155273, -3.2967777252197266, -5.821068286895752, -8.769477844238281, 14.211455345153809, -12.689577102661133, -7.484536647796631, -7.017589092254639, -1.5577620267868042, 2.273195743560791, -7.027873516082764, -7.292969226837158, -23.786775588989258, 11.443657875061035, 3.6145434379577637, 9.835585594177246, 4.515695571899414, -16.26057243347168, 8.723514556884766, 15.061809539794922, -15.523185729980469, 10.61935806274414, -7.538400650024414, -5.417693138122559, -12.907052040100098, -9.390027046203613, 0.3111083209514618, 5.219118595123291, 5.002248764038086, 5.91801118850708, -3.80513858795166, 5.029507160186768, 6.62286376953125, 10.036029815673828, 5.034188747406006, 15.522226333618164, 0.18276527523994446, -22.512971878051758, 6.964269161224365, 4.67215633392334, 9.203619003295898, -10.967479705810547, 9.452373504638672, 3.901911973953247, -10.145501136779785, -12.1591796875, 9.380866050720215, 21.807682037353516, 7.179355621337891, -8.015305519104004, 3.8648996353149414, -9.253989219665527, -3.0231845378875732, 3.1113195419311523, -2.627997398376465, 11.576828956604004, -16.311012268066406, 1.1766531467437744, 10.53796672821045, 1.2971587181091309, 9.989298820495605, -3.6253597736358643, 7.30477237701416, -11.201099395751953, 0.2502371072769165, -1.9444782733917236, 6.41162633895874, -5.927343368530273, 0.6646490097045898, -32.842567443847656, -4.3099684715271, 6.251766204833984, 12.150525093078613, 4.237642288208008, 0.159569650888443, 18.972570419311523, -0.11073726415634155, -2.4103944301605225, 2.5171687602996826, -1.1210094690322876, -10.806588172912598, -17.200105667114258, -10.994330406188965, 2.5206408500671387, 5.507440090179443, 4.362733840942383, -6.065855026245117, -5.342257499694824, 8.109458923339844, -7.613446235656738, -5.22642183303833, 13.984103202819824, 0.6351501941680908, -6.050806522369385, -7.207634449005127, 11.504749298095703, -0.6041852235794067, 1.1989097595214844, 1.2987689971923828, -20.272123336791992, -9.341919898986816, 1.6233123540878296, -1.4009331464767456, 3.285822868347168, -9.462340354919434, -9.122678756713867, 3.819066286087036, 16.351375579833984, -9.966642379760742, -0.7766729593276978, -9.455220222473145, -16.270641326904297, -1.173201560974121, -2.944765090942383, -25.35865592956543, -6.9018707275390625, -7.789505958557129, -16.279617309570312, -9.677894592285156, 16.044601440429688, -24.039514541625977, -15.610302925109863, 8.784468650817871, 0.8144177198410034, 3.586009979248047, -0.7846565246582031, -0.7175472378730774, -7.074311256408691, 10.615128517150879, -20.808330535888672, 2.071743965148926, -22.45538330078125, -24.46059799194336, -3.784304141998291, 14.68397331237793, 3.9118993282318115, -1.7123908996582031, 16.05791664123535, -2.1835594177246094, -8.809282302856445, 5.051942348480225, -15.517745971679688, 20.703475952148438, 1.9659448862075806, -2.641336441040039, -7.636081695556641, 6.04949951171875, 16.008516311645508, -9.67093276977539, -1.4592180252075195, -0.7920074462890625, -12.379613876342773, 9.143681526184082, 0.2076014131307602, -7.924002647399902, 5.331914901733398, -11.792040824890137, 0.5514712333679199, -0.9214505553245544, -12.084979057312012, -8.798929214477539, -0.37182652950286865, -21.466075897216797, -2.4777934551239014, 1.4123029708862305, -1.0657038688659668, -2.209071159362793, -15.73668384552002, -4.306127548217773, -2.403156042098999, 5.016021728515625, 0.3472904562950134, 9.129379272460938, -0.8371604084968567, 2.016357421875, 6.010630130767822, -0.9702655076980591, 12.662163734436035, -18.28770637512207, -14.60197925567627, 9.897676467895508, -17.255155563354492, -8.010652542114258, -7.359323024749756, 2.342358350753784, -8.605147361755371, -8.500996589660645, 21.634387969970703, -2.894033193588257, 1.1243361234664917, 7.95775842666626, 2.933835744857788, -4.291114330291748, -10.102176666259766, -5.094078063964844, 4.491186141967773, 3.8142523765563965, 8.502561569213867, 8.56490707397461, 23.040876388549805, 15.057724952697754, 2.28043794631958, 3.4635376930236816, 2.7503302097320557, 0.4640912711620331, 2.127347946166992, 2.923311948776245, -5.137898921966553, -9.43885612487793, -6.088061809539795, 6.4762420654296875, 12.017607688903809, 3.3724331855773926, -0.2124653160572052, 7.488424777984619, -0.20919345319271088, 3.0468907356262207, 10.876055717468262, -16.7128963470459, -9.417702674865723, 22.981136322021484, -0.20296774804592133, -6.173885345458984, -9.967830657958984, -3.121188163757324, -10.247116088867188, -4.67878532409668, -3.3170011043548584, 12.411999702453613, -4.922591209411621, 6.925405502319336, -8.339187622070312, 29.154430389404297, -12.22667121887207, -1.8801302909851074, 1.077556848526001, -1.4615159034729004, 8.209985733032227, 7.094600200653076, 1.9794279336929321, 18.468976974487305, -5.0058817863464355, -0.7823818922042847, -1.7851572036743164, 1.0399457216262817, 3.948119640350342, -0.03048275038599968, -4.20331335067749, -5.288782119750977, -2.699899911880493, -8.555606842041016, -6.250882148742676, -11.742453575134277, -22.293912887573242, 5.972004413604736, -9.340829849243164, 9.191637992858887, -9.619874000549316, 3.7250864505767822, -12.163530349731445, 12.113075256347656, -8.376140594482422, -15.359477996826172, -4.639111518859863, -10.458722114562988, 27.575252532958984, -11.902928352355957, 7.474851131439209, -24.3857479095459, 7.003595352172852, -13.597907066345215, -4.36124849319458, -10.847614288330078, 5.0140581130981445, -8.368133544921875, 1.064753770828247, 4.395893573760986, 6.005400657653809, -2.36541748046875, 13.405547142028809, 13.4033203125, -6.537146091461182, 3.8688955307006836, 9.813178062438965, 21.775541305541992, -6.813920974731445, 8.31039810180664, 2.347778558731079, -8.33349609375, 1.2076433897018433, -14.371338844299316, -2.8978495597839355, 21.738086700439453, 8.398672103881836, 13.022541999816895, 5.116376876831055, -2.5698087215423584, -10.45046615600586, -19.26479148864746, -12.997017860412598, 21.00782585144043, -6.1302595138549805, 1.5886269807815552, 11.00669002532959, -12.359016418457031, 4.230717658996582, -11.782562255859375, -1.438320279121399, -13.425568580627441, 18.0990047454834, 3.2199456691741943, 14.316267013549805, -9.863604545593262, 3.442986249923706, -9.25966739654541, 27.36741065979004, -3.898710012435913, 2.943227767944336, 8.611726760864258, -4.876682281494141, 7.410628795623779, -1.9759893417358398, 1.2761828899383545, -7.798181056976318, 8.28492546081543, 13.915634155273438, -15.021522521972656, 1.1049151420593262, -3.6154356002807617, 2.4277398586273193, 8.38050651550293, -1.8971813917160034, -0.650519073009491, 0.5205398201942444, -1.1006834506988525, 6.129403114318848, -11.340677261352539, -1.0692509412765503, 3.3304386138916016, 11.77038288116455, 18.17634391784668, 2.398481607437134, 0.5196447968482971, -11.739745140075684, -9.5999116897583, 3.264646291732788, -1.791522741317749, 5.646658897399902, -25.046405792236328, -6.138640403747559, 1.332397222518921, -2.7826759815216064, -4.1018853187561035, -9.725404739379883, -4.241588592529297, -6.0817461013793945, -6.420896053314209, 0.21276569366455078, 6.586654186248779, -1.1691279411315918, -11.19957160949707, 13.813036918640137, -8.141751289367676, 0.255646288394928, 3.1376218795776367, -0.7471924424171448, -13.880900382995605, -6.379398345947266, -15.663984298706055, 14.512016296386719, -9.688313484191895, 16.62215232849121, 6.84537935256958, 7.354400157928467, -0.0735107958316803, -10.684832572937012, -4.094187259674072, -1.9780957698822021, 21.8087215423584, 3.6924169063568115, -11.176275253295898, -2.357574224472046, 16.00199317932129, 3.8454012870788574, 4.143129348754883, -3.269796371459961, -12.527693748474121, 17.099536895751953, 13.070573806762695, -14.987131118774414, -0.29166319966316223, -2.9673635959625244, 14.028003692626953, -0.5164957046508789, -12.378192901611328, 18.29201889038086, 1.7392634153366089, 4.516894817352295, 3.97873592376709, 4.469831466674805, -12.673334121704102, -21.104522705078125, 3.04919695854187, 7.692819595336914, 2.231710910797119, 17.046815872192383, -3.0289483070373535, -15.313302993774414, 13.355976104736328, 9.211955070495605, 1.2021503448486328, -23.328758239746094, 16.97412872314453, -5.856672286987305, 8.77159595489502, -5.979753017425537, 18.966466903686523, 4.534366607666016, -8.925362586975098, -19.910274505615234, 9.986799240112305, -14.978036880493164, 1.3502027988433838, 9.71831226348877, -12.258896827697754, 9.483452796936035, 3.4744203090667725, -5.195221424102783, -0.46996378898620605, 6.0166778564453125, -3.729355573654175, 8.609618186950684, -2.352665424346924, 1.019131064414978, 6.454718589782715, 10.504817008972168, 2.5093979835510254, 6.023409366607666, 24.77570343017578, 5.493402004241943, -17.853031158447266, 15.26832389831543, 1.0316219329833984, 10.290576934814453, 0.79909348487854, 3.272226095199585, 12.635761260986328, 4.828564167022705, -7.311751842498779, 13.153042793273926, -8.254538536071777, 9.323755264282227, 7.223562717437744, -0.6054180264472961, 0.3390863537788391, -13.896683692932129, -5.937627792358398, 32.059635162353516, -20.79542350769043, 10.635472297668457, -9.651137351989746, -4.164019584655762, -20.65115737915039, -12.045954704284668, -2.22806453704834, 5.560035705566406, 11.84247875213623, 14.312896728515625, -0.3794628977775574, -8.695124626159668, 11.311494827270508, 2.0938100814819336, -12.515172004699707, 19.822729110717773, 10.26724624633789, 13.512787818908691, 0.633019745349884, -6.693089485168457, 7.72761344909668, -3.6459455490112305, 16.310697555541992, -3.562620162963867, -3.4603123664855957, 13.453269958496094, 13.9203462600708, 4.175082683563232, 11.709165573120117, 4.505607604980469, -6.605130195617676, 12.209156036376953, -11.339348793029785, 0.009437286294996738, -2.9834165573120117, -5.797454833984375, -8.067882537841797, -0.02543770521879196, -4.939843654632568, -1.9959545135498047, -4.785612106323242, -8.081220626831055, -4.288569450378418, 5.111197471618652, -1.210629940032959, -10.894905090332031, -4.2731146812438965, -12.443920135498047, -3.823197364807129, 1.7079405784606934, -3.9581151008605957, -1.1590977907180786, 11.099838256835938, -4.472960948944092, 3.7371432781219482, -8.071561813354492, 14.859630584716797, -1.7062448263168335, 0.13789039850234985, -8.438759803771973, 7.096531391143799, 8.703790664672852, 13.859257698059082, -4.917749404907227, 5.606589317321777, -18.450952529907227, -14.012884140014648, 12.14013957977295, 12.456632614135742, 0.3764084577560425, 1.329507827758789, 23.995046615600586, 7.421968460083008, 11.502613067626953, 6.3019843101501465, 0.14956195652484894, -11.509525299072266, 4.851983547210693, 0.9253427982330322, 1.13618004322052, 5.380305767059326, 0.24907556176185608, 15.43053150177002, -11.598944664001465, 5.8188347816467285, -8.86640739440918, -2.3223867416381836, -12.005228042602539, -0.2542625069618225, 10.42023754119873, 5.624534606933594, -1.1412853002548218, 17.014556884765625, -9.21849250793457, 17.383953094482422, -2.296834945678711, 12.951557159423828, 1.065969705581665, -3.236926555633545, 16.846115112304688, -6.665053367614746, 4.796401023864746, 8.210223197937012, 11.591888427734375, -9.588465690612793, 10.529173851013184, -12.270979881286621, 2.746734619140625, 6.995095252990723, 6.404300212860107, 1.9682978391647339, 0.9437960386276245, 4.468640327453613, -5.5225605964660645, 12.995673179626465, 11.410577774047852, -0.11157919466495514, 8.86427116394043, -2.243817090988159, 5.473150253295898, -4.448737144470215, 5.905905723571777, 6.956019878387451, -1.4976130723953247, -15.460731506347656, 15.892536163330078, 9.340658187866211, 7.584746360778809, 10.006086349487305, 2.312100410461426, 8.30052375793457, 5.5910725593566895, -12.046727180480957, -8.785898208618164, -15.422517776489258, -9.154146194458008, -6.311972141265869, -7.890686988830566, -23.125288009643555, -1.8966059684753418, -10.667154312133789, -14.523221969604492, 2.2705276012420654, 10.53663158416748, 12.37212085723877, 6.569485664367676, 8.788581848144531, 24.506338119506836, -9.51006031036377, 10.09832763671875, -5.386055946350098, 5.977812767028809, -0.4224016070365906, 2.505669116973877, -1.2294015884399414, -3.8827786445617676, -7.953977108001709, 11.500761032104492, 9.190258026123047, 8.010358810424805, -0.38722389936447144, -13.975317001342773, -0.4325640797615051, 11.15965747833252, -14.240066528320312, 11.360538482666016, 6.911733150482178, 0.6100064516067505, -8.89665412902832, 12.88301944732666, -14.308891296386719, -2.8828277587890625, -6.6967339515686035, 0.9448068141937256, 3.624891757965088, 1.6903796195983887, -8.43845272064209, -6.4141364097595215, 2.0365612506866455, 4.252477169036865, 22.14722442626953, 16.401227951049805, 0.3560815453529358, 3.5864129066467285, 10.33694839477539, 8.425357818603516, 0.4891345500946045, 10.041590690612793, 2.957092761993408, -9.775871276855469, -23.316139221191406, -18.250415802001953, -9.816120147705078, -5.992862701416016, -2.8872828483581543, 10.30975341796875, -12.83775806427002, -16.15945053100586, 2.4609246253967285, -1.7895927429199219, -6.057880401611328, -5.067339897155762, -27.97154998779297, -11.021215438842773, 5.132483959197998, 5.062865257263184, 16.270679473876953, 10.900311470031738, -2.1503148078918457, -13.150928497314453, -3.522019863128662, 8.195850372314453, 13.168209075927734, 30.26092529296875, 24.086362838745117, 11.77417278289795, 0.1156081110239029, -4.4213457107543945, -3.2970213890075684, -19.810697555541992, 3.753800392150879, -3.0723485946655273, -7.831154823303223, -0.4717518389225006, -2.0168356895446777, -10.12610912322998, 6.199302673339844, 4.115211486816406, 1.5012788772583008, 15.329486846923828, 12.727231979370117, 1.3933165073394775, -9.23058032989502, -12.86644458770752, 4.810985565185547, -0.13460476696491241, 13.760037422180176, -3.621786594390869, 6.772653579711914, -3.744842767715454, -1.5357880592346191, 15.647089004516602, -1.863682746887207, -3.386115789413452, 3.587627649307251, 26.022090911865234, -10.301438331604004, 6.757077693939209, -16.188623428344727, -2.4164535999298096, 4.119992733001709, 5.468494892120361, 14.14996337890625, -12.307600975036621, -10.033336639404297, -0.9598445892333984, 1.4112192392349243, -1.7902575731277466, -3.20778489112854, -6.755789756774902, 3.560774326324463, -3.7905871868133545, -4.9127197265625, 12.434986114501953, 11.302579879760742, 5.928386688232422, 6.0622663497924805, 8.706445693969727, 21.594850540161133, 3.388097047805786, -1.6480796337127686, -0.23712335526943207, 3.7718729972839355, -8.234540939331055, -0.45484864711761475, -10.268181800842285, -3.195887565612793, 2.9128458499908447, 0.5275471210479736, -13.713993072509766, -12.16372013092041, -6.352856159210205, 1.1636954545974731, -13.655940055847168, -20.796344757080078, -2.5366435050964355, 6.42122745513916, 12.026107788085938, 6.180893898010254, 4.228753089904785, 2.4038820266723633, 10.55416488647461, -2.2807211875915527, -3.9587559700012207, -11.14574909210205, -8.730965614318848, -7.012988090515137, -12.678824424743652, 1.5266807079315186, 5.376710414886475, -8.723023414611816, 5.831720352172852, -3.00935959815979, 16.660369873046875, -5.13240909576416, 0.5136486291885376, 3.5991735458374023, 6.944477081298828, 13.476865768432617, 1.9965550899505615, 24.719505310058594, -3.7019457817077637, -1.7555885314941406, 20.035537719726562, -0.8876824975013733, -1.9892652034759521, 5.691561698913574, -9.23289966583252, 11.330902099609375, -8.4118013381958, -14.667533874511719, -9.000264167785645, 2.8638246059417725, 9.973633766174316, 7.24539852142334, 10.884308815002441, -2.008838653564453, -5.300279140472412, 9.864057540893555, -15.169844627380371, 1.8107011318206787, -2.561492443084717, 2.3330326080322266, -22.00355339050293, -10.095943450927734, -0.8072855472564697, 0.5688369870185852, -22.125431060791016, 5.215394973754883, 4.326442241668701, -0.29161810874938965, -6.775400638580322, -5.382023811340332, 20.77783203125, -1.0058238506317139, 1.1494951248168945, 15.324167251586914, 5.151960849761963, 0.6159069538116455, -16.30377769470215, 6.73778772354126, -12.563815116882324, -7.826854705810547, 7.037959098815918, -1.80385160446167, -7.240720748901367, 4.84393835067749, 13.947457313537598, 6.8468451499938965, 2.477673053741455, -2.9912924766540527, 6.377152919769287, 8.082361221313477, -12.503913879394531, 11.23095989227295, -20.112815856933594, 8.900593757629395, -6.220034599304199, -5.59481143951416, 11.694684982299805, 8.944696426391602, 1.3508516550064087, -4.9542717933654785, -3.325272560119629, 11.768926620483398, -8.72378158569336, -14.135198593139648, -0.183535635471344, 1.8864874839782715, -2.0585670471191406, -17.34888458251953, 8.04578971862793, -10.240715026855469, -6.617374420166016, 9.203269958496094, 0.49380505084991455, -11.175924301147461, 5.319176197052002, -0.5128491520881653, 8.891707420349121, 7.5238447189331055, 23.152381896972656, 7.436093330383301, 12.982646942138672, -0.3401566743850708, 0.06848414242267609, -5.896354675292969, 0.09946131706237793, 0.5950472354888916, 4.868138790130615, -23.835115432739258, 4.375186920166016, -7.747373580932617, 14.096678733825684, -10.288253784179688, -2.9311749935150146, -8.972139358520508, 6.258694648742676, -8.504767417907715, -11.127552032470703, -8.027113914489746, 1.4440327882766724, -0.7291163206100464, -11.697486877441406, -23.280197143554688, 4.289237022399902, 5.825205326080322, -13.20197582244873, -3.2683050632476807, 0.289657324552536, -9.84695816040039, -4.3815484046936035, -21.002315521240234, -11.913835525512695, -1.8222023248672485, -9.018461227416992, 6.091117858886719, 4.048206329345703, -1.2841932773590088, 2.06404447555542, 7.491430282592773, -2.8181257247924805, 8.058107376098633, 0.26576289534568787, 6.500925064086914, -12.569669723510742, -11.527517318725586, -0.4822390079498291, -11.665888786315918, 2.8922784328460693, -1.626425862312317, -1.4399700164794922, 16.367786407470703, -22.355682373046875, 9.095612525939941, 1.691270351409912, 14.361310005187988, -9.87317180633545, -13.316779136657715, -7.092940807342529, -7.7528977394104, 7.953953266143799, -5.834044456481934, -0.7560476660728455, -0.8286639451980591, 0.11121728271245956, -0.8978110551834106, -6.497506618499756, -2.223237991333008, 9.509828567504883, 4.7756266593933105, -4.301054000854492, 1.3220906257629395, 8.219572067260742, -20.514904022216797, -7.8663105964660645, -1.1124783754348755, -3.400719165802002, 9.916525840759277, -0.9361160397529602, -2.5119152069091797, -17.579362869262695, 18.60202407836914, 5.84044885635376, -16.451988220214844, -7.140471458435059, -11.829496383666992, 14.175836563110352, -1.339928150177002, 0.2750627398490906, -9.786738395690918, -36.20698928833008, 0.7267102599143982, -6.586709499359131, -3.0119142532348633, -7.6667375564575195, -5.438216209411621, -9.722293853759766, -1.135421633720398, -0.17026865482330322, 10.107373237609863, 5.7218499183654785, -5.464080810546875, -9.399559020996094, 22.400402069091797, -2.4988484382629395, 17.684968948364258, 27.10561752319336, 2.7362990379333496, -13.948848724365234, -20.86296844482422, -0.11498148739337921, 5.920909881591797, -16.002487182617188, -16.94525909423828, -2.3404970169067383, -8.471585273742676, 5.10137939453125, -6.350849151611328, 4.6983747482299805, 1.4784175157546997, 6.11763858795166, 3.205258846282959, 1.742949366569519, 2.5990848541259766, -3.986159324645996, -1.8850429058074951, -17.0124568939209, 16.313962936401367, 4.067742347717285, -4.369336128234863, -29.42837905883789, -8.702319145202637, -0.3826282024383545, 17.274394989013672, 5.647140026092529, 1.9925521612167358, 6.181826591491699, 18.630943298339844, -18.74546241760254, 0.32503989338874817, 7.251543045043945, 21.413280487060547, 14.679688453674316, -11.031121253967285, 8.972423553466797, 4.870873928070068, -2.1473379135131836, -20.61053466796875, 5.97136116027832, -3.0752718448638916, 8.250946998596191, -13.095498085021973, -10.779241561889648, 8.157143592834473, -19.340669631958008, 0.5257306098937988, -0.7335860729217529, -3.8840603828430176, 6.909445762634277, 18.642154693603516, -2.759078025817871, -1.551393747329712, -0.8556633591651917, 1.457226276397705, 4.895830154418945, 9.033852577209473, 3.870278835296631, 4.098076343536377, -2.706873655319214, 20.674612045288086, -6.163947582244873, -8.542797088623047, -3.067880868911743, -7.149297714233398, -14.387860298156738, 4.742978096008301, -9.466902732849121, 21.392608642578125, -9.592284202575684, -4.3306450843811035, -17.00986099243164, 15.177192687988281, -8.26494312286377, -15.076568603515625, 0.43925827741622925, -12.532278060913086, 2.0921401977539062, -1.376239538192749, 5.4870758056640625, 18.56957244873047, 6.605382442474365, -9.429166793823242, -22.28141975402832, 20.212099075317383, 5.419073104858398, -7.942624568939209, 0.5966618657112122, 2.9946084022521973, -11.669612884521484, 5.944716930389404, 9.768342971801758, 1.5089902877807617, 3.6432266235351562, -6.341388702392578, 0.3621792495250702, -19.106897354125977, -4.949197292327881, 30.594223022460938, -3.377711534500122, -3.4111766815185547, -4.693877220153809, 8.080738067626953, 4.107133865356445, 12.425713539123535, -5.533687591552734, -4.95624303817749, 1.7265779972076416, -3.8218605518341064, -9.477357864379883, -8.006447792053223, -5.571761608123779, 20.95780372619629, -7.087430000305176, 20.2121524810791, -4.369194984436035, -16.06027603149414, 7.367630481719971, 5.220363616943359, 18.093460083007812, -9.139049530029297, 18.420209884643555, -20.461185455322266, -18.778345108032227, 8.317510604858398, 17.227834701538086, 9.444320678710938, -11.923905372619629, 6.064218997955322, 14.614946365356445, 15.343072891235352, 12.418264389038086, 15.146108627319336, 4.271810531616211, -9.929362297058105, 13.74992561340332, 12.24215030670166, 11.164251327514648, -25.73695945739746, -7.832720756530762, 15.819195747375488, -6.58768892288208, 15.736106872558594, -11.569561004638672, 7.2779951095581055, -23.513492584228516, 9.372886657714844, 3.3138749599456787, -7.59551477432251, -2.676067590713501, -0.26510876417160034, -7.946999549865723, -7.072619915008545, 18.601957321166992, 6.0769500732421875, -1.7258617877960205, 11.782918930053711, 5.507257461547852, -14.939521789550781, -13.898876190185547, 3.759140729904175, -10.725544929504395, 4.629993915557861, -4.212923526763916, -10.342370986938477, -6.115350723266602, -22.59984588623047, 14.191579818725586, 3.200421094894409, -0.22232361137866974, 20.360034942626953, 4.306911468505859, -15.458268165588379, 9.758988380432129, 9.839149475097656, -2.037381649017334, 18.569753646850586, -4.4246015548706055, -1.803495168685913, 7.082545757293701, -0.80218505859375, -11.736434936523438, -3.2642862796783447, -2.4032833576202393, 17.18505859375, -0.1979409158229828, -17.255964279174805, -5.866593837738037, -6.165271759033203, 0.713519811630249, -10.177040100097656, -11.110294342041016, 6.221227169036865, -9.072040557861328, 11.848670959472656, 1.183150053024292, -1.5473966598510742, -2.751396894454956, 3.618718147277832, 3.5399489402770996, -10.088493347167969, 11.603609085083008, -3.9099037647247314, 6.308030128479004, -0.42340177297592163, 19.354366302490234, -0.9258327484130859, -7.618842124938965, 0.28369569778442383, 10.260379791259766, -1.7037205696105957, -17.200496673583984, 0.5855533480644226, -5.587982654571533, 13.996274948120117, 3.915363311767578, 0.7646948099136353, 12.559279441833496, -0.061791446059942245, -19.97377586364746, -3.0293009281158447, -5.850195407867432, 2.7315499782562256, -12.279170989990234, -1.3719524145126343, 0.45530009269714355, -5.206542015075684, -9.82948112487793, 9.300185203552246, -11.842392921447754, 13.531726837158203, 6.524452209472656, 3.110285997390747, -5.101561546325684, -1.0557678937911987, -0.4517906606197357, -7.514428615570068, -7.414149284362793, 3.5593364238739014, -8.640755653381348, 1.9303935766220093, -6.290757179260254, -10.289266586303711, 6.574607849121094, -5.471183776855469, -9.737839698791504, -5.431044578552246, 5.984399318695068, -7.877867221832275, -2.770174026489258, -16.142576217651367, 1.0444695949554443, -1.810110330581665, -11.680566787719727, 4.949958801269531, 5.243166923522949, 8.256294250488281, -6.580135345458984, -12.49672794342041, 13.574313163757324, -19.291996002197266, 0.7795774340629578, 0.2723875939846039, 3.6712005138397217, 0.4334484338760376, 1.7813526391983032, -3.127002477645874, 2.618365526199341, -1.6345752477645874, 2.9724714756011963, 26.08333969116211, -13.741447448730469, -25.627582550048828, 2.6748504638671875, 23.17019271850586, -9.940633773803711, 13.243528366088867, 1.665132999420166, 5.816178798675537, -4.250171661376953, 15.727677345275879, -0.3125949501991272, 16.077295303344727, 26.120304107666016, 22.64889144897461, 12.011948585510254, -18.17704200744629, 3.681046485900879, 9.638303756713867, -1.6676331758499146, -3.149973154067993, -7.867739200592041, 25.183469772338867, 5.267429351806641, -2.2660627365112305, -1.7540174722671509, 7.384726524353027, 7.925978660583496, -8.380304336547852, 0.9688316583633423, 5.12282657623291, 1.4531254768371582, 22.1047420501709, 5.031144618988037, 3.2613108158111572, -29.800016403198242, -4.031001091003418, 13.732489585876465, 18.934383392333984, 7.2232255935668945, 9.984505653381348, -4.227020263671875, -6.774265289306641, -4.939998626708984, 10.56076431274414, 1.7609221935272217, -10.447964668273926, 12.758132934570312, -6.027139663696289, -9.368393898010254, -1.1091874837875366, 6.191572189331055, -2.830008029937744, 7.869701385498047, 0.08070570230484009, 12.151970863342285, -0.5127962231636047, 8.497847557067871, -3.601687431335449, 5.94345760345459, 1.4307548999786377, -9.760507583618164, 14.475579261779785, -2.564167022705078, 4.387433052062988, -4.009769439697266, 1.2702834606170654, 0.41530388593673706, -4.684991359710693, 2.08950138092041, 0.7976912260055542, 6.974712371826172, -1.1180250644683838, 9.050192832946777, -2.0821282863616943, -2.5063772201538086, -1.4229199886322021, -2.5268197059631348, 0.551192045211792, 5.387238502502441, 3.5350728034973145, -7.529898166656494, -14.517683982849121, -15.982955932617188, 18.362045288085938, -11.678017616271973, -1.3047499656677246, 18.71311378479004, 6.03640604019165, 2.8897082805633545, -4.927563190460205, -4.86883544921875, 2.3759684562683105, 7.7248759269714355, 8.974559783935547, 7.035151481628418, -9.195660591125488, -4.055195331573486, -0.5745723247528076, 3.8415675163269043, 16.5751895904541, 0.795710563659668, -3.5489063262939453, 4.270069122314453, 3.018346071243286, 9.580093383789062, 0.33744585514068604, -3.7360951900482178, -13.347969055175781, -2.2392044067382812, 2.631986141204834, -2.2885942459106445, 7.622123718261719, 4.713441371917725, 5.261385440826416, -15.602354049682617, 12.442463874816895, 6.878667831420898, -24.523582458496094, -5.753818511962891, -21.669879913330078, -7.28330659866333, -9.632562637329102, -12.461710929870605, 6.307991027832031, 8.861387252807617, -3.40964412689209, -11.196757316589355, 11.473451614379883, -6.524898529052734, -1.3749736547470093, 5.081327438354492, -1.2070868015289307, -5.705610275268555, -1.9587171077728271, 3.7669994831085205, 5.965849876403809, -3.523397922515869, -14.504460334777832, 4.031966686248779, 1.1013927459716797, 11.42698860168457, 1.8150920867919922, -4.988216400146484, -7.03873872756958, 6.3561482429504395, -17.996246337890625, -0.013332095928490162, -12.000751495361328, 3.6476447582244873, 0.6119566559791565, 0.3784039616584778, 4.007514953613281, 10.829051971435547, 12.447999954223633, -4.594736576080322, 11.541654586791992, 1.4547615051269531, 4.099932670593262, 1.5994335412979126, 2.948031425476074, -5.807047367095947, -11.139829635620117, -7.0575408935546875, 4.288738250732422, 17.250757217407227, -8.736309051513672, 10.554069519042969, -13.30649185180664, -1.6412569284439087, 9.57398796081543, 10.686080932617188, 0.44956499338150024, 21.84391975402832, 2.496298313140869, -1.659654974937439, -15.248804092407227, 0.5227898955345154, -2.234498977661133, -4.047673225402832, 7.127902984619141, -4.544915199279785, -8.344030380249023, -6.103084564208984, 8.218732833862305, -1.9335378408432007, 12.05308723449707, 9.111830711364746, -2.3786706924438477, -22.17400550842285, 3.571650505065918, 3.1110310554504395, 11.421317100524902, 17.007577896118164, 5.9243574142456055, -5.677471160888672, -7.514703273773193, -3.577117919921875, -2.9684033393859863, -3.98911714553833, -1.0291996002197266, 20.327016830444336, -15.507525444030762, -26.383752822875977, -2.291673183441162, -4.0703444480896, -13.73855972290039, 11.497869491577148, 5.064010143280029, 11.940406799316406, 10.516992568969727, 7.415699481964111, -1.9066542387008667, -7.1855058670043945, 2.318603277206421, 8.37322998046875, -13.581680297851562, -8.786214828491211, -6.476312637329102, -5.712485313415527, -16.221311569213867, -3.6413071155548096, -3.7972397804260254, 1.887952208518982, -16.2734375, -13.105597496032715, 6.053276062011719, 12.89808177947998, 19.317001342773438, 12.131742477416992, -3.397061347961426, 8.666900634765625, -14.044998168945312, 10.684943199157715, -5.328255653381348, -25.379335403442383, 15.434012413024902, -0.934731125831604, 11.698561668395996, -1.7282543182373047, 11.730876922607422, 8.602639198303223, 6.530765533447266, -17.25364112854004, -8.757296562194824, -18.40692710876465, 5.048892974853516, -8.300695419311523, 9.49557876586914, -6.04954195022583, -1.8710912466049194, 6.404600620269775, -10.44277286529541, 2.3965179920196533, -27.28862762451172, -8.521018981933594, 8.720730781555176, 25.571258544921875, -20.882884979248047, 15.217966079711914, 13.571852684020996, -4.5561299324035645, -9.500408172607422, -2.0630428791046143, -2.3860726356506348, 3.629288673400879, -15.359533309936523, 0.8322611451148987, -5.779122352600098, -3.545258045196533, 6.82414436340332, 3.730783224105835, -22.26871109008789, 1.0328088998794556, 0.18634849786758423, 6.067897796630859, 1.7617100477218628, 1.952265739440918, -3.5078506469726562, -0.013409251347184181, 8.25894832611084, -14.744540214538574, -2.8767333030700684, -4.042214393615723, -7.377128601074219, -22.35913848876953, 11.253101348876953, -4.09514856338501, 14.818035125732422, 24.339088439941406, -10.377752304077148, 7.950567722320557, 0.1556381732225418, 5.445554733276367, 6.6630754470825195, 7.808454513549805, 5.341863632202148, 1.186737298965454, -9.454049110412598, -5.827914237976074, -12.167655944824219, -13.706136703491211, 0.05181040242314339, -7.786538124084473, -18.10171127319336, -19.64006805419922, -5.0864739418029785, -10.695060729980469, 4.001084327697754, 0.09174515306949615, 4.595861434936523, 14.014400482177734, -8.479148864746094, 4.23084831237793, -6.41331672668457, -7.538557529449463, -1.5447862148284912, -16.839351654052734, 2.999626636505127, -0.02918354421854019, -2.946690320968628, 1.5233397483825684, -7.402975082397461, 5.092693328857422, -22.783889770507812, -13.800086975097656, 2.4888501167297363, -4.3392510414123535, 3.6014649868011475, -22.968395233154297, -3.937016725540161, -5.7733893394470215, -1.1278280019760132, -15.436517715454102, 10.674222946166992, 3.8631412982940674, 2.767746925354004, 2.446415424346924, -4.681304931640625, -8.835166931152344, 12.114352226257324, 8.404465675354004, -1.4954544305801392, -10.798262596130371, -2.8368544578552246, 12.943426132202148, 5.5704755783081055, 16.26553726196289, 13.1455659866333, 13.896419525146484, 5.288125038146973, -0.6717730164527893, -2.076619863510132, -9.772506713867188, -10.416084289550781, 3.668466567993164, -7.44147253036499, -21.233003616333008, 5.050149440765381, -12.57225513458252, -4.30360221862793, -5.639366626739502, -7.225747585296631, 12.295690536499023, 2.6994335651397705, -18.43494987487793, 27.370662689208984, -0.7033764123916626, 11.863767623901367, 10.21872615814209, 0.8144978284835815, -15.179229736328125, -21.58576202392578, 19.09490394592285, -16.20724105834961, 9.09949779510498, -17.120086669921875, -11.844596862792969, 7.879543781280518, -7.4833478927612305, -1.3426313400268555, -7.697803497314453, -12.07089614868164, 10.06969928741455, 0.7320594787597656, -3.064851760864258, -10.824068069458008, -14.40213394165039, -6.861987113952637, 1.2512403726577759, 3.5226564407348633, 2.576310157775879, 6.081559181213379, -17.393230438232422, 8.501214981079102, -7.445382595062256, -6.132390022277832, 12.152018547058105, 5.99171257019043, -3.899355411529541, -0.17828045785427094, -11.548127174377441, -6.545117378234863, 3.64888072013855, -2.853086233139038, 9.329594612121582, -5.722122669219971, 2.0281496047973633, -17.486326217651367, -1.7037571668624878, 1.3471238613128662, -3.4930901527404785, -19.412748336791992, 0.5482879877090454, -19.24567222595215, -5.817552089691162, 3.9046146869659424, -2.172464609146118, -10.88624095916748, 2.1149630546569824, -19.613143920898438, 8.769637107849121, 2.9668617248535156, -29.732208251953125, -0.3125297427177429, 8.500313758850098, 1.8779553174972534, -3.218362331390381, -13.828412055969238, -16.074750900268555, -0.0871368870139122, -0.08957859873771667, 12.156814575195312, -12.80866813659668, 14.739485740661621, 8.803182601928711, 4.848250865936279, 1.6142005920410156, -5.469601631164551, -9.608049392700195, 2.9831016063690186, -8.729645729064941, 4.387038230895996, 15.920764923095703, 15.348712921142578, 12.50909423828125, 3.0078043937683105, -2.0390138626098633, 13.103903770446777, 10.961518287658691, -8.521706581115723, -9.645744323730469, 1.0426862239837646, 2.2139501571655273, -10.164091110229492, -17.883621215820312, -3.125903367996216, -4.1029815673828125, 3.6049821376800537, -11.955653190612793, 3.3230433464050293, -4.496783256530762, 4.011340141296387, 15.439367294311523, 5.298108100891113, 0.152707040309906, 9.517738342285156, -13.908512115478516, -3.8995492458343506, -0.33567342162132263, -2.328907012939453, -5.965681552886963, 18.816673278808594, 2.6629960536956787, 5.805485248565674, 1.7060905694961548, -16.25807762145996, 7.728652477264404, 7.2483086585998535, -1.1601574420928955, -3.337568998336792, 1.5682413578033447, -17.245637893676758, 14.325133323669434, -5.080684661865234, 2.605301856994629, 0.9320124387741089, -12.781847953796387, 2.1668002605438232, -0.4102155864238739, -0.16621243953704834, 4.6113128662109375, -10.131563186645508, -13.244146347045898, -0.7369828224182129, 10.645597457885742, 6.985719680786133, 5.986456394195557, -9.276491165161133, -12.297550201416016, 6.791382789611816, 8.954679489135742, 1.101131558418274, 0.6070429086685181, -2.8917720317840576, -16.206974029541016, 3.8583998680114746, 6.779938220977783, 4.709714412689209, 1.1184313297271729, 10.559196472167969, -10.238896369934082, -16.472991943359375, 8.453923225402832, -0.547663152217865, 7.935877799987793, 0.8104293942451477, 6.225788116455078, 13.008624076843262, 2.442784547805786, -13.318492889404297, 21.120336532592773, 0.6078953742980957, 9.010428428649902, -1.3257641792297363, -7.590569496154785, -3.2903685569763184, -27.27506446838379, 4.304587364196777, -6.675900936126709, 16.955373764038086, 6.770962715148926, 13.155926704406738, 6.711690902709961, -6.002954483032227, 1.5852017402648926, 11.145414352416992, -1.2652268409729004, 2.751530647277832, 1.5525743961334229, 5.146524429321289, 0.8884449005126953, 14.018365859985352, 9.234130859375, 1.9823789596557617, 4.936093807220459, -5.17559289932251, 0.3729194104671478, 4.431071758270264, 4.292192459106445, -19.593128204345703, -5.102509498596191, -10.45566463470459, 21.875865936279297, 13.379317283630371, 6.8057756423950195, -5.279231071472168, -0.42756855487823486, 4.039519309997559, 14.485179901123047, -11.05892276763916, -8.206778526306152, -11.733354568481445, -9.831121444702148, -6.082732677459717, -2.9245080947875977, 10.451299667358398, -6.338350296020508, 1.9523077011108398, -2.7185444831848145, -14.980827331542969, 10.838888168334961, 8.963630676269531, -0.1821439564228058, -3.3607590198516846, 0.6870440244674683, 6.374863624572754, -6.942564010620117, -6.397459506988525, 3.5073461532592773, -10.298813819885254, -14.107093811035156, 1.9509592056274414, -5.584399223327637, -16.027099609375, 11.781814575195312, 2.8465347290039062, -11.859697341918945, -14.769221305847168, -8.090218544006348, -3.7174859046936035, 8.590662956237793, -9.132673263549805, 0.9743029475212097, -15.410872459411621, 2.839690685272217, 4.40329647064209, 13.822632789611816, 7.727031230926514, -11.075702667236328, 0.03181760758161545, 6.272437572479248, -6.664356231689453, -5.597661972045898, -0.7747805118560791, -0.11088944226503372, -14.2384672164917, 2.035369634628296, 8.205453872680664, -3.2415456771850586, 3.38303542137146, 4.410938262939453, 0.5522937774658203, -6.277346134185791, -6.039618492126465, 15.010719299316406, 12.646085739135742, -15.72288703918457, -5.8492937088012695, 4.261425018310547, 1.3448359966278076, 21.166399002075195, -9.965283393859863, 4.995134353637695, 2.280412197113037, 0.9823150634765625, -8.275250434875488, -9.791752815246582, -3.3713083267211914, -20.125049591064453, 13.691197395324707, 1.703682541847229, 10.966222763061523, 2.446817398071289, 20.857799530029297, 2.913400650024414, 1.5534634590148926, 17.023014068603516, 7.955162048339844, 24.658893585205078, 1.773536205291748, 1.7405240535736084, 8.10336685180664, 2.540921688079834, 2.7990784645080566, 6.620864391326904, -11.641444206237793, -0.9722158312797546, 5.763930320739746, -7.89383602142334, -11.872995376586914, -23.54067611694336, -4.828378677368164, -4.18026876449585, -5.583347797393799, 14.861868858337402, 19.287460327148438, -9.05677318572998, -10.554512023925781, 2.364997386932373, 5.690217018127441, -9.807229995727539, -5.63599967956543, 1.5059900283813477, -17.529678344726562, -10.235051155090332, 4.098169326782227, 7.666697978973389, -1.1883788108825684, -3.8025436401367188, 7.3822760581970215, -1.0219933986663818, -14.588294982910156, 3.8699331283569336, 10.167337417602539, 3.756096839904785, -4.240225791931152, -4.0988054275512695, -13.79909610748291, -12.122966766357422, -6.397968292236328, -4.418105602264404, -7.275631427764893, -1.1462255716323853, -1.8242766857147217, 1.9775280952453613, -15.945684432983398, 6.3838300704956055, 18.871234893798828, 0.493889719247818, 9.252317428588867, 1.280484914779663, -18.34337615966797, 10.518221855163574, -1.8157484531402588, -14.728693008422852, -19.70973014831543, 7.0564470291137695, 5.412766933441162, -5.91716194152832, -6.857494354248047, 7.795241355895996, -1.5071009397506714, -5.81726598739624, -11.11290168762207, -1.632643461227417, -18.987567901611328, 10.078054428100586, -0.8050659894943237, 0.22476330399513245, -10.632294654846191, -1.054194688796997, -8.288461685180664, -3.717014789581299, -13.897735595703125, -1.4400107860565186, -8.150824546813965, -4.674559116363525, -7.8940205574035645, -17.012069702148438, 17.10578155517578, -17.83889389038086, -17.352388381958008, -14.58333969116211, 11.504348754882812, 2.9045677185058594, -7.275852680206299, -5.745667934417725, 11.067200660705566, -2.682347297668457, -9.210350036621094, -8.774454116821289, 15.08658504486084, -14.852656364440918, 20.28232765197754, -0.8611783385276794, 24.145204544067383, 0.03685019537806511, -4.534202575683594, -18.39912986755371, -20.971378326416016, -4.597622871398926, -10.286064147949219, -8.47969913482666, -9.331793785095215, -6.011675834655762, -14.338034629821777, 13.763265609741211, 1.9944331645965576, 12.549516677856445, 2.0504367351531982, -7.525149345397949, 9.39047908782959, 15.84646987915039, -1.2671546936035156, 5.069642066955566, 3.223477602005005, 0.13150395452976227, 9.63729476928711, 5.104351043701172, 11.516213417053223, -10.033733367919922, -7.262334823608398, 11.536914825439453, -2.7149007320404053, 4.439160346984863, -12.413204193115234, 5.821258544921875, 13.450096130371094, -7.85462760925293, 5.004929542541504, 2.944873332977295, -0.8884031772613525, -12.492597579956055, 5.223593711853027, 6.92197322845459, 0.9462087750434875, -9.519886016845703, 8.945523262023926, 3.187730550765991, -1.7627626657485962, 7.983493804931641, -5.540889739990234, 1.1966636180877686, 2.598850965499878, 4.178835868835449, -9.447782516479492, -1.0008028745651245, 1.0562258958816528, 2.795088052749634, 5.295595169067383, -1.2456873655319214, 9.07156753540039, -2.4767181873321533, -9.085067749023438, 3.1043946743011475, -6.744693756103516, -7.227990627288818, 20.590232849121094, 1.1877236366271973, 0.9998034834861755, 11.993304252624512, -9.525846481323242, -9.28317642211914, 20.62982749938965, -1.5028266906738281, -4.4167914390563965, 8.215929985046387, -21.518455505371094, 1.896612524986267, -3.8491673469543457, 10.26319408416748, 6.765595436096191, 7.722954273223877, 11.868877410888672, 3.5038585662841797, -9.111401557922363, 8.05947494506836, 6.542637825012207, 12.183818817138672, 5.978481769561768, 2.291829824447632, 17.093889236450195, -10.732231140136719, 5.985800743103027, -8.281452178955078, -11.071218490600586, 2.2741949558258057, -1.1123195886611938, -20.49468994140625, 7.117647647857666, -2.6293344497680664, 7.363972187042236, 15.751018524169922, -0.4913283586502075, -7.7067179679870605, 0.43100205063819885, -10.265456199645996, -4.60082483291626, 4.547321796417236, 1.5338993072509766, 1.5189967155456543, -10.335545539855957, 11.003397941589355, -36.69750213623047, -1.61932373046875, -3.4135231971740723, -11.252573013305664, 1.9284958839416504, -6.199658393859863, -11.842275619506836, -5.735298156738281, 6.224808692932129, 6.684168815612793, 4.766373157501221, -8.19234561920166, -5.164010047912598, 9.064750671386719, -7.741446018218994, 4.743967533111572, -6.51292085647583, 3.035581111907959, -1.6629304885864258, -5.654815673828125, 1.8262336254119873, 17.62242889404297, 5.244608402252197, 9.236296653747559, -1.1464056968688965, 4.651510238647461, 7.953803539276123, -9.682217597961426, 1.9628793001174927, -15.230489730834961, -2.300896644592285, 14.088359832763672, -2.6998605728149414, 9.608625411987305, -17.654659271240234, 9.100448608398438, 4.476500034332275, -4.073386192321777, -0.622489333152771, -10.732141494750977, 0.2124275118112564, -8.917216300964355, 0.7035949230194092, 19.972360610961914, -2.5762786865234375, -6.300234317779541, -1.6849857568740845, 9.389374732971191, 3.3887264728546143, -2.685488700866699, -9.998003005981445, -0.032197389751672745, -2.697265625, 8.369897842407227, -11.90698528289795, 14.586555480957031, 1.9877357482910156, 15.59173583984375, -13.060450553894043, -3.7160568237304688, 21.443119049072266, 6.561738014221191, -12.108722686767578, -9.430818557739258, -8.698579788208008, -18.267776489257812, 8.85710334777832, -4.679156303405762, -12.735084533691406, 2.4184470176696777, 9.333182334899902, 1.5015997886657715, -20.51285743713379, -16.75098991394043, -4.537631988525391, -5.7129292488098145, -3.2708330154418945, 1.2701241970062256, -1.8756260871887207, 1.479509949684143, -11.09266185760498, 8.17699146270752, -4.549189567565918, -3.924996852874756, 9.61779499053955, -3.1095824241638184, -15.945213317871094, 4.579936504364014, 14.858009338378906, 1.4220242500305176, 4.15861701965332, 1.945488691329956, 16.601165771484375, -10.700312614440918, -0.9465459585189819, 9.459413528442383, -7.285336494445801, 11.551774978637695, 7.045868873596191, -2.4443514347076416, 4.734280586242676, 10.93410587310791, -6.935035705566406, 7.209507942199707, -4.883551120758057, -4.658847808837891, 12.141715049743652, 3.7224292755126953, -14.73276138305664, -4.109946250915527, -2.3465917110443115, -2.905109405517578, -8.339864730834961, -1.7031090259552002, 11.986858367919922, 9.459665298461914, -13.075324058532715, 0.6066425442695618, 23.46194076538086, 7.4938459396362305, 7.813416004180908, 3.0492167472839355, 1.4138824939727783, 11.852439880371094, -19.14198875427246, 5.720455646514893, 4.455227851867676, -15.31979751586914, 3.544795036315918, -12.837488174438477], [1.0011664628982544, 4.2943034172058105, 14.702625274658203, -8.327272415161133, 5.243749618530273, 1.441731333732605, -5.670051574707031, -9.308947563171387, -10.623565673828125, -0.18394994735717773, 0.1373283565044403, -4.154370307922363, 12.20654010772705, 2.3568172454833984, -3.6896564960479736, -9.809220314025879, 6.360347270965576, -19.68508529663086, 1.4454896450042725, 17.455602645874023, -7.916691780090332, 4.344573020935059, -5.607848644256592, 2.765791177749634, 3.429342746734619, 3.339461088180542, -4.276838779449463, -5.796773433685303, 0.6585407853126526, 0.6907516717910767, -0.9394608736038208, 3.099609851837158, -3.421138286590576, -20.886991500854492, -1.8350660800933838, -5.785431385040283, 6.48973274230957, -30.266719818115234, 4.467088222503662, 2.8169987201690674, -10.071069717407227, -16.323772430419922, 1.3496885299682617, -4.839325428009033, -20.599895477294922, -15.259078979492188, -4.115743160247803, -9.935298919677734, 25.980493545532227, 10.714237213134766, -1.0216032266616821, 9.509790420532227, -3.127793312072754, 20.36590576171875, 9.830791473388672, -1.6386661529541016, -15.79617977142334, 5.044492721557617, -22.661090850830078, 7.145084857940674, -9.550128936767578, 26.76656723022461, -2.924359083175659, -7.616414546966553, 16.607784271240234, 1.2324447631835938, -1.009918212890625, -18.50407600402832, -7.199512004852295, 3.059664011001587, 15.963811874389648, 9.580208778381348, 2.3132100105285645, 1.2437800168991089, 9.328256607055664, 3.519010305404663, -3.3623433113098145, -15.125495910644531, 1.5178513526916504, -7.209586143493652, -7.4660868644714355, -3.5195932388305664, -11.355668067932129, 11.967631340026855, 2.9443206787109375, -0.8884971141815186, -18.108552932739258, 1.0025897026062012, 4.705737113952637, -18.420852661132812, -14.880657196044922, -3.3310458660125732, 10.366215705871582, 28.46524429321289, 12.024378776550293, -7.867338180541992, -0.34587177634239197, 23.555877685546875, 6.2747602462768555, 5.894920825958252, 4.643808364868164, -3.445729970932007, -22.66435432434082, -11.754472732543945, -6.8834614753723145, 5.239040851593018, 4.811223030090332, 2.195063829421997, -4.459115505218506, 12.94930648803711, 5.906621932983398, 1.978756308555603, -6.57578706741333, 1.600766658782959, 7.468903541564941, 8.36428451538086, -1.7388285398483276, 4.7256598472595215, -1.201965570449829, 7.183425426483154, -2.2449166774749756, 20.4794979095459, 16.591808319091797, -11.018237113952637, 2.989025354385376, 6.336806297302246, 8.99654769897461, -14.385148048400879, -8.955524444580078, 9.574433326721191, 3.6857450008392334, 15.087854385375977, 12.492429733276367, -7.843838691711426, 1.005283236503601, -4.3027262687683105, 4.3154401779174805, 18.67170524597168, -1.3737592697143555, 11.648097038269043, 4.791006565093994, 4.525938034057617, 1.4413954019546509, 0.8939042687416077, 8.140241622924805, -3.8050484657287598, 8.416245460510254, 2.05694317817688, -7.870660781860352, -9.954275131225586, 13.433685302734375, -14.428627014160156, 5.418674468994141, 35.710540771484375, -10.44961929321289, 5.073147296905518, -13.072733879089355, 6.545960903167725, 7.363007068634033, 3.400024175643921, 7.512924671173096, -9.706722259521484, 1.2335931062698364, -10.039941787719727, 6.74705696105957, -13.955547332763672, 5.207956314086914, 13.351619720458984, 13.605497360229492, 1.7836582660675049, 2.0677664279937744, -4.99813175201416, 1.7238028049468994, -8.542757034301758, -18.75617218017578, -5.556015491485596, 6.454176902770996, -11.102746963500977, 7.031160354614258, 21.826396942138672, 3.015354633331299, 2.4931821823120117, -19.897544860839844, -3.8240742683410645, -14.11462688446045, 6.578568935394287, -7.718748092651367, -0.45085614919662476, -6.718062400817871, -0.44342243671417236, -2.1347036361694336, 7.429747581481934, 22.763751983642578, 4.475053310394287, -7.4392170906066895, -4.541886329650879, -3.478916883468628, -12.575631141662598, 0.38847190141677856, -8.30168628692627, -1.7656431198120117, 2.7872300148010254, 19.397552490234375, -0.891141951084137, 11.25973892211914, 12.646049499511719, -10.478742599487305, -0.9891366362571716, -6.211699485778809, 15.990407943725586, 8.292285919189453, 5.039886951446533, 0.45210742950439453, -16.761606216430664, -3.032733201980591, -7.609503746032715, -0.17931726574897766, -12.369598388671875, -15.758055686950684, -10.761555671691895, 6.338927745819092, 5.469971656799316, 2.0471954345703125, 1.6163033246994019, -4.216435432434082, 7.057180404663086, -7.438172817230225, -1.4026014804840088, 6.851369857788086, 7.46417236328125, -9.391039848327637, -3.1287336349487305, 7.026418685913086, 1.3078274726867676, 3.6996958255767822, 8.617795944213867, 2.897815227508545, -9.556333541870117, -10.219422340393066, 1.9814388751983643, 15.853767395019531, -3.4663190841674805, 9.425997734069824, -1.4806697368621826, 16.918704986572266, -18.960176467895508, -12.502294540405273, 11.423295974731445, 11.018522262573242, -4.0460286140441895, 9.773102760314941, -3.690822124481201, -0.9118455648422241, 5.717279434204102, -0.5026708841323853, -3.5863614082336426, -14.723223686218262, -11.92502212524414, -7.6671247482299805, 15.23904037475586, -10.734562873840332, 6.39585542678833, -10.314704895019531, 4.206962585449219, -0.9262713193893433, -13.866442680358887, 9.716344833374023, -6.117427349090576, -13.242015838623047, -2.2971444129943848, 0.8353642225265503, 5.720811367034912, -10.10250473022461, -12.197799682617188, 14.141170501708984, 12.94390869140625, -0.3271155059337616, 5.568323135375977, 10.59363842010498, -1.9657268524169922, -5.558412551879883, 1.226067304611206, -2.7808051109313965, -3.365243673324585, 9.673866271972656, 12.699336051940918, 4.551826477050781, 8.737569808959961, -10.68564510345459, 5.0737690925598145, 7.767857551574707, 4.101675033569336, -15.888066291809082, -10.175168991088867, 5.058632850646973, -15.818835258483887, 2.7452924251556396, -0.739749550819397, 19.5765380859375, -6.512164115905762, -5.531241416931152, 12.072297096252441, 12.279394149780273, -10.943412780761719, -1.1741087436676025, 2.2873570919036865, -2.6582913398742676, 0.061007481068372726, 6.125863075256348, -22.210769653320312, 6.866125583648682, -1.0325466394424438, 1.9849056005477905, 8.451637268066406, 2.38742733001709, -2.5565221309661865, 15.896520614624023, -7.7423810958862305, -12.406805038452148, -27.202476501464844, 9.895909309387207, -9.325984954833984, 11.413458824157715, 9.283061027526855, 9.58759880065918, -9.819621086120605, 5.024166584014893, 15.422775268554688, -5.0563788414001465, -8.807815551757812, 1.257985234260559, -9.158782005310059, -4.578941345214844, -0.4150208532810211, -1.337777853012085, 2.602476119995117, -8.485060691833496, -8.160910606384277, -0.702039361000061, 11.198450088500977, 3.174729824066162, -25.135862350463867, 0.25722262263298035, -5.064250946044922, -5.280744552612305, -2.475930690765381, -4.077360153198242, -2.6546249389648438, 6.70642614364624, 12.5191068649292, -5.523533821105957, -1.0212255716323853, 14.616849899291992, -11.851070404052734, -2.113612651824951, 2.6235594749450684, -6.852779865264893, -4.039059162139893, 14.794652938842773, 5.81179666519165, -13.111681938171387, 29.89886474609375, 4.090696334838867, 4.0357818603515625, -4.177395820617676, 6.814570903778076, 5.225217342376709, -0.07518814504146576, 0.03909142687916756, 0.4383036196231842, -9.438523292541504, 9.070842742919922, -6.314816951751709, -17.933300018310547, -24.515913009643555, 1.3966994285583496, -18.402873992919922, -1.0751774311065674, 1.5373389720916748, -1.810228705406189, 15.273232460021973, -4.449548721313477, 0.33414262533187866, 4.351093292236328, 12.443727493286133, -15.70425796508789, 22.234804153442383, -4.430244445800781, 16.96260643005371, -8.374855041503906, -4.949112892150879, 4.292941570281982, -3.1154088973999023, 3.569965362548828, -2.841996192932129, 19.79199981689453, -8.388998985290527, 3.4815683364868164, 12.561229705810547, 3.124642848968506, -18.2451114654541, 12.450746536254883, -12.483760833740234, -8.567229270935059, 15.577240943908691, 15.108234405517578, -1.4106422662734985, 14.253585815429688, -15.074087142944336, 1.4997622966766357, -6.306988716125488, -12.796698570251465, -21.76607322692871, 12.516311645507812, -0.6269795298576355, 9.31704330444336, 5.6409759521484375, -1.7163276672363281, -0.4165904223918915, -15.581361770629883, -3.0232138633728027, 10.653637886047363, 15.032827377319336, -8.554729461669922, 16.068771362304688, -12.176041603088379, 2.0165460109710693, 1.7897484302520752, 9.565719604492188, -7.828795909881592, 7.94379997253418, -5.0328450202941895, -13.796319961547852, -7.11090612411499, -0.6304771900177002, 9.171321868896484, 6.520865440368652, -6.901420593261719, 5.110605239868164, 7.438300132751465, 8.044586181640625, 1.8750971555709839, -6.4786834716796875, -9.471653938293457, 2.6350607872009277, 6.0611982345581055, -9.018678665161133, -5.949237823486328, 13.706695556640625, 10.143108367919922, 5.095095157623291, 1.3579764366149902, -3.3628299236297607, -7.915380001068115, 10.357213973999023, -9.413320541381836, 19.271770477294922, 16.393531799316406, -10.489933013916016, -19.14827537536621, 0.658096969127655, -11.985315322875977, -6.017026424407959, 2.741669178009033, -14.39623737335205, 2.06363844871521, 12.651167869567871, 12.988670349121094, 10.399803161621094, -2.7114546298980713, -4.55896520614624, -5.629481315612793, 1.5104272365570068, 7.3324480056762695, -1.0778383016586304, -13.742773056030273, -13.115434646606445, -7.890486240386963, 0.41551798582077026, 6.205468654632568, -0.9629606604576111, 6.895302772521973, -1.6489851474761963, 1.9811434745788574, 4.163074970245361, -2.2510740756988525, 8.326912879943848, 13.939552307128906, 5.184896469116211, -3.692483425140381, 1.0984200239181519, 10.077363014221191, 1.0987567901611328, -2.5835747718811035, -5.468294620513916, -1.2715119123458862, -0.6453773379325867, -16.043298721313477, 6.35883092880249, -12.178367614746094, 9.079306602478027, -31.980953216552734, -0.6293895840644836, 10.74455738067627, 10.003727912902832, -15.773073196411133, -13.364974975585938, 4.3677191734313965, -12.415122985839844, 9.25002384185791, 14.821839332580566, -10.475194931030273, -11.931879043579102, -3.6909008026123047, 7.753500461578369, 5.539817810058594, 3.249100685119629, -14.184505462646484, 12.822731971740723, -0.046690016984939575, 6.032681465148926, 1.5818082094192505, 7.904952049255371, 12.213667869567871, 7.267633438110352, -28.321269989013672, -7.860369682312012, 6.788084030151367, -11.460514068603516, 21.107540130615234, -10.100327491760254, 1.7150018215179443, 5.4294657707214355, 8.599398612976074, 20.796525955200195, -18.895530700683594, 10.429698944091797, 6.825595855712891, 0.5723238587379456, -2.1801843643188477, -7.122612476348877, 8.743491172790527, -4.154279708862305, 6.6297526359558105, 21.73153305053711, 12.998477935791016, 8.070439338684082, 9.678642272949219, -31.57518768310547, 8.285265922546387, -4.215909957885742, -11.335564613342285, 8.626561164855957, 22.58392333984375, -9.750539779663086, 1.4135708808898926, -7.880410194396973, -4.639289855957031, 11.810359001159668, -4.785608768463135, -2.2155210971832275, -0.43245139718055725, -15.577011108398438, -4.350296974182129, 6.750743389129639, 7.087165832519531, -1.0475058555603027, 10.604087829589844, 13.379358291625977, -19.345935821533203, -2.990269899368286, 3.5460457801818848, 7.629205703735352, 0.745956301689148, 10.518058776855469, -4.707045555114746, 9.780787467956543, -8.356537818908691, -12.443586349487305, 14.831147193908691, 3.015627384185791, 0.4731608033180237, -13.913557052612305, 10.499329566955566, -2.79215669631958, 7.576263427734375, -3.2632181644439697, -3.783674478530884, -7.730087757110596, -6.089231491088867, -0.40562212467193604, 0.1208442896604538, -0.3567288815975189, -5.447304725646973, 1.750908374786377, 5.412106990814209, -6.320662021636963, 4.1267595291137695, 12.673624992370605, -25.755834579467773, 8.240842819213867, -13.155717849731445, 12.299833297729492, 1.543676733970642, 9.658055305480957, -6.594353199005127, -1.6436195373535156, -16.6529541015625, -2.3886265754699707, 2.1348955631256104, 6.992921352386475, -7.409975051879883, 3.3236262798309326, -5.835900783538818, 6.700926780700684, 9.494885444641113, -3.2508561611175537, -3.0514230728149414, 1.5145193338394165, -19.476030349731445, -9.386926651000977, 2.418046236038208, -4.4069085121154785, 11.846871376037598, -15.240554809570312, 8.13332462310791, -7.077491283416748, -13.096420288085938, 2.470531940460205, -7.8400726318359375, -13.62753677368164, 4.28726863861084, -10.568461418151855, -10.923477172851562, 2.9910266399383545, -7.682100296020508, -4.047544479370117, 20.33812713623047, 5.858068466186523, -1.1890610456466675, -25.653514862060547, -1.9485254287719727, -2.5466456413269043, 3.3897175788879395, -7.63947057723999, 15.697751998901367, -0.6773058176040649, -3.3010780811309814, 15.849489212036133, 3.972895622253418, 0.940939724445343, -5.75806188583374, 1.7854671478271484, -4.021643161773682, -10.63016128540039, 6.533995628356934, 3.3675217628479004, 5.535937309265137, 3.3243417739868164, -0.8814805746078491, -8.456031799316406, 1.8156697750091553, -4.454656600952148, 12.578858375549316, 17.948122024536133, -16.514001846313477, -0.09800122678279877, -4.5359392166137695, 2.2765588760375977, 16.694847106933594, 8.010807991027832, -19.887088775634766, 10.64410400390625, -5.793825149536133, 2.8619062900543213, -6.842186450958252, -7.135735511779785, -6.5685625076293945, 1.3550338745117188, 15.515236854553223, -3.7732484340667725, -4.239421844482422, -11.016143798828125, -3.0393431186676025, 1.0249189138412476, -6.1315484046936035, 6.477965354919434, -6.09017276763916, -1.1254146099090576, 15.386551856994629, 22.870346069335938, 23.096017837524414, -6.230463981628418, -3.9926578998565674, -10.55660343170166, -8.873588562011719, -2.4583566188812256, 10.739141464233398, 12.981420516967773, -6.41448450088501, -2.7590954303741455, 9.691437721252441, 11.746907234191895, 3.147879123687744, -6.586940765380859, 3.047468662261963, 18.401836395263672, 18.427812576293945, 0.09706955403089523, -11.680347442626953, -4.23945951461792, -15.920166015625, 0.96329265832901, -2.9102392196655273, -3.594059705734253, 5.6113128662109375, -3.5251200199127197, 10.541773796081543, -1.1899046897888184, 2.091057538986206, 10.719432830810547, -12.67754077911377, 19.192089080810547, -1.4953019618988037, -9.295442581176758, 4.217721939086914, 20.541200637817383, -4.070459365844727, 7.580109596252441, 13.60969352722168, 14.426338195800781, 4.251387596130371, -10.833999633789062, -3.6936073303222656, 14.95560073852539, -12.1671781539917, 12.433353424072266, 2.9314141273498535, 4.9387898445129395, 2.2439870834350586, -8.1334228515625, 23.87679672241211, -12.903157234191895, -1.4571363925933838, -13.850232124328613, 3.586580276489258, 5.089611530303955, -25.7413387298584, 4.266404628753662, -1.6117143630981445, 8.088007926940918, -0.03324505686759949, -10.225214004516602, 0.033943962305784225, -6.833864212036133, 7.561312675476074, 14.37222957611084, 15.090620040893555, -1.6209231615066528, -9.428776741027832, -0.8155035376548767, 12.673778533935547, 3.3283746242523193, 11.842281341552734, -1.0741931200027466, 0.13634087145328522, 3.3880958557128906, 8.829096794128418, 41.707740783691406, 16.52594757080078, -1.7065125703811646, 4.026254177093506, -5.3307342529296875, 9.67951488494873, 6.339507102966309, -2.2740695476531982, 1.8733105659484863, -3.3369944095611572, -3.8241794109344482, 12.357474327087402, -0.1505403369665146, -7.053835868835449, -10.916975975036621, -8.016772270202637, -9.379642486572266, 8.396048545837402, -7.221827507019043, 2.2993485927581787, 0.9229159951210022, 5.342927932739258, -11.099428176879883, -6.471750736236572, 8.144948959350586, 6.558026313781738, -7.246293544769287, 15.047754287719727, 2.8812315464019775, -1.0928593873977661, 2.2712812423706055, -9.939226150512695, 2.5337796211242676, 1.3082964420318604, 25.21567153930664, -3.6716439723968506, 0.36504581570625305, -16.707582473754883, -14.650660514831543, -2.0332236289978027, 19.120046615600586, -5.372013568878174, 6.291032314300537, 0.2245960384607315, 4.962411880493164, 11.009799003601074, -2.118352174758911, 2.8491604328155518, -1.5281736850738525, -19.43448829650879, 4.659384727478027, 0.9828895330429077, 0.027758989483118057, 8.73379898071289, -4.031735420227051, 14.53529167175293, -0.6743910312652588, -8.64138412475586, 3.7992193698883057, 19.777801513671875, -5.8923187255859375, 1.8058171272277832, -16.290843963623047, 12.861993789672852, -10.620708465576172, -10.770665168762207, -1.6011022329330444, -8.072815895080566, -1.5937919616699219, -11.64299488067627, -5.436157703399658, 1.7869658470153809, 10.195327758789062, -9.070866584777832, -14.976458549499512, -5.009026527404785, 1.2399871349334717, -2.513451337814331, 2.7012827396392822, -5.523824214935303, 1.9421157836914062, 11.220560073852539, -10.181873321533203, -1.9389369487762451, -0.042741499841213226, -4.497631549835205, -6.157995223999023, -24.810457229614258, -6.9032721519470215, -0.022427385672926903, 1.0871001482009888, 16.15306854248047, -14.054773330688477, -17.980031967163086, -0.16983425617218018, -4.98403263092041, 4.52894401550293, -7.933023452758789, -10.992746353149414, -1.8227667808532715, -5.4554219245910645, 9.068720817565918, -6.6853861808776855, -3.9360311031341553, -2.366781711578369, -10.692968368530273, -18.69544792175293, -1.2961552143096924, -0.3454574942588806, -8.513452529907227, -1.1295591592788696, -11.676105499267578, -0.771488606929779, -7.205549240112305, -12.702692031860352, 5.374324798583984, 12.5498046875, 0.11566705256700516, 5.104114532470703, 19.010194778442383, 4.329902172088623, 16.16928482055664, -1.5509026050567627, -7.910445213317871, 18.626300811767578, -15.925674438476562, -13.929924964904785, 5.603170871734619, 2.0340416431427, -12.090991973876953, 9.680398941040039, 0.41087883710861206, -5.327016830444336, 0.6921893358230591, -0.7373170852661133, -6.936557292938232, 1.0018188953399658, -8.071434020996094, 2.931497573852539, 14.833799362182617, 7.999205112457275, 3.535144805908203, -0.056789666414260864, 22.89128303527832, -4.362571716308594, -12.583284378051758, 4.7268452644348145, -1.8166512250900269, 3.0049684047698975, -6.589200019836426, -20.10564613342285, 0.3463236093521118, 11.475215911865234, -14.030447959899902, 3.1219425201416016, 7.957686424255371, 1.7787121534347534, 11.90646743774414, 0.980087161064148, -17.193851470947266, -13.022943496704102, -14.101119041442871, 6.6079230308532715, -0.4039260149002075, 3.7575249671936035, -3.8071985244750977, -13.548425674438477, 1.110276222229004, 6.873037815093994, 9.045049667358398, -4.210652828216553, 11.976358413696289, 1.9486470222473145, -1.4543081521987915, 17.49736785888672, 1.274308204650879, -2.991844654083252, 6.614889621734619, 8.638103485107422, 0.8984789848327637, -0.4859517812728882, -16.917638778686523, 1.4699934720993042, -15.96113395690918, -14.399496078491211, -0.40086066722869873, -9.797101020812988, -10.094139099121094, -7.065870761871338, -3.413741111755371, -0.24730658531188965, -11.239112854003906, 3.797872304916382, 8.85879898071289, 1.3916130065917969, -4.557418346405029, -0.984912633895874, 5.546941757202148, -6.934060096740723, -0.009824913926422596, -16.42853546142578, -1.3340510129928589, 1.2524805068969727, 17.824499130249023, 13.18934440612793, 13.781328201293945, -9.781222343444824, -6.891760349273682, 11.628499031066895, 2.4854788780212402, -6.845176696777344, -18.480972290039062, 5.480370998382568, -3.5533976554870605, -10.428506851196289, 5.118258476257324, 2.0210471153259277, 7.2883806228637695, 4.917799472808838, -4.398996829986572, -10.991921424865723, 7.622363090515137, 11.38694953918457, 1.3003590106964111, -3.1980278491973877, 7.195038795471191, 3.8206539154052734, 8.331611633300781, -4.161907196044922, -0.10831766575574875, -11.437889099121094, -11.556257247924805, -8.95272159576416, 0.9165737628936768, 4.7861127853393555, 20.596729278564453, -6.028733253479004, -6.108744144439697, 11.11656379699707, -15.069631576538086, -9.969111442565918, 3.075442314147949, 1.9313688278198242, -9.452230453491211, 14.518860816955566, 13.315670013427734, -10.074136734008789, -7.659940242767334, -14.304591178894043, 1.7289215326309204, -2.258867025375366, -4.291237831115723, 24.40578269958496, 1.8718559741973877, -14.295001029968262, -8.14332389831543, 1.3984471559524536, -0.6311147212982178, 6.870515823364258, 3.8837785720825195, 13.24653434753418, 15.3304443359375, 11.479520797729492, 12.68959903717041, -13.19672679901123, -2.387356758117676, 5.62998104095459, 12.188214302062988, -8.385172843933105, 12.885854721069336, -9.800838470458984, 1.0740950107574463, 13.657487869262695, -4.09822416305542, 3.795713186264038, -13.089942932128906, 2.284740447998047, 6.860539436340332, 3.458421468734741, 0.9616255760192871, -5.358816146850586, -13.379571914672852, 2.2644665241241455, -8.697368621826172, -2.1594367027282715, -6.148974418640137, 10.4555082321167, 9.692415237426758, 15.361184120178223, 4.182155609130859, -13.419488906860352, -1.9909281730651855, -6.475430011749268, -15.672846794128418, -2.072573184967041, -11.71030044555664, 21.266372680664062, 26.27475357055664, -6.6708664894104, 5.69497013092041, -2.049407720565796, 0.946202278137207, -0.03064396232366562, 4.0366315841674805, -2.2638163566589355, 6.937456130981445, -20.14499282836914, -8.703878402709961, 12.811808586120605, 1.6900469064712524, 1.7032585144042969, 1.3338477611541748, -14.173722267150879, -6.91362190246582, 7.68295955657959, 3.407627582550049, -16.07588005065918, 16.484663009643555, 14.502047538757324, -9.796623229980469, 4.87415885925293, -13.558825492858887, -13.322962760925293, -18.853628158569336, 16.226306915283203, 13.094033241271973, 15.574273109436035, -17.20543670654297, 6.877455234527588, -9.340206146240234, 19.64704704284668, 14.952593803405762, 10.980502128601074, 15.573202133178711, 8.304784774780273, 6.179929733276367, -2.6066651344299316, -0.15717414021492004, 3.136141061782837, -4.540107727050781, -12.396121978759766, 3.661902904510498, -11.536444664001465, 17.660476684570312, 19.3768367767334, 9.820996284484863, -1.2694694995880127, 6.782298564910889, 5.729949951171875, 13.409311294555664, -18.49939727783203, 19.2203426361084, 10.54969310760498, 7.270383358001709, -4.02446174621582, -8.79057788848877, 7.503838062286377, -3.751221179962158, -22.77924346923828, -9.100898742675781, -1.566143274307251, -16.23052215576172, -14.087150573730469, 5.7016143798828125, -8.672846794128418, -1.9803733825683594, 8.06237506866455, -5.9426374435424805, -1.4924346208572388, 12.702020645141602, 15.69688606262207, -22.0280818939209, 5.735405921936035, 12.542388916015625, -0.3201252222061157, -0.39503857493400574, -7.2702765464782715, 13.772493362426758, -3.118156909942627, 12.840962409973145, 10.07326602935791, -10.67651653289795, 4.247948169708252, -7.389342308044434, 6.661430358886719, -5.5111846923828125, -8.935089111328125, 12.317312240600586, 3.221346139907837, 3.1495940685272217, 9.015828132629395, 8.23416805267334, 1.2544217109680176, 0.3101491928100586, 1.4125475883483887, 3.4127488136291504, 2.3286304473876953, 16.468814849853516, 0.30660271644592285, -0.41650229692459106, 20.692224502563477, -6.827373027801514, 5.566685199737549, -16.644298553466797, -20.583498001098633, -9.335203170776367, 3.9558398723602295, 0.2512464225292206, 17.063461303710938, 1.1935844421386719, 8.240291595458984, -1.469175934791565, -4.888973712921143, -3.539854049682617, -6.776296615600586, 7.991244316101074, -9.187921524047852, -3.9764740467071533, -2.4473934173583984, -9.402539253234863, 16.584320068359375, 14.629664421081543, 2.565795421600342, 8.114580154418945, -18.26368522644043, -1.1016172170639038, 13.895779609680176, 0.3967157304286957, -10.272665023803711, -10.002054214477539, 10.020805358886719, -7.845001697540283, 9.703226089477539, -15.564298629760742, 4.506742477416992, 6.126296043395996, 2.365417003631592, -0.3984302580356598, 6.786625862121582, -4.350921630859375, 2.6092028617858887, 2.430004835128784, 4.108833312988281, -6.506424903869629, 10.000615119934082, -18.37602424621582, -10.714892387390137, 3.8750076293945312, -1.7963241338729858, -13.239755630493164, 0.424510657787323, -11.469731330871582, -5.4660725593566895, 6.4949564933776855, 8.462916374206543, 7.488128662109375, -10.349616050720215, -2.2312753200531006, 18.80453109741211, -6.44711446762085, 11.970516204833984, -3.5185790061950684, -3.181724786758423, -9.048015594482422, -5.979887962341309, -3.826902151107788, -9.82804012298584, -4.307531356811523, 17.796890258789062, -11.221331596374512, -13.665958404541016, -19.846378326416016, -23.04294776916504, 8.940991401672363, 7.70713996887207, 8.369100570678711, 6.332118511199951, 13.2802152633667, 7.108899116516113, 9.174135208129883, 13.210782051086426, 12.609418869018555, 10.749828338623047, 7.789896011352539, -7.304010391235352, -4.772518157958984, 6.164392471313477, 13.769434928894043, -16.565509796142578, 15.64736557006836, 5.659041404724121, 11.978808403015137, -5.892796039581299, -1.8859964609146118, -6.263275146484375, 4.825348377227783, -4.082566261291504, -13.209895133972168, -0.9348500967025757, 0.49980515241622925, 3.5834107398986816, -4.660025119781494, 3.1298484802246094, 6.828549385070801, 17.5782413482666, 2.0701417922973633, -2.354353904724121, 3.498133420944214, -12.705991744995117, -19.426115036010742, 6.180757522583008, 5.951085090637207, 0.4311440587043762, -13.794734954833984, 11.10890007019043, -7.148586273193359, 19.665332794189453, 8.672181129455566, 2.490596055984497, 7.044482231140137, -1.9090006351470947, -2.214930534362793, -1.4337468147277832, -5.497999668121338, -5.508369445800781, -6.356513977050781, 1.1332805156707764, -27.90158462524414, -12.981525421142578, -3.8343029022216797, 6.334260940551758, -3.421051502227783, 1.085623025894165, -20.659313201904297, 10.059504508972168, -1.7266573905944824, -7.571349620819092, -1.0867159366607666, 10.996511459350586, 6.329337120056152, 8.264418601989746, 11.284460067749023, -5.6816534996032715, 10.786054611206055, -20.164310455322266, -8.55752182006836, 6.784197807312012, 8.864760398864746, -9.386719703674316, 2.960949182510376, 9.720952987670898, -0.6491400003433228, 3.842409133911133, 4.449416637420654, 13.799137115478516, -5.559676170349121, -12.116127014160156, 15.696415901184082, 5.184920310974121, -18.160930633544922, 2.3282198905944824, -11.30322265625, -21.639060974121094, 22.181608200073242, 1.3290268182754517, 12.00912094116211, -17.188430786132812, 2.789783477783203, -3.453221321105957, -1.2970393896102905, 15.683989524841309, -7.881669044494629, -20.556276321411133, 9.144593238830566, -13.580488204956055, 11.420049667358398, 10.908726692199707, -15.257804870605469, 6.077724456787109, 2.5444040298461914, -22.574771881103516, -13.324450492858887, -0.24571238458156586, 2.498443603515625, -0.7063808441162109, 8.857178688049316, -13.668325424194336, 3.333465576171875, 0.28100213408470154, -11.572784423828125, -8.981011390686035, 4.508780479431152, 7.886513710021973, 3.8600850105285645, 6.922606945037842, 13.121282577514648, 1.4562095403671265, -3.8410840034484863, -19.96247673034668, 15.644926071166992, -4.205902099609375, 3.164565086364746, -0.4929920732975006, -2.6209828853607178, -3.8383235931396484, -1.8740055561065674, 0.6721839308738708, 1.1522893905639648, -18.491662979125977, 9.993083000183105, -7.648932933807373, 1.4850603342056274, -0.721580982208252, 12.370464324951172, 0.9223146438598633, 1.08246648311615, -16.764827728271484, -2.438699960708618, 9.554642677307129, -1.555334448814392, 13.887578964233398, 0.17747604846954346, 11.512818336486816, 15.289472579956055, 8.860114097595215, -8.273694038391113, 20.94384002685547, 3.54339599609375, -8.238313674926758, 19.28933334350586, -6.058420181274414, -1.0495028495788574, 11.837343215942383, 6.110076427459717, -8.519540786743164, 26.661813735961914, -13.06967830657959, -7.32328987121582, 5.29308557510376, 20.142887115478516, -18.31292724609375, -8.653584480285645, 0.6868692636489868, 4.836298942565918, -2.7911524772644043, 3.6645779609680176, -5.679132461547852, -5.6606621742248535, -16.14521026611328, 7.0822978019714355, -8.47113037109375, 2.130063056945801, 18.472314834594727, 19.8984375, 2.8748483657836914, -0.39755627512931824, -9.783323287963867, 1.0688085556030273, 1.0635992288589478, -13.729333877563477, -3.525085926055908, 3.8495631217956543, 5.050685882568359, -8.546062469482422, -4.804742813110352, -9.371772766113281, 7.3037333488464355, -7.084360122680664, 4.666515350341797, -0.8020729422569275, 7.11711311340332, 20.966827392578125, -8.628888130187988, 2.4801807403564453, -2.0851526260375977, 15.062323570251465, 6.836960792541504, -1.004189372062683, -2.4046435356140137, 16.480478286743164, 21.80366325378418, -14.331742286682129, -1.1351604461669922, -11.178092956542969, -1.3763176202774048, -0.4123879671096802, -6.192481994628906, 12.379056930541992, -6.19788122177124, 7.56259822845459, -17.252347946166992, -7.294090270996094, 2.5105652809143066, 9.036367416381836, -5.621562957763672, 10.406951904296875, 3.281351089477539, -13.755483627319336, 20.588848114013672, -10.034969329833984, 18.29319190979004, -13.312379837036133, 10.98570442199707, 10.802069664001465, 9.756990432739258, -1.9169986248016357, 7.820345878601074, -23.73904800415039, 1.2255356311798096, -0.7542782425880432, -3.50780987739563, 17.44838523864746, 7.356935024261475, 6.776459693908691, 4.559332370758057, 5.504540920257568, 6.699918746948242, -14.652517318725586, 0.11759816110134125, 16.055667877197266, 0.7300403714179993, -6.551684856414795, -0.492870956659317, 1.011993169784546, 1.7834315299987793, 4.014559745788574, 5.264632225036621, 8.616015434265137, -8.877126693725586, -4.836510181427002, -11.24463939666748, -1.4871816635131836, 10.996366500854492, 6.939728736877441, -9.63578987121582, -14.774092674255371, -10.026350021362305, 9.725872039794922, 10.550166130065918, 14.37520980834961, -18.885787963867188, -23.107269287109375, 16.481369018554688, -1.511951208114624, 0.7370592355728149, 1.3803045749664307, -4.659588813781738, 4.186490058898926, -16.246803283691406, -18.388368606567383, -9.989316940307617, -9.325739860534668, 1.9319349527359009, -5.474113464355469, -14.853185653686523, 4.330742359161377, -15.985515594482422, 5.447478771209717, 12.906858444213867, 10.03793716430664, -7.349147319793701, 9.36895751953125, -12.219954490661621, 3.382997512817383, -12.184894561767578, -2.8808999061584473, -12.72158432006836, 4.766139030456543, -2.8957595825195312, -6.068986892700195, 0.7252219319343567, -4.3798627853393555, 2.84277081489563, -11.749638557434082, 11.062446594238281, -13.036141395568848, -2.539890766143799, 14.638652801513672, 1.745046615600586, -16.640077590942383, -11.213582992553711, -0.2522367537021637, -12.087142944335938, -7.209600448608398, -2.661050796508789, -0.7533038854598999, -7.246253967285156, 1.7528138160705566, -1.5128707885742188, 9.840071678161621, -15.120453834533691, -14.229360580444336, -14.235973358154297, -16.177019119262695, -10.484827995300293, -14.210786819458008, -6.0156354904174805, -13.934502601623535, 21.81686019897461, -9.144619941711426, 6.513039588928223, -6.2831501960754395, -9.440909385681152, -12.873454093933105, -0.9661520719528198, -3.914673089981079, -13.455589294433594, 0.8963514566421509, 0.29955607652664185, -1.832084059715271, -12.321294784545898, -0.1773732602596283, 7.302512168884277, -3.1297457218170166, -11.026832580566406, -9.397650718688965, 22.592147827148438, -1.665144681930542, -2.988018751144409, 2.1008429527282715, -7.209349155426025, -12.939337730407715, 0.6766600608825684, 2.932656764984131, 6.753701210021973, 0.7043557167053223, 0.5344133377075195, 1.763790249824524, 6.682568550109863, -1.5027234554290771, -14.084662437438965, -4.222869396209717, 10.448196411132812, 5.845725059509277, 5.66246223449707, -4.414167881011963, 0.9369438886642456, -0.6673505902290344, -2.248112916946411, 4.683722496032715, -2.836456298828125, 5.296311378479004, 1.4662115573883057, -3.4047274589538574, 0.8164504170417786, -5.589393615722656, -7.418948173522949, 1.2875754833221436, 10.024650573730469, -20.19866180419922, -1.2772679328918457, 17.27252960205078, 0.8836112022399902, 8.136356353759766, -10.503569602966309, -4.964229583740234, 8.774202346801758, 10.495889663696289, -5.908756732940674, -17.296260833740234, -3.5911033153533936, -6.038277626037598, -2.7920331954956055, -14.972352981567383, -10.482802391052246, 17.77362632751465, -15.801010131835938, 2.6251957416534424, 6.573124885559082, 1.6754554510116577, -6.434907913208008, -14.27425765991211, -13.188281059265137, 3.483445405960083, -2.025911808013916, -9.244730949401855, 1.663881778717041, 12.612557411193848, -3.5899875164031982, 3.6578938961029053, 1.5096889734268188, 6.472180366516113, 3.5411453247070312, 11.519659042358398, -13.033184051513672, -0.9947347044944763, 7.157830238342285, -16.576807022094727, 5.501819133758545, -17.237499237060547, 5.903668403625488, 11.73194694519043, -9.29754638671875, 15.350951194763184, -10.608392715454102, 8.507740020751953, -4.482237339019775, -6.911506175994873, -17.631513595581055, -17.620927810668945, 3.164034128189087, 2.0349864959716797, 27.279756546020508, -17.934242248535156, 1.5854253768920898, 9.282154083251953, 0.5097302794456482, 7.6564435958862305, -19.10854148864746, -5.804103851318359, -7.4168548583984375, 11.652745246887207, -4.641699314117432, -2.3867015838623047, 1.4594333171844482, -6.264754295349121, -6.175531387329102, -16.03432846069336, 14.414148330688477, -6.865100860595703, 16.031339645385742, 2.3374927043914795, 6.969333648681641, 4.852411270141602, 13.641075134277344, -17.300870895385742, 22.67388343811035, 6.127819538116455, -11.156085968017578, 6.394622325897217, -8.209090232849121, 8.665484428405762, -5.230679035186768, 2.3033084869384766, 5.6273298263549805, -23.398534774780273, 8.820091247558594, -0.8548462390899658, 3.3347365856170654, -1.0067888498306274, 3.538036823272705, 1.935152530670166, -5.560791492462158, -2.4156436920166016, 15.112038612365723, -1.8828438520431519, -8.900681495666504, 4.144107818603516, -9.143494606018066, -21.811161041259766, 33.938392639160156, 1.4750146865844727, -3.06355619430542, 14.052165985107422, 21.908935546875, -5.627828598022461, 2.8689489364624023, 7.794651508331299, -10.7481050491333, 2.3524811267852783, 14.259662628173828, 14.009465217590332, 8.39249038696289, 6.080151557922363, 5.430807113647461, -4.127132892608643, -3.2503068447113037, -17.556283950805664, 6.110682964324951, -3.5044219493865967, -1.9031624794006348, -5.252199172973633, 3.783139228820801, 14.239602088928223, -5.862061977386475, -15.85915470123291, -6.78926420211792, -13.519087791442871, -14.293556213378906, 0.7991362810134888, 6.841415882110596, -14.760763168334961, 7.162862300872803, -4.118101119995117, 10.976814270019531, 5.620666027069092, 10.319786071777344, 14.20780086517334, -8.228801727294922, 9.499580383300781, 9.123860359191895, 7.188465118408203, -7.995142936706543, -8.91131591796875, 0.36549878120422363, -1.6917455196380615, 11.600353240966797, 0.28629231452941895, -11.756001472473145, 3.734623908996582, 0.625737190246582, -3.1478004455566406, 3.373490333557129, -0.21820597350597382, -1.3119679689407349, -10.912169456481934, 16.964218139648438, 3.9890589714050293, -2.5559825897216797, -11.319254875183105, 5.671520233154297, 1.851447582244873, -7.72327995300293, 1.3766448497772217, -4.5390849113464355, 6.792637825012207, -24.005281448364258, 1.0245119333267212, 1.5127032995224, -14.204477310180664, 4.030904293060303, -7.631515979766846, -7.962899208068848, 2.158282995223999, 2.2619268894195557, -3.429853677749634, 8.561540603637695, 4.557863235473633, -4.978862762451172, 9.90050220489502, -3.8066749572753906, -22.88710594177246, 27.336181640625, -8.88061809539795, 6.321139812469482, -5.303668975830078, 1.5778963565826416, 1.9933110475540161, 11.43234634399414, 14.047821044921875, -12.31356430053711, 13.737771987915039, 4.331423759460449, 9.019118309020996, -7.137007236480713, -4.278514862060547, -0.5456771850585938, 6.3645782470703125, -6.906898498535156, 3.464700222015381, 8.65246868133545, 3.6194639205932617, -5.127215385437012, -4.63215970993042, -9.824600219726562, 6.492757797241211, -2.27276873588562, -6.521385192871094, 3.502378225326538, 3.446800470352173, 6.590548038482666, -21.663299560546875, 12.63869857788086, 8.635093688964844, 4.734312534332275, -8.943594932556152, -7.396295070648193, -11.900124549865723, -7.204791069030762, 0.05212852358818054, -28.469097137451172, -6.688967704772949, 13.381482124328613, -9.707925796508789, -6.1216325759887695, 11.02346420288086, -5.691534996032715, 4.232987403869629, -5.410416126251221, 3.2682301998138428, -12.036304473876953, -1.7112470865249634, 8.59337329864502, -5.932280540466309, 13.327555656433105, -5.6580281257629395, 1.5611255168914795, 9.301170349121094, 10.787712097167969, -0.9370366334915161, -0.10109403729438782, -11.02917194366455, 4.47385311126709, 28.807109832763672, -5.674271583557129, -2.7811596393585205, -0.037964362651109695, 3.6295933723449707, -9.483800888061523, -8.66447639465332, 18.13287925720215, 16.644641876220703, 10.176438331604004, -6.397182941436768, -0.6695504188537598, 11.799394607543945, -1.3212859630584717, 13.919487953186035, -6.551723957061768, -1.7272839546203613, 5.323310852050781, 18.193241119384766, -10.993739128112793, 12.457626342773438, -16.97991180419922, -8.099251747131348, -13.344629287719727, -2.7530713081359863, 0.49942654371261597, -7.62932825088501, 5.622247695922852, -16.202966690063477, -7.7726640701293945, 1.7249765396118164, 2.943937301635742, -0.039783868938684464, 4.790954113006592, 14.320552825927734, 2.9284005165100098, 15.274048805236816, 10.38177490234375, -6.442742347717285, -2.310584306716919, -10.073695182800293, -4.116089344024658, 1.8412643671035767, -8.98396110534668, -17.90384864807129, -5.533302307128906, 1.525134563446045, -15.907011985778809, 4.857623100280762, -8.373848915100098, -4.699560642242432, -3.5388686656951904, -8.849699020385742, -10.884603500366211, 6.899763107299805, 0.44273555278778076, 7.3859429359436035, 4.375699520111084, -9.22275161743164, 10.692514419555664, -1.8583765029907227, 3.2781002521514893, 17.91753387451172, 4.275241851806641, 1.2592273950576782, 0.15451154112815857, 1.9726042747497559, 3.8098368644714355, -13.65963077545166, -1.1547203063964844, -8.834360122680664, -14.016753196716309, 18.871219635009766, 6.439140319824219, -5.121800422668457, -5.303940296173096, -10.878682136535645, 5.390311241149902, -25.408260345458984, -20.141332626342773, -3.613067865371704, 6.12222957611084, 0.1822499930858612, -11.062259674072266, -1.3366351127624512, -4.765768527984619, -2.7420644760131836, 12.122758865356445, 8.339522361755371, 2.6819825172424316, 0.34708914160728455, 1.3805007934570312, 3.951537847518921, -0.4419916868209839, 13.566373825073242, -13.759075164794922, -2.823699474334717, -16.06218147277832, 13.19148063659668, -12.848804473876953, 7.3986101150512695, 6.174567222595215, -6.8880720138549805, 5.465970516204834, 0.15723204612731934, 1.5945820808410645, -0.550469696521759, -3.9356112480163574, -13.92166519165039, 12.550250053405762, -10.98033332824707, 5.244916915893555, 16.56805419921875, -6.575295925140381, 5.529762268066406, 12.240276336669922, -5.073311805725098, 7.330499172210693, -5.1409196853637695, 1.1233932971954346, -5.96110725402832, -6.937877655029297, -2.270442485809326, 0.06799422949552536, 0.5405570268630981, 6.829813003540039, 9.573389053344727, -8.182198524475098, -10.565509796142578, -3.262751340866089, -6.870477199554443, 7.577486991882324, -13.126029968261719, 7.5602006912231445, -5.979591369628906, -4.637110710144043, -3.7764346599578857, 2.777632713317871, -2.808117628097534, -1.4865610599517822, -4.1380414962768555, -8.229520797729492, -3.8502612113952637, -5.381767749786377, -10.607767105102539, 24.874876022338867, 1.4840306043624878, 10.24666690826416, -0.3940466642379761, 8.759261131286621, 2.9510045051574707, -12.267232894897461, 11.86158275604248, -21.66876792907715, 9.795689582824707, 9.903533935546875, 0.4273093342781067, -2.118684768676758, 7.128157615661621, 2.6799097061157227, -4.297630310058594, -18.5607967376709, 3.317870855331421, 3.5097696781158447, -14.624134063720703, -9.474322319030762, -10.998468399047852, -8.5524263381958, -5.913875579833984, -7.960333824157715, 9.576166152954102, 2.7268481254577637, -3.8216946125030518, -8.138360977172852, -9.313102722167969, -0.3565533757209778, 16.99431800842285, 6.987077713012695, -9.474505424499512, 4.8901753425598145, 4.005840301513672, -1.1651928424835205, -0.533048152923584, -15.975835800170898, -9.911795616149902, 10.578280448913574, -7.534797191619873, 7.344696521759033, 7.845107078552246, 13.529111862182617, 5.185449123382568, -9.84153938293457, -3.8130736351013184, 4.609853267669678, 1.553687572479248, 3.7853450775146484, 1.73359215259552, -4.010821342468262, -2.644890785217285, -11.797560691833496, -15.858888626098633, 3.6734108924865723, 7.4169816970825195, 14.146713256835938, -13.587278366088867, 2.3657987117767334, -16.29553985595703, -3.3112683296203613, -23.548200607299805, 11.950703620910645, 3.6681885719299316, -7.008868217468262, -2.907284736633301, -5.375080585479736, 12.012247085571289, -0.42535436153411865, -2.040459394454956, 7.063319206237793, -11.58481502532959, 4.570306301116943, -2.3484482765197754, -3.7630395889282227, 1.123836874961853, -7.199809551239014, -4.4387078285217285, 9.712679862976074, 5.13693380355835, -23.26066780090332, -8.629128456115723, 11.819389343261719, -3.467419385910034, 4.671836853027344, 8.752079963684082, 9.490407943725586, 7.569970607757568, 1.8298499584197998, -5.528735637664795, -2.9172215461730957, 23.40628433227539, 11.045924186706543, 13.283756256103516, -8.550630569458008, -3.240492105484009, -4.659804821014404, -7.906982898712158, -13.449931144714355, 5.255217552185059, -5.030848979949951, -12.666412353515625, -6.136305332183838, 7.962069988250732, -1.3631409406661987, -0.9415533542633057, -16.64307403564453, -1.1158467531204224, 11.550525665283203, 1.725547432899475, 4.394476890563965, -25.448402404785156, -0.99919593334198, -14.158327102661133, -2.9784932136535645, -6.810060501098633, -9.796630859375, -5.625572204589844, -4.404397010803223, 9.962968826293945, 0.6309143304824829, -7.1646928787231445, -13.406320571899414, 6.168773651123047, -8.543142318725586, 23.207134246826172, 16.686824798583984, 18.766357421875, 4.139987468719482, 11.401531219482422, -16.588176727294922, 5.204349040985107, -1.0621553659439087, -0.5812878608703613, -9.02174186706543, -0.4036194384098053, 19.068429946899414, 7.451406002044678, 1.1376041173934937, 6.471999168395996, 4.204289436340332, -9.215381622314453, 3.8443515300750732, 10.907861709594727, -25.212671279907227, -13.860902786254883, 20.938968658447266, 8.221138000488281, -5.247900485992432, -13.252574920654297, 0.4413840174674988, -1.1126728057861328, 16.850502014160156, -17.050159454345703, -6.287781715393066, 14.897308349609375, 15.84901237487793, 0.3501438796520233, -6.3669633865356445, 0.0752146989107132, -0.6779062747955322, 3.397820472717285, -2.3809428215026855, 1.5723353624343872, -4.244404315948486, 5.489540100097656, 8.869830131530762, 3.407505750656128, 3.119554042816162, 9.638310432434082, 1.6481056213378906, 12.191390037536621, -13.527337074279785, -8.735648155212402, 11.363276481628418, 9.630173683166504, 7.4553728103637695, 9.328146934509277, -0.7790177464485168, 9.93465518951416, 6.57684326171875, -15.935033798217773, -5.869803428649902, 0.49518224596977234, -29.932727813720703, 6.902217864990234, 1.7646210193634033, -7.199382781982422, -4.453434467315674, 5.0774688720703125, 8.027532577514648, -13.267041206359863, 6.00773286819458, -2.5355868339538574, -2.067913770675659, -9.796167373657227, 3.560718297958374, -15.164918899536133, -0.8559714555740356, 2.1829745769500732, -18.656261444091797, 6.635776519775391, 12.819046974182129, 17.968852996826172, 5.274399757385254, -3.4426844120025635, -8.801840782165527, -2.7169528007507324, 5.488787651062012, -7.506486892700195, 2.669929027557373, 9.72577953338623, 0.9159111976623535, 2.8797051906585693, 3.4826228618621826, 8.972855567932129, 8.47268295288086, 4.111931324005127, -6.910150051116943, 7.795854568481445, 2.9255313873291016, 8.341218948364258, 15.712091445922852, -5.762199401855469, -2.977872371673584, 7.156876564025879, -0.6012744307518005, -1.5016294717788696, 9.542844772338867, -8.4122314453125, -8.99065113067627, -8.9440336227417, 11.009803771972656, -7.280087471008301, 1.8208537101745605, 0.41408929228782654, -8.854836463928223, 12.290143013000488, 4.132220268249512, 11.287627220153809, 0.6261541843414307, 7.442195892333984, 0.376480370759964, -17.9178409576416, 8.971707344055176, 0.10580446571111679, 0.32150089740753174, 3.3880221843719482, 4.304774284362793, 17.053007125854492, 0.027367547154426575, 6.122731685638428, 15.200671195983887, 1.7355824708938599, 6.277790546417236, 8.435051918029785, -11.361490249633789, 14.097437858581543, -11.881690979003906, -9.368961334228516, -3.3251867294311523, 0.8353659510612488, 0.6815505027770996, -16.137374877929688, -0.021263737231492996, 15.886743545532227, 2.4744632244110107, -5.086005210876465, 2.337036609649658, 7.095052719116211, 4.243005752563477, -4.435074329376221, 21.175622940063477, 4.108420372009277, 4.017712593078613, 18.04861068725586, 4.261712074279785, 5.903937339782715, 6.5692877769470215, -5.8341064453125, 1.4490559101104736, -6.8839616775512695, -2.243037462234497, -10.086044311523438, -1.4147312641143799, 18.99884796142578, -9.107349395751953, 9.266693115234375, 9.19047737121582, 0.660463809967041, -13.559088706970215, 6.4433393478393555, -4.617003917694092, 7.604074954986572, -6.0941290855407715, -9.98200798034668, -21.766366958618164, -0.27098262310028076, 5.504039764404297, 5.9030890464782715, 4.985902786254883, -14.183975219726562, 0.2295227199792862, 16.50870132446289, 8.803656578063965, -2.1056432723999023, 10.006206512451172, 13.456950187683105, 0.776116132736206, -14.55836296081543, -13.759220123291016, 6.132476806640625, -4.826224327087402, 1.0761966705322266, -12.33461856842041, -3.570617198944092, -7.087979316711426, -16.48764991760254, 4.9660868644714355, 6.1767144203186035, -4.905933856964111, -10.786591529846191, -8.428622245788574, 4.026928424835205, -16.240617752075195, 8.332377433776855, -4.302807807922363, -5.335612773895264, -4.790290832519531, 9.551468849182129, 10.533994674682617, 2.4146411418914795, -14.303949356079102, 4.0062336921691895, -7.23545503616333, -16.52634048461914, -7.255928993225098, -13.58203411102295, 3.1954245567321777, 11.81889533996582, 2.8593969345092773, 18.904064178466797, -2.5297486782073975, 4.221009731292725, -6.64530086517334, -2.3727526664733887, 6.222914695739746, 12.818227767944336, -10.557653427124023, 0.8047264814376831, 7.69017219543457, -6.097769260406494, 7.872203826904297, -6.509210586547852, 3.028273582458496, 9.783183097839355, -1.7725298404693604, -12.444904327392578, 3.873643636703491, -17.345678329467773, -22.582290649414062, 1.8653666973114014, -3.151257038116455, -2.3682737350463867, 7.036528587341309, 12.02964973449707, 26.845233917236328, 8.019271850585938, -2.2328240871429443, -7.629619598388672, 3.7907309532165527, -7.0794243812561035, 15.103135108947754, -0.05296584591269493, 9.735541343688965, -0.2509314715862274, -4.982847690582275, -17.524812698364258, -8.5906343460083, -2.8596208095550537, 4.238198280334473, -19.5545654296875, 17.5466365814209, 10.091086387634277, 16.76654815673828, -11.190160751342773, -0.18540814518928528, -7.242264270782471, 0.7168588042259216, 2.91353702545166, 7.917174816131592, 15.801839828491211, -30.468416213989258, 10.034290313720703, 0.2258792370557785, -9.394699096679688, 6.260994911193848, 6.529858589172363, 5.541050434112549, -13.40035629272461, 22.582969665527344, 10.826109886169434, -18.786579132080078, 0.807794451713562, 4.3328986167907715, 19.925405502319336, 10.230437278747559, 14.289870262145996, 13.047304153442383, 2.67877459526062, -23.424880981445312, 6.649309158325195, -12.896726608276367, 14.634623527526855, 0.09599290043115616, -11.428021430969238, 2.5424158573150635, 7.928426265716553, -14.976293563842773, -9.994913101196289, 15.121006965637207, 5.4601850509643555, -20.218069076538086, -4.449643135070801, 7.47114896774292, -16.48294448852539, -13.34321403503418, 0.6192627549171448, -1.3461310863494873, -20.088560104370117, -10.29232406616211, 0.01595642790198326, -0.8880898952484131, -2.023923635482788, 8.637748718261719, 3.368983268737793, 8.596988677978516, -4.997419357299805, 0.4056466817855835, 6.979432106018066, 3.4017465114593506, -0.1546674370765686, 11.873616218566895, 21.83397674560547, -17.663169860839844, -5.3840179443359375, -3.0155677795410156, -1.4666616916656494, 14.620619773864746, 10.117911338806152, 3.012655258178711, 12.59430980682373, 2.228278636932373, 8.847305297851562, 8.341177940368652, -12.620594024658203, -24.374706268310547, 1.6363294124603271, 0.5708488821983337, 4.901628017425537, 4.9055633544921875, -2.8937034606933594, 18.376314163208008, -8.45576286315918, -1.5307071208953857, -2.286956548690796, -11.309832572937012, 13.275896072387695, -9.243109703063965, 23.487518310546875, -8.972578048706055, -1.252718210220337, -5.0265793800354, 4.198665142059326, 13.062450408935547, -2.3195157051086426, 3.9240097999572754, 0.1531873494386673, 1.8922816514968872, -3.639816999435425, 4.6361308097839355, -10.072481155395508, 15.943904876708984, 8.88453483581543, 1.9578876495361328, 15.660987854003906, -2.6517691612243652, -5.0215654373168945, -0.06588424742221832, 17.176992416381836, -8.297229766845703, -0.5235801935195923], [-14.500327110290527, -7.849574565887451, -9.452073097229004, -29.51045799255371, 0.2063635289669037, -9.739053726196289, 0.057000257074832916, 8.620946884155273, 17.67742347717285, -3.1394457817077637, -7.948550701141357, 1.3396515846252441, 4.142878532409668, -7.486001014709473, 1.021344542503357, -2.5614583492279053, -7.103565216064453, -0.6288098692893982, 20.411693572998047, -0.6993030309677124, 10.841046333312988, -3.0888726711273193, -11.462841033935547, -11.11483383178711, -7.599196434020996, 2.2666330337524414, -4.541720867156982, 2.747299909591675, -1.5850526094436646, -8.673395156860352, 5.498588562011719, 6.5294694900512695, -10.540580749511719, -0.9522430896759033, 0.9269619584083557, 7.10808801651001, 3.8691246509552, -1.8124463558197021, -0.8979387283325195, 16.81186294555664, -5.404560089111328, -6.435876846313477, 3.709887981414795, -2.542278289794922, 6.933196067810059, 6.108715057373047, 12.127586364746094, -6.132597923278809, 9.195793151855469, -7.756262302398682, 3.909234046936035, 10.043511390686035, -8.474732398986816, 5.004474639892578, -10.989564895629883, -5.20970344543457, 2.5862395763397217, 8.053133010864258, -7.7728800773620605, 3.5975427627563477, 2.420823335647583, -7.768065452575684, -13.96292781829834, 3.0993807315826416, -2.324275016784668, -1.6627521514892578, -5.226373672485352, -4.403231620788574, 9.95407485961914, -13.977272033691406, 6.170380115509033, 11.071686744689941, 5.363109111785889, -0.7370325922966003, 20.664785385131836, 0.948483943939209, -7.741079807281494, 7.788479328155518, -9.559150695800781, 3.282625198364258, -15.74963092803955, 16.23670196533203, -17.520183563232422, -1.7101106643676758, 26.202075958251953, 1.183741569519043, -21.57926368713379, -10.914507865905762, 12.388076782226562, -6.935062408447266, 20.102130889892578, -26.497119903564453, -11.305610656738281, -6.241959571838379, 1.5746058225631714, 3.0433261394500732, -10.313798904418945, -0.40505367517471313, 17.23836898803711, 6.849331378936768, -6.341365814208984, 14.975013732910156, -2.2741074562072754, 8.62200927734375, -7.730632305145264, 0.8411476612091064, 4.835114002227783, -19.07488250732422, 8.759779930114746, -2.5962841510772705, 19.22516632080078, 3.882575273513794, -11.837349891662598, -12.105841636657715, 8.425249099731445, 5.261622428894043, 15.527262687683105, 12.040351867675781, -12.86160659790039, 2.4761860370635986, -6.570310115814209, -13.160859107971191, -4.3121724128723145, 5.84132194519043, 5.045370101928711, -1.2289972305297852, -10.176313400268555, 3.632387638092041, -4.808516979217529, -11.125480651855469, -9.114131927490234, -10.269627571105957, 2.7255148887634277, -0.8965734243392944, 2.365967273712158, -8.651862144470215, 14.470473289489746, 14.81161880493164, -14.609393119812012, -14.421709060668945, -4.260366439819336, -19.328388214111328, -9.840311050415039, -18.78338623046875, 12.160970687866211, -11.213951110839844, -2.532562732696533, 5.119934558868408, 4.4805707931518555, 1.8885756731033325, 12.454937934875488, 40.13080596923828, 6.325798034667969, -5.672990798950195, 1.2198206186294556, -5.358275890350342, 4.904065132141113, -14.36271858215332, 9.28551959991455, 10.459427833557129, -0.9045416116714478, 1.4420866966247559, -6.526298522949219, -10.491758346557617, 7.668862819671631, -8.671156883239746, 14.186363220214844, 1.3549352884292603, 2.189215898513794, -2.4343478679656982, 7.173867702484131, -1.2289050817489624, 9.038690567016602, 7.192561626434326, 2.7944512367248535, -8.82020378112793, -8.359580039978027, 1.534740924835205, 10.8350830078125, -1.7509784698486328, -9.09310531616211, 16.25408363342285, -4.907837867736816, 28.258798599243164, 1.5106472969055176, 1.518615484237671, 10.785001754760742, -12.798669815063477, -4.870733261108398, 4.45387077331543, 11.496658325195312, 15.94570255279541, 4.748228073120117, -4.521258354187012, 7.180761337280273, -16.4780216217041, -11.516341209411621, -3.7450952529907227, 13.267894744873047, -0.33266812562942505, -15.388275146484375, -8.606513023376465, 15.824710845947266, -6.0198588371276855, -8.524779319763184, 7.411221981048584, -0.5399022102355957, 21.070249557495117, -6.823632717132568, 8.040815353393555, -9.478578567504883, 2.4243738651275635, -2.102379322052002, 9.268108367919922, 7.023834705352783, -2.05576491355896, 15.976319313049316, -8.456245422363281, 7.106854438781738, 4.806798934936523, -4.930474281311035, -2.982177257537842, -5.506387233734131, -1.897659420967102, 4.443239688873291, -0.6309697031974792, 3.2980597019195557, 2.730822801589966, 18.625789642333984, 3.7935426235198975, -4.307397365570068, -5.488824844360352, 3.6960432529449463, -1.4349974393844604, 7.979140281677246, -11.231451988220215, 4.422545433044434, 12.967957496643066, 10.60253620147705, -3.2920899391174316, 17.121458053588867, 21.763639450073242, 11.542197227478027, 10.255058288574219, 1.6940512657165527, -4.9852705001831055, -7.5124664306640625, 6.19841194152832, -14.402178764343262, -7.846592903137207, 6.102827072143555, 7.69425630569458, 12.832878112792969, 19.524600982666016, 6.453686237335205, -6.952019214630127, -6.546527862548828, -1.6321707963943481, -16.096559524536133, -22.114452362060547, 1.047875165939331, 11.24912166595459, -4.494197845458984, -6.351711273193359, -2.348802089691162, 15.5849027633667, 10.108497619628906, 13.9524507522583, -6.859469890594482, -21.640748977661133, 2.5206613540649414, 1.255871057510376, -5.6246161460876465, 1.561430811882019, -4.369632720947266, -4.513607501983643, 10.173958778381348, -13.49687385559082, 19.923721313476562, 2.0368311405181885, -7.802553653717041, 4.818881511688232, -1.559389591217041, 0.40525782108306885, -8.99384880065918, 6.348199367523193, -6.755357265472412, 0.9720409512519836, 15.706342697143555, 13.31775188446045, 11.849782943725586, -2.876164436340332, 8.728538513183594, -25.92685317993164, -14.457598686218262, 7.007638931274414, 2.6847121715545654, 5.613677024841309, -20.152597427368164, -10.712651252746582, 6.708102703094482, 3.4059886932373047, 6.929318428039551, 12.707633018493652, -2.301300525665283, 7.546482086181641, -4.165575981140137, -17.279483795166016, 6.420813083648682, 1.100540041923523, 6.2717437744140625, 3.198117971420288, 6.1658220291137695, -3.7772321701049805, 10.989652633666992, -10.163461685180664, -8.401164054870605, 0.09305423498153687, 4.619599342346191, -0.771182119846344, -8.972322463989258, -0.8229435682296753, 6.997673511505127, 5.501228332519531, -7.420323371887207, -7.941885948181152, -1.3133926391601562, 0.03512677177786827, -1.9700038433074951, -1.4690985679626465, 3.995234966278076, 9.753767967224121, 15.491589546203613, -14.32847785949707, -6.949469566345215, 5.8846306800842285, 4.339820384979248, 6.124691963195801, 3.3444976806640625, -10.870407104492188, 8.386279106140137, 4.041003704071045, -4.963497638702393, 14.349054336547852, 7.857872009277344, -11.808229446411133, -19.128381729125977, 14.50589370727539, -1.716957688331604, 0.7547143697738647, 1.598848581314087, 4.729764461517334, 3.505406618118286, -7.450797080993652, -11.233041763305664, 6.67929744720459, -4.307697772979736, 13.6143798828125, -1.3184843063354492, -1.7237108945846558, -2.3345961570739746, 1.6324437856674194, -8.051833152770996, -10.139838218688965, 1.096195936203003, -9.774857521057129, 8.323115348815918, -7.584817409515381, -16.491987228393555, 7.520444393157959, 2.621384620666504, 8.045014381408691, -6.4744720458984375, -10.935800552368164, -7.718071937561035, 3.414803981781006, -5.386960506439209, 2.872802734375, -2.6268813610076904, 19.79990005493164, -12.018633842468262, 9.248467445373535, -15.589468002319336, 6.725587368011475, 1.5405446290969849, 10.870394706726074, -16.950424194335938, 6.7052154541015625, -3.12203311920166, -13.738532066345215, -4.856273651123047, 17.623573303222656, 9.098637580871582, 0.5324791669845581, -11.451322555541992, -1.7719876766204834, 8.34144401550293, 10.328239440917969, 15.289844512939453, 11.345331192016602, 2.3946990966796875, 10.887575149536133, -1.1668612957000732, -2.043449640274048, 4.345734119415283, -20.294044494628906, -8.419107437133789, 7.887368202209473, -12.154470443725586, 3.436490297317505, -2.336045265197754, 1.9596869945526123, -20.49817657470703, -14.692651748657227, -1.1617262363433838, -5.503916263580322, 1.792991042137146, 2.047041893005371, 8.323627471923828, -2.4977784156799316, 5.944952964782715, 2.48222017288208, 10.218348503112793, 10.532764434814453, 16.05858612060547, -12.804689407348633, 2.2565672397613525, 13.388443946838379, -10.650218963623047, -0.4217998683452606, 0.3922345042228699, 9.443788528442383, 1.355555534362793, -10.372766494750977, -5.389645099639893, -11.51060676574707, -0.7454991340637207, 2.423332452774048, -8.313953399658203, -15.332484245300293, 14.819798469543457, 0.8585103750228882, 6.861478805541992, -6.796723365783691, -11.60140609741211, -6.983283042907715, 2.9504384994506836, 3.764692783355713, 2.2355082035064697, -10.780282020568848, -12.068095207214355, -12.256926536560059, 5.89721155166626, -15.426675796508789, -7.520771026611328, 3.3051905632019043, -13.173437118530273, -17.976577758789062, 0.8570270538330078, -1.7897109985351562, 14.359859466552734, -2.420543670654297, 17.26736831665039, -10.986871719360352, 5.6866912841796875, -17.100561141967773, -6.380743026733398, -1.0000540018081665, 11.280766487121582, 5.855703830718994, -2.619654893875122, -13.018546104431152, -17.788135528564453, 1.7491573095321655, -8.524712562561035, 14.415933609008789, -20.918773651123047, -2.36460542678833, -0.5950588583946228, -3.024885892868042, -15.353818893432617, 0.8637269735336304, 1.0236430168151855, 1.9407451152801514, -2.185965061187744, 5.890491008758545, 9.204203605651855, 1.1760658025741577, -4.928175926208496, -3.0952212810516357, 6.06395959854126, -5.286895751953125, 3.9009313583374023, 22.483123779296875, -6.908834457397461, -6.942226409912109, 1.2133517265319824, -13.76705265045166, -0.14468979835510254, 15.433065414428711, 1.5589290857315063, -14.522363662719727, 14.254398345947266, 3.9673945903778076, -5.118285179138184, 1.6024529933929443, -1.0099369287490845, -0.8398265838623047, -6.791004657745361, -13.317533493041992, 19.01449966430664, -0.9863990545272827, 0.9274476766586304, 4.84254789352417, 1.2869619131088257, 15.529291152954102, 5.629726409912109, -11.695040702819824, -7.930989742279053, -12.13779354095459, 8.628103256225586, -10.586838722229004, 2.1690330505371094, 10.018354415893555, -1.57212233543396, -9.329854965209961, 3.4386415481567383, 3.3499207496643066, 7.891324996948242, 7.431282043457031, -4.584344387054443, 11.487885475158691, -14.824739456176758, -7.073509693145752, 2.8552463054656982, -4.824115753173828, -0.34651613235473633, -9.285820960998535, -5.458449363708496, 6.355199337005615, -1.3695811033248901, -20.088863372802734, -10.776978492736816, -2.5771780014038086, -1.4203606843948364, -2.0238442420959473, -2.2626945972442627, 9.895919799804688, -0.7574462890625, -14.756202697753906, 5.475092887878418, -2.644718647003174, -0.5573604106903076, 2.9624404907226562, -11.134244918823242, -3.259415864944458, -12.625551223754883, 9.06511402130127, -8.931845664978027, 0.385059118270874, 2.2442708015441895, 14.345917701721191, 1.9805994033813477, -8.892155647277832, 18.44918441772461, 1.705930233001709, -12.343205451965332, 7.522089004516602, -4.257559299468994, -0.24999411404132843, -6.62705659866333, 2.448132276535034, -13.271482467651367, -1.8275883197784424, 12.177993774414062, 13.549223899841309, 9.626862525939941, 14.237615585327148, 15.717775344848633, -14.760437965393066, -1.1542537212371826, 3.2932088375091553, -21.099939346313477, -14.869478225708008, 9.922107696533203, 3.109562635421753, 4.175014495849609, 11.151474952697754, 15.594827651977539, -3.6556122303009033, -10.782313346862793, 9.405304908752441, 8.540050506591797, -0.3832608461380005, 3.5550642013549805, 13.69655990600586, 12.701720237731934, 13.691627502441406, -9.289701461791992, 0.2304503470659256, 1.717740774154663, 20.995681762695312, 12.495445251464844, -6.200113773345947, 10.017682075500488, -11.054306030273438, 3.3544788360595703, 0.544352650642395, 0.11201829463243484, -13.517473220825195, 0.33014023303985596, -5.8175201416015625, 4.926477432250977, 5.557538032531738, 24.751436233520508, -11.214252471923828, 3.5365524291992188, -13.438769340515137, -5.128479957580566, 11.412054061889648, 6.369150161743164, 14.88718032836914, -11.557920455932617, 14.413381576538086, -6.593227863311768, 20.504940032958984, 12.877689361572266, 4.165363788604736, -12.557960510253906, -9.239221572875977, 13.096105575561523, 6.619464874267578, -10.548891067504883, -8.204267501831055, -10.158309936523438, 16.583215713500977, 3.6736345291137695, -6.444647789001465, 1.1813263893127441, -6.886354446411133, 20.42903709411621, 2.3874175548553467, -2.3947417736053467, 23.420927047729492, -0.33396977186203003, -11.73364543914795, -1.9156126976013184, 3.505185842514038, 2.7123894691467285, 11.528106689453125, -15.082293510437012, 14.184663772583008, 2.1631393432617188, -5.684335708618164, 10.235862731933594, -10.296124458312988, 12.303343772888184, 5.210563659667969, 11.890642166137695, -7.031917095184326, -11.425586700439453, -4.408247470855713, 15.856133460998535, 3.320788860321045, -11.927680969238281, -7.049129486083984, 16.327823638916016, 16.92662811279297, -0.9808776378631592, -9.535972595214844, -6.457090377807617, 1.9308505058288574, 9.397334098815918, 5.377728462219238, -0.4401942491531372, 15.880192756652832, 12.928471565246582, -1.1232603788375854, 8.576200485229492, 19.366426467895508, 30.45846176147461, -13.927423477172852, -14.655750274658203, -0.1525726020336151, -4.344883918762207, -13.207826614379883, -14.262347221374512, -10.773581504821777, 19.594486236572266, -5.512815475463867, 6.411225318908691, 2.426288604736328, -4.790944576263428, 16.4164981842041, 3.3808743953704834, -2.973607063293457, 5.837060928344727, -2.465097427368164, -2.5342979431152344, -12.566765785217285, 10.354257583618164, -2.2441699504852295, 7.330100059509277, 6.861142635345459, 0.9007105231285095, 4.543299198150635, 8.484140396118164, 0.8751713633537292, 11.406584739685059, 17.208017349243164, -8.699320793151855, -3.756500244140625, 6.639163017272949, -8.071599960327148, 4.185702323913574, -11.546839714050293, 5.393799781799316, -16.4218692779541, -20.651775360107422, -4.675196647644043, 13.860325813293457, -3.6685924530029297, 20.021053314208984, 4.096773147583008, 16.072280883789062, 4.398391246795654, -15.579273223876953, 20.04091453552246, -5.143452167510986, 14.506982803344727, -4.160928249359131, -10.745731353759766, -1.7103701829910278, -8.504220962524414, 11.443832397460938, -8.981010437011719, -4.953944683074951, -2.5369632244110107, 8.51826286315918, -1.2707033157348633, 13.464040756225586, -3.6969900131225586, -1.2222976684570312, 7.270888805389404, 8.983519554138184, -6.024820804595947, -2.779193878173828, 9.029684066772461, 13.259310722351074, 8.503854751586914, 15.169538497924805, -18.541948318481445, -4.299439430236816, -11.823076248168945, -7.876863479614258, 8.374015808105469, 15.754825592041016, 2.8181843757629395, 6.181008815765381, 9.667268753051758, 11.060972213745117, 2.1127853393554688, -8.052299499511719, 7.130105972290039, -5.4135236740112305, -5.559939384460449, -21.13072395324707, -3.027007579803467, 14.757472038269043, -1.9802534580230713, 2.025907278060913, 11.096555709838867, -6.933817386627197, -7.212822437286377, 1.2377686500549316, 7.3618927001953125, -5.840884208679199, -17.169179916381836, 1.6819292306900024, -1.5808736085891724, -15.05774974822998, -6.808094024658203, -2.237654685974121, 2.1445579528808594, -7.867266654968262, 3.636894464492798, -1.029637336730957, -13.785116195678711, 2.746413230895996, -3.432403087615967, 6.75802755355835, 13.047260284423828, 3.1722073554992676, -0.8725973963737488, -7.0604424476623535, 6.129922389984131, 7.316315650939941, 5.290212631225586, -3.1423192024230957, 3.162454128265381, 8.89725399017334, 14.79143238067627, 22.905391693115234, -9.971259117126465, 4.777868747711182, -19.36547088623047, 4.059574127197266, 2.3335728645324707, -4.6096906661987305, 8.026931762695312, -0.7646832466125488, 10.550783157348633, -8.053775787353516, 3.7970759868621826, -9.680054664611816, -10.950712203979492, 2.583955764770508, 3.725813865661621, 19.984455108642578, -18.13072395324707, -18.50628662109375, 24.218034744262695, -5.842977523803711, 3.4815332889556885, 4.260348320007324, -14.654544830322266, -8.669228553771973, 1.768748164176941, 7.218661308288574, -14.035751342773438, 9.988971710205078, -7.004817962646484, 5.8140764236450195, -10.861502647399902, 14.94622802734375, 3.9255034923553467, 10.506385803222656, -4.227684497833252, 17.711605072021484, 0.5063429474830627, 2.4989778995513916, 10.99803352355957, -2.516798496246338, -3.8813817501068115, -17.974178314208984, -7.53846549987793, -8.512101173400879, -3.5536468029022217, -6.630448818206787, 6.694693088531494, -8.124728202819824, -3.9372518062591553, 15.15976333618164, 2.053231954574585, 12.639692306518555, -1.9198764562606812, 7.730661392211914, -8.757501602172852, 9.325859069824219, 21.833486557006836, -22.943260192871094, 16.898822784423828, 14.067188262939453, 0.9157989025115967, -15.702123641967773, 3.7106027603149414, 8.10595417022705, 7.263533592224121, -6.361218452453613, 8.584955215454102, -19.37003517150879, 5.444636344909668, 24.08684730529785, -4.033097267150879, -6.086775779724121, -1.7452795505523682, 4.271798610687256, -7.144554138183594, -8.644432067871094, 5.836742401123047, -2.8940927982330322, -1.9669814109802246, -4.776143550872803, 1.1450495719909668, 15.375946998596191, -8.566762924194336, 8.717784881591797, -0.5982562303543091, 1.7742358446121216, 6.01785945892334, 15.944262504577637, 4.340704441070557, -4.449689865112305, -3.893857955932617, 4.767791748046875, -3.931110382080078, 17.2288875579834, 1.3818938732147217, -14.341964721679688, -0.715639054775238, 19.255998611450195, 6.330785751342773, 7.940108776092529, -9.657227516174316, -3.6790874004364014, -10.28192138671875, 11.491565704345703, -3.6007914543151855, -0.48814499378204346, -6.689183712005615, 12.120988845825195, 0.12138888239860535, -1.3902578353881836, 0.1259748488664627, 9.158517837524414, -15.093896865844727, -6.170113563537598, -4.857269287109375, 15.228614807128906, -4.044612884521484, 9.217690467834473, -2.0081326961517334, 11.800634384155273, -0.7538847923278809, -0.09702415764331818, -0.9573037624359131, 5.3567891120910645, -11.754661560058594, 19.3328914642334, 0.07766570150852203, 0.12027962505817413, -14.888161659240723, -11.91718578338623, -2.1971561908721924, 8.420352935791016, -2.7780089378356934, -14.00133991241455, 1.6762275695800781, 5.340906143188477, -10.636998176574707, -6.377039909362793, 22.89624786376953, 1.441714882850647, -0.35148417949676514, 3.780963897705078, -11.55870532989502, -34.08435821533203, 7.375326633453369, -11.548215866088867, -3.4029083251953125, 9.972684860229492, 4.857550144195557, -5.1845383644104, -5.3465962409973145, -3.1468141078948975, 5.517919540405273, -0.5205531716346741, -4.593956470489502, 2.704572916030884, 26.742237091064453, 13.860749244689941, 1.3666821718215942, 2.4017796516418457, -12.357932090759277, -1.6766022443771362, -2.284252882003784, -10.888177871704102, 11.213723182678223, -1.4061665534973145, -5.289128303527832, -3.3069581985473633, 13.134899139404297, 8.749788284301758, 7.789573669433594, -3.1999006271362305, 3.905691146850586, -8.694819450378418, 0.001226111315190792, 3.280000925064087, 14.551223754882812, 11.553717613220215, 16.87457847595215, 2.775590181350708, -2.2652082443237305, 1.6087435483932495, 2.5592756271362305, 15.687050819396973, -2.10672664642334, 3.1772878170013428, -3.410412311553955, -7.41732120513916, -10.462644577026367, -22.463550567626953, -5.978362560272217, -6.179428577423096, 1.1988060474395752, -10.518044471740723, -11.941452026367188, 9.327835083007812, -11.34809398651123, -17.57384490966797, 14.977107048034668, 7.342419624328613, -6.168608665466309, -6.525983810424805, -24.195356369018555, 8.93658447265625, -10.308631896972656, 6.772939682006836, -15.313642501831055, -21.178760528564453, 5.513062477111816, 14.044393539428711, 11.401313781738281, 21.908227920532227, 2.9042932987213135, -3.91654372215271, -5.298369407653809, -0.35931500792503357, 24.505496978759766, 9.809391975402832, -10.96116828918457, 23.34415626525879, -1.8253607749938965, -12.249713897705078, 4.12929630279541, -15.111251831054688, -4.36124849319458, 1.570097804069519, 10.668561935424805, -2.9088246822357178, 15.97764778137207, -0.4034161865711212, -7.759593963623047, 1.5242440700531006, -12.048225402832031, -2.871459484100342, 0.829215943813324, -4.759183406829834, -2.158446788787842, -22.218549728393555, -12.204011917114258, -33.149070739746094, 4.4867353439331055, -16.995323181152344, -3.439340591430664, -11.131742477416992, -11.791717529296875, 5.935744762420654, -4.447556972503662, -3.970353364944458, 12.28042221069336, 25.9896240234375, 10.85629940032959, -10.298815727233887, 1.2826037406921387, -9.301777839660645, 0.6899968385696411, 11.113690376281738, 3.717200756072998, 18.046539306640625, 6.52160120010376, 6.5393781661987305, 11.20103645324707, -7.304079055786133, 4.5531816482543945, 20.941139221191406, 13.715578079223633, -15.431242942810059, 0.1431589275598526, 1.1795505285263062, 2.2557036876678467, -7.969454765319824, -16.861087799072266, 3.9294145107269287, -28.708736419677734, 4.087893962860107, 8.287996292114258, -5.664830207824707, 3.005635976791382, 5.194842338562012, 8.690397262573242, -0.4757143557071686, -4.499985694885254, -14.158287048339844, 23.981006622314453, -8.884910583496094, -9.398504257202148, -0.839735746383667, -2.8885746002197266, 4.393625259399414, 12.376045227050781, 11.015863418579102, 5.7948102951049805, -9.864371299743652, -6.467619895935059, -25.619102478027344, -0.5763381123542786, 3.501331329345703, -8.277366638183594, 11.343111038208008, -14.242152214050293, -0.8928701877593994, -23.000850677490234, 7.143965721130371, -6.684640884399414, -14.217920303344727, 9.589957237243652, 4.913238525390625, 14.794268608093262, -9.355181694030762, 6.103878021240234, -9.877010345458984, 0.0168780367821455, -6.760408878326416, -14.384833335876465, -20.280406951904297, 4.486350059509277, -21.255369186401367, -14.716886520385742, 2.515486240386963, 1.9889085292816162, 0.37639319896698, 5.65679931640625, 2.8995859622955322, -1.4898040294647217, -3.7268333435058594, -2.9350218772888184, 8.920907020568848, 11.059555053710938, -11.999698638916016, 12.699246406555176, -4.808415412902832, 13.764504432678223, 18.4534912109375, 5.051112174987793, 2.064101457595825, -5.767452239990234, -0.7820584774017334, 0.4323164224624634, -17.56766128540039, -9.767560958862305, -19.749921798706055, -9.04980754852295, 1.3474440574645996, -14.136913299560547, 14.464969635009766, 1.9535540342330933, 10.179798126220703, 4.458274841308594, -5.307268142700195, -10.40672492980957, 16.886123657226562, 6.62050199508667, -11.30622673034668, -4.067016124725342, 20.37668228149414, 10.91275405883789, -1.5816091299057007, 9.02636432647705, -6.914519309997559, 10.182929992675781, -0.3903595805168152, -10.298452377319336, -5.883878707885742, 1.4125233888626099, -12.590906143188477, 19.78398895263672, 12.68278980255127, -26.02128791809082, -12.577493667602539, -0.208171084523201, 14.315156936645508, 4.2974772453308105, -2.6745076179504395, -1.1017886400222778, -12.904735565185547, 5.3861517906188965, 6.317122459411621, -2.7999587059020996, 13.500797271728516, 16.93155288696289, 7.601624011993408, -10.875946044921875, -8.397232055664062, 7.868375778198242, 5.209502696990967, 20.53215217590332, 4.507565975189209, 9.178339958190918, -2.7333295345306396, 11.792762756347656, -0.9810045957565308, -4.727283477783203, 9.035168647766113, -3.1229875087738037, -5.177748203277588, -10.948777198791504, -14.770551681518555, 18.05147933959961, 16.324779510498047, -7.503206729888916, 6.977734088897705, -12.161722183227539, 10.207242965698242, 6.81044340133667, -13.507621765136719, 13.550653457641602, 2.2703709602355957, -18.137798309326172, -1.5155922174453735, 14.40640640258789, 1.3657034635543823, 1.4506573677062988, -10.248547554016113, 3.8382911682128906, -19.44312286376953, -18.43970489501953, -10.51611328125, -5.727174758911133, -18.121356964111328, 10.552444458007812, -8.6801176071167, 2.98738956451416, -8.93591594696045, 11.426430702209473, -17.8500919342041, 5.246086597442627, 9.416617393493652, -5.587574005126953, -7.053901672363281, 7.097386837005615, -15.143884658813477, -4.670632362365723, -1.8022115230560303, -4.50828742980957, -1.1042258739471436, -0.12006793171167374, 3.802190065383911, 2.2735753059387207, -1.5424097776412964, -8.59920883178711, 1.399720311164856, 4.972781181335449, -17.608652114868164, 3.2625393867492676, -1.837813377380371, 6.601188659667969, 22.315319061279297, 2.721266746520996, -11.105392456054688, 12.690372467041016, -15.245896339416504, -0.6289775371551514, 1.2930927276611328, -12.90890121459961, 12.292322158813477, -13.354169845581055, 23.610809326171875, -10.324188232421875, 3.2800676822662354, -5.898286819458008, -14.81657886505127, -8.408506393432617, 5.1691460609436035, -12.428911209106445, 7.869147777557373, 7.725497245788574, -7.857716083526611, -15.010367393493652, 2.0404436588287354, -5.182468891143799, 19.46270179748535, 2.532320976257324, -4.45059061050415, -8.856144905090332, -2.882739305496216, 0.6118541359901428, -8.93203353881836, 1.5282179117202759, 7.080653190612793, -10.066482543945312, -11.78783893585205, 0.5114292502403259, 7.682555198669434, 14.557882308959961, 4.728872299194336, 10.193931579589844, -16.162057876586914, -14.952908515930176, 2.6139917373657227, -4.897887706756592, 13.467361450195312, 3.3321542739868164, 18.546953201293945, -14.394169807434082, 7.905900955200195, 5.75716495513916, 11.2929048538208, 11.677396774291992, -5.4815568923950195, 10.204323768615723, 7.052757740020752, -11.400018692016602, 3.47135329246521, -3.530669689178467, -0.613385796546936, -5.692472457885742, 16.26537322998047, -10.717286109924316, -3.592778444290161, -14.789629936218262, 8.094529151916504, -2.777113676071167, -3.232693910598755, 0.8217529058456421, -10.821550369262695, 6.878709316253662, -7.2580108642578125, -6.4366888999938965, 0.013512087985873222, -2.552945375442505, -9.905643463134766, 7.850805282592773, 10.704530715942383, -4.1620941162109375, -5.543742656707764, -5.335031032562256, 6.872480392456055, 0.5161232352256775, 2.8452560901641846, 5.9880218505859375, 15.095283508300781, -17.67071533203125, 13.652511596679688, 5.002823829650879, -21.701480865478516, -8.764612197875977, -1.2504600286483765, 2.554471969604492, 4.5608978271484375, -11.172393798828125, -2.890432119369507, -5.408997535705566, -3.378023862838745, 0.4309384524822235, -2.6873974800109863, -2.660823345184326, 29.52090072631836, 5.601188659667969, 10.073610305786133, -6.543555736541748, -10.584664344787598, 6.825058460235596, 4.951966762542725, 0.5909255743026733, -2.1543335914611816, -17.66263198852539, -14.104657173156738, 18.185914993286133, -1.3867130279541016, -0.5883286595344543, -14.03805160522461, 4.449758052825928, 8.035022735595703, 9.865484237670898, 16.781963348388672, -5.884149551391602, 1.447223424911499, -2.1044154167175293, 0.8841195702552795, -3.5711841583251953, 9.427993774414062, 9.717704772949219, -11.295161247253418, 7.8601274490356445, -4.409143924713135, 0.4547399580478668, 7.318406581878662, 0.018756086006760597, 11.64106559753418, -4.266706466674805, 13.20712947845459, 15.422019004821777, -7.268369674682617, 12.637680053710938, -4.254528045654297, 18.95917510986328, -17.87290382385254, -24.801008224487305, 8.149168968200684, -14.01524543762207, -12.553921699523926, 7.239765167236328, 19.899799346923828, -0.20190902054309845, 1.2916861772537231, -17.51167869567871, 12.114694595336914, -8.570096969604492, -5.487455368041992, 7.441310405731201, 22.8120059967041, -3.4070920944213867, 14.06040096282959, 5.264939785003662, 16.631303787231445, -13.07655143737793, -5.495677947998047, 5.729378700256348, 9.828703880310059, 16.126216888427734, -10.938483238220215, 7.6082587242126465, -15.404293060302734, 0.33488380908966064, 10.404851913452148, 6.452746391296387, 16.710731506347656, -21.14569664001465, 0.09335491061210632, -7.653443336486816, -11.788143157958984, 12.805174827575684, -9.896327018737793, -1.8905107975006104, 6.8190531730651855, -20.48676300048828, 15.537238121032715, 8.15626335144043, 11.515892028808594, 18.333650588989258, -0.11976871639490128, -6.7708024978637695, -9.318025588989258, 2.711299419403076, -3.0929086208343506, -13.215030670166016, 4.793391704559326, -4.011178016662598, 2.5910239219665527, 3.3780746459960938, -8.150766372680664, -3.0285418033599854, 3.820035696029663, 2.779496669769287, -1.0389286279678345, -11.942899703979492, -3.867743968963623, -14.959959983825684, 16.002304077148438, 21.40266227722168, 8.040101051330566, 9.818342208862305, 2.2854628562927246, -13.954734802246094, -5.961992263793945, 5.394717216491699, 8.7730712890625, -12.376605987548828, -4.606313228607178, -0.9719828367233276, 19.43178939819336, 4.268819808959961, -1.4570565223693848, 1.5263539552688599, 4.803605556488037, 32.87687683105469, -14.277939796447754, -21.1558895111084, 4.908522129058838, -0.056463614106178284, -1.4230997562408447, 4.0132622718811035, 5.368828773498535, 5.408668041229248, -7.383246421813965, 0.6249250769615173, -8.280314445495605, -2.319495916366577, 8.433553695678711, -17.257579803466797, -5.314234733581543, 6.505184173583984, -15.663915634155273, -11.869794845581055, -4.981976509094238, -7.1713714599609375, 8.586956024169922, 4.093897342681885, 14.157628059387207, -2.839940071105957, -6.443798065185547, 10.696955680847168, -3.3488223552703857, 3.215113639831543, -2.9289231300354004, 19.05978012084961, -7.545753002166748, -9.06479549407959, 10.233778953552246, -3.1446104049682617, 2.9363789558410645, -2.467674970626831, -4.351362705230713, 7.862945556640625, 9.0540132522583, -5.080985069274902, -6.445537567138672, 8.41079044342041, -10.631834030151367, -17.48611068725586, -8.105788230895996, 5.788546562194824, -1.4606101512908936, 16.79867172241211, 7.776623249053955, -4.556460857391357, -12.75137996673584, 19.33005714416504, -12.82345199584961, -7.859947681427002, 4.049330711364746, 1.187629222869873, -1.4708638191223145, -6.305137634277344, 6.08095121383667, -6.315657615661621, 10.85750961303711, -11.863567352294922, 10.821948051452637, 2.947833299636841, -3.8748834133148193, -15.436697959899902, 2.1661362648010254, -6.572101593017578, 1.4900468587875366, 1.1511658430099487, 1.5402841567993164, 14.960824012756348, -2.634965181350708, 12.845922470092773, -8.845314025878906, -11.035694122314453, 0.9220666885375977, 3.7473959922790527, 16.16134262084961, -27.144105911254883, 4.177135467529297, -3.529649019241333, -12.437357902526855, 2.3870997428894043, -5.2785539627075195, 2.1485767364501953, 0.032363034784793854, -6.413045406341553, -16.637666702270508, 2.881190776824951, 7.559903144836426, -19.184837341308594, -10.084309577941895, -1.049983263015747, -7.613559246063232, 1.6518713235855103, -0.08772369474172592, -10.01364803314209, -14.458556175231934, 13.708806991577148, 7.78881311416626, 6.410388946533203, -14.423022270202637, 1.1400340795516968, -8.827256202697754, 6.259247779846191, -4.211367607116699, 2.619290828704834, 5.674813270568848, -6.1227946281433105, 25.60558319091797, 6.776365280151367, -14.783098220825195, -4.51366662979126, -10.945425987243652, 5.299370288848877, -19.451292037963867, 6.697688579559326, -18.386323928833008, 0.25990617275238037, 0.43230152130126953, -4.287777423858643, -0.7899835109710693, 2.7618086338043213, 7.9860124588012695, -2.705618381500244, -6.711423397064209, 6.886416435241699, 7.082016468048096, -3.4933652877807617, 12.395963668823242, -16.82566261291504, -2.9838523864746094, 9.257487297058105, 6.59035587310791, 0.9175187945365906, -0.3003372550010681, 0.38464832305908203, -10.870420455932617, -2.3519532680511475, 2.7626471519470215, 3.500535011291504, 12.889453887939453, 1.7909634113311768, -7.60905647277832, -1.3894962072372437, -11.591750144958496, 6.886730670928955, 5.077178478240967, -9.220232963562012, -1.2668932676315308, -19.46335792541504, 7.46674108505249, 20.53480339050293, -16.381547927856445, 3.0736615657806396, -1.1616085767745972, 2.6572539806365967, 0.6995660662651062, 9.12515640258789, 2.653870105743408, -0.16623689234256744, 19.36948585510254, 4.570253372192383, 22.116966247558594, -13.538888931274414, -5.666372299194336, -11.347994804382324, 8.366304397583008, 8.426454544067383, 4.066805839538574, -2.568791389465332, 8.214540481567383, 5.514660358428955, 6.334596157073975, 16.611146926879883, 2.4818665981292725, -8.412205696105957, -17.102174758911133, 0.10553538054227829, 0.35278216004371643, -4.908031463623047, -7.859575271606445, -2.3712894916534424, -15.121192932128906, -13.396073341369629, 17.935575485229492, -8.521522521972656, -0.6623644828796387, -7.288729667663574, -15.602425575256348, -6.1423540115356445, 0.22854678332805634, 17.168704986572266, -8.655974388122559, -20.456130981445312, -2.178102493286133, 11.469050407409668, -6.013370513916016, -17.641984939575195, -11.073265075683594, 5.969777584075928, 0.2312811315059662, -1.4374886751174927, 8.60006046295166, 6.823058605194092, 13.36046028137207, 7.2002410888671875, 6.506328105926514, -13.344247817993164, -1.9866300821304321, -3.220503330230713, 24.470619201660156, -13.859128952026367, 4.947958946228027, -0.4280269145965576, -9.68351936340332, 0.3812142014503479, 21.439485549926758, 15.387511253356934, 13.406848907470703, 5.375901222229004, -6.927648067474365, -18.69829559326172, -2.2401089668273926, -8.023655891418457, -3.270944118499756, -6.865331649780273, -11.349542617797852, -4.444277763366699, -11.431650161743164, 7.730650901794434, -21.188161849975586, 0.12141700834035873, -11.673986434936523, -0.658798098564148, 2.199932098388672, -20.958831787109375, -6.551837921142578, -10.5433931350708, 10.29997444152832, -9.051200866699219, 7.174045562744141, -10.240382194519043, -4.979931831359863, -18.9107723236084, -3.010995864868164, 10.036625862121582, -14.83364200592041, 14.77567195892334, -11.430015563964844, -10.464715003967285, -15.026237487792969, -3.9053735733032227, -3.9253594875335693, -11.820852279663086, 15.998148918151855, -17.312664031982422, -3.907395839691162, 12.320151329040527, -12.045403480529785, -0.7859474420547485, 2.2322211265563965, 13.870454788208008, 18.8612060546875, -1.9268717765808105, -1.6538066864013672, -23.441936492919922, 5.5623931884765625, -5.16798734664917, 6.2447099685668945, -0.5444019436836243, -8.004446029663086, 12.856120109558105, -5.21444845199585, -5.407649993896484, -2.3880598545074463, 9.046210289001465, 7.3232526779174805, -1.2541664838790894, -10.079108238220215, -0.1273878514766693, 1.1296871900558472, -11.172300338745117, 0.6803274154663086, 6.39223575592041, -12.16113567352295, -4.696138381958008, 18.16417121887207, -0.6581680774688721, -4.6049699783325195, -9.232715606689453, -5.324304580688477, -20.85397720336914, 4.218231201171875, 12.128872871398926, 7.076859474182129, -20.64067840576172, 7.562847137451172, -4.976032257080078, 10.884407043457031, 11.147111892700195, -11.822752952575684, 1.572234034538269, -6.8585944175720215, -1.7975960969924927, 16.58913803100586, -0.018164243549108505, -5.1799445152282715, -13.899903297424316, -19.220190048217773, 8.220654487609863, 5.026423454284668, -14.452241897583008, 3.0434415340423584, -25.46450424194336, 0.13166582584381104, -7.623903751373291, 4.385211944580078, -3.8571083545684814, 4.479952335357666, 19.733598709106445, 6.323332786560059, 17.620952606201172, -15.675012588500977, -0.36942943930625916, 2.350919723510742, 22.845582962036133, -2.392812728881836, 8.616378784179688, 2.9829025268554688, -4.866811752319336, -13.38943099975586, -4.111254692077637, -10.154848098754883, 29.70682144165039, -9.304734230041504, -31.48433494567871, 12.886141777038574, -8.040045738220215, 7.697706699371338, 7.745375156402588, -5.20069694519043, -10.851335525512695, 8.642122268676758, 6.478051662445068, -20.998714447021484, 7.091879844665527, 0.4953804016113281, -17.884939193725586, -0.07061479985713959, 13.42756175994873, -14.44114875793457, 1.684242606163025, 6.697752475738525, -1.9978094100952148, -0.5255419611930847, 2.4640374183654785, 7.158329963684082, -7.191066265106201, -11.360645294189453, 3.448456287384033, 7.312917709350586, -2.698514223098755, 0.7179871201515198, 0.9889438152313232, -2.1837563514709473, 11.819315910339355, 17.175962448120117, -8.683906555175781, -9.16026782989502, -4.330955505371094, 2.346519708633423, 4.6853742599487305, -11.442626953125, -7.214841365814209, 1.5244686603546143, 10.513415336608887, 5.1867241859436035, 15.373697280883789, 10.250239372253418, 1.4826287031173706, -12.633310317993164, -12.45480728149414, -6.213753700256348, 4.765524864196777, 4.434267044067383, -21.25019073486328, -5.79073429107666, -5.515338897705078, -16.83534812927246, 7.356393337249756, 4.126729965209961, -6.290380954742432, 6.916274070739746, 4.644412040710449, 0.7698986530303955, -13.544361114501953, -23.54245376586914, -8.819588661193848, 5.289444923400879, -4.4005842208862305, 7.603703022003174, -3.4708380699157715, -6.739190101623535, -16.60087776184082, 5.727364540100098, 7.322070598602295, -1.3583241701126099, 5.628715515136719, 8.60523796081543, 11.133377075195312, 12.558703422546387, -0.5405163168907166, -1.7267776727676392, 13.955148696899414, 14.295181274414062, -13.773058891296387, -3.086483955383301, 10.069069862365723, -6.754473686218262, 6.984942436218262, -6.182195663452148, -5.913729667663574, -10.54571533203125, -2.655803680419922, -14.62552547454834, -32.04716491699219, 9.812395095825195, -10.305581092834473, -3.7768683433532715, 6.7276105880737305, -0.45284125208854675, -10.547725677490234, -2.6870388984680176, -13.590611457824707, 4.939543724060059, 4.200249195098877, -14.561031341552734, -8.236732482910156, -1.9586858749389648, 9.430102348327637, -2.942619800567627, -11.452837944030762, -1.7963284254074097, 10.489715576171875, -1.58587646484375, -4.697021484375, -2.2685751914978027, 0.376406729221344, 6.8188066482543945, 2.835855007171631, -3.2391927242279053, 3.799349784851074, -1.9823921918869019, -11.284951210021973, -12.940042495727539, 7.216833114624023, -10.93600845336914, 9.548585891723633, 7.5695648193359375, 18.32292938232422, 0.37635886669158936, -13.490951538085938, 7.577803134918213, 6.589414596557617, 16.726457595825195, -12.695025444030762, 10.031448364257812, 1.6625319719314575, -4.283899784088135, -8.739620208740234, -12.060236930847168, 1.0461140871047974, -5.940083980560303, -18.733856201171875, 0.19637644290924072, -0.42935359477996826, 6.526899814605713, -8.338545799255371, -0.05826018378138542, -1.4823708534240723, -4.7336602210998535, -3.3318943977355957, -3.983182668685913, 9.248235702514648, -12.724080085754395, 3.247077703475952, -5.900521755218506, 1.4309182167053223, 6.355038642883301, 12.325715065002441, 19.043935775756836, -13.473026275634766, -8.678855895996094, 5.800047397613525, -11.959385871887207, -3.4324746131896973, 2.260617971420288, -5.5739850997924805, -3.280656099319458, 12.651581764221191, -11.308499336242676, -1.56168532371521, 12.126163482666016, -5.2045063972473145, 12.153020858764648, 6.657094955444336, 1.5172905921936035, -3.988708972930908, -6.227286338806152, 12.294636726379395, 7.118382453918457, -5.963654518127441, -3.0813941955566406, -1.0764120817184448, 16.067922592163086, -5.440843105316162, 4.262747287750244, 18.4872989654541, 2.1382763385772705, -3.190800189971924, -0.14593079686164856, 17.692913055419922, -18.327232360839844, -5.4465532302856445, -20.066753387451172, 4.443475246429443, -17.31894874572754, -20.059995651245117, -4.127893924713135, 14.558813095092773, -12.576513290405273, 4.64816427230835, 23.765872955322266, -6.342863082885742, -11.264615058898926, -1.883166790008545, 12.084202766418457, 0.7714568972587585, -10.301042556762695, 13.64631175994873, -0.8698319792747498, 2.2926764488220215, 14.037193298339844, 12.920655250549316, -4.392674922943115, 3.633002996444702, 1.7043484449386597, -5.0558319091796875, 6.073525428771973, -1.357712984085083, -10.25032901763916, -11.239765167236328, 10.297334671020508, -4.979705333709717, -10.669946670532227, 7.814556121826172, 10.708686828613281, 9.739609718322754, 6.111981391906738, 16.920894622802734, -5.5283684730529785, 5.920485973358154, -1.9043351411819458, 11.589717864990234, -0.8621898889541626, -2.708662986755371, 15.827107429504395, 3.3663668632507324, 13.388248443603516, 13.057860374450684, 9.391002655029297, -1.5645676851272583, 18.73443031311035, -3.397160053253174, 6.5533647537231445, 11.625652313232422, -13.484891891479492, 12.697774887084961, -13.292646408081055, -1.003283977508545, 4.06857967376709, -3.4293601512908936, 9.98338794708252, 9.92266845703125, -5.8858113288879395, -7.406872749328613, 5.516839504241943, 1.1640772819519043, 14.09461498260498, -17.85460662841797, -12.795125961303711, -23.831880569458008, -9.916353225708008, 8.912921905517578, 7.8321638107299805, -20.556737899780273, -3.177450656890869, 3.3546903133392334, -6.975704193115234, -13.522056579589844, 2.156715154647827, 4.138487815856934, 2.218681573867798, 2.663137435913086, -3.4538826942443848, -3.6627700328826904, 12.818037033081055, 8.383625984191895, 13.738288879394531, -5.2276811599731445, 9.602760314941406, 0.544764518737793, -13.607723236083984, 1.708754539489746, 14.95175838470459, -1.097467064857483, -8.339540481567383, 8.511534690856934, -13.879474639892578, 12.026483535766602, -10.689399719238281, -1.3292099237442017, 14.503640174865723, 12.508526802062988, -1.8026548624038696, -11.80183219909668, 8.778244972229004, -19.781970977783203, 2.838470935821533, 7.163876533508301, 0.12005172669887543, -8.503807067871094, 11.478205680847168, -0.7358390688896179, 13.352141380310059, -19.243961334228516, -9.906982421875, -2.8557050228118896, 7.159567832946777, 2.9698801040649414, -2.804555654525757, -16.42673110961914, 6.248416423797607, -15.7139310836792, -17.735177993774414, 5.807030200958252, 12.570147514343262, 10.970817565917969, 2.0973782539367676, -0.7458782196044922, -6.359047889709473, 5.39894962310791, -6.168848037719727, 13.963208198547363, 16.56690216064453, -14.240317344665527, -0.3178219795227051, -14.22433090209961, -11.177389144897461, -11.954809188842773, 12.561348915100098, 0.7770572900772095, 16.11850929260254, 12.202235221862793, 1.2965120077133179, -11.817159652709961, 8.951212882995605, -3.536996603012085, 9.664485931396484, 6.112860679626465, -11.919293403625488, -10.545363426208496, 0.3582032322883606, 6.749668121337891, -17.513347625732422, -3.839057683944702, -24.018861770629883, -5.183339595794678, -0.13070517778396606, 6.407722473144531, 8.614683151245117, 13.167176246643066, -0.8148090839385986, 13.880414962768555, -11.952948570251465, 14.085208892822266, 20.42351531982422, 3.1377339363098145, -8.415217399597168, 19.452926635742188, 5.467884540557861, -12.720968246459961, -3.1963083744049072, -8.385414123535156, 12.795515060424805, 7.254084587097168, -26.440460205078125, 4.291518688201904, 3.267378807067871, -1.7570548057556152, 8.620833396911621, 2.586155414581299, 11.581451416015625, 4.485087871551514, -8.876116752624512, 10.432716369628906, 15.194472312927246, -15.062798500061035, -14.491628646850586, -1.3022079467773438, 0.9485891461372375, -2.4930613040924072, 1.0026581287384033, -1.1916108131408691, 12.826364517211914, 11.257268905639648, -7.422192573547363, 14.21474838256836, -20.16883659362793, 8.30812931060791, 2.3677561283111572, 4.007610321044922, 3.6509227752685547, 6.690609455108643, 6.496840476989746, 9.849709510803223, 5.747183799743652, 5.037003517150879, -16.727251052856445, 5.2396697998046875, 5.864007949829102, -17.050090789794922, 19.391389846801758, -5.027334690093994, -5.525143623352051, -8.259879112243652, -10.962189674377441, 13.264473915100098, 6.799591064453125, -4.779129981994629, -6.993927001953125, 25.44793128967285, 12.388254165649414, -18.30622100830078, 15.788546562194824, -0.665081262588501, 0.9072211384773254, 12.33795166015625, -6.661223888397217, -4.050654411315918, 4.580060958862305, 29.687185287475586, 4.708280086517334, -1.6464102268218994, 4.987636566162109, 14.006988525390625, 4.423152923583984, 23.87830352783203, 31.477022171020508, -2.6262869834899902, 9.49521255493164, 17.9554443359375, -1.354903221130371, 20.73422622680664, -11.263326644897461, 5.118364334106445, -0.5541716814041138, 18.893444061279297, -12.886872291564941, -1.578853726387024, 1.9456446170806885, -22.310331344604492, 1.7880792617797852, -3.5281050205230713, -9.723541259765625, -20.266752243041992, -0.4340684115886688, 1.2132899761199951, 1.685382604598999, 16.194238662719727, -2.4905807971954346, -9.478055000305176, -10.591009140014648, -8.006673812866211, 13.391965866088867, 0.15637324750423431, 2.16191029548645, -4.920805931091309, -3.8546764850616455, -4.363792419433594, 4.450281143188477, 14.40097713470459, 0.273224800825119, 1.628122091293335, 1.1108965873718262, -5.97860860824585, -4.849287509918213, 10.003815650939941, 0.010089945048093796, 9.502288818359375, 1.2468336820602417, 5.527223587036133, 13.48772144317627, -2.9819211959838867, 13.432254791259766, -8.523857116699219, -8.300926208496094, -12.127996444702148, -7.143420696258545, 5.687637805938721, -1.917447805404663, 3.0982182025909424, -7.8805999755859375, 8.154471397399902, 3.533104658126831, -6.547779083251953, 18.26997184753418, -6.73241662979126, -2.676595687866211, -5.660747528076172, 0.8440737724304199, -1.095595121383667, 6.631526470184326, -15.578545570373535, -7.348984718322754, -14.126606941223145, -9.213159561157227, 6.387288570404053, -1.851388931274414, 7.883903503417969, 9.24353313446045, 3.8699402809143066, -7.943689823150635, 11.296744346618652, -0.9753235578536987, 0.14928843080997467, -1.3660612106323242, 0.16766995191574097, -11.111401557922363, -9.820433616638184, 11.897102355957031, -4.328055381774902, 6.689624309539795, -3.869260311126709, -0.7052446603775024, 0.835660994052887, -0.9732350707054138, -1.3420383930206299, -3.856572151184082, -12.536850929260254, 2.931671142578125, 3.477320432662964, -12.297883987426758, -0.5492211580276489, -7.125694274902344, 4.764513969421387, -18.600582122802734, -9.914812088012695, 1.4089508056640625, 1.6656115055084229, 15.369222640991211, 18.91692352294922, 1.8943451642990112, 6.536484718322754, 9.546112060546875, -32.00444412231445, -9.268095970153809, 6.796927452087402, -9.657540321350098, 14.909036636352539, 18.89681625366211, 1.991015076637268, 6.913552761077881, -12.9033842086792, 19.77897834777832, 9.58400821685791, -15.557191848754883, -17.759117126464844, 12.862726211547852, -25.180294036865234, -2.197227954864502, 14.21495532989502, 14.205162048339844, -7.129680633544922, 1.0961787700653076, -6.538772106170654, -10.024168014526367, 2.368966579437256, -30.246562957763672, 8.670713424682617, 10.677103042602539, -4.912091255187988, -9.50959300994873, 8.187592506408691, 9.078337669372559, 16.28447723388672, -4.270516395568848, -4.8058013916015625, 4.542305946350098, 6.678185939788818, 9.139805793762207, -4.1549272537231445, 10.973079681396484, -2.1558470726013184, 10.759820938110352, 3.538144588470459, 9.793729782104492, 12.950445175170898, 2.4964120388031006, 4.667061805725098, 8.702371597290039, -0.5267854928970337, 1.162137508392334, 0.6715932488441467, 2.0382282733917236, -0.7057926058769226, 10.373064041137695, 8.177207946777344, -11.27888298034668, -4.580440998077393, -1.5747735500335693, 13.242535591125488, 0.7245650291442871, 8.170855522155762, -4.732264518737793, 14.365492820739746, 2.3947484493255615, -1.5187370777130127, -3.1356284618377686, 3.598684310913086, -13.157276153564453, -10.842508316040039, -13.66921329498291, -3.3801794052124023, -6.8361287117004395, 14.943388938903809, -11.677600860595703, -0.557937502861023, -1.7636163234710693, -4.043529033660889, -14.8427095413208, -4.960822105407715, 9.888285636901855, -7.161874294281006, -14.294509887695312, 1.8649637699127197, 1.7699037790298462, -6.139870643615723, -2.6249442100524902, 8.184556007385254, -4.77452278137207, 27.00946617126465, 2.6479053497314453, -2.8088083267211914, 8.217269897460938, -5.90106725692749, -25.370052337646484, -6.447163105010986, -0.04835280030965805, 4.779362201690674, 1.2655510902404785, -10.219484329223633, 2.6014339923858643, 0.48248571157455444, 5.840584754943848, -7.01516580581665, 9.477849960327148, -14.014448165893555, 6.086116313934326, -6.516098499298096, -23.834091186523438, 11.96720027923584, 3.852381467819214, 10.983118057250977, 2.727863311767578, 0.7604125738143921, -2.9227848052978516, 2.537569999694824, 10.006860733032227, 6.552274703979492, 19.6324520111084, -2.8640496730804443, -1.7634437084197998, -7.097332000732422, 0.447546124458313, -2.072554349899292, 5.053613662719727, -10.248237609863281, -3.60209321975708, 5.375035285949707, 3.5359299182891846, 17.79973793029785, -8.158527374267578, -4.101558685302734, 3.079007863998413, -6.2656073570251465, -11.093059539794922, 15.861493110656738, 0.9139804840087891, -16.09227180480957, -4.505466938018799, 0.5933041572570801, 14.005294799804688, -16.845735549926758, 2.4099478721618652, -5.3328728675842285, -2.4904274940490723, 11.584449768066406, 9.770392417907715, 9.69066047668457, 9.471179008483887, -11.153584480285645, 3.3100697994232178, -7.135595798492432, -11.463010787963867, 5.319252967834473, 16.15146827697754, 7.362043380737305, 7.912470817565918, 15.223675727844238, -1.5163301229476929, 10.267276763916016, -7.969079971313477, -2.9503118991851807, -14.125791549682617, 10.19786262512207, -4.445580959320068, 1.3085960149765015, -0.3911783993244171, 0.221143439412117]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[0.0020766931120306253, 0.025308916345238686, -0.007417789660394192, -0.015006269328296185, -0.05664638429880142, 0.012524297460913658, -0.028528941795229912, -0.010467144660651684, -0.021330324932932854, -0.00036467568133957684, -0.009207883849740028, -0.009078464470803738, -0.01893692836165428, 0.012497290037572384, -0.02708679437637329, -0.012034905143082142, 0.02221539430320263, -0.02501140534877777, 0.04126160591840744, -0.009613302536308765, 0.026044568046927452, 0.03593366965651512, -0.003516016062349081, 0.006315524224191904, 0.028012074530124664, 0.011228960007429123, 0.052014242857694626, 0.01807723566889763, -0.004076117184013128, 0.023545490577816963, 0.003014711895957589, 0.03873823583126068, -0.02918730303645134, 0.03481662645936012, 0.0429118387401104, 0.03374839201569557, 0.001837691175751388, -0.04208501800894737, -0.0171057116240263, 5.0190628826385364e-05, 0.02986784465610981, 0.004229269456118345, -0.012036200612783432, 0.006992236245423555, 0.028885891661047935, -0.007905333302915096, 0.0022244106512516737, 0.005993967410176992, 0.010579226538538933, 0.018600743263959885, 0.025076694786548615, -0.03145899623632431, 0.011324470862746239, -0.001630339422263205, -0.01770123280584812, 0.005141620524227619, 0.0011948286555707455, -0.004625392146408558, 0.011229715310037136, -0.005675912834703922, 0.006136657204478979, 0.0023053281474858522, 0.001164378598332405, -0.0008220323943533003, -0.007063242606818676, 0.02568769082427025, 0.012726988643407822, 0.006988574750721455, 0.016892891377210617, -0.01274782419204712, 0.02937220223248005, -0.033099330961704254, -0.006916855927556753, -0.03420441225171089, -0.011718910187482834, 0.006749841850250959, 0.0005326432874426246, 0.03990159556269646, 0.004364604130387306, 0.008572325110435486, -0.0018664054805412889, -0.020988445729017258, 0.01534728892147541, 0.028009720146656036, -0.02422136254608631, -0.03935606777667999, -0.03583524376153946, -0.0072029042057693005, 0.0048830523155629635, 0.012933382764458656, -0.04642513394355774, -0.03243756666779518, 0.0020366625394672155, 0.007947234436869621, -0.008883371017873287, -0.012683176435530186, 0.00880425050854683, -0.014553295448422432, -0.03490797430276871, 0.027010636404156685, 0.01782119832932949, -0.0076819309033453465, -0.0027226360980421305, -0.017168113961815834, 0.03229086846113205, 0.009489241056144238, 0.019837483763694763, -0.0047716968692839146, -0.02123524621129036, 0.0450257882475853, 0.00833583902567625, 0.021022271364927292, 0.027932526543736458, 0.0387696772813797, 0.01104042585939169, -0.02995235286653042, 0.011717836372554302, -0.010268647223711014, -0.004347575828433037, 0.0219134371727705, 0.006201567128300667, -0.005190298892557621, -0.0017897096695378423, 0.010305440984666348, 0.028556523844599724, 0.02674076147377491, -0.02754092961549759, 0.011350679211318493, 0.00734324473887682, 0.015320200473070145, 0.015388567000627518, 0.013488800264894962, 0.02115754783153534, 0.005401626694947481, 0.024981610476970673, 0.0401386097073555, -0.006313582882285118, -0.022298475727438927, -0.005253298208117485, 0.0007917027687653899, -0.004504464101046324, -0.016543244943022728, -0.011484665796160698, 0.015440245158970356, 0.019699202850461006, -0.01647253893315792, -0.0027709186542779207, -0.027559379115700722, -0.013554152101278305, 0.0036881393752992153, -0.007708315271884203, -0.0015672362642362714, -0.027096914127469063, -0.0038844516966491938, -0.03289277106523514, 2.6221407551929587e-06, -0.0009751452598720789, -0.017645051702857018, -0.004124690778553486, 0.004806617274880409, 0.008988665416836739, 0.04186186566948891, 0.032536376267671585, -0.00774860754609108, 0.0375145822763443, 0.004489325452595949, 0.0511673204600811, 0.00887651089578867, -0.00913748238235712, 0.01224853191524744, 0.002192714484408498, 0.029890062287449837, -0.0045829289592802525, -0.006827042903751135, -0.045495908707380295, -0.004586999770253897, -0.012196068651974201, -0.011991825886070728, 0.02292151004076004, 0.0170891135931015, 0.001765084802173078, -0.0023149382323026657, -0.015537058003246784, -0.019951792433857918, 0.011739668436348438, 0.00602682726457715, -0.02091498114168644, -0.0008906711009331048, -0.005937086883932352, -0.0005840025842189789, 0.032828349620103836, -0.014374712482094765, 0.02743195928633213, 0.02076445333659649, 0.0015747920842841268, 0.01878930814564228, 0.009412148036062717, -0.003932576160877943, -0.03533506765961647, -0.025993959978222847, -0.019561633467674255, -0.00724487379193306, -0.0010776554699987173, 0.027704564854502678, 0.00205163168720901, 0.03638072311878204, 0.02562471106648445, 0.02357039786875248, 0.00042688543908298016, 0.003947553224861622, 0.016222836449742317, -0.0036196247674524784, 0.0043713366612792015, -0.030735448002815247, 0.01475515216588974, 0.01057917159050703, -0.005960928741842508, 0.03126779571175575, 0.03552450239658356, 0.012453364208340645, -0.010200617834925652, 0.012770069763064384, 0.008921793662011623, 0.02122492343187332, -0.004981458652764559, 0.014112100005149841, 0.01719013787806034, 0.01332925446331501, 0.011253902688622475, 0.003003613790497184, -0.006838053930550814, -0.019373465329408646, -0.04061742499470711, -0.039368532598018646, -0.013533412478864193, -0.008738585747778416, 0.026545705273747444, 0.012638621032238007, -0.014143523760139942, -0.0077766734175384045, -0.012979190796613693, -0.014836899004876614, 0.039980094879865646, -0.014286814257502556, 0.008648425340652466, -0.004915173165500164, -0.022586697712540627, -0.0032596366945654154, -0.03168163821101189, -0.0036580590531229973, 0.01702851988375187, -0.020275987684726715, 0.021116217598319054, -0.014237414114177227, -0.010484463535249233, -0.0021177709568291903, 0.023077771067619324, -0.017969798296689987, 0.008581705391407013, -0.020198505371809006, 0.015360265970230103, 0.04355183243751526, -0.036214910447597504, -0.0059788827784359455, -0.013616150245070457, -0.011444573290646076, -0.035054996609687805, -0.00024725424009375274, -0.017775751650333405, -0.012909354642033577, 0.006733363959938288, -0.007818109355866909, -0.014019614085555077, 0.03603605553507805, 0.002649006899446249, -0.006173837464302778, -0.001061999355442822, -0.00029368611285462976, -0.003897172398865223, 0.010154600255191326, -0.018093034625053406, 0.026527877897024155, 0.001656007720157504, 0.022456243634223938, -0.006307733710855246, 0.034993141889572144, 0.011181829497218132, 0.003032577922567725, 0.033645134419202805, 0.003364947624504566, 0.017690282315015793, -0.004554384853690863, 0.05818430334329605, 0.0016696309903636575, -0.007645998150110245, 0.007297041360288858, 0.02272924594581127, -0.014828418381512165, 0.021595466881990433, -0.00657039787620306, -0.011601246893405914, -0.017477355897426605, 0.028323084115982056, -0.02529001794755459, -0.014916499145328999, -0.013985884375870228, -0.003104581730440259, 0.004530423786491156, -0.014006380923092365, -0.014534709975123405, -0.04740646481513977, 0.022806931287050247, 0.007203696761280298, 0.019602082669734955, 0.00899967085570097, -0.03240692615509033, 0.017385751008987427, 0.03001781925559044, -0.030937330797314644, 0.021164121106266975, -0.015023848973214626, -0.010797330178320408, -0.025723440572619438, -0.018714094534516335, 0.0006200313218869269, 0.010401575826108456, 0.009969360195100307, 0.011794451624155045, -0.007583548314869404, 0.010023685172200203, 0.013199206441640854, 0.02000156231224537, 0.0100330151617527, 0.030935417860746384, 0.0003642467490863055, -0.044867802411317825, 0.013879618607461452, 0.009311493486166, 0.01834258809685707, -0.021857919171452522, 0.018838349729776382, 0.007776415441185236, -0.02021973580121994, -0.024232948198914528, 0.018695836886763573, 0.043462179601192474, 0.014308280311524868, -0.01597430929541588, 0.007702650967985392, -0.018442975357174873, -0.006025133188813925, 0.006200783886015415, -0.0052375346422195435, 0.023072337731719017, -0.03250744938850403, 0.002345041139051318, 0.02100191079080105, 0.002585205715149641, 0.019908428192138672, -0.007225253619253635, 0.014558233320713043, -0.022323518991470337, 0.0004987164284102619, -0.0038752977270632982, 0.012778215110301971, -0.01181305106729269, 0.0013246292946860194, -0.06545443832874298, -0.008589661680161953, 0.012459618039429188, 0.02421570010483265, 0.008445518091320992, 0.0003180184285156429, 0.037811871618032455, -0.0002206966746598482, -0.004803857300430536, 0.005016656126827002, -0.002234144601970911, -0.021537266671657562, -0.034279391169548035, -0.021911432966589928, 0.005023575853556395, 0.010976193472743034, 0.00869482196867466, -0.01208910159766674, -0.01064698863774538, 0.016161954030394554, -0.01517341285943985, -0.010416131466627121, 0.027869977056980133, 0.0012658388586714864, -0.012059110216796398, -0.014364640228450298, 0.022928684949874878, -0.0012041263980790973, 0.0023893979378044605, 0.002588415052741766, -0.040401846170425415, -0.01861821860074997, 0.003235221840441227, -0.0027920256834477186, 0.0065485648810863495, -0.01885821297764778, -0.018181275576353073, 0.007611305918544531, 0.032587893307209015, -0.019863275811076164, -0.0015478902496397495, -0.01884402334690094, -0.03242699056863785, -0.0023381621576845646, -0.0058688451535999775, -0.05053918436169624, -0.013755260035395622, -0.015524295158684254, -0.03244487941265106, -0.01928780786693096, 0.031976498663425446, -0.047910165041685104, -0.03111095353960991, 0.01750723272562027, 0.0016231146873906255, 0.007146830670535564, -0.0015638013137504458, -0.0014300541952252388, -0.014098930172622204, 0.021155692636966705, -0.041470494121313095, 0.004128935281187296, -0.04475302994251251, -0.048749376088380814, -0.00754202576354146, 0.029264800250530243, 0.007796320132911205, -0.0034127533435821533, 0.03200303763151169, -0.004351780749857426, -0.017556684091687202, 0.010068397969007492, -0.03092649020254612, 0.041261523962020874, 0.003918080125004053, -0.005264119245111942, -0.015218524262309074, 0.012056505307555199, 0.03190458193421364, -0.01927393302321434, -0.0029081860557198524, -0.0015784514835104346, -0.024672267958521843, 0.01822313480079174, 0.0004137445648666471, -0.015792343765497208, 0.01062637660652399, -0.02350124903023243, 0.00109906867146492, -0.0018364285351708531, -0.024085069075226784, -0.01753605157136917, -0.0007410412072204053, -0.042781367897987366, -0.004938181955367327, 0.002814685460180044, -0.002123921876773238, -0.004402624908834696, -0.031362827867269516, -0.008582007139921188, -0.00478943157941103, 0.00999680906534195, 0.0006921414169482887, 0.01819463074207306, -0.0016684402944520116, 0.004018551204353571, 0.011979039758443832, -0.0019337155390530825, 0.025235382840037346, -0.0364469513297081, -0.029101388528943062, 0.019725827500224113, -0.034389104694128036, -0.015965035185217857, -0.014666951261460781, 0.004668263252824545, -0.017149848863482475, -0.016942279413342476, 0.04311680793762207, -0.00576773751527071, 0.002240774454548955, 0.01585961878299713, 0.005847062915563583, -0.008552086539566517, -0.02013339102268219, -0.01015237346291542, 0.008950824849307537, 0.0076017118990421295, 0.01694539748132229, 0.017069650813937187, 0.045919906347990036, 0.0300096794962883, 0.004544857423752546, 0.006902746390551329, 0.0054813409224152565, 0.0009249225840903819, 0.004239752423018217, 0.005826089531183243, -0.010239707306027412, -0.018811410292983055, -0.012133358977735043, 0.012906992807984352, 0.02395079843699932, 0.006721177138388157, -0.0004234382067807019, 0.0149242477491498, -0.00041691746446304023, 0.006072379183024168, 0.0216757133603096, -0.03330839425325394, -0.01876925118267536, 0.04580084607005119, -0.00040450977394357324, -0.012304402887821198, -0.019865643233060837, -0.006220452021807432, -0.020422251895070076, -0.009324705228209496, -0.006610702257603407, 0.024736812338232994, -0.00981060415506363, 0.013802164234220982, -0.01661979779601097, 0.05810406804084778, -0.024367457255721092, -0.003747053910046816, 0.002147544641047716, -0.0029127656016498804, 0.016362302005290985, 0.014139365404844284, 0.0039449515752494335, 0.03680822253227234, -0.009976600296795368, -0.0015592679847031832, -0.00355777470394969, 0.0020725864451378584, 0.007868506014347076, -6.075137935113162e-05, -0.00837710127234459, -0.010540413670241833, -0.005380834918469191, -0.01705111563205719, -0.012457855977118015, -0.023402424529194832, -0.04443122446537018, 0.01190205942839384, -0.018616046756505966, 0.01831871084868908, -0.019172174856066704, 0.007424006704241037, -0.024241618812084198, 0.024141063913702965, -0.016693444922566414, -0.03061106614768505, -0.009245635941624641, -0.020843978971242905, 0.05495680496096611, -0.023722246289253235, 0.014897196553647518, -0.048600200563669205, 0.013957994990050793, -0.027100296691060066, -0.008691862225532532, -0.021619031205773354, 0.009992895647883415, -0.016677485778927803, 0.0021220282651484013, 0.008760908618569374, 0.01196861732751131, -0.004714219365268946, 0.0267169289290905, 0.026712490245699883, -0.013028373010456562, 0.007710614707320929, 0.019557423889636993, 0.04339812323451042, -0.013579978607594967, 0.01656242087483406, 0.00467906566336751, -0.016608454287052155, 0.0024068038910627365, -0.028641728684306145, -0.005775343626737595, 0.04332347586750984, 0.016738349571824074, 0.025953609496355057, 0.010196814313530922, -0.005121566355228424, -0.0208275243639946, -0.0383942611515522, -0.02590274065732956, 0.04186808317899704, -0.01221745740622282, 0.0031660948880016804, 0.021936064586043358, -0.02463121898472309, 0.008431716822087765, -0.02348235994577408, -0.0028665373101830482, -0.026756830513477325, 0.03607087582349777, 0.006417273078113794, 0.02853197045624256, -0.01965792290866375, 0.0068617877550423145, -0.01845429092645645, 0.05454258248209953, -0.007770034018903971, 0.005865781102329493, 0.01716296188533306, -0.009719109162688255, 0.014769203029572964, -0.0039380984380841255, 0.002543401438742876, -0.01554158516228199, 0.01651165448129177, 0.027733519673347473, -0.02993752807378769, 0.0022020689211785793, -0.0072054751217365265, 0.004838426597416401, 0.01670214533805847, -0.003781036240980029, -0.0012964686611667275, 0.0010374231496825814, -0.002193635329604149, 0.012215751223266125, -0.022601693868637085, -0.002130991080775857, 0.006637482903897762, 0.0234580859541893, 0.036225009709596634, 0.004780115559697151, 0.0010356394341215491, -0.02339702658355236, -0.01913239061832428, 0.006506360601633787, -0.003570461180061102, 0.011253653094172478, -0.04991687461733818, -0.012234160676598549, 0.0026554351206868887, -0.005545805208384991, -0.0081749577075243, -0.019382495433092117, -0.00845338311046362, -0.012120772153139114, -0.012796689756214619, 0.00042403684346936643, 0.013127041049301624, -0.002330043585970998, -0.022320473566651344, 0.027529045939445496, -0.016226312145590782, 0.0005094968364574015, 0.006253203842788935, -0.0014891363680362701, -0.027664296329021454, -0.01271398551762104, -0.031217938289046288, 0.02892209403216839, -0.01930857263505459, 0.033127546310424805, 0.013642674311995506, 0.014657140709459782, -0.0001465052191633731, -0.0212946105748415, -0.008159615099430084, -0.003942296840250492, 0.04346425086259842, 0.007358897011727095, -0.022274043411016464, -0.0046985880471765995, 0.031891580671072006, 0.0076637910678982735, 0.008257156237959862, -0.006516624242067337, -0.024967389181256294, 0.03407895937561989, 0.026049334555864334, -0.02986898645758629, -0.0005812776507809758, -0.005913883447647095, 0.02795746922492981, -0.0010293633677065372, -0.024669436737895012, 0.03645554929971695, 0.0034663095138967037, 0.009002061560750008, 0.007929523475468159, 0.008908265270292759, -0.025257645174860954, -0.04206079989671707, 0.006076975259929895, 0.015331601724028587, 0.0044477456249296665, 0.03397388756275177, -0.006036620121449232, -0.030519040301442146, 0.02661813423037529, 0.018359201028943062, 0.002395856427028775, -0.04649364575743675, 0.03382902592420578, -0.011672205291688442, 0.017481576651334763, -0.01191750168800354, 0.03779970481991768, 0.009036881849169731, -0.017788030207157135, -0.03968069329857826, 0.01990344747900963, -0.02985086292028427, 0.0026909213047474623, 0.019368359819054604, -0.024431683123111725, 0.01890029013156891, 0.006924435030668974, -0.01035394985228777, -0.0009366263984702528, 0.011991092003881931, -0.007432514801621437, 0.017158759757876396, -0.004688804969191551, 0.0020311034750193357, 0.012864097021520138, 0.020935844630002975, 0.005001169163733721, 0.012004507705569267, 0.04937737435102463, 0.010948216542601585, -0.03558065742254257, 0.030429396778345108, 0.00205599726177752, 0.020508868619799614, 0.00159257382620126, 0.00652146665379405, 0.02518276497721672, 0.0096232108771801, -0.0145721435546875, 0.026213694363832474, -0.016451094299554825, 0.018582016229629517, 0.014396384358406067, -0.0012065833434462547, 0.0006757908267900348, -0.027695752680301666, -0.011833547614514828, 0.06389407068490982, -0.041444770991802216, 0.021196236833930016, -0.01923448219895363, -0.008298789151012897, -0.041157253086566925, -0.024007294327020645, -0.004440478514879942, 0.011081015691161156, 0.023601772263646126, 0.028525253757834435, -0.0007562602986581624, -0.017329171299934387, 0.02254353277385235, 0.004172912333160639, -0.024942433461546898, 0.0395062156021595, 0.02046237140893936, 0.026930656284093857, 0.0012615929590538144, -0.013339164666831493, 0.015400945208966732, -0.007266280706971884, 0.03250682353973389, -0.007100215181708336, -0.006896318402141333, 0.02681203931570053, 0.027742911130189896, 0.008320838212966919, 0.023336080834269524, 0.008979566395282745, -0.013163863681256771, 0.024332551285624504, -0.02259904518723488, 1.880828131106682e-05, -0.005945876706391573, -0.01155418623238802, -0.016079092398285866, -5.069672624813393e-05, -0.009844987653195858, -0.0039778887294232845, -0.009537608362734318, -0.01610567606985569, -0.008547014556825161, 0.010186491534113884, -0.002412755973637104, -0.02171328105032444, -0.008516213856637478, -0.024800429120659828, -0.007619539275765419, 0.003403883893042803, -0.007888427004218102, -0.002310053678229451, 0.02212170697748661, -0.008914502337574959, 0.007448035292327404, -0.016086425632238388, 0.029614880681037903, -0.003400504356250167, 0.0002748121914919466, -0.016818242147564888, 0.01414321456104517, 0.017346441745758057, 0.027621163055300713, -0.009800954721868038, 0.011173795908689499, -0.03677229955792427, -0.0279273372143507, 0.024195002391934395, 0.024825764819979668, 0.000750172883272171, 0.002649676753208041, 0.047821544110774994, 0.014791802503168583, 0.022924426943063736, 0.012559700757265091, 0.0002980733406729996, -0.022938203066587448, 0.009669885039329529, 0.0018441856373101473, 0.002264379058033228, 0.010722817853093147, 0.0004964015097357333, 0.030752673745155334, -0.02311641350388527, 0.011596796102821827, -0.017670534551143646, -0.004628460388630629, -0.023926127701997757, -0.0005067389574833214, 0.020767278969287872, 0.011209560558199883, -0.002274553757160902, 0.033909596502780914, -0.018372230231761932, 0.034645795822143555, -0.0045775361359119415, 0.025812137871980667, 0.0021244515664875507, -0.006451115943491459, 0.03357389569282532, -0.013283289037644863, 0.00955911073833704, 0.01636277511715889, 0.023102350533008575, -0.019109578803181648, 0.020984387025237083, -0.024455763399600983, 0.005474175326526165, 0.013941054232418537, 0.01276361383497715, 0.003922769799828529, 0.001880962518043816, 0.008905891329050064, -0.011006328277289867, 0.025900060310959816, 0.022741002961993217, -0.0002223746123490855, 0.017666276544332504, -0.004471872933208942, 0.0109078548848629, -0.008866224437952042, 0.01177032571285963, 0.013863177970051765, -0.002984706312417984, -0.030812861397862434, 0.03167343884706497, 0.018615704029798508, 0.015116214752197266, 0.019941886886954308, 0.004607959650456905, 0.016542742028832436, 0.011142871342599392, -0.02400883287191391, -0.017510080710053444, -0.03073670156300068, -0.01824399083852768, -0.012579606845974922, -0.01572594605386257, -0.04608813673257828, -0.003779889550060034, -0.021259378641843796, -0.028944427147507668, 0.0045251064002513885, 0.020999249070882797, 0.02465733513236046, 0.013092825189232826, 0.017515430226922035, 0.04884053394198418, -0.018953317776322365, 0.020125720649957657, -0.01073427777737379, 0.011913634836673737, -0.0008418360957875848, 0.004993737675249577, -0.0024501674342900515, -0.00773828336969018, -0.015852082520723343, 0.02292073704302311, 0.018315959721803665, 0.015964450314641, -0.000771727820392698, -0.027852466329932213, -0.0008620896842330694, 0.022240925580263138, -0.028380105271935463, 0.022641276940703392, 0.013774915598332882, 0.0012157279998064041, -0.017730815336108208, 0.025675544515252113, -0.02851727232336998, -0.005745405331254005, -0.013346427120268345, 0.0018829769687727094, 0.007224320899695158, 0.003368885489180684, -0.01681763119995594, -0.012783217243850231, 0.004058816935867071, 0.00847508292645216, 0.04413887858390808, 0.032687246799468994, 0.0007096618646755815, 0.007147633470594883, 0.020601285621523857, 0.016791533678770065, 0.0009748332086019218, 0.020012645050883293, 0.005893413908779621, -0.01948307268321514, -0.046468496322631836, -0.03637263551354408, -0.019563287496566772, -0.011943629011511803, -0.005754284560680389, 0.020547086372971535, -0.025585338473320007, -0.03220539167523384, 0.00490456260740757, -0.00356661481782794, -0.012073207646608353, -0.010099085047841072, -0.055746618658304214, -0.021965013816952705, 0.010228915140032768, 0.010090166702866554, 0.032427068799734116, 0.021724054589867592, -0.004285525064915419, -0.026209479197859764, -0.0070192995481193066, 0.016334129497408867, 0.026243919506669044, 0.060309287160634995, 0.048003535717725754, 0.023465638980269432, 0.0002304041408933699, -0.00881163403391838, -0.006570883095264435, -0.0394822359085083, 0.00748123275116086, -0.00612311577424407, -0.015607301145792007, -0.0009401899296790361, -0.00401950441300869, -0.020181087777018547, 0.012355059385299683, 0.00820151623338461, 0.002992012305185199, 0.03055129386484623, 0.025365063920617104, 0.0027768458239734173, -0.018396321684122086, -0.025642510503530502, 0.009588177315890789, -0.0002682640333659947, 0.027423419058322906, -0.007218132261186838, 0.013497733511030674, -0.007463380228728056, -0.0030607881490141153, 0.03118426725268364, -0.0037142743822187185, -0.006748446263372898, 0.007150054443627596, 0.051861394196748734, -0.020530516281723976, 0.013466691598296165, -0.032263532280921936, -0.004815933294594288, 0.008211045525968075, 0.01089857704937458, 0.028200531378388405, -0.02452874928712845, -0.019996196031570435, -0.0019129469292238355, 0.0028125254902988672, -0.0035679396241903305, -0.006393037270754576, -0.01346412394195795, 0.007096536457538605, -0.0075545478612184525, -0.009790930896997452, 0.02478262409567833, 0.022525766864418983, 0.011815129779279232, 0.012081949040293694, 0.017351733520627022, 0.043038010597229004, 0.00675239460542798, -0.0032845826353877783, -0.0004725810722447932, 0.007517250720411539, -0.01641123928129673, -0.0009065022459253669, -0.020464235916733742, -0.006369325798004866, 0.00580523069947958, 0.0010513885645195842, -0.027331653982400894, -0.024241996929049492, -0.012661087326705456, 0.0023192167282104492, -0.027215955778956413, -0.04144660755991936, -0.005055468529462814, 0.012797350063920021, 0.023967739194631577, 0.01231837086379528, 0.008427801541984081, 0.004790878389030695, 0.02103419415652752, -0.004545421805232763, -0.0078897038474679, -0.022213205695152283, -0.017400601878762245, -0.013976714573800564, -0.025268588215112686, 0.0030426373705267906, 0.01071565318852663, -0.01738477312028408, 0.011622476391494274, -0.005997580476105213, 0.03320370987057686, -0.01022876612842083, 0.0010236891685053706, 0.007173065096139908, 0.013840173371136189, 0.026859065517783165, 0.003979085478931665, 0.04926536977291107, -0.007377887610346079, -0.0034988452680408955, 0.03993033990263939, -0.001769129536114633, -0.003964556846767664, 0.011343143880367279, -0.018400944769382477, 0.02258221246302128, -0.01676451414823532, -0.02923203818500042, -0.017937306314706802, 0.00570753263309598, 0.019877208396792412, 0.014439902268350124, 0.02169216237962246, -0.004003566689789295, -0.010563326999545097, 0.019658826291561127, -0.030233129858970642, 0.0036086831241846085, -0.00510499207302928, 0.004649677313864231, -0.0438525453209877, -0.02012096904218197, -0.0016089003765955567, 0.0011336782481521368, -0.044095445424318314, 0.01039415504783392, 0.008622493594884872, -0.0005811877781525254, -0.013503208756446838, -0.010726242326200008, 0.04140971228480339, -0.0020045824348926544, 0.002290915697813034, 0.030540691688656807, 0.010267732664942741, 0.00122748757712543, -0.03249303251504898, 0.013428246602416039, -0.025039376690983772, -0.015598731115460396, 0.014026480726897717, -0.003595032263547182, -0.014430579729378223, 0.009653851389884949, 0.027796942740678787, 0.013645595870912075, 0.004937942139804363, -0.005961573217064142, 0.012709510512650013, 0.016107948496937752, -0.024919996038079262, 0.022383028641343117, -0.0400843508541584, 0.017738666385412216, -0.012396377511322498, -0.01115032285451889, 0.02330722287297249, 0.017826562747359276, 0.002692214446142316, -0.009873743169009686, -0.006627187132835388, 0.0234551839530468, -0.017386283725500107, -0.028171107172966003, -0.0003657820343505591, 0.003759723622351885, -0.004102673847228289, -0.03457590565085411, 0.016035063192248344, -0.020409494638442993, -0.013188266195356846, 0.018341893330216408, 0.000984141486696899, -0.022273344919085503, 0.010600988753139973, -0.0010220959084108472, 0.017720956355333328, 0.014994839206337929, 0.046142131090164185, 0.01481995265930891, 0.025874098762869835, -0.0006779239629395306, 0.0001364872296107933, -0.01175129134207964, 0.00019822397734969854, 0.0011859146179631352, 0.00970208179205656, -0.047502804547548294, 0.008719640783965588, -0.015440327115356922, 0.0280943363904953, -0.020504239946603775, -0.00584176043048501, -0.017881255596876144, 0.012473425827920437, -0.016949793323874474, -0.022176939994096756, -0.015997841954231262, 0.002877922262996435, -0.001453111064620316, -0.023312807083129883, -0.04639686644077301, 0.008548344485461712, 0.011609491892158985, -0.02631121687591076, -0.006513652391731739, 0.0005772800068370998, -0.019624747335910797, -0.008732318878173828, -0.04185710474848747, -0.02374398335814476, -0.0036316048353910446, -0.01797357387840748, 0.012139449827373028, 0.00806797668337822, -0.002559365937486291, 0.004113590344786644, 0.014930238015949726, -0.005616455804556608, 0.01605961099267006, 0.0005296589806675911, 0.012956185266375542, -0.025051046162843704, -0.022974060848355293, -0.0009610906126908958, -0.023249832913279533, 0.005764240399003029, -0.0032414270099252462, -0.0028698251117020845, 0.03262059763073921, -0.04455433040857315, 0.018127333372831345, 0.003370660590007901, 0.028621740639209747, -0.019676990807056427, -0.026540016755461693, -0.014136058278381824, -0.015451336279511452, 0.015852034091949463, -0.011627108789980412, -0.001506784581579268, -0.0016515071038156748, 0.00022165333211887628, -0.001789315603673458, -0.01294937264174223, -0.004430859349668026, 0.018952855840325356, 0.009517707861959934, -0.008571895770728588, 0.002634894335642457, 0.016381407156586647, -0.04088570550084114, -0.01567736454308033, -0.002217142377048731, -0.006777550559490919, 0.019763395190238953, -0.0018656564643606544, -0.005006185732781887, -0.03503524139523506, 0.03707338124513626, 0.011639872565865517, -0.032788410782814026, -0.014230785891413689, -0.02357589825987816, 0.028252096846699715, -0.002670444082468748, 0.0005481933476403356, -0.0195047315210104, -0.07215964794158936, 0.0014483158010989428, -0.013127151876688004, -0.006002671550959349, -0.015279620885848999, -0.01083823200315237, -0.019376294687390327, -0.0022628677543252707, -0.0003393412916921079, 0.020143749192357063, 0.011403507553040981, -0.010889779776334763, -0.01873309165239334, 0.04464345425367355, -0.004980144090950489, 0.03524571284651756, 0.05402083322405815, 0.0054533774964511395, -0.027799716219305992, -0.041579388082027435, -0.00022915529552847147, 0.011800228618085384, -0.0318925678730011, -0.033771488815546036, -0.004664553329348564, -0.016883663833141327, 0.010166924446821213, -0.012657087296247482, 0.009363746270537376, 0.0029464501421898603, 0.012192304246127605, 0.0063880025409162045, 0.003473655553534627, 0.005179912783205509, -0.0079443184658885, -0.0037568446714431047, -0.03390541300177574, 0.0325133316218853, 0.008106911554932594, -0.008707980625331402, -0.05865004286170006, -0.017343509942293167, -0.0007625686703249812, 0.034427449107170105, 0.011254612356424332, 0.003971107769757509, 0.012320229783654213, 0.037131015211343765, -0.03735924884676933, 0.0006477966089732945, 0.014452148228883743, 0.04267614707350731, 0.029256261885166168, -0.021984755992889404, 0.01788182184100151, 0.009707532823085785, -0.004279592074453831, -0.04107629507780075, 0.011900776997208595, -0.006128941662609577, 0.01644393615424633, -0.026099007576704025, -0.021482765674591064, 0.016256988048553467, -0.038545481860637665, 0.0010477682808414102, -0.001462019165046513, -0.007740837521851063, 0.01377035677433014, 0.03715335950255394, -0.005498775281012058, -0.003091889899224043, -0.0017053162446245551, 0.0029042165260761976, 0.009757270105183125, 0.018004247918725014, 0.00771337142214179, 0.008167366497218609, -0.005394733045250177, 0.041203998029232025, -0.012284597381949425, -0.01702558621764183, -0.006114211864769459, -0.014248375780880451, -0.0286746546626091, 0.00945264007896185, -0.018867306411266327, 0.04263494908809662, -0.019117187708616257, -0.008630869910120964, -0.03390023857355118, 0.030247775837779045, -0.016471831128001213, -0.030047234147787094, 0.0008754310547374189, -0.024976525455713272, 0.00416958425194025, -0.002742811804637313, 0.010935608297586441, 0.037008706480264664, 0.013164366595447063, -0.018792100250720978, -0.04440632835030556, 0.040282219648361206, 0.010800080373883247, -0.015829456970095634, 0.0011891325702890754, 0.005968181416392326, -0.02325725369155407, 0.011847675777971745, 0.019468069076538086, 0.0030073807574808598, 0.007260861806571484, -0.012638232670724392, 0.0007218144019134343, -0.038079582154750824, -0.009863629937171936, 0.06097353994846344, -0.006731696892529726, -0.006798392161726952, -0.009354783222079277, 0.016104714944958687, 0.008185417391359806, 0.02476414479315281, -0.011028504930436611, -0.009877671487629414, 0.003441027831286192, -0.007616874761879444, -0.018888143822550774, -0.015956655144691467, -0.011104385368525982, 0.0417683906853199, -0.014125075191259384, 0.04028232768177986, -0.008707699365913868, -0.03200773894786835, 0.014683508314192295, 0.010404057800769806, 0.036059826612472534, -0.018213903531432152, 0.03671102970838547, -0.040778644382953644, -0.037424784153699875, 0.016576595604419708, 0.03433465585112572, 0.01882230117917061, -0.023764053359627724, 0.012085841037333012, 0.02912723273038864, 0.0305783711373806, 0.02474929764866829, 0.030185826122760773, 0.008513614535331726, -0.01978897675871849, 0.027403267100453377, 0.024398306384682655, 0.022250080481171608, -0.051293134689331055, -0.015610422007739544, 0.03152727335691452, -0.013129102997481823, 0.03136167675256729, -0.023057853803038597, 0.014504866674542427, -0.04686181619763374, 0.018679935485124588, 0.006604472175240517, -0.015137676149606705, -0.005333337467163801, -0.0005283553036861122, -0.015838176012039185, -0.014095558784902096, 0.037073247134685516, 0.012111213058233261, -0.0034396005794405937, 0.023483069613575935, 0.010975830256938934, -0.029774103313684464, -0.027700120583176613, 0.007491875905543566, -0.02137574926018715, 0.009227464906871319, -0.008396253921091557, -0.020612092688679695, -0.012187744490802288, -0.04504094272851944, 0.02828347310423851, 0.006378361023962498, -0.0004430855333339423, 0.040577054023742676, 0.008583569899201393, -0.03080795146524906, 0.019449425861239433, 0.019609184935688972, -0.004060451872646809, 0.037009067833423615, -0.008818123489618301, -0.003594321897253394, 0.014115341007709503, -0.0015987352235242724, -0.023390429094433784, -0.006505643017590046, -0.0047896853648126125, 0.034249402582645416, -0.0003944914205931127, -0.03439071774482727, -0.011691978201270103, -0.012287236750125885, 0.0014220275916159153, -0.020282592624425888, -0.022142546251416206, 0.012398754246532917, -0.01808035559952259, 0.02361411228775978, 0.002357989316806197, -0.0030839238315820694, -0.005483467131853104, 0.007212017197161913, 0.007055032067000866, -0.02010612189769745, 0.02312570996582508, -0.00779234291985631, 0.012571750208735466, -0.0008438294171355665, 0.038572780787944794, -0.0018451621290296316, -0.015184166841208935, 0.0005653986008837819, 0.02044868655502796, -0.0033954735845327377, -0.03428016975522041, 0.0011669935192912817, -0.011136713437736034, 0.027894234284758568, 0.0078032235614955425, 0.0015240181237459183, 0.025030337274074554, -0.00012314884224906564, -0.03980724886059761, -0.0060373228043317795, -0.011659297160804272, 0.005443912465125322, -0.024472087621688843, -0.002734267618507147, 0.0009074019617401063, -0.01037651114165783, -0.019589915871620178, 0.01853504218161106, -0.023601600900292397, 0.026968400925397873, 0.013003074564039707, 0.006198724266141653, -0.010167287662625313, -0.0021041196305304766, -0.000900407787412405, -0.014976073056459427, -0.014776218682527542, 0.007093670777976513, -0.017220815643668175, 0.003847227431833744, -0.012537325732409954, -0.020506257191300392, 0.013103033415973186, -0.010903935879468918, -0.019407277926802635, -0.010823939926922321, 0.011926761828362942, -0.01570039801299572, -0.005520889535546303, -0.03217175975441933, 0.002081602346152067, -0.003607505699619651, -0.023279085755348206, 0.009865147061645985, 0.010449503548443317, 0.01645459234714508, -0.013114049099385738, -0.024905674159526825, 0.02705327607691288, -0.038448479026556015, 0.0015536787686869502, 0.0005428618169389665, 0.007316613104194403, 0.0008638521539978683, 0.003550192341208458, -0.006232039537280798, 0.0052183386869728565, -0.0032576685771346092, 0.005924063269048929, 0.051983460783958435, -0.027386369183659554, -0.05107514560222626, 0.005330911837518215, 0.04617762938141823, -0.019811440259218216, 0.026394030079245567, 0.003318569390103221, 0.011591502465307713, -0.008470488712191582, 0.031344879418611526, -0.0006229940918274224, 0.03204165771603584, 0.05205712839961052, 0.04513868689537048, 0.023939520120620728, -0.03622640296816826, 0.007336236070841551, 0.01920890435576439, -0.0033235521987080574, -0.006277819629758596, -0.015680212527513504, 0.050190042704343796, 0.010497858747839928, -0.004516207613050938, -0.0034957141615450382, 0.014717579819262028, 0.015796281397342682, -0.016701743006706238, 0.0019308578921481967, 0.010209668427705765, 0.0028960437048226595, 0.04405421391129494, 0.010026948526501656, 0.006499712821096182, -0.059390705078840256, -0.008033687248826027, 0.027368517592549324, 0.03773576393723488, 0.01439571287482977, 0.019898876547813416, -0.008424348197877407, -0.013500945642590523, -0.00984529685229063, 0.02104734629392624, 0.0035094749182462692, -0.020822538062930107, 0.025426648557186127, -0.012011942453682423, -0.018670979887247086, -0.0022105835378170013, 0.012339652515947819, -0.005640137009322643, 0.01568412221968174, 0.00016084450180642307, 0.02421858161687851, -0.0010219903197139502, 0.016936002299189568, -0.007178075145930052, 0.011845165863633156, 0.002851459663361311, -0.019452454522252083, 0.02884947694838047, -0.005110322497785091, 0.00874404702335596, -0.007991372607648373, 0.002531643956899643, 0.0008276905282400548, -0.009337073192000389, 0.004164325539022684, 0.0015897791599854827, 0.01390043180435896, -0.0022281967103481293, 0.018036814406514168, -0.004149631131440401, -0.004995148628950119, -0.002835844876244664, -0.005035890266299248, 0.0010985123226419091, 0.010736634954810143, 0.007045313715934753, -0.015006903558969498, -0.028933390974998474, -0.03185364231467247, 0.03659510985016823, -0.023274004459381104, -0.002600334817543626, 0.03729477897286415, 0.012030409649014473, 0.005759118124842644, -0.009820513427257538, -0.009703470394015312, 0.004735247232019901, 0.015395489521324635, 0.017886077985167503, 0.014020885340869427, -0.01832672767341137, -0.008081905543804169, -0.0011451086029410362, 0.007656150497496128, 0.03303394839167595, 0.001585831749252975, -0.007072883658111095, 0.008510143496096134, 0.006015490274876356, 0.019092893227934837, 0.0006725213606841862, -0.007445946801453829, -0.02660217694938183, -0.004462679848074913, 0.005245484411716461, -0.004561112727969885, 0.015190706588327885, 0.009393773972988129, 0.010485813021659851, -0.0310951117426157, 0.024797527119517326, 0.01370901707559824, -0.048874903470277786, -0.011467220261693, -0.043187543749809265, -0.014515452086925507, -0.019197462126612663, -0.02483588643372059, 0.012571671977639198, 0.017660528421401978, -0.006795337423682213, -0.02231486514210701, 0.02286630868911743, -0.013003963977098465, -0.002740289084613323, 0.010126961395144463, -0.0024056946858763695, -0.011371142230927944, -0.0039036753587424755, 0.007507537957280874, 0.01188979297876358, -0.007022046018391848, -0.028907036408782005, 0.008035611361265182, 0.0021950488444417715, 0.022773709148168564, 0.003617434296756983, -0.00994139350950718, -0.01402803510427475, 0.012667648494243622, -0.03586608171463013, -2.6570542104309425e-05, -0.02391720563173294, 0.007269666995853186, 0.00121961475815624, 0.0007541498634964228, 0.007986879907548428, 0.021582037210464478, 0.02480856142938137, -0.009157197549939156, 0.02300223708152771, 0.0028993042651563883, 0.008171065710484982, 0.003187631955370307, 0.005875354632735252, -0.011573303490877151, -0.022201409563422203, -0.014065506868064404, 0.008547350764274597, 0.034380339086055756, -0.0174112506210804, 0.021034004166722298, -0.026519514620304108, -0.0032709850929677486, 0.01908072456717491, 0.021297099068760872, 0.0008959720726124942, 0.04353439807891846, 0.004975061863660812, -0.003307651961222291, -0.03039049543440342, 0.001041907467879355, -0.0044533018954098225, -0.008066914044320583, 0.01420573703944683, -0.009057905524969101, -0.016629450023174286, -0.012163299135863781, 0.016379734501242638, -0.0038534938357770443, 0.024021508172154427, 0.01815965585410595, -0.004740632604807615, -0.044192250818014145, 0.007118212524801493, 0.0062002092599868774, 0.02276240661740303, 0.033895689994096756, 0.011807099916040897, -0.011315061710774899, -0.0149766206741333, -0.007129108998924494, -0.005915955640375614, -0.007950212806463242, -0.002051169751212001, 0.040511250495910645, -0.03090612031519413, -0.05258217453956604, -0.004567248746752739, -0.008112097159028053, -0.02738061547279358, 0.02291497401893139, 0.010092448443174362, 0.023796940222382545, 0.020960109308362007, 0.014779307879507542, -0.003799915313720703, -0.014320538379251957, 0.004620919935405254, 0.01668764278292656, -0.02706795744597912, -0.017510712146759033, -0.012907133437693119, -0.011384843848645687, -0.03232868015766144, -0.0072570363990962505, -0.007567806169390678, 0.0037626426201313734, -0.03243256360292435, -0.026119137182831764, 0.012064031325280666, 0.02570556290447712, 0.03849831223487854, 0.024178266525268555, -0.0067702606320381165, 0.01727292127907276, -0.027991339564323425, 0.021294832229614258, -0.010619083419442177, -0.05058039724826813, 0.03075961023569107, -0.0018628963734954596, 0.0233149491250515, -0.0034443687181919813, 0.023379351943731308, 0.01714484952390194, 0.013015656732022762, -0.03438608720898628, -0.01745307818055153, -0.0366845577955246, 0.010062321089208126, -0.016543082892894745, 0.01892445795238018, -0.012056590057909489, -0.0037290393374860287, 0.012764212675392628, -0.02081219106912613, 0.004776202142238617, -0.05438556894659996, -0.01698218286037445, 0.017380204051733017, 0.050962895154953, -0.04161908105015755, 0.030329035595059395, 0.027048371732234955, -0.009080256335437298, -0.018934082239866257, -0.004111594054847956, -0.0047553847543895245, 0.007233083713799715, -0.030611176043748856, 0.001658676192164421, -0.011517650447785854, -0.007065612822771072, 0.01360035315155983, 0.007435359992086887, -0.044381000101566315, 0.002058363053947687, 0.00037138801417313516, 0.012093172408640385, 0.003511045128107071, 0.0038908179849386215, -0.006991060916334391, -2.6724310373538174e-05, 0.016459882259368896, -0.029385508969426155, -0.005733259487897158, -0.008056034334003925, -0.014702437445521355, -0.04456121847033501, 0.022427156567573547, -0.008161530829966068, 0.02953198365867138, 0.048507209867239, -0.02068260684609413, 0.01584528759121895, 0.00031018309528008103, 0.010852857492864132, 0.013279346749186516, 0.015562059357762337, 0.010646204464137554, 0.002365138614550233, -0.018841689452528954, -0.011614890769124031, -0.024249842390418053, -0.02731599658727646, 0.00010325686889700592, -0.01551838032901287, -0.036076270043849945, -0.0391421765089035, -0.010137218981981277, -0.021314995363354683, 0.007974063977599144, 0.0001828458480304107, 0.00915944017469883, 0.027930358424782753, -0.016898736357688904, 0.008431977592408657, -0.012781583704054356, -0.015024160966277122, -0.0030787212308496237, -0.03356041759252548, 0.005978182889521122, -5.8162091590929776e-05, -0.005872682202607393, 0.0030359788797795773, -0.014753948897123337, 0.010149613954126835, -0.0454077385365963, -0.02750323712825775, 0.004960217513144016, -0.008648021146655083, 0.007177631836384535, -0.04577545076608658, -0.007846378721296787, -0.011506224051117897, -0.002247733762487769, -0.03076460398733616, 0.02127346582710743, 0.007699146401137114, 0.00551605224609375, 0.004875646438449621, -0.009329726919531822, -0.017608271911740303, 0.024143608286976814, 0.016749896109104156, -0.0029804043006151915, -0.021520674228668213, -0.005653781816363335, 0.025795932859182358, 0.01110182236880064, 0.03241682052612305, 0.026198793202638626, 0.027695225551724434, 0.010539104230701923, -0.0013388271909207106, -0.004138652700930834, -0.01947636716067791, -0.02075900137424469, 0.007311164401471615, -0.01483067311346531, -0.04231685772538185, 0.010064824484288692, -0.025056198239326477, -0.008576974272727966, -0.011239119805395603, -0.014400739222764969, 0.02450501173734665, 0.005379905458539724, -0.03674040734767914, 0.05454906448721886, -0.001401812070980668, 0.02364419959485531, 0.020365672186017036, 0.0016232742927968502, -0.03025183454155922, -0.04301989823579788, 0.038055676966905594, -0.03230063617229462, 0.018135078251361847, -0.03411991521716118, -0.023605993017554283, 0.01570373959839344, -0.014914129860699177, -0.002675831550732255, -0.015341535210609436, -0.02405700273811817, 0.020068665966391563, 0.0014589766506105661, -0.006108175031840801, -0.02157210372388363, -0.028703102841973305, -0.01367577351629734, 0.0024936916306614876, 0.00702056847512722, 0.005134523380547762, 0.012120399624109268, -0.03466428443789482, 0.016942713409662247, -0.014838466420769691, -0.012221704237163067, 0.02421867661178112, 0.011941337026655674, -0.007771320175379515, -0.0003553086135070771, -0.023015135899186134, -0.013044259510934353, 0.007272130344063044, -0.005686131305992603, 0.01859365403652191, -0.011404051445424557, 0.004042052663862705, -0.03484982252120972, -0.0033955464605242014, 0.0026847850531339645, -0.006961643695831299, -0.038689132779836655, 0.0010927246185019612, -0.03835615515708923, -0.011594239622354507, 0.007781801745295525, -0.004329668823629618, -0.02169601246714592, 0.00421507004648447, -0.03908851742744446, 0.017477672547101974, 0.005912883207201958, -0.05925556644797325, -0.0006228641723282635, 0.016940917819738388, 0.003742719069123268, -0.006414117757230997, -0.027559688314795494, -0.032036587595939636, -0.00017366169777233154, -0.000178527960088104, 0.024228235706686974, -0.025527363643050194, 0.029375435784459114, 0.01754452846944332, 0.0096624456346035, 0.003217062447220087, -0.01090078242123127, -0.019148608669638634, 0.005945248994976282, -0.01739797182381153, 0.00874326005578041, 0.031729698181152344, 0.03058961033821106, 0.02493032068014145, 0.005994481034576893, -0.0040637049823999405, 0.02611576020717621, 0.021846039220690727, -0.016983553767204285, -0.019223732873797417, 0.002078048186376691, 0.004412348847836256, -0.020256785675883293, -0.0356416217982769, -0.006229849066585302, -0.008177142590284348, 0.007184641435742378, -0.02382732555270195, 0.006622744258493185, -0.008961979299783707, 0.007994502782821655, 0.030770283192396164, 0.010559000074863434, 0.0003043414035346359, 0.018968621268868446, -0.027719326317310333, -0.00777170667424798, -0.0006689889705739915, -0.004641455132514238, -0.011889457702636719, 0.037501171231269836, 0.00530728604644537, 0.011570191010832787, 0.003400197019800544, -0.032401952892541885, 0.015403016470372677, 0.01444570254534483, -0.0023121654521673918, -0.006651693489402533, 0.0031254668720066547, -0.03437013551592827, 0.028549641370773315, -0.010125680826604366, 0.0051923030987381935, 0.0018574780551716685, -0.025473911315202713, 0.0043183802627027035, -0.0008175496477633715, -0.00033125735353678465, 0.00919023435562849, -0.02019195817410946, -0.02639526128768921, -0.0014687888324260712, 0.021216416731476784, 0.013922369107604027, 0.011930861510336399, -0.018487820401787758, -0.02450871840119362, 0.013535059988498688, 0.01784645766019821, 0.002194528467953205, 0.0012098216684535146, -0.005763231310993433, -0.03230010345578194, 0.007689696736633778, 0.013512251898646355, 0.009386345744132996, 0.0022290064953267574, 0.02104422077536583, -0.02040587179362774, -0.03283027186989784, 0.01684846356511116, -0.0010914793238043785, 0.015816010534763336, 0.001615166082046926, 0.012407843954861164, 0.02592587098479271, 0.004868410062044859, -0.026543430984020233, 0.04209231585264206, 0.0012115206336602569, 0.017957564443349838, -0.002642215695232153, -0.015127819962799549, -0.0065576243214309216, -0.05435853824019432, 0.008578937500715256, -0.013304907828569412, 0.03379164636135101, 0.013494363985955715, 0.02621944062411785, 0.013376235961914062, -0.011963741853833199, 0.0031592685263603926, 0.022212538868188858, -0.0025215663481503725, 0.005483733490109444, 0.003094242885708809, 0.010256897658109665, 0.0017706489888951182, 0.02793826162815094, 0.018403397873044014, 0.00395083287730813, 0.009837514720857143, -0.01031483057886362, 0.0007432192796841264, 0.008831017650663853, 0.008554235100746155, -0.03904862701892853, -0.010169177316129208, -0.02083788439631462, 0.04359806701540947, 0.026664653792977333, 0.013563744723796844, -0.010521379299461842, -0.0008521337294951081, 0.008050663396716118, 0.02886861003935337, -0.022040164098143578, -0.016355909407138824, -0.02338428981602192, -0.01959318481385708, -0.012122738175094128, -0.005828473251312971, 0.020829185843467712, -0.012632177211344242, 0.0038909015711396933, -0.005417992826551199, -0.029856422916054726, 0.021601639688014984, 0.017864298075437546, -0.00036300846841186285, -0.006697910837829113, 0.0013692619977518916, 0.012704947963356972, -0.013836360536515713, -0.012749981135129929, 0.006990055553615093, -0.020525284111499786, -0.02811509370803833, 0.0038882142398506403, -0.011129571124911308, -0.03194161877036095, 0.023480869829654694, 0.005673074163496494, -0.023636087775230408, -0.029434697702527046, -0.016123607754707336, -0.007408858742564917, 0.017120981588959694, -0.01820119470357895, 0.0019417620496824384, -0.03071349300444126, 0.005659434478729963, 0.008775662630796432, 0.02754816971719265, 0.015399784781038761, -0.02207360602915287, 6.341171683743596e-05, 0.012500815093517303, -0.013281899504363537, -0.011156003922224045, -0.0015441186260432005, -0.0002209999511251226, -0.02837691828608513, 0.004056442063301802, 0.016353270038962364, -0.006460321601480246, 0.006742306984961033, 0.00879089254885912, 0.001100708032026887, -0.012510597705841064, -0.012036812491714954, 0.02991599775850773, 0.025203339755535126, -0.03133533149957657, -0.011657499708235264, 0.008492915891110897, 0.002680225297808647, 0.042184118181467056, -0.019860567525029182, 0.009955180808901787, 0.004544805735349655, 0.0019577299244701862, -0.016492372378706932, -0.019514724612236023, -0.0067189354449510574, -0.040108732879161835, 0.027286222204566002, 0.0033953976817429066, 0.021855413913726807, 0.0048764473758637905, 0.04156908765435219, 0.005806336645036936, 0.003096014726907015, 0.033926453441381454, 0.015854444354772568, 0.049144573509693146, 0.0035346143413335085, 0.0034688219893723726, 0.016149813309311867, 0.005063994787633419, 0.005578495096415281, 0.013195221312344074, -0.023201115429401398, -0.0019376024138182402, 0.01148737221956253, -0.01573222316801548, -0.023662589490413666, -0.046915993094444275, -0.009622841142117977, -0.008331174030900002, -0.011127475649118423, 0.02961934171617031, 0.03843943774700165, -0.018049927428364754, -0.021034885197877884, 0.004713382106274366, 0.011340463533997536, -0.019545570015907288, -0.011232409626245499, 0.0030014014337211847, -0.034936223179101944, -0.020398207008838654, 0.008167551830410957, 0.015279542654752731, -0.0023684101179242134, -0.007578376680612564, 0.014712695963680744, -0.0020368078257888556, -0.029074115678668022, 0.007712682243436575, 0.02026325650513172, 0.0074858092702925205, -0.008450666442513466, -0.008168819360435009, -0.027501262724399567, -0.024160778149962425, -0.01275099441409111, -0.008805177174508572, -0.014500156044960022, -0.002284399699419737, -0.0036357389762997627, 0.003941165283322334, -0.03177936002612114, 0.012722818180918694, 0.03760991245508194, 0.0009843101724982262, 0.0184396430850029, 0.002551975194364786, -0.03655790165066719, 0.020962560549378395, -0.0036187423393130302, -0.029353925958275795, -0.03928101062774658, 0.014063326641917229, 0.010787512175738811, -0.011792759411036968, -0.013666818849742413, 0.015535726211965084, -0.0030036154203116894, -0.011593669652938843, -0.02214774303138256, -0.0032538187224417925, -0.037841759622097015, 0.02008531615138054, -0.0016044769436120987, 0.00044794779387302697, -0.02118990384042263, -0.0021009843330830336, -0.01651870273053646, -0.007407919969409704, -0.02769784815609455, -0.0028699063695967197, -0.016244394704699516, -0.009316282346844673, -0.01573259010910988, -0.03390464186668396, 0.03409140557050705, -0.035552479326725006, -0.03458288684487343, -0.0290642399340868, 0.022927885875105858, 0.005788732785731554, -0.01450059749186039, -0.011450976133346558, 0.022056661546230316, -0.005345852579921484, -0.018356002867221832, -0.01748727262020111, 0.030067196115851402, -0.02960098162293434, 0.04042218625545502, -0.0017163074808195233, 0.048120804131031036, 7.344154437305406e-05, -0.009036554954946041, -0.036669015884399414, -0.04179544746875763, -0.009162950329482555, -0.020499875769019127, -0.0168998334556818, -0.018598036840558052, -0.011981123127043247, -0.0285753533244133, 0.02742985263466835, 0.003974856808781624, 0.025010880082845688, 0.004086470231413841, -0.014997439458966255, 0.018714996054768562, 0.03158162906765938, -0.0025254085194319487, 0.010103672742843628, 0.006424312479794025, 0.00026208418421447277, 0.01920689269900322, 0.010172847658395767, 0.022951532155275345, -0.01999698579311371, -0.014473656192421913, 0.02299278974533081, -0.005410730838775635, 0.008847137913107872, -0.024739213287830353, 0.011601625941693783, 0.026805713772773743, -0.015654081478714943, 0.009974702261388302, 0.005869060754776001, -0.0017705658683553338, -0.024897443130612373, 0.010410495102405548, 0.013795323669910431, 0.0018857710529118776, -0.01897290162742138, 0.017828209325671196, 0.006353069096803665, -0.0035131429322063923, 0.015910908579826355, -0.011042858473956585, 0.0023849215358495712, 0.005179446656256914, 0.008328317664563656, -0.01882920041680336, -0.0019945758394896984, 0.0021050325594842434, 0.00557054253295064, 0.010553992353379726, -0.002482624491676688, 0.01807941310107708, -0.0049360389821231365, -0.01810631901025772, 0.006186983082443476, -0.01344201061874628, -0.014405209571123123, 0.04103583097457886, 0.002367104170843959, 0.0019925839733332396, 0.02390236221253872, -0.018984779715538025, -0.018501143902540207, 0.041114743798971176, -0.002995097078382969, -0.008802557364106178, 0.016374148428440094, -0.042885757982730865, 0.0037799025885760784, -0.0076712965965271, 0.020454294979572296, 0.013483666814863682, 0.015391659922897816, 0.023654382675886154, 0.006983104627579451, -0.018158800899982452, 0.016062337905168533, 0.013039317913353443, 0.024282053112983704, 0.011914968490600586, 0.0045675612054765224, 0.034067705273628235, -0.021389074623584747, 0.011929554864764214, -0.01650473289191723, -0.02206466905772686, 0.0045324149541556835, -0.0022168257273733616, -0.040845416486263275, 0.01418529823422432, -0.005240199621766806, 0.01467621698975563, 0.03139139711856842, -0.000979205477051437, -0.015359301120042801, 0.0008589765639044344, -0.020458804443478584, -0.00916933175176382, 0.00906270183622837, 0.0030570239759981632, 0.0030273234006017447, -0.02059848979115486, 0.021929504349827766, -0.07313722372055054, -0.00322727276943624, -0.006803068332374096, -0.022426104173064232, 0.0038434453308582306, -0.012355768121778965, -0.023601366207003593, -0.011430309154093266, 0.01240589190274477, 0.013321385718882084, 0.009499265812337399, -0.01632714457809925, -0.010291745886206627, 0.01806582696735859, -0.015428513288497925, 0.00945461168885231, -0.012980092316865921, 0.006049839314073324, -0.0033141798339784145, -0.011269910261034966, 0.0036396391224116087, 0.035121072083711624, 0.010452376678586006, 0.018407713621854782, -0.002284758724272251, 0.009270346723496914, 0.015851736068725586, -0.019296424463391304, 0.003911970648914576, -0.03035399504005909, -0.004585630726069212, 0.028077756986021996, -0.005380756221711636, 0.01914975605905056, -0.035185307264328, 0.018136972561478615, 0.008921555243432522, -0.008118159137666225, -0.001240606070496142, -0.02138889580965042, 0.0004233628569636494, -0.017771795392036438, 0.0014022475806996226, 0.039804428815841675, -0.005134460516273975, -0.01255621388554573, -0.0033581354655325413, 0.01871279440820217, 0.0067536490969359875, -0.0053521133959293365, -0.01992577686905861, -6.416861288016662e-05, -0.005375584587454796, 0.01668100245296955, -0.023730332031846046, 0.029070649296045303, 0.003961509093642235, 0.031073950231075287, -0.026029160246253014, -0.0074060107581317425, 0.042735613882541656, 0.013077383860945702, -0.024132389575242996, -0.01879539154469967, -0.017336057499051094, -0.0364072322845459, 0.017651991918683052, -0.00932544469833374, -0.025380713865160942, 0.00481990585103631, 0.01860080473124981, 0.002992651890963316, -0.040881626307964325, -0.03338431566953659, -0.009043389931321144, -0.011385728605091572, -0.0065186903811991215, 0.0025313266087323427, -0.003738077124580741, 0.0029486273415386677, -0.022107405588030815, 0.016296545043587685, -0.009066424332559109, -0.007822423242032528, 0.019168030470609665, -0.0061973221600055695, -0.03177842125296593, 0.009127701632678509, 0.029611650854349136, 0.00283405976369977, 0.008288022130727768, 0.0038773114793002605, 0.033085718750953674, -0.02132546156644821, -0.0018864430021494627, 0.01885238103568554, -0.014519497752189636, 0.023022405803203583, 0.014042245224118233, -0.004871532786637545, 0.009435305371880531, 0.021791405975818634, -0.013821356929838657, 0.014368373900651932, -0.009732798673212528, -0.009284970350563526, 0.024198142811655998, 0.0074187107384204865, -0.029362034052610397, -0.008191023021936417, -0.0046767001040279865, -0.005789812188595533, -0.016621146351099014, -0.0033942547161132097, 0.023889517411589622, 0.018852882087230682, -0.026058802381157875, 0.0012090237578377128, 0.04675907641649246, 0.014935052022337914, 0.01557194720953703, 0.0060770148411393166, 0.0028178333304822445, 0.02362162433564663, -0.03814951702952385, 0.011400728486478329, 0.00887916050851345, -0.030531983822584152, 0.0070646898820996284, -0.02558480016887188], [0.0020157943945378065, 0.008646347559988499, 0.029602939262986183, -0.016766512766480446, 0.010558005422353745, 0.0029028479475528, -0.011416342109441757, -0.018743062391877174, -0.021389974281191826, -0.00037037325091660023, 0.0002765032113529742, -0.008364599198102951, 0.02457720786333084, 0.0047453236766159534, -0.00742892362177372, -0.019750334322452545, 0.012806214392185211, -0.03963485360145569, 0.0029104151763021946, 0.03514590859413147, -0.015939829871058464, 0.008747562766075134, -0.011291099712252617, 0.005568770691752434, 0.00690479576587677, 0.006723823957145214, -0.008611183613538742, -0.011671489104628563, 0.0013259361730888486, 0.0013907910324633121, -0.0018915535183623433, 0.006240896414965391, -0.006888276431709528, -0.04205482453107834, -0.0036948062479496002, -0.011648653075098991, 0.013066724874079227, -0.06094039976596832, 0.008994240313768387, 0.005671874154359102, -0.02027755416929722, -0.032867033034563065, 0.002717524766921997, -0.00974371936172247, -0.0414767749607563, -0.03072332963347435, -0.008286826312541962, -0.020004186779260635, 0.05231031775474548, 0.02157253585755825, -0.0020569427870213985, 0.019147448241710663, -0.006297642365098, 0.04100564867258072, 0.019793765619397163, -0.0032993655186146498, -0.031804751604795456, 0.010156813077628613, -0.04562687873840332, 0.01438624132424593, -0.01922866702079773, 0.05389303341507912, -0.005888038780540228, -0.015335237607359886, 0.033438872545957565, 0.002481460804119706, -0.0020334157161414623, -0.03725695610046387, -0.01449582725763321, 0.006160467863082886, 0.0321422703564167, 0.019289230927824974, 0.004657523240894079, 0.002504283795133233, 0.018781939521431923, 0.007085336372256279, -0.0067698960192501545, -0.030454367399215698, 0.0030561115127056837, -0.014516111463308334, -0.015032561495900154, -0.007086510304361582, -0.022864023223519325, 0.024096177890896797, 0.005928230006247759, -0.0017889407463371754, -0.03646058961749077, 0.0020186600740998983, 0.009474746882915497, -0.03708938881754875, -0.029961397871375084, -0.006706880405545235, 0.020871814340353012, 0.0573132261633873, 0.024210434406995773, -0.01584045961499214, -0.0006963940686546266, 0.04742848500609398, 0.01263388991355896, 0.011869103647768497, 0.009350056760013103, -0.006937790662050247, -0.04563345015048981, -0.023666994646191597, -0.013859476894140244, 0.010548524558544159, 0.009687136858701706, 0.00441964203491807, -0.00897818710654974, 0.026072727516293526, 0.011892663314938545, 0.003984118811786175, -0.013239990919828415, 0.0032230569049715996, 0.015038232319056988, 0.016841033473610878, -0.003501037135720253, 0.00951485987752676, -0.0024200924672186375, 0.014463437721133232, -0.004520018119364977, 0.04123435914516449, 0.033406708389520645, -0.022184623405337334, 0.006018240470439196, 0.012758816592395306, 0.018114062026143074, -0.028963716700673103, -0.018031463027000427, 0.019277602434158325, 0.007421047892421484, 0.03037857823073864, 0.02515283040702343, -0.015793144702911377, 0.002024083398282528, -0.008663306012749672, 0.008688905276358128, 0.03759446740150452, -0.002765989862382412, 0.023452812805771828, 0.009646432474255562, 0.009112730622291565, 0.0029021715745329857, 0.0017998277908191085, 0.016389936208724976, -0.007661258801817894, 0.016945654526352882, 0.00414154352620244, -0.01584714837372303, -0.02004239335656166, 0.02704799734055996, -0.02905125916004181, 0.010910207405686378, 0.07190123945474625, -0.021039742976427078, 0.010214507579803467, -0.02632124163210392, 0.01317993737757206, 0.014825016260147095, 0.006845764350146055, 0.015126867219805717, -0.019543958827853203, 0.002483772812411189, -0.020214878022670746, 0.013584833592176437, -0.028098737820982933, 0.01048593781888485, 0.02688276208937168, 0.02739393152296543, 0.0035912992898374796, 0.004163335543125868, -0.010063467547297478, 0.0034707835875451565, -0.017200378701090813, -0.0377645380795002, -0.011186736635863781, 0.012995135970413685, -0.02235477976500988, 0.014156860299408436, 0.043946266174316406, 0.006071253214031458, 0.00501988735049963, -0.04006262868642807, -0.007699566427618265, -0.02841903641819954, 0.013245591893792152, -0.015541281551122665, -0.0009077744325622916, -0.01352645456790924, -0.0008928070310503244, -0.004298110026866198, 0.014959394000470638, 0.04583358019590378, 0.009010277688503265, -0.014978460967540741, -0.009144841693341732, -0.007004610728472471, -0.025320351123809814, 0.0007821671315468848, -0.016714995726943016, -0.003555026836693287, 0.005611936561763287, 0.039055921137332916, -0.0017942660488188267, 0.022670874372124672, 0.025462135672569275, -0.021098380908370018, -0.0019915730226784945, -0.012506920844316483, 0.03219582140445709, 0.01669606752693653, 0.010147538967430592, 0.0009102938347496092, -0.03374858573079109, -0.006106243934482336, -0.015321323648095131, -0.0003610455896705389, -0.02490551583468914, -0.03172799199819565, -0.021667810156941414, 0.012763087637722492, 0.01101349201053381, 0.004121917299926281, 0.003254339098930359, -0.008489564061164856, 0.01420924998819828, -0.01497635804116726, -0.002824062015861273, 0.013794861733913422, 0.01502870675176382, -0.01890834979712963, -0.006299535743892193, 0.014147313311696053, 0.0026332398410886526, 0.007449137046933174, 0.017351465299725533, 0.005834593903273344, -0.01924115978181362, -0.020576253533363342, 0.003989520017057657, 0.03192070126533508, -0.006979245692491531, 0.01897873543202877, -0.0029812483116984367, 0.034064896404743195, -0.03817528858780861, -0.02517269179224968, 0.023000188171863556, 0.022185197100043297, -0.008146459236741066, 0.01967761293053627, -0.007431270554661751, -0.00183595169801265, 0.011511432006955147, -0.0010121006052941084, -0.00722094438970089, -0.02964441291987896, -0.02401038631796837, -0.01543734036386013, 0.030682982876896858, -0.021613460034132004, 0.012877708300948143, -0.020768100395798683, 0.008470491506159306, -0.0018649970879778266, -0.027919331565499306, 0.019563334062695503, -0.012317108921706676, -0.026662081480026245, -0.00462517561390996, 0.00168196065351367, 0.01151854358613491, -0.020340846851468086, -0.024559609591960907, 0.028472481295466423, 0.026061858981847763, -0.0006586293457075953, 0.01121151726692915, 0.021329717710614204, -0.003957884386181831, -0.011191562749445438, 0.0024686201941221952, -0.005599000491201878, -0.006775735877454281, 0.019477805122733116, 0.025569425895810127, 0.009164855815470219, 0.017592623829841614, -0.02151496708393097, 0.01021575927734375, 0.01564016006886959, 0.008258500136435032, -0.03198976069688797, -0.02048715204000473, 0.010185283608734608, -0.03185036778450012, 0.005527497734874487, -0.0014894456835463643, 0.039416298270225525, -0.013111889362335205, -0.011136854998767376, 0.02430691570043564, 0.024723894894123077, -0.022033968940377235, -0.0023640042636543512, 0.004605469759553671, -0.005352325737476349, 0.00012283526302780956, 0.012334093451499939, -0.044720180332660675, 0.01382457185536623, -0.0020789767149835825, 0.003996499814093113, 0.01701691374182701, 0.004806955810636282, -0.005147418938577175, 0.03200678154826164, -0.01558886468410492, -0.02498042955994606, -0.054770711809396744, 0.01992487721145153, -0.01877736486494541, 0.022980380803346634, 0.018690939992666245, 0.01930411159992218, -0.0197712741792202, 0.01011588703840971, 0.03105292282998562, -0.010180745273828506, -0.017734060063958168, 0.0025328851770609617, -0.018440712243318558, -0.009219449944794178, -0.0008356220205314457, -0.0026935432106256485, 0.005239944439381361, -0.017084209248423576, -0.016431551426649094, -0.0014135182136669755, 0.022547472268342972, 0.006392146460711956, -0.05060969665646553, 0.0005179038271307945, -0.010196594521403313, -0.01063249260187149, -0.004985152278095484, -0.00820954330265522, -0.005344943609088659, 0.013503025285899639, 0.025206543505191803, -0.011121336370706558, -0.002056182362139225, 0.02943023480474949, -0.023861488327383995, -0.004255644511431456, 0.005282395053654909, -0.013797701336443424, -0.008132426999509335, 0.02978823147714138, 0.011701737530529499, -0.026399660855531693, 0.06019974499940872, 0.008236395195126534, 0.008125828579068184, -0.00841096043586731, 0.013720769435167313, 0.010520691983401775, -0.00015138725575525314, 7.870847184676677e-05, 0.0008825006079860032, -0.019003955647349358, 0.018263651058077812, -0.012714541517198086, -0.03610772639513016, -0.0493614636361599, 0.002812178572639823, -0.0370531901717186, -0.0021648115944117308, 0.0030953488312661648, -0.00364479748532176, 0.030751826241612434, -0.008958925493061543, 0.0006727780564688146, 0.008760690689086914, 0.025054771453142166, -0.03161967173218727, 0.044768571853637695, -0.008920056745409966, 0.0341532900929451, -0.01686231605708599, -0.009964770637452602, 0.008643604815006256, -0.006272707134485245, 0.007187931798398495, -0.005722205154597759, 0.03985011950135231, -0.016890794038772583, 0.007009949069470167, 0.025291355326771736, 0.0062912991270422935, -0.03673554211854935, 0.02506890334188938, -0.025135375559329987, -0.01724965125322342, 0.03136393055319786, 0.030419612303376198, -0.002840251661837101, 0.028698822483420372, -0.030350858345627785, 0.0030196900479495525, -0.012698779813945293, -0.02576545998454094, -0.04382481053471565, 0.025200914591550827, -0.0012623893562704325, 0.018759362399578094, 0.011357799172401428, -0.0034557327162474394, -0.0008387822308577597, -0.031372226774692535, -0.0060870773158967495, 0.021450523287057877, 0.030267784371972084, -0.01722448505461216, 0.03235360234975815, -0.024515800178050995, 0.00406020600348711, 0.0036035615485161543, 0.01926005817949772, -0.015762856230139732, 0.01599441096186638, -0.010133360512554646, -0.0277781430631876, -0.014317424036562443, -0.0012694316683337092, 0.018465960398316383, 0.013129409402608871, -0.013895636424422264, 0.010289927013218403, 0.014976614154875278, 0.016197338700294495, 0.003775406628847122, -0.013044478371739388, -0.019070662558078766, 0.005305551923811436, 0.012203894555568695, -0.018158620223402977, -0.011978467926383018, 0.027597688138484955, 0.020422598347067833, 0.010258697904646397, 0.0027342119719833136, -0.00677087577059865, -0.015937188640236855, 0.020853688940405846, -0.01895321160554886, 0.03880266472697258, 0.033007487654685974, -0.021120911464095116, -0.03855401650071144, 0.001325042569078505, -0.024131782352924347, -0.012114956974983215, 0.005520202219486237, -0.028986044228076935, 0.004155024420469999, 0.02547244168817997, 0.026151984930038452, 0.02093943953514099, -0.005459366831928492, -0.009179229848086834, -0.011334655806422234, 0.003041163319721818, 0.014763486571609974, -0.0021701690275222063, -0.027670329436659813, -0.026407217606902122, -0.0158870667219162, 0.0008366229594685137, 0.012494374997913837, -0.001938869128935039, 0.0138833187520504, -0.003320142161101103, 0.003988924901932478, 0.00838212575763464, -0.004532415885478258, 0.016765788197517395, 0.02806653268635273, 0.010439508594572544, -0.007434615399688482, 0.0022116091568022966, 0.020290223881602287, 0.0022122871596366167, -0.005201887805014849, -0.011010115034878254, -0.0025601203087717295, -0.0012994323624297976, -0.032302312552928925, 0.012803161516785622, -0.02452048286795616, 0.018280692398548126, -0.06439191848039627, -0.0012672417797148228, 0.021633584052324295, 0.0201419647783041, -0.031758230179548264, -0.02690965309739113, 0.008794166147708893, -0.024997176602482796, 0.01862442120909691, 0.02984297089278698, -0.021091237664222717, -0.02402419224381447, -0.007431428879499435, 0.015611252747476101, 0.011154122650623322, 0.006541888229548931, -0.02855973318219185, 0.025817876681685448, -9.400781709700823e-05, 0.012146477587521076, 0.0031848850194364786, 0.015916192904114723, 0.024591557681560516, 0.014632985927164555, -0.057023342698812485, -0.015826428309082985, 0.013667439110577106, -0.023075124248862267, 0.042498890310525894, -0.020336462184786797, 0.0034530633129179478, 0.010931935161352158, 0.017314422875642776, 0.04187267646193504, -0.03804512694478035, 0.020999634638428688, 0.013742967508733273, 0.0011523431167006493, -0.004389683250337839, -0.014340993948280811, 0.01760454662144184, -0.008364417590200901, 0.013348647393286228, 0.04375526309013367, 0.02617173083126545, 0.016249392181634903, 0.019487421959638596, -0.06357493251562119, 0.016681933775544167, -0.008488506078720093, -0.022823546081781387, 0.017369113862514496, 0.04547150433063507, -0.019632183015346527, 0.002846148330718279, -0.015866778790950775, -0.009340958669781685, 0.02377951890230179, -0.0096355639398098, -0.004460831638425589, -0.0008707174565643072, -0.03136346861720085, -0.008759086951613426, 0.013592256233096123, 0.014269624836742878, -0.002109096385538578, 0.021350756287574768, 0.02693861350417137, -0.038951992988586426, -0.006020746659487486, 0.007139771245419979, 0.015360992401838303, 0.001501942635513842, 0.021177541464567184, -0.009477381594479084, 0.01969308592379093, -0.016825435683131218, -0.025054486468434334, 0.029861710965633392, 0.006071802228689194, 0.0009526836220175028, -0.02801419422030449, 0.021139830350875854, -0.005621856078505516, 0.015254396013915539, -0.00657031312584877, -0.007618223782628775, -0.015564112924039364, -0.012260337360203266, -0.0008166981860995293, 0.00024331343593075871, -0.0007182543049566448, -0.010967852547764778, 0.0035253590904176235, 0.010896984487771988, -0.012726310640573502, 0.008309006690979004, 0.02551765739917755, -0.051857978105545044, 0.016592491418123245, -0.026488324627280235, 0.024765048176050186, 0.0031081095803529024, 0.01944597065448761, -0.013277373276650906, -0.0033093388192355633, -0.033529821783304214, -0.004809370264410973, 0.004298496525734663, 0.014079867862164974, -0.014919583685696125, 0.006691941525787115, -0.011750269681215286, 0.013491952791810036, 0.019117437303066254, -0.006545422598719597, -0.006143874954432249, 0.003049402730539441, -0.03921393305063248, -0.01890006847679615, 0.004868605174124241, -0.008873071521520615, 0.023853033781051636, -0.03068603202700615, 0.016376009210944176, -0.01425014529377222, -0.02636893279850483, 0.004974282346665859, -0.015785561874508858, -0.02743830718100071, 0.008632183074951172, -0.021279023960232735, -0.02199382893741131, 0.006022270303219557, -0.015467492863535881, -0.008149511180818081, 0.04094971716403961, 0.011794903315603733, -0.0023941099643707275, -0.0516519621014595, -0.003923250362277031, -0.005127532873302698, 0.006825012620538473, -0.01538166031241417, 0.03160657361149788, -0.0013637185329571366, -0.006646541878581047, 0.03191208839416504, 0.007999209687113762, 0.0018945311894640326, -0.011593545787036419, 0.003594941459596157, -0.008097360841929913, -0.021403253078460693, 0.013155845925211906, 0.006780322641134262, 0.011146309785544872, 0.006693382281810045, -0.0017748133977875113, -0.017025761306285858, 0.003655752632766962, -0.008969210088253021, 0.02532684989273548, 0.036137569695711136, -0.033250048756599426, -0.00019732015789486468, -0.0091328676789999, 0.0045837280340492725, 0.03361416980624199, 0.016129327937960625, -0.04004157707095146, 0.021431326866149902, -0.011665552854537964, 0.005762293469160795, -0.01377637218683958, -0.014367416501045227, -0.013225444592535496, 0.002728287363424897, 0.031239088624715805, -0.0075972313061356544, -0.008535846136510372, -0.022180408239364624, -0.006119552534073591, 0.002063618740066886, -0.012345540337264538, 0.01304303202778101, -0.012262232601642609, -0.0022659613750874996, 0.03097998909652233, 0.046048201620578766, 0.046502579003572464, -0.012544701807200909, -0.008039000444114208, -0.021255148574709892, -0.01786649040877819, -0.004949768073856831, 0.021622680127620697, 0.02613738738000393, -0.012915217317640781, -0.005555289331823587, 0.01951318420469761, 0.023651761934161186, 0.006338084116578102, -0.013262447901070118, 0.006135913077741861, 0.03705110028386116, 0.037103403359651566, 0.00019544428505469114, -0.023517746478319168, -0.00853592250496149, -0.032054390758275986, 0.0019395375857129693, -0.005859609227627516, -0.007236444391310215, 0.011298074387013912, -0.007097638212144375, 0.02122529037296772, -0.002395808696746826, 0.004210230894386768, 0.02158299647271633, -0.025525541976094246, 0.03864223137497902, -0.003010709537193179, -0.01871586963534355, 0.008492155000567436, 0.041358593851327896, -0.008195649832487106, 0.015262139961123466, 0.027402380481362343, 0.02904665097594261, 0.0085599385201931, -0.02181367203593254, -0.007436878047883511, 0.030112292617559433, -0.024497954174876213, 0.02503388375043869, 0.005902243312448263, 0.009943985380232334, 0.004518146626651287, -0.016376206651329994, 0.0480746366083622, -0.02597980760037899, -0.0029338651802390814, -0.02788669243454933, 0.007221384905278683, 0.01024765707552433, -0.051828790456056595, 0.008590174838900566, -0.0032450994476675987, 0.016284765675663948, -6.693712202832103e-05, -0.020587913691997528, 6.834432861069217e-05, -0.013759614899754524, 0.015224293805658817, 0.028937704861164093, 0.03038414567708969, -0.0032636409159749746, -0.01898433081805706, -0.0016419722232967615, 0.025517966598272324, 0.00670150201767683, 0.02384379133582115, -0.002162829739972949, 0.00027451495407149196, 0.006821747403591871, 0.017776908352971077, 0.08397627621889114, 0.033274099230766296, -0.0034359705168753862, 0.00810664426535368, -0.010733144357800484, 0.01948917843401432, 0.01276425365358591, -0.004578715655952692, 0.00377180939540267, -0.006718857679516077, -0.007699777837842703, 0.02488110587000847, -0.00030310481088235974, -0.01420251652598381, -0.021980740129947662, -0.016141336411237717, -0.018885402008891106, 0.016904989257454872, -0.014540757983922958, 0.00462961383163929, 0.0018582413904368877, 0.010757695883512497, -0.02234809659421444, -0.01303051970899105, 0.016399413347244263, 0.013204230926930904, -0.01459001936018467, 0.030297838151454926, 0.005801203660666943, -0.0022004132624715567, 0.00457310164347291, -0.02001209370791912, 0.005101628135889769, 0.002634183969348669, 0.050770387053489685, -0.007392656058073044, 0.0007349999505095184, -0.0336398109793663, -0.029498310759663582, -0.004093785770237446, 0.03849717974662781, -0.010816258378326893, 0.012666652910411358, 0.0004522119415923953, 0.009991547092795372, 0.022167634218931198, -0.004265187308192253, 0.005736629944294691, -0.0030768949072808027, -0.03913028910756111, 0.00938141904771328, 0.0019789948128163815, 5.5891221563797444e-05, 0.01758503168821335, -0.008117680437862873, 0.02926602214574814, -0.0013578498037531972, -0.017398959025740623, 0.00764952227473259, 0.03982153162360191, -0.011863864958286285, 0.003635914996266365, -0.03280073031783104, 0.02589692734181881, -0.02138422057032585, -0.02168615162372589, -0.003223732579499483, -0.016254177317023277, -0.0032090137246996164, -0.02344254031777382, -0.01094540860503912, 0.0035979589447379112, 0.020527740940451622, -0.018263697624206543, -0.030154287815093994, -0.01008540391921997, 0.002496646950021386, -0.00506069790571928, 0.00543888658285141, -0.011121921241283417, 0.003910345025360584, 0.02259198948740959, -0.020500650629401207, -0.0039039445109665394, -8.605769107816741e-05, -0.009055737406015396, -0.012398789636790752, -0.04995451122522354, -0.01389936450868845, -4.515632463153452e-05, 0.0021888173650950193, 0.03252332657575607, -0.028298525139689445, -0.03620181977748871, -0.00034195207990705967, -0.010035079903900623, 0.009118783287703991, -0.015972713008522987, -0.022133300080895424, -0.003670042147859931, -0.010984196327626705, 0.018259378150105476, -0.01346066314727068, -0.007924985140562057, -0.004765386693179607, -0.02152971178293228, -0.03764227032661438, -0.0026097383815795183, -0.0006955599528737366, -0.017141375690698624, -0.002274306258186698, -0.023509206250309944, -0.0015533504774793983, -0.01450798287987709, -0.025576181709766388, 0.010820912197232246, 0.025268351659178734, 0.00023288934607990086, 0.010276857763528824, 0.03827599808573723, 0.00871802307665348, 0.03255597874522209, -0.0031226584687829018, -0.015927253291010857, 0.03750304877758026, -0.03206548094749451, -0.028047150000929832, 0.01128168124705553, 0.004095432814210653, -0.024344557896256447, 0.019490959122776985, 0.0008272822597064078, -0.010725659318268299, 0.0013936856994405389, -0.0014845479745417833, -0.01396638248115778, 0.0020171080250293016, -0.016251394525170326, 0.005902411416172981, 0.029867051169276237, 0.01610596664249897, 0.007117822766304016, -0.00011434291809564456, 0.04609035700559616, -0.008783801458775997, -0.025335760787129402, 0.0095172468572855, -0.003657728899270296, 0.006050341296941042, -0.013266997411847115, -0.040481630712747574, 0.0006973037961870432, 0.023104725405573845, -0.028249546885490417, 0.006285862065851688, 0.016022371128201485, 0.0035813406575471163, 0.023973027244210243, 0.0019733523949980736, -0.034618888050317764, -0.02622099034488201, -0.028391839936375618, 0.013304694555699825, -0.0008132831426337361, 0.0075655728578567505, -0.007665588054805994, -0.027279021218419075, 0.0022354810498654842, 0.013838489539921284, 0.018211716786026955, -0.008477921597659588, 0.024113748222589493, 0.003923495300114155, -0.002928170608356595, 0.03523000329732895, 0.0025657506193965673, -0.006023917347192764, 0.013318722136318684, 0.01739235408604145, 0.0018090388039126992, -0.0009784376015886664, -0.03406275063753128, 0.0029597522225230932, -0.03213687986135483, -0.028992604464292526, -0.0008071112097240984, -0.01972593180835247, -0.02032400295138359, -0.014226747676730156, -0.006873382721096277, -0.0004979384248144925, -0.02262934483587742, 0.0076468102633953094, 0.017836712300777435, 0.002801937283948064, -0.009176114574074745, -0.001983068184927106, 0.01116846688091755, -0.01396135427057743, -1.9781931769102812e-05, -0.033077966421842575, -0.0026860395446419716, 0.0025218017399311066, 0.0358886644244194, 0.02655602991580963, 0.027747957035899162, -0.01969396136701107, -0.01387618575245142, 0.02341335266828537, 0.005004377104341984, -0.013782392255961895, -0.037210434675216675, 0.011034430004656315, -0.0071545736864209175, -0.02099723368883133, 0.010305335745215416, 0.004069268703460693, 0.014674759469926357, 0.009901722893118858, -0.008857142180204391, -0.02213163860142231, 0.015347215346992016, 0.02292700670659542, 0.0026182024739682674, -0.006439055781811476, 0.014486820437014103, 0.007692679762840271, 0.01677524857223034, -0.00837977509945631, -0.0002180917508667335, -0.023029569536447525, -0.023267898708581924, -0.018025819212198257, 0.0018454715609550476, 0.00963657908141613, 0.041470397263765335, -0.012138527818024158, -0.01229962520301342, 0.022382598370313644, -0.03034188598394394, -0.020072266459465027, 0.006192236673086882, 0.0038887064438313246, -0.019031554460525513, 0.029232939705252647, 0.026810379698872566, -0.02028372883796692, -0.015422875061631203, -0.02880151942372322, 0.003481089835986495, -0.004548106342554092, -0.008640174753963947, 0.04913971945643425, 0.003768880618736148, -0.028782209381461143, -0.016396140679717064, 0.0028156975749880075, -0.001270715263672173, 0.013833411037921906, 0.007819777354598045, 0.02667117863893509, 0.030867019668221474, 0.023113392293453217, 0.025549819692969322, -0.026570893824100494, -0.004806813318282366, 0.011335661634802818, 0.024540308862924576, -0.016883092001080513, 0.025944970548152924, -0.019733456894755363, 0.002162632066756487, 0.02749861218035221, -0.0082515524700284, 0.007642462849617004, -0.026355890557169914, 0.004600201267749071, 0.013813324272632599, 0.006963344290852547, 0.0019361809827387333, -0.010789685882627964, -0.026939043775200844, 0.0045593804679811, -0.017511680722236633, -0.004347908776253462, -0.012380626983940601, 0.021051598712801933, 0.019515153020620346, 0.03092891164124012, 0.008420543745160103, -0.02701941318809986, -0.004008626099675894, -0.013037927448749542, -0.03155642747879028, -0.004173013847321272, -0.023578055202960968, 0.04281868785619736, 0.052902791649103165, -0.013431428000330925, 0.011466513387858868, -0.0041263713501393795, 0.0019051270792260766, -6.169996049720794e-05, 0.008127539418637753, -0.004558071494102478, 0.013968192040920258, -0.040560852736234665, -0.01752478815615177, 0.02579588256776333, 0.0034028179943561554, 0.0034294186625629663, 0.002685630228370428, -0.028538022190332413, -0.013920202851295471, 0.015469223260879517, 0.0068610734306275845, -0.03236791491508484, 0.033190976828336716, 0.029199086129665375, -0.019724970683455467, 0.009813854470849037, -0.02729996107518673, -0.026825062930583954, -0.03796076029539108, 0.03267078846693039, 0.026364127174019814, 0.03135795518755913, -0.03464221581816673, 0.013847383670508862, -0.01880599930882454, 0.03955826535820961, 0.030106237158179283, 0.022108646109700203, 0.03135579824447632, 0.01672123372554779, 0.012442953884601593, -0.005248378962278366, -0.0003164616064168513, 0.006314449943602085, -0.009141260758042336, -0.02495891973376274, 0.0073730433359742165, -0.023228006437420845, 0.035558413714170456, 0.03901421278715134, 0.019774043932557106, -0.002556008053943515, 0.01365579105913639, 0.011536943726241589, 0.02699892222881317, -0.03724753484129906, 0.03869912028312683, 0.021241234615445137, 0.014638522639870644, -0.008103035390377045, -0.0176993515342474, 0.015108571387827396, -0.007552880793809891, -0.0458647720515728, -0.018324166536331177, -0.003153344616293907, -0.032679278403520584, -0.02836371399462223, 0.011479891836643219, -0.01746230758726597, -0.0039873747155070305, 0.016233155503869057, -0.01196517888456583, -0.0030049362685531378, 0.025574831292033195, 0.0316048301756382, -0.044352348893880844, 0.011547928676009178, 0.0252534206956625, -0.0006445547915063798, -0.0007953887688927352, -0.014638307504355907, 0.027730168774724007, -0.006278240121901035, 0.025854581966996193, 0.020281976088881493, -0.021496588364243507, 0.00855301320552826, -0.014878040179610252, 0.01341242901980877, -0.011096471920609474, -0.017990317195653915, 0.024800240993499756, 0.0064860060811042786, 0.006341536995023489, 0.018152881413698196, 0.016579050570726395, 0.002525710267946124, 0.0006244686082936823, 0.002844088012352586, 0.006871385034173727, 0.004688571207225323, 0.03315906599164009, 0.0006173279834911227, -0.0008386048139072955, 0.04166267439723015, -0.013746545650064945, 0.011208219453692436, -0.03351239487528801, -0.041443757712841034, -0.018795926123857498, 0.007964869029819965, 0.0005058710812591016, 0.03435635566711426, 0.002403217600658536, 0.016591381281614304, -0.0029581061098724604, -0.009843683801591396, -0.007127304561436176, -0.01364370621740818, 0.01608993671834469, -0.018499381840229034, -0.008006415329873562, -0.0049276938661932945, -0.018931502476334572, 0.03339162841439247, 0.029456036165356636, 0.005166090093553066, 0.016338268294930458, -0.036772940307855606, -0.0022180466912686825, 0.027978399768471718, 0.0007987656281329691, -0.02068345434963703, -0.020138593390583992, 0.020176349207758904, -0.01579548604786396, 0.01953691989183426, -0.03133787214756012, 0.009074081666767597, 0.012334965169429779, 0.004762638825923204, -0.0008022177498787642, 0.013664503581821918, -0.008760345168411732, 0.005253488663583994, 0.004892683122307062, 0.008272913284599781, -0.01310033444315195, 0.020135696977376938, -0.03699912875890732, -0.021573854610323906, 0.007802118081599474, -0.003616801230236888, -0.026657531037926674, 0.0008547292090952396, -0.023093681782484055, -0.011005640961229801, 0.013077243231236935, 0.017039623111486435, 0.015076941810548306, -0.020838391035795212, -0.0044925520196557045, 0.0378619059920311, -0.012980915606021881, 0.0241019856184721, -0.007084468379616737, -0.006406230386346579, -0.01821768842637539, -0.012040180154144764, -0.007705260068178177, -0.01978822611272335, -0.008672980591654778, 0.035833075642585754, -0.02259354293346405, -0.02751566655933857, -0.039959605783224106, -0.04639572650194168, 0.01800220087170601, 0.015517909079790115, 0.016850730404257774, 0.012749377638101578, 0.02673899382352829, 0.014313383027911186, 0.018471624702215195, 0.02659919485449791, 0.025388382375240326, 0.021644197404384613, 0.01568453386425972, -0.014706228859722614, -0.009609206579625607, 0.012411669827997684, 0.02772401086986065, -0.03335375711321831, 0.031505122780799866, 0.01139417290687561, 0.024118682369589806, -0.011864825151860714, -0.0037973516155034304, -0.012610765174031258, 0.009715577587485313, -0.008220026269555092, -0.026597408577799797, -0.0018822699785232544, 0.0010063305962830782, 0.0072150034829974174, -0.009382707998156548, 0.006301780231297016, 0.013748914003372192, 0.03539283573627472, 0.004168118350207806, -0.004740363918244839, 0.007043302059173584, -0.025582825765013695, -0.03911342844367027, 0.01244462002068758, 0.011982187628746033, 0.0008680851897224784, -0.027774952352046967, 0.022367168217897415, -0.014393291436135769, 0.03959508240222931, 0.017460966482758522, 0.005014680325984955, 0.014183683320879936, -0.0038436693139374256, -0.004459642805159092, -0.002886771457269788, -0.01106992457062006, -0.011090803891420364, -0.012798496522009373, 0.002281798981130123, -0.05617832764983177, -0.026137597858905792, -0.0077201612293720245, 0.012753691524267197, -0.006888101808726788, 0.002185843186452985, -0.04159640893340111, 0.020254267379641533, -0.0034765310119837523, -0.015244502574205399, -0.0021880436688661575, 0.022140879184007645, 0.012743777595460415, 0.016639959067106247, 0.02272064983844757, -0.011439701542258263, 0.021717136725783348, -0.040599748492240906, -0.01723010651767254, 0.013659615069627762, 0.01784871518611908, -0.018899651244282722, 0.005961710587143898, 0.019572611898183823, -0.0013070082059130073, 0.007736482657492161, 0.008958659134805202, 0.027783814817667007, -0.011194107122719288, -0.024395165964961052, 0.03160388395190239, 0.01043955609202385, -0.03656604886054993, 0.0046877446584403515, -0.022758426144719124, -0.04356907680630684, 0.04466146603226662, 0.0026759235188364983, 0.024179713800549507, -0.03460797294974327, 0.005617077928036451, -0.006952873896807432, -0.002611518604680896, 0.03157886490225792, -0.01586931385099888, -0.041388947516679764, 0.018412142992019653, -0.02734357677400112, 0.022993652150034904, 0.02196413092315197, -0.03072076290845871, 0.012237168848514557, 0.0051230196841061115, -0.04545307904481888, -0.026828059926629066, -0.0004947285633534193, 0.005030480679124594, -0.0014222596073523164, 0.017833448946475983, -0.02752043306827545, 0.006711752153933048, 0.0005657825968228281, -0.023301174864172935, -0.01808278076350689, 0.009078185074031353, 0.0158790685236454, 0.007772072218358517, 0.013938293792307377, 0.02641899138689041, 0.0029319990426301956, -0.0077338144183158875, -0.04019336402416229, 0.03150021284818649, -0.008468355983495712, 0.006371680181473494, -0.0009926127968356013, -0.005277207121253014, -0.00772825675085187, -0.0037732087075710297, 0.0013534058816730976, 0.0023200721479952335, -0.037231963127851486, 0.0201205313205719, -0.015400712378323078, 0.002990088425576687, -0.001452864264138043, 0.02490725927054882, 0.0018570305546745658, 0.002179487608373165, -0.033755071461200714, -0.004910190124064684, 0.019237754866480827, -0.0031315817032009363, 0.027961887419223785, 0.0003573384019546211, 0.023180436342954636, 0.030784524977207184, 0.017839359119534492, -0.016658633947372437, 0.04216928780078888, 0.007134435698390007, -0.016587398946285248, 0.03883802890777588, -0.012198301032185555, -0.0021131171379238367, 0.023833848536014557, 0.012302307412028313, -0.017153633758425713, 0.0536821186542511, -0.02631508931517601, -0.014745047315955162, 0.010657341219484806, 0.040556613355875015, -0.03687208518385887, -0.01742352358996868, 0.001382974092848599, 0.00973762571811676, -0.005619834177196026, 0.007378429174423218, -0.011434625834226608, -0.011397436261177063, -0.032507505267858505, 0.01425982266664505, -0.017056161537766457, 0.004288766533136368, 0.03719300404191017, 0.04006442427635193, 0.005788351409137249, -0.0008004580158740282, -0.019698191434144974, 0.0021519882138818502, 0.002141499426215887, -0.027643270790576935, -0.007097569759935141, 0.007750886958092451, 0.01016928255558014, -0.017207033932209015, -0.009674089029431343, -0.018869556486606598, 0.01470567099750042, -0.01426397543400526, 0.009395775385200977, -0.0016149304574355483, 0.014329921454191208, 0.04221557080745697, -0.01737379841506481, 0.004993709735572338, -0.004198341630399227, 0.030327172949910164, 0.013765850104391575, -0.0020218808203935623, -0.00484161963686347, 0.033182550221681595, 0.043900493532419205, -0.028856186196208, -0.002285584108904004, -0.022506484761834145, -0.00277114100754261, -0.0008303208160214126, -0.012468227185308933, 0.024924561381340027, -0.01247909851372242, 0.015226881951093674, -0.034736666828393936, -0.01468625571578741, 0.005054887384176254, 0.018194235861301422, -0.011318712495267391, 0.020953834056854248, 0.006606822833418846, -0.027695920318365097, 0.041454531252384186, -0.02020486630499363, 0.03683235123753548, -0.0268037561327219, 0.02211911976337433, 0.021749382838606834, 0.019645171239972115, -0.003859772812575102, 0.015745842829346657, -0.04779728874564171, 0.0024675496388226748, -0.0015186984091997147, -0.00706278532743454, 0.03513137996196747, 0.014812789857387543, 0.013644034042954445, 0.009179968386888504, 0.011083095334470272, 0.013489923439919949, -0.02950204908847809, 0.0002367775305174291, 0.03232721611857414, 0.001469896757043898, -0.013191462494432926, -0.0009923690231516957, 0.0020375933963805437, 0.0035908427089452744, 0.008083098568022251, 0.01060005184262991, 0.017347879707813263, -0.01787361316382885, -0.00973805133253336, -0.022640472277998924, -0.002994359703734517, 0.022140588611364365, 0.013972767628729343, -0.01940114051103592, -0.02974683605134487, -0.020187512040138245, 0.019582515582442284, 0.021242188289761543, 0.028943706303834915, -0.03802550956606865, -0.04652523621916771, 0.033184342086315155, -0.0030442317947745323, 0.001484028878621757, 0.0027791685424745083, -0.009381829760968685, 0.008429271169006824, -0.03271205723285675, -0.037023983895778656, -0.02011294849216938, -0.018776871263980865, 0.003889846382662654, -0.011021831072866917, -0.02990608476102352, 0.008719715289771557, -0.03218596801161766, 0.010968203656375408, 0.025987260043621063, 0.02021084353327751, -0.014797110110521317, 0.018863888457417488, -0.02460421621799469, 0.0068114823661744595, -0.024533625692129135, -0.005800535902380943, -0.025614220649003983, 0.009596362709999084, -0.005830455105751753, -0.01221957616508007, 0.0014601950533688068, -0.008818616159260273, 0.00572376511991024, -0.023657260462641716, 0.022273637354373932, -0.02624756470322609, -0.005113932304084301, 0.029474133625626564, 0.003513556905090809, -0.03350389376282692, -0.022577941417694092, -0.0005078650428913534, -0.024336807429790497, -0.014516140334308147, -0.0053578815422952175, -0.0015167365781962872, -0.014589940197765827, 0.003529195673763752, -0.003046083264052868, 0.01981245167553425, -0.03044421412050724, -0.02865004725754261, -0.028663361445069313, -0.03257155045866966, -0.02111063338816166, -0.02861264906823635, -0.012112156488001347, -0.028056366369128227, 0.043927066028118134, -0.01841219700872898, 0.013113652355968952, -0.012650782242417336, -0.019008759409189224, -0.025920001789927483, -0.0019452949054539204, -0.007881982252001762, -0.0270920991897583, 0.0018047550693154335, 0.000603139924351126, -0.0036888020113110542, -0.024808259680867195, -0.00035713144461624324, 0.014703212305903435, -0.0063015734776854515, -0.02220192924141884, -0.018921660259366035, 0.04548806697130203, -0.0033526786137372255, -0.006016213912516832, 0.004229933489114046, -0.014515633694827557, -0.026052655652165413, 0.0013624184066429734, 0.005904745310544968, 0.013598211109638214, 0.0014181820442900062, 0.0010760122677311301, 0.003551296191290021, 0.013454989530146122, -0.003025652375072241, -0.02835870534181595, -0.008502518758177757, 0.02103687822818756, 0.011770050972700119, 0.011401060968637466, -0.00888768769800663, 0.0018864857265725732, -0.0013436742592602968, -0.00452645355835557, 0.009430421516299248, -0.005711051169782877, 0.010663836263120174, 0.0029521374963223934, -0.006855234503746033, 0.0016438786406069994, -0.011253940872848034, -0.01493765041232109, 0.0025924635119736195, 0.020184090360999107, -0.04066891223192215, -0.0025717096868902445, 0.03477730229496956, 0.001779103302396834, 0.016382113099098206, -0.02114836871623993, -0.00999520719051361, 0.01766638085246086, 0.021132905036211014, -0.011896961368620396, -0.03482508286833763, -0.00723049184307456, -0.012157744728028774, -0.005621607415378094, -0.030146021395921707, -0.021106554195284843, 0.035786233842372894, -0.03181447833776474, 0.005285689607262611, 0.013234631158411503, 0.003373438725247979, -0.012956338003277779, -0.028740445151925087, -0.026553889736533165, 0.007013728376477957, -0.004079063888639212, -0.018613765016198158, 0.0033501358702778816, 0.025394700467586517, -0.007228245493024588, 0.007364971097558737, 0.0030396769288927317, 0.013031384907662868, 0.007129904348403215, 0.02319420874118805, -0.026241609826683998, -0.0020028443541377783, 0.014411902986466885, -0.033376503735780716, 0.01107761450111866, -0.03470677137374878, 0.011886716820299625, 0.023621639236807823, -0.018720105290412903, 0.030908308923244476, -0.02135942317545414, 0.017129873856902122, -0.009024742059409618, -0.01391594298183918, -0.03550009801983833, -0.035478781908750534, 0.006370611488819122, 0.004097335040569305, 0.0549263134598732, -0.03610962629318237, 0.0031921681948006153, 0.018689114600419998, 0.001026314334012568, 0.01541583426296711, -0.03847401216626167, -0.011686248704791069, -0.014933435246348381, 0.023462170735001564, -0.009345809929072857, -0.00480549456551671, 0.002938489895313978, -0.012613743543624878, -0.012434097938239574, -0.03228425234556198, 0.029022106900811195, -0.013822508975863457, 0.03227823227643967, 0.004706414882093668, 0.014032375998795033, 0.009770067408680916, 0.027465565130114555, -0.0348343662917614, 0.045652635395526886, 0.012338032945990562, -0.02246217429637909, 0.012875225394964218, -0.016528557986021042, 0.01744748279452324, -0.010531689040362835, 0.004637586884200573, 0.011330323293805122, -0.04711168259382248, 0.017758775502443314, -0.001721186563372612, 0.006714311428368092, -0.0020271148532629013, 0.007123645395040512, 0.003896324662491679, -0.011196352541446686, -0.004863767884671688, 0.030427271500229836, -0.0037910041864961386, -0.01792103983461857, 0.008343936875462532, -0.018409930169582367, -0.04391559213399887, 0.06833311170339584, 0.0029698621947318316, -0.006168304476886988, 0.028293274343013763, 0.04411245509982109, -0.011331328190863132, 0.005776473321020603, 0.015694107860326767, -0.021640727296471596, 0.004736593458801508, 0.02871105819940567, 0.02820729836821556, 0.01689782552421093, 0.012242055498063564, 0.010934635996818542, -0.008309758268296719, -0.006544316653162241, -0.03534862771630287, 0.012303529307246208, -0.007055963855236769, -0.0038319146260619164, -0.01057501882314682, 0.007617145776748657, 0.028670666739344597, -0.011802944354712963, -0.031931549310684204, -0.013669815845787525, -0.027219951152801514, -0.028779301792383194, 0.0016090176068246365, 0.013774820603430271, -0.029719997197389603, 0.014422034844756126, -0.008291573263704777, 0.02210122160613537, 0.011316906660795212, 0.02077833004295826, 0.028606636449694633, -0.016568247228860855, 0.0191268902271986, 0.01837039925158024, 0.014473585411906242, -0.016097787767648697, -0.01794245094060898, 0.0007359120063483715, -0.003406238043680787, 0.023356683552265167, 0.0005764340749010444, -0.023670071735978127, 0.007519463077187538, 0.0012598879402503371, -0.0063379257917404175, 0.006792339961975813, -0.0004393458948470652, -0.0026415763422846794, -0.02197106182575226, 0.03415653482079506, 0.008031753823161125, -0.005146332550793886, -0.022790705785155296, 0.011419299058616161, 0.0037277895025908947, -0.015550405718386173, 0.0027717999182641506, -0.00913920160382986, 0.013676607981324196, -0.04833333194255829, 0.0020627991762012243, 0.003045746125280857, -0.028599945828318596, 0.008116007782518864, -0.015365644358098507, -0.016032865270972252, 0.004345586057752371, 0.004554267041385174, -0.006905824411660433, 0.017238197848200798, 0.009177010506391525, -0.010024670511484146, 0.019934125244617462, -0.007664533797651529, -0.04608194902539253, 0.055039919912815094, -0.017880642786622047, 0.01272727269679308, -0.010678649879992008, 0.0031770088244229555, 0.004013423807919025, 0.023018410429358482, 0.028284527361392975, -0.02479269541800022, 0.02766025997698307, 0.008721087127923965, 0.01815950684249401, -0.01436997763812542, -0.008614557795226574, -0.0010986914858222008, 0.012814733199775219, -0.013906666077673435, 0.006975986063480377, 0.01742127723991871, 0.007287594489753246, -0.010323370806872845, -0.009326602332293987, -0.01978130079805851, 0.013072815723717213, -0.004576096776872873, -0.01313045620918274, 0.00705184880644083, 0.006939946208149195, 0.013269711285829544, -0.04361788183450699, 0.025447335094213486, 0.017386293038725853, 0.009532282128930092, -0.018007444217801094, -0.014892039820551872, -0.023960256949067116, -0.014506456442177296, 0.00010495795868337154, -0.057320985943078995, -0.013467874377965927, 0.026942890137434006, -0.019546382129192352, -0.012325575575232506, 0.02219514735043049, -0.011459597386419773, 0.008522890508174896, -0.010893579572439194, 0.006580404471606016, -0.02423444762825966, -0.0034455033019185066, 0.017302291467785835, -0.011944325640797615, 0.026834310963749886, -0.011392133310437202, 0.0031432416290044785, 0.01872740313410759, 0.02172047458589077, -0.0018866724567487836, -0.00020354737353045493, -0.022206639871001244, 0.009007860906422138, 0.0580015555024147, -0.011424838565289974, -0.00559971434995532, -7.643918797839433e-05, 0.007307989522814751, -0.01909511908888817, -0.017445454373955727, 0.036509569734334946, 0.033513084053993225, 0.020489707589149475, -0.01288038119673729, -0.0013481035130098462, 0.023757440969347954, -0.0026603376027196646, 0.02802613563835621, -0.013191541656851768, -0.0034777927212417126, 0.01071819756180048, 0.03663110360503197, -0.02213529869914055, 0.025082755833864212, -0.03418813273310661, -0.0163074042648077, -0.026868687942624092, -0.005543159786611795, 0.0010055683087557554, -0.015361239202320576, 0.011320090852677822, -0.03262379392981529, -0.015649838373064995, 0.0034731468185782433, 0.00592745840549469, -8.010266174096614e-05, 0.009646326303482056, 0.028833657503128052, 0.005896175745874643, 0.030753469094634056, 0.020903142169117928, -0.012972112745046616, -0.004652236122637987, -0.020282840356230736, -0.008287522941827774, 0.0037072859704494476, -0.018088718876242638, -0.03604843094944954, -0.011141004040837288, 0.0030707758851349354, -0.03202790766954422, 0.009780560620129108, -0.016860291361808777, -0.009462310932576656, -0.007125320378690958, -0.01781838946044445, -0.021915558725595474, 0.013892299495637417, 0.0008914240752346814, 0.014871195890009403, 0.008810234256088734, -0.01856951043009758, 0.02152879908680916, -0.003741740481927991, 0.006600277032703161, 0.036075983196496964, 0.008607967756688595, 0.0025353862438350916, 0.00031110062263906, 0.003971731755882502, 0.007670900318771601, -0.02750292606651783, -0.002324966713786125, -0.017787504941225052, -0.028221972286701202, 0.03799617663025856, 0.012964860536158085, -0.010312467813491821, -0.010679196566343307, -0.021903637796640396, 0.010853099636733532, -0.051158156245946884, -0.04055348038673401, -0.007274716626852751, 0.012326777912676334, 0.00036695049493573606, -0.022273261100053787, -0.002691242378205061, -0.009595616720616817, -0.0055209980346262455, 0.02440851740539074, 0.016791176050901413, 0.005400026682764292, 0.0006988451932556927, 0.002779563656076789, 0.007956207729876041, -0.0008899263339117169, 0.027315158396959305, -0.02770315296947956, -0.005685365758836269, -0.03234033286571503, 0.026560330763459206, -0.025870371609926224, 0.014896701090037823, 0.012432156130671501, -0.013868759386241436, 0.011005435138940811, 0.00031657819636166096, 0.0032106046564877033, -0.0011083409190177917, -0.007924140430986881, -0.02803051844239235, 0.025269249454140663, -0.022108307108283043, 0.010560356080532074, 0.03335887938737869, -0.013239001855254173, 0.011133876629173756, 0.024645132943987846, -0.010214838199317455, 0.014759562909603119, -0.010350963100790977, 0.00226189149543643, -0.012002366594970226, -0.013969040475785732, -0.004571413155645132, 0.0001369027013424784, 0.0010883823269978166, 0.013751458376646042, 0.019275499507784843, -0.016474412754178047, -0.021273082122206688, -0.006569372955709696, -0.013833333738148212, 0.015256859362125397, -0.026428550481796265, 0.015222054906189442, -0.012039583176374435, -0.009336571209132671, -0.007603646721690893, 0.005592613015323877, -0.005653992760926485, -0.0029931101016700268, -0.00833172257989645, -0.016569694504141808, -0.007752292323857546, -0.010835898108780384, -0.02135816402733326, 0.05008421465754509, 0.002988015301525593, 0.020631108433008194, -0.0007933916058391333, 0.017636297270655632, 0.0059416876174509525, -0.024699408560991287, 0.023882653564214706, -0.043628890067338943, 0.019723089411854744, 0.019940229132771492, 0.0008603641763329506, -0.004265856929123402, 0.014352159574627876, 0.0053958529606461525, -0.008653045631945133, -0.037371158599853516, 0.006680353078991175, 0.007066730875521898, -0.029444901272654533, -0.019076034426689148, -0.022144820541143417, -0.017219847068190575, -0.011907268315553665, -0.016027700155973434, 0.019281091168522835, 0.005490360781550407, -0.007694775238633156, -0.01638614945113659, -0.01875142753124237, -0.0007179009262472391, 0.03421713784337044, 0.014068102464079857, -0.019076403230428696, 0.009846103377640247, 0.008065542206168175, -0.0023460525553673506, -0.0010732635855674744, -0.032166481018066406, -0.019956862553954124, 0.021298794075846672, -0.015170905739068985, 0.014788148924708366, 0.015795698389410973, 0.02724013291299343, 0.010440620593726635, -0.01981540583074093, -0.0076774172484874725, 0.00928169023245573, 0.003128265729174018, 0.0076215872541069984, 0.003490493865683675, -0.008075571618974209, -0.005325344391167164, -0.023753749206662178, -0.031931012868881226, 0.007396213710308075, 0.014933690428733826, 0.028483640402555466, -0.027357248589396477, 0.0047634076327085495, -0.032810185104608536, -0.006667059380561113, -0.04741302505135536, 0.02406209520995617, 0.007385699078440666, -0.014111976139247417, -0.005853660404682159, -0.010822433978319168, 0.024186009541153908, -0.0008564279414713383, -0.004108354449272156, 0.014221610501408577, -0.023325396701693535, 0.009202064014971256, -0.004728473257273436, -0.0075766765512526035, 0.0022627846337854862, -0.014496426098048687, -0.00893709808588028, 0.01955595426261425, 0.010342937894165516, -0.04683409258723259, -0.017374282702803612, 0.02379770018160343, -0.006981461308896542, 0.009406490251421928, 0.0176218394190073, 0.019108422100543976, 0.015241725370287895, 0.0036843037232756615, -0.011131810024380684, -0.0058736675418913364, 0.04712728410959244, 0.022240370512008667, 0.026746124029159546, -0.017216231673955917, -0.0065245553851127625, -0.00938226468861103, -0.015920281410217285, -0.027080707252025604, 0.010581095702946186, -0.01012934185564518, -0.025503134354948997, -0.012355118058621883, 0.01603119634091854, -0.002744610421359539, -0.001895766705274582, -0.03350992873311043, -0.0022466969676315784, 0.02325635775923729, 0.0034742963034659624, 0.008848041296005249, -0.051238980144262314, -0.0020118269603699446, -0.02850702404975891, -0.0059970347210764885, -0.013711688108742237, -0.01972498558461666, -0.011326785199344158, -0.008868014439940453, 0.020059898495674133, 0.0012703117681667209, -0.014425721019506454, -0.026992900297045708, 0.012420491315424442, -0.017201153561472893, 0.04672630876302719, 0.03359801694750786, 0.03778504580259323, 0.008335640653967857, 0.0229563657194376, -0.033399395644664764, 0.010478674434125423, -0.002138592302799225, -0.001170391566120088, -0.01816478930413723, -0.0008126658503897488, 0.03839325159788132, 0.015003002248704433, 0.0022905042860656977, 0.01303101982921362, 0.008465109393000603, -0.018554670736193657, 0.0077403937466442585, 0.02196238934993744, -0.05076434835791588, -0.02790817618370056, 0.04215947911143303, 0.016552815213799477, -0.010566363111138344, -0.026683341711759567, 0.0008887028088793159, -0.0022403064649552107, 0.03392757102847099, -0.03432957082986832, -0.01266010757535696, 0.029994923621416092, 0.031911127269268036, 0.0007049957057461143, -0.012819536030292511, 0.00015144071949180216, -0.0013649275060743093, 0.006841327529400587, -0.004793899599462748, 0.003165811998769641, -0.00854587834328413, 0.011052891612052917, 0.017858922481536865, 0.006860828027129173, 0.006281053181737661, 0.019406216219067574, 0.003318371484056115, 0.024546703323721886, -0.027236560359597206, -0.01758875511586666, 0.0228793416172266, 0.019389832392334938, 0.015010989271104336, 0.018781717866659164, -0.0015685100806877017, 0.02000289037823677, 0.013242117129266262, -0.032084327191114426, -0.011818530969321728, 0.0009970226092264056, -0.06026792526245117, 0.013897242024540901, 0.0035529688466340303, -0.014495567418634892, -0.008966748602688313, 0.0102232089266181, 0.01616300269961357, -0.026712467893958092, 0.012096244841814041, -0.005105266813188791, -0.004163632169365883, -0.019724052399396896, 0.007169313263148069, -0.030533742159605026, -0.001723452121950686, 0.00439530098810792, -0.037563372403383255, 0.01336077693849802, 0.02581045590341091, 0.0361793115735054, 0.010619718581438065, -0.006931658368557692, -0.017722029238939285, -0.005470437463372946, 0.011051376350224018, -0.01511390507221222, 0.005375757347792387, 0.01958232931792736, 0.0018441375577822328, 0.0057981302961707115, 0.0070120724849402905, 0.01806635782122612, 0.017059288918972015, 0.008279151283204556, -0.013913212344050407, 0.015696531161665916, 0.0058903987519443035, 0.016794592142105103, 0.03163544461131096, -0.011601876467466354, -0.0059957848861813545, 0.014409983530640602, -0.0012106334324926138, -0.0030234495643526316, 0.019214000552892685, -0.01693757250905037, -0.018102189525961876, -0.01800832711160183, 0.022167643532156944, -0.014658061787486076, 0.0036661901976913214, 0.0008337463368661702, -0.01782873272895813, 0.024745536968111992, 0.008320001885294914, 0.02272702567279339, 0.001260727527551353, 0.014984458684921265, 0.0007580228266306221, -0.03607660159468651, 0.018064046278595924, 0.00021303155517671257, 0.0006473246612586081, 0.006821599323302507, 0.00866742990911007, 0.03433530777692795, 5.510307528311387e-05, 0.012327788397669792, 0.030605727806687355, 0.003494501346722245, 0.012639991007745266, 0.0169835202395916, -0.022875744849443436, 0.02838442660868168, -0.02392314188182354, -0.018863895907998085, -0.00669508334249258, 0.001681964029558003, 0.0013722650473937392, -0.03249172866344452, -4.281338260625489e-05, 0.03198709711432457, 0.004982197657227516, -0.010240395553410053, 0.004705496598035097, 0.014285503886640072, 0.008543062023818493, -0.0089297816157341, 0.0426359698176384, 0.00827208161354065, 0.00808944646269083, 0.0363399013876915, 0.008580726571381092, 0.011887257918715477, 0.013226904906332493, -0.011746657080948353, 0.00291759567335248, -0.013860483653843403, -0.0045162346214056015, -0.02030770294368267, -0.0028484847862273455, 0.03825315088033676, -0.018337154760956764, 0.0186579842120409, 0.01850452832877636, 0.0013298081466928124, -0.02730049006640911, 0.012973315082490444, -0.009296087548136711, 0.015310392715036869, -0.01227019913494587, -0.020098231732845306, -0.04382539913058281, -0.0005456088110804558, 0.011082085780799389, 0.011885549873113632, 0.010038845241069794, -0.02855866588652134, 0.0004621315747499466, 0.03323937579989433, 0.017725685611367226, -0.0042395987547934055, 0.02014695480465889, 0.02709484100341797, 0.001562667777761817, -0.02931247465312481, -0.02770344354212284, 0.012347409501671791, -0.009717341512441635, 0.002166863763704896, -0.02483508549630642, -0.007189244497567415, -0.014271262101829052, -0.03319698944687843, 0.009998946450650692, 0.01243648026138544, -0.009877831675112247, -0.021718217059969902, -0.016970574855804443, 0.008108002133667469, -0.03269960358738899, 0.016776790842413902, -0.008663470856845379, -0.010742967016994953, -0.009644990786910057, 0.019231364130973816, 0.021209627389907837, 0.004861749242991209, -0.028800226747989655, 0.008066334761679173, -0.01456819660961628, -0.03327489271759987, -0.014609419740736485, -0.027346689254045486, 0.00643381429836154, 0.023796705529093742, 0.005757241044193506, 0.038062307983636856, -0.005093512125313282, 0.008498774841427803, -0.013379952870309353, -0.004777409136295319, 0.01252950169146061, 0.025808807462453842, -0.02125726267695427, 0.0016202732222154737, 0.015483745373785496, -0.012277527712285519, 0.015850255265831947, -0.013105942867696285, 0.006097264587879181, 0.01969790831208229, -0.0035688928328454494, -0.025057140737771988, 0.007799371611326933, -0.03492458537220955, -0.04546821862459183, 0.0037558148615062237, -0.006344885099679232, -0.00476839067414403, 0.014167669229209423, 0.02422104775905609, 0.05405142530798912, 0.01614636927843094, -0.004495670553296804, -0.015361825935542583, 0.007632431574165821, -0.014254037290811539, 0.030409343540668488, -0.00010664386354619637, 0.019601985812187195, -0.0005052369087934494, -0.010032693855464458, -0.035285260528326035, -0.017296776175498962, -0.00575769180431962, 0.008533382788300514, -0.03937205672264099, 0.03532920032739639, 0.02031785622239113, 0.033758535981178284, -0.022530782967805862, -0.0003733092453330755, -0.014581906609237194, 0.0014433563919737935, 0.005866249091923237, 0.015940802171826363, 0.03181614726781845, -0.06134650483727455, 0.020203499123454094, 0.0004547956050373614, -0.01891571842133999, 0.012606173753738403, 0.013147516176104546, 0.011156604625284672, -0.026980891823768616, 0.04546958580613136, 0.02179778553545475, -0.03782575950026512, 0.0016264503356069326, 0.00872405618429184, 0.04011872410774231, 0.020598432049155235, 0.02877187915146351, 0.02627003937959671, 0.005393567495048046, -0.04716472700238228, 0.01338802371174097, -0.025966860353946686, 0.029466021806001663, 0.00019327650079503655, -0.023009702563285828, 0.005119016394019127, 0.01596345752477646, -0.03015395626425743, -0.020124215632677078, 0.03044532798230648, 0.010993787087500095, -0.04070798680186272, -0.008959115482866764, 0.015042753890156746, -0.033187516033649445, -0.02686583809554577, 0.0012468519853428006, -0.0027103619650006294, -0.04044722765684128, -0.02072303742170334, 3.212740193703212e-05, -0.0017881208332255483, -0.004075060598552227, 0.017391638830304146, 0.00678326515480876, 0.01730957068502903, -0.01006203331053257, 0.0008167476044036448, 0.014052708633244038, 0.006849232129752636, -0.0003114145074505359, 0.023906882852315903, 0.043961528688669205, -0.03556383401155472, -0.010840428061783314, -0.006071682553738356, -0.002953043906018138, 0.029437825083732605, 0.02037186548113823, 0.006065818015486002, 0.025357959792017937, 0.004486518446356058, 0.01781357079744339, 0.016794510185718536, -0.025410883128643036, -0.049077149480581284, 0.003294660709798336, 0.0011493732454255223, 0.009869161993265152, 0.00987708568572998, -0.0058263149112463, 0.03699971362948418, -0.01702522113919258, -0.0030819957610219717, -0.004604663234204054, -0.02277173474431038, 0.02673029713332653, -0.018610499799251556, 0.04729084670543671, -0.01806579902768135, -0.0025222802069038153, -0.010120744816958904, 0.008453784510493279, 0.02630053646862507, -0.004670219030231237, 0.007900780998170376, 0.00030843442073091865, 0.003810006659477949, -0.007328574545681477, 0.009334598667919636, -0.020280394703149796, 0.03210218995809555, 0.01788852922618389, 0.0039421007968485355, 0.03153255209326744, -0.005339193623512983, -0.010110650211572647, -0.00013265435700304806, 0.03458494320511818, -0.01670602336525917, -0.0010542003437876701], [-0.028571633622050285, -0.015466904267668724, -0.01862448826432228, -0.058147791773080826, 0.0004066213732585311, -0.019189957529306412, 0.00011231404641875997, 0.016986826434731483, 0.03483182564377785, -0.006186004728078842, -0.015661928802728653, 0.002639666898176074, 0.008163182064890862, -0.014750514179468155, 0.002012470504269004, -0.005047130864113569, -0.013996957801282406, -0.001239015138708055, 0.040219467133283615, -0.0013779158471152186, 0.0213613398373127, -0.006086355075240135, -0.022586531937122345, -0.021900814026594162, -0.014973555691540241, 0.004466203507035971, -0.008949065580964088, 0.00541331572458148, -0.003123208414763212, -0.017090171575546265, 0.01083449088037014, 0.01286575198173523, -0.02076929807662964, -0.0018763121915981174, 0.0018264979589730501, 0.014005869626998901, 0.007623773533850908, -0.0035712679382413626, -0.0017693102126941085, 0.033126313239336014, -0.010649215430021286, -0.012681335210800171, 0.007310011424124241, -0.0050093382596969604, 0.013661259785294533, 0.012036691419780254, 0.02389635518193245, -0.012083751149475574, 0.018119510263204575, -0.015283040702342987, 0.007702805567532778, 0.019789865240454674, -0.016698723658919334, 0.009860881604254246, -0.021653981879353523, -0.010265267454087734, 0.005095960106700659, 0.01586799882352352, -0.015315784141421318, 0.007088645361363888, 0.004770021885633469, -0.015306297689676285, -0.02751273475587368, 0.0061070602387189865, -0.004579782020300627, -0.0032763085328042507, -0.010298115201294422, -0.00867618527263403, 0.019613638520240784, -0.02754099853336811, 0.012158197350800037, 0.021815795451402664, 0.010567540302872658, -0.0014522586716338992, 0.040718164294958115, 0.0018689051503315568, -0.015253124758601189, 0.015346521511673927, -0.01883547566831112, 0.006468127481639385, -0.03103327751159668, 0.03199300915002823, -0.03452199697494507, -0.003369624260812998, 0.05162891373038292, 0.0023324599023908377, -0.04252006113529205, -0.021506087854504585, 0.02440962754189968, -0.01366493757814169, 0.03960949927568436, -0.05221027135848999, -0.02227672189474106, -0.012299238704144955, 0.0031026238575577736, 0.0059966095723211765, -0.020322443917393684, -0.0007981230155564845, 0.033966705203056335, 0.01349601149559021, -0.012495109811425209, 0.02950696274638176, -0.0044809309765696526, 0.01698892004787922, -0.015232538804411888, 0.001657408312894404, 0.009527171961963177, -0.03758539631962776, 0.017260383814573288, -0.0051157521083951, 0.0378815196454525, 0.007650277111679316, -0.023324469104409218, -0.02385350875556469, 0.01660121977329254, 0.010367569513618946, 0.030595120042562485, 0.023724466562271118, -0.025342676788568497, 0.0048791090957820415, -0.01294622477144003, -0.025932328775525093, -0.00849676039069891, 0.011509817093610764, 0.009941463358700275, -0.0024216321762651205, -0.020051540806889534, 0.007157303858548403, -0.009474764578044415, -0.02192179299890995, -0.01795860379934311, -0.02023540809750557, 0.005370390135794878, -0.0017666199710220098, 0.004661932587623596, -0.017047742381691933, 0.028512809425592422, 0.029185006394982338, -0.028786538168787956, -0.028416723012924194, -0.008394681848585606, -0.038084909319877625, -0.019389476627111435, -0.037011027336120605, 0.02396213449537754, -0.022096116095781326, -0.004990194458514452, 0.01008838601410389, 0.008828574791550636, 0.0037212742026895285, 0.024541372433304787, 0.07907426357269287, 0.012464434839785099, -0.011178134940564632, 0.002403550548478961, -0.01055801659822464, 0.009663034230470657, -0.028300488367676735, 0.01829630881547928, 0.02060939371585846, -0.001782320556230843, 0.0028415063861757517, -0.012859503738582134, -0.020673098042607307, 0.015110827051103115, -0.017085760831832886, 0.027952995151281357, 0.00266978214494884, 0.004313659388571978, -0.004796670749783516, 0.014135482721030712, -0.0024214505683630705, 0.017809953540563583, 0.014172317460179329, 0.005506223067641258, -0.01737944409251213, -0.016471825540065765, 0.003024073550477624, 0.0213495884090662, -0.0034501508343964815, -0.017917172983288765, 0.03202725946903229, -0.009670468047261238, 0.0556815043091774, 0.0029765991494059563, 0.0029922998510301113, 0.021250907331705093, -0.025218665599822998, -0.0095973564311862, 0.008775965310633183, 0.022653166204690933, 0.03141961991786957, 0.009355970658361912, -0.008908746764063835, 0.014149066060781479, -0.032468508929014206, -0.02269194833934307, -0.007379384711384773, 0.026143232360482216, -0.000655493582598865, -0.030321257188916206, -0.016958385705947876, 0.0311812162399292, -0.011861608363687992, -0.01679733581840992, 0.014603168703615665, -0.001063830335624516, 0.041517093777656555, -0.013445374555885792, 0.015843726694583893, -0.018676714971661568, 0.00477701798081398, -0.00414255540817976, 0.018262000754475594, 0.013839855790138245, -0.004050706047564745, 0.03147994726896286, -0.016662295907735825, 0.014003438875079155, 0.009471379220485687, -0.009715070948004723, -0.005876121111214161, -0.01084985677152872, -0.003739172825589776, 0.008755017071962357, -0.0012432709336280823, 0.006498539820313454, 0.00538084888830781, 0.036700498312711716, 0.007474845740944147, -0.008487352170050144, -0.010815251618623734, 0.007282731588929892, -0.0028275377117097378, 0.015722202137112617, -0.02213059924542904, 0.008714240975677967, 0.025552231818437576, 0.020891375839710236, -0.006486776750534773, 0.033736344426870346, 0.042883358895778656, 0.022742895409464836, 0.020206701010465622, 0.0033379807136952877, -0.009823041968047619, -0.0148026617243886, 0.012213431298732758, -0.028378240764141083, -0.015461028553545475, 0.01202508993446827, 0.015160863287746906, 0.025286071002483368, 0.0384715273976326, 0.012716427445411682, -0.013698349706828594, -0.012899363413453102, -0.0032160505652427673, -0.03171687200665474, -0.043574605137109756, 0.002064746804535389, 0.022165415808558464, -0.008855425752699375, -0.012515494599938393, -0.004628110211342573, 0.030708694830536842, 0.019917914643883705, 0.027492091059684753, -0.013515989296138287, -0.04264121502637863, 0.0049667442217469215, 0.0024745846167206764, -0.011082816869020462, 0.0030766637064516544, -0.008609981276094913, -0.008893671445548534, 0.0200469009578228, -0.026594415307044983, 0.0392579585313797, 0.004013398662209511, -0.015374253503978252, 0.009495186619460583, -0.003072641557082534, 0.0007985252887010574, -0.017721597105264664, 0.012508574873209, -0.013310844078660011, 0.0019153221510350704, 0.030947983264923096, 0.026241471990942955, 0.023348966613411903, -0.005667231976985931, 0.017198825255036354, -0.051086608320474625, -0.028487440198659897, 0.01380794309079647, 0.005289991851896048, 0.011061262339353561, -0.0397089421749115, -0.021108347922563553, 0.01321773324161768, 0.006711204536259174, 0.013653619214892387, 0.02503928542137146, -0.004534512758255005, 0.014869686216115952, -0.008207905106246471, -0.03404771909117699, 0.012651653960347176, 0.002168518491089344, 0.012357925996184349, 0.00630161352455616, 0.012149215675890446, -0.007442707661539316, 0.0216541551053524, -0.02002621814608574, -0.016553763300180435, 0.00018335528147872537, 0.009102518670260906, -0.0015195473097264767, -0.017679180949926376, -0.0016215387731790543, 0.013788307085633278, 0.010839692316949368, -0.014621102251112461, -0.015648795291781425, -0.002587925875559449, 6.921424937900156e-05, -0.003881721291691065, -0.002894731005653739, 0.007872262969613075, 0.01921895146369934, 0.030524829402565956, -0.02823301963508129, -0.013693325221538544, 0.011595153249800205, 0.008551239036023617, 0.012068172916769981, 0.006590041797608137, -0.021419191733002663, 0.016524434089660645, 0.00796244665980339, -0.009780140593647957, 0.0282735638320446, 0.015483253635466099, -0.02326708845794201, -0.03769081085920334, 0.028582602739334106, -0.003383115865290165, 0.001487098983488977, 0.0031503920909017324, 0.009319589473307133, 0.006907098926603794, -0.014681147411465645, -0.022133732214570045, 0.013160974718630314, -0.008487943559885025, 0.026825951412320137, -0.002597958780825138, -0.0033964223694056273, -0.004600118845701218, 0.0032165884040296078, -0.01586543768644333, -0.01997966878116131, 0.002159958705306053, -0.019260507076978683, 0.01639997586607933, -0.01494522299617529, -0.032496027648448944, 0.014818381518125534, 0.005165210459381342, 0.015852000564336777, -0.012757384218275547, -0.021548043936491013, -0.015207789838314056, 0.00672857416793704, -0.010614537633955479, 0.005660607945173979, -0.005176041275262833, 0.03901398181915283, -0.023681672289967537, 0.018223300576210022, -0.030717691406607628, 0.013252184726297855, 0.0030355092603713274, 0.021419167518615723, -0.03339933604001999, 0.013212043792009354, -0.006151694804430008, -0.027070581912994385, -0.009568864479660988, 0.03472571820020676, 0.017928073182702065, 0.00104920391459018, -0.022563835605978966, -0.0034915476571768522, 0.01643609069287777, 0.020350897684693336, 0.030127309262752533, 0.022354988381266594, 0.004718546289950609, 0.021453019231557846, -0.0022991986479610205, -0.004026439972221851, 0.008562890812754631, -0.03998764976859093, -0.016589118167757988, 0.015541372820734978, -0.02394932694733143, 0.006771305110305548, -0.004602973815053701, 0.0038613928481936455, -0.04038987308740616, -0.028950592502951622, -0.0022890805266797543, -0.010844988748431206, 0.0035329328384250402, 0.004033518023788929, 0.016400983557105064, -0.004921655170619488, 0.011714012362062931, 0.00489099882543087, 0.020134367048740387, 0.020753895863890648, 0.03164204582571983, -0.0252305269241333, 0.004446369595825672, 0.026380764320492744, -0.020985329523682594, -0.0008311199489980936, 0.000772863975726068, 0.01860816404223442, 0.0026710042729973793, -0.020438633859157562, -0.01061982661485672, -0.022680649533867836, -0.0014689412200823426, 0.004774965811520815, -0.016381921246647835, -0.030211327597498894, 0.029201123863458633, 0.0016916199820116162, 0.013519947417080402, -0.013392352499067783, -0.022859562188386917, -0.013759952038526535, 0.005813582334667444, 0.0074179996736347675, 0.0044048745185136795, -0.021241609007120132, -0.02377913147211075, -0.024151207879185677, 0.011619942262768745, -0.030396923422813416, -0.014819025062024593, 0.006512590683996677, -0.02595711313188076, -0.035421282052993774, 0.0016886972589418292, -0.003526469925418496, 0.02829485386610031, -0.004769471008330584, 0.03402384743094444, -0.021648675203323364, 0.011205130256712437, -0.03369516879320145, -0.012572699226439, -0.001970519544556737, 0.02222776971757412, 0.011538155376911163, -0.005161802284419537, -0.02565191313624382, -0.03504997491836548, 0.003446562448516488, -0.016797205433249474, 0.028405344113707542, -0.04121862351894379, -0.004659249447286129, -0.0011725117219612002, -0.005960274487733841, -0.030253365635871887, 0.0017018988728523254, 0.002016999525949359, 0.003824069397523999, -0.00430725421756506, 0.011606700718402863, 0.018136082217097282, 0.002317335456609726, -0.009710541926324368, -0.006098864134401083, 0.01194850541651249, -0.01041736826300621, 0.007686445955187082, 0.04430104047060013, -0.013613257557153702, -0.013679053634405136, 0.0023908040020614862, -0.027126779779791832, -0.000285098678432405, 0.030409513041377068, 0.0030717342160642147, -0.028615053743124008, 0.028087051585316658, 0.007817406207323074, -0.010085135698318481, 0.00315749435685575, -0.0019899928011000156, -0.0016548052662983537, -0.01338108442723751, -0.026241041719913483, 0.037466417998075485, -0.001943613518960774, 0.0018274550093337893, 0.009541819803416729, 0.0025358465500175953, 0.030599117279052734, 0.011092886328697205, -0.023044060915708542, -0.01562732644379139, -0.023916466161608696, 0.017000926658511162, -0.020860444754362106, 0.004273890983313322, 0.01974029652774334, -0.0030977304559201, -0.01838366873562336, 0.00677554402500391, 0.006600727327167988, 0.015549169853329659, 0.014642694965004921, -0.009033052250742912, 0.02263587899506092, -0.029210859909653664, -0.013937735930085182, 0.005626014433801174, -0.00950550101697445, -0.0006827798788435757, -0.018296902999281883, -0.01075540017336607, 0.012522367760539055, -0.0026986405719071627, -0.03958335891366005, -0.02123509906232357, -0.005078105255961418, -0.0027986972127109766, -0.003987809177488089, -0.0044584427960217, 0.019499048590660095, -0.0014924820279702544, -0.02907581441104412, 0.010788194835186005, -0.005211187992244959, -0.0010982302483171225, 0.005837231408804655, -0.02193906158208847, -0.0064223953522741795, -0.024877550080418587, 0.017862018197774887, -0.017599424347281456, 0.0007587255095131695, 0.004422140307724476, 0.028267383575439453, 0.0039025989826768637, -0.017521219328045845, 0.03635251522064209, 0.0033613871783018112, -0.024321213364601135, 0.014821622520685196, -0.008389150723814964, -0.0004925916437059641, -0.013058038428425789, 0.004823831841349602, -0.026150302961468697, -0.003601103788241744, 0.02399567700922489, 0.026697566732764244, 0.018968895077705383, 0.02805398404598236, 0.030970510095357895, -0.029084159061312675, -0.002274356549605727, 0.006488981656730175, -0.0415755957365036, -0.029299013316631317, 0.01955064944922924, 0.006127122789621353, 0.008226502686738968, 0.021973012015223503, 0.030728252604603767, -0.007203065790235996, -0.02124560996890068, 0.01853233575820923, 0.016827426850795746, -0.0007551821763627231, 0.007004945073276758, 0.02698788046836853, 0.02502763457596302, 0.026978161185979843, -0.018304549157619476, 0.0004540823574643582, 0.003384658833965659, 0.04137016460299492, 0.024621188640594482, -0.012216784991323948, 0.019738972187042236, -0.021781548857688904, 0.0066097090020775795, 0.0010725995525717735, 0.0002207223151344806, -0.026635006070137024, 0.0006505126366391778, -0.011462917551398277, 0.009707195684313774, 0.010950645431876183, 0.04877055436372757, -0.022096708416938782, 0.006968468893319368, -0.026479925960302353, -0.010105223394930363, 0.022486459463834763, 0.012549856677651405, 0.029333895072340965, -0.022773876786231995, 0.02840031497180462, -0.01299138180911541, 0.040403202176094055, 0.025374367833137512, 0.008207486942410469, -0.024744369089603424, -0.018205082044005394, 0.025804737582802773, 0.013043079525232315, -0.020785672590136528, -0.01616579480469227, -0.02001606673002243, 0.03267578408122063, 0.007238577585667372, -0.01269861776381731, 0.0023277008440345526, -0.013568962924182415, 0.040253639221191406, 0.0047041988000273705, -0.004718630574643612, 0.04614889994263649, -0.0006580583867616951, -0.023120127618312836, -0.0037745481822639704, 0.006906663998961449, 0.0053445277735590935, 0.0227151308208704, -0.029718346893787384, 0.02794964611530304, 0.004262277856469154, -0.011200489476323128, 0.020168878138065338, -0.020287618041038513, 0.024242669343948364, 0.010266962461173534, 0.023429475724697113, -0.01385578140616417, -0.022513125091791153, -0.008686068467795849, 0.031243132427334785, 0.00654332572594285, -0.02350245788693428, -0.013889696449041367, 0.0321725569665432, 0.03335244953632355, -0.0019327340414747596, -0.018789805471897125, -0.012723134830594063, 0.003804573090746999, 0.018516629934310913, 0.010596346110105515, -0.0008673644624650478, 0.03129053860902786, 0.0254744291305542, -0.0022132869344204664, 0.016898658126592636, 0.03815985843539238, 0.06001574918627739, -0.027442777529358864, -0.028877882286906242, -0.0003006310435011983, -0.008561216294765472, -0.026024874299764633, -0.02810271456837654, -0.021228404715657234, 0.038609232753515244, -0.010862523689866066, 0.0126327620819211, 0.004780790768563747, -0.00944013986736536, 0.03234728053212166, 0.006661719176918268, -0.005859233904629946, 0.011501421220600605, -0.004857260268181562, -0.004993613809347153, -0.02476171962916851, 0.02040216326713562, -0.004421941936016083, 0.014443324878811836, 0.01351928524672985, 0.0017747717211022973, 0.008952176198363304, 0.01671726070344448, 0.0017244491027668118, 0.02247568406164646, 0.033906903117895126, -0.017141254618763924, -0.0074018570594489574, 0.01308189332485199, -0.015904385596513748, 0.008247562684118748, -0.02275204285979271, 0.01062801294028759, -0.03235786408185959, -0.0406925268471241, -0.009212068282067776, 0.02731056697666645, -0.007228642236441374, 0.039449743926525116, 0.00807233527302742, 0.03166903182864189, 0.008666647598147392, -0.030697602778673172, 0.03948887810111046, -0.010134724900126457, 0.02858474850654602, -0.008198747411370277, -0.02117352932691574, -0.0033701357897371054, -0.016756827011704445, 0.02254907600581646, -0.017696300521492958, -0.009761317633092403, -0.004998865537345409, 0.01678449660539627, -0.0025038104504346848, 0.026529721915721893, -0.007284597493708134, -0.0024084311444312334, 0.014326654374599457, 0.017701243981719017, -0.011871385388076305, -0.005476159974932671, 0.01779220812022686, 0.0261263195425272, 0.016756106168031693, 0.029890256002545357, -0.036535296589136124, -0.008471671491861343, -0.02329634316265583, -0.015520674176514149, 0.01650026999413967, 0.031043514609336853, 0.005552987102419138, 0.012179140001535416, 0.01904851198196411, 0.02179468423128128, 0.004163059871643782, -0.0158663559705019, 0.014049253426492214, -0.010666877962648869, -0.010955377481877804, -0.041636254638433456, -0.00596445519477129, 0.029078315943479538, -0.003901917254552245, 0.003991873934864998, 0.021864797919988632, -0.013662484474480152, -0.014212239533662796, 0.002438915427774191, 0.014505969360470772, -0.01150895468890667, -0.03383037447929382, 0.003314095316454768, -0.003114973893389106, -0.0296699870377779, -0.013414757326245308, -0.004409104119986296, 0.004225664772093296, -0.015501764602959156, 0.007166184484958649, -0.002028810791671276, -0.027162373065948486, 0.005411568563431501, -0.006763251964002848, 0.013316106051206589, 0.02570849098265171, 0.006250558886677027, -0.0017193772364407778, -0.013911987654864788, 0.012078478932380676, 0.01441616378724575, 0.010423904284834862, -0.006191666703671217, 0.006231341045349836, 0.017531264573335648, 0.02914523147046566, 0.0451330840587616, -0.019647499546408653, 0.009414374828338623, -0.038157977163791656, 0.007999037392437458, 0.004598102532327175, -0.009082994423806667, 0.015816371887922287, -0.0015067418571561575, 0.02078939974308014, -0.015869265422225, 0.007481807842850685, -0.01907370612025261, -0.02157742530107498, 0.005091459956020117, 0.007341392338275909, 0.03937762975692749, -0.03572501614689827, -0.036465030163526535, 0.04771953076124191, -0.011513079516589642, 0.006860058754682541, 0.008394645527005196, -0.028875505551695824, -0.017081961035728455, 0.0034851643722504377, 0.014223744161427021, -0.027656227350234985, 0.019682399928569794, -0.013802384957671165, 0.011456131935119629, -0.021401645615696907, 0.029450243338942528, 0.0077348630875349045, 0.020701918751001358, -0.008330284617841244, 0.03489917889237404, 0.0009977046865969896, 0.0049240184016525745, 0.021670669317245483, -0.004959132522344589, -0.007647925056517124, -0.035416554659605026, -0.014853890985250473, -0.016772355884313583, -0.007002152036875486, -0.013064722530543804, 0.013191310688853264, -0.016009069979190826, -0.007758012507110834, 0.02987099438905716, 0.004045715089887381, 0.024905415251851082, -0.0037829496432095766, 0.01523259561508894, -0.01725589483976364, 0.01837579347193241, 0.04302098602056503, -0.04520769789814949, 0.03329766169190407, 0.02771817147731781, 0.0018045020988211036, -0.03093966841697693, 0.007311420049518347, 0.0159720778465271, 0.01431216113269329, -0.012534228153526783, 0.016915908083319664, -0.038166970014572144, 0.010728182271122932, 0.04746103659272194, -0.007946867495775223, -0.011993462219834328, -0.003438921645283699, 0.00841720774769783, -0.014077723026275635, -0.017033101990818977, 0.011500793509185314, -0.005702557973563671, -0.0038757657166570425, -0.009410975500941277, 0.0022562206722795963, 0.030296966433525085, -0.016880061477422714, 0.017177637666463852, -0.0011788118863478303, 0.0034959774930030107, 0.011857668869197369, 0.03141678124666214, 0.008552980609238148, -0.008767726831138134, -0.007672508247196674, 0.009394519031047821, -0.007745911367237568, 0.03394802287220955, 0.0027229017578065395, -0.02825959399342537, -0.0014101045671850443, 0.037942271679639816, 0.012474263086915016, 0.01564529351890087, -0.01902872696518898, -0.007249321788549423, -0.020259631797671318, 0.022643130272626877, -0.00709504634141922, -0.00096184725407511, -0.013180455192923546, 0.023883353918790817, 0.00023918623628560454, -0.0027393822092562914, 0.0002482224954292178, 0.0180460624396801, -0.029741210862994194, -0.012157672084867954, -0.009570826776325703, 0.030006662011146545, -0.007969558238983154, 0.01816265657544136, -0.003956850618124008, 0.023252123966813087, -0.001485464395955205, -0.00019117766350973397, -0.0018862838624045253, 0.0105550866574049, -0.02316153794527054, 0.03809378296136856, 0.00015303351392503828, 0.00023700053861830384, -0.029335828498005867, -0.02348177880048752, -0.00432930514216423, 0.016591573134064674, -0.00547382514923811, -0.02758842334151268, 0.003302860539406538, 0.010523791424930096, -0.0209592804312706, -0.012565402314066887, 0.045115064829587936, 0.0028407739009708166, -0.000692569010425359, 0.007450060453265905, -0.022775422781705856, -0.06716026365756989, 0.014532439410686493, -0.02275475487112999, -0.006705135107040405, 0.019650308415293694, 0.009571379981935024, -0.010215681977570057, -0.010535002686083317, -0.00620052358135581, 0.010872581042349339, -0.0010257047833874822, -0.009051991626620293, 0.005329125560820103, 0.05269325152039528, 0.027311401441693306, 0.0026929283048957586, 0.004732497967779636, -0.024350229650735855, -0.0033035988453775644, -0.004500921815633774, -0.021454207599163055, 0.02209566719830036, -0.002770728897303343, -0.01042176689952612, -0.006516073364764452, 0.025881176814436913, 0.017240697517991066, 0.015348677523434162, -0.006305126007646322, 0.0076958248391747475, -0.017132384702563286, 2.4159458007488865e-06, 0.00646295677870512, 0.02867192029953003, 0.022765595465898514, 0.03324988856911659, 0.005469059105962515, -0.004463396035134792, 0.0031698893290013075, 0.005042830016463995, 0.03090996854007244, -0.004151121713221073, 0.006260569207370281, -0.006719920784235001, -0.014615186490118504, -0.020615730434656143, -0.04426247254014015, -0.0117798438295722, -0.012176026590168476, 0.002362143015488982, -0.020724892616271973, -0.023529592901468277, 0.018379688262939453, -0.022360432893037796, -0.03462773188948631, 0.029511086642742157, 0.014467599801719189, -0.012154706753790379, -0.01285888347774744, -0.04767484590411186, 0.01760876178741455, -0.020312262699007988, 0.013345488347113132, -0.030174201354384422, -0.04173090681433678, 0.010863009840250015, 0.027673255652189255, 0.02246529795229435, 0.04316825792193413, 0.005722657311707735, -0.0077172089368104935, -0.010439976118505001, -0.0007079989882186055, 0.04828594997525215, 0.01932855322957039, -0.021598028019070625, 0.045997630804777145, -0.003596714697778225, -0.024136995896697044, 0.008136419579386711, -0.029775407165288925, -0.00859346054494381, 0.003093741135671735, 0.02102147415280342, -0.005731585901230574, 0.03148256614804268, -0.00079489650670439, -0.015289604663848877, 0.003003390273079276, -0.023739980533719063, -0.005657961126416922, 0.001633897889405489, -0.00937755685299635, -0.004253031685948372, -0.04377971962094307, -0.024046944454312325, -0.06531736254692078, 0.008840722031891346, -0.03348780795931816, -0.006776921451091766, -0.021934131160378456, -0.023234553635120392, 0.011695869266986847, -0.008763523772358894, -0.007823236286640167, 0.024197503924369812, 0.05121029540896416, 0.02139139361679554, -0.020292920991778374, 0.0025272590573877096, -0.018328344449400902, 0.0013595788041129708, 0.021898560225963593, 0.0073244208469986916, 0.03555913642048836, 0.012850248254835606, 0.012885276228189468, 0.022070668637752533, -0.01439205277711153, 0.008971648290753365, 0.04126269370317459, 0.02702535316348076, -0.03040592186152935, 0.00028208221192471683, 0.0023242016322910786, 0.00444466806948185, -0.015703117474913597, -0.03322330862283707, 0.0077425697818398476, -0.05656806752085686, 0.008054839447140694, 0.01633077673614025, -0.011162055656313896, 0.005922344513237476, 0.010235984809696674, 0.017123671248555183, -0.0009373537614010274, -0.008866830728948116, -0.027897672727704048, 0.047252487391233444, -0.01750694401562214, -0.01851893588900566, -0.0016546262195333838, -0.005691685248166323, 0.00865725614130497, 0.02438592165708542, 0.02170580066740513, 0.011418169364333153, -0.01943688653409481, -0.012743882834911346, -0.05048021301627159, -0.001135624130256474, 0.0068990690633654594, -0.01630983129143715, 0.02235061302781105, -0.02806292288005352, -0.0017593230586498976, -0.04532117769122124, 0.014076563529670238, -0.013171503320336342, -0.028015175834298134, 0.018896177411079407, 0.009681109338998795, 0.029150819405913353, -0.018433572724461555, 0.012027161195874214, -0.01946179009974003, 3.325670331832953e-05, -0.013320798054337502, -0.028344063088297844, -0.03996077924966812, 0.008839963003993034, -0.041881855577230453, -0.028998345136642456, 0.004956546705216169, 0.00391897140070796, 0.0007416500593535602, 0.011146231554448605, 0.005713381804525852, -0.0029355292208492756, -0.007343401201069355, -0.005783205386251211, 0.017577871680259705, 0.021791892126202583, -0.02364436350762844, 0.025022760033607483, -0.009474564343690872, 0.027121758088469505, 0.036361001431941986, 0.009952777065336704, 0.0040671322494745255, -0.011364263482391834, -0.001540978206321597, 0.0008518419344909489, -0.03461555019021034, -0.01924612931907177, -0.03891550377011299, -0.01783185824751854, 0.0026550213806331158, -0.027855558320879936, 0.0285019651055336, 0.0038493082392960787, 0.0200584065169096, 0.008784642443060875, -0.010457510128617287, -0.020505545660853386, 0.0332726389169693, 0.013045123778283596, -0.022277936339378357, -0.008013701997697353, 0.0401504822075367, 0.021502632647752762, -0.003116423264145851, 0.01778566651046276, -0.013624458573758602, 0.020064577460289, -0.0007691695936955512, -0.02029220573604107, -0.011593671515583992, 0.00278325448743999, -0.02480928599834442, 0.03898262977600098, 0.02499033510684967, -0.0512726865708828, -0.024782856926321983, -0.0004101830127183348, 0.02820677123963833, 0.008467805571854115, -0.005269884597510099, -0.0021709788125008345, -0.02542765997350216, 0.010612944141030312, 0.012447340413928032, -0.005517075303941965, 0.02660214714705944, 0.03336215391755104, 0.014978338964283466, -0.021430104970932007, -0.016546014696359634, 0.015503950417041779, 0.010264871641993523, 0.04045682027935982, 0.008881766349077225, 0.018085120245814323, -0.005385788157582283, 0.02323661372065544, -0.0019329842180013657, -0.009314700961112976, 0.01780301332473755, -0.006153575144708157, -0.010202302597463131, -0.021573612466454506, -0.029104087501764297, 0.03556887060403824, 0.032166559249162674, -0.014784416183829308, 0.01374901831150055, -0.02396361529827118, 0.02011248469352722, 0.013419385999441147, -0.02661559358239174, 0.02670038491487503, 0.00447356840595603, -0.035738952457904816, -0.0029863426461815834, 0.028386570513248444, 0.0026910000015050173, 0.002858394291251898, -0.02019387111067772, 0.007563019171357155, -0.038310982286930084, -0.0363338366150856, -0.020721087232232094, -0.011284899897873402, -0.03570655733346939, 0.020792674273252487, -0.017103416845202446, 0.005886391270905733, -0.017607444897294044, 0.022514788433909416, -0.035172052681446075, 0.01033695787191391, 0.018554626032710075, -0.011009829118847847, -0.013899100013077259, 0.013984783552587032, -0.029839707538485527, -0.009203074499964714, -0.003551101079210639, -0.008883188478648663, -0.0021757809445261955, -0.00023658342252019793, 0.007491884753108025, 0.0044798823073506355, -0.00303918425925076, -0.01694399304687977, 0.0027580270543694496, 0.00979843270033598, -0.034696318209171295, 0.006428549997508526, -0.0036212513223290443, 0.013007068075239658, 0.043970394879579544, 0.005362019408494234, -0.021882209926843643, 0.025005275383591652, -0.030040713027119637, -0.0012393455253913999, 0.0025479267351329327, -0.02543586678802967, 0.02422095090150833, -0.026313230395317078, 0.046523045748472214, -0.020342914387583733, 0.006463088095188141, -0.011622061021625996, -0.029194779694080353, -0.016568230465054512, 0.01018535252660513, -0.02449008822441101, 0.015505471266806126, 0.015222420915961266, -0.015482946299016476, -0.029576623812317848, 0.004020516760647297, -0.01021160464733839, 0.038349561393260956, 0.004989718087017536, -0.008769501931965351, -0.01745026372373104, -0.00568018713966012, 0.0012056053383275867, -0.017599795013666153, 0.0030112206004559994, 0.013951811008155346, -0.019835127517580986, -0.023226911202073097, 0.0010077268816530704, 0.015137806534767151, 0.028685040771961212, 0.009317832067608833, 0.02008625492453575, -0.031845930963754654, -0.029463406652212143, 0.005150643177330494, -0.009650861844420433, 0.02653626538813114, 0.006565720308572054, 0.03654515743255615, -0.028362460434436798, 0.01557788997888565, 0.011343993246555328, 0.022251686081290245, 0.023009294643998146, -0.010800931602716446, 0.020106732845306396, 0.01389684621244669, -0.022462746128439903, 0.006839999929070473, -0.006956877652555704, -0.00120862340554595, -0.011216522194445133, 0.032049503177404404, -0.021117480471730232, -0.007079257629811764, -0.029141679406166077, 0.015949565917253494, -0.0054720607586205006, -0.006369742099195719, 0.001619192655198276, -0.02132292464375496, 0.01355389878153801, -0.014301279559731483, -0.012682936154305935, 2.6624395104590803e-05, -0.00503035681322217, -0.019518209621310234, 0.015469329431653023, 0.02109234593808651, -0.00820104405283928, -0.01092346291989088, -0.010512215085327625, 0.013541624881327152, 0.0010169759625568986, 0.005606329534202814, 0.011798876337707043, 0.029743943363428116, -0.034818608313798904, 0.026901086792349815, 0.009857629425823689, -0.042760882526636124, -0.017269905656576157, -0.0024639228358864784, 0.005033364985138178, 0.008986852131783962, -0.022014230489730835, -0.0056953453458845615, -0.01065795961767435, -0.006656102370470762, 0.0008491267217323184, -0.005295283161103725, -0.005242920946329832, 0.05816836655139923, 0.011036654934287071, 0.019849173724651337, -0.01289350725710392, -0.020856160670518875, 0.013448184356093407, 0.009757420048117638, 0.0011643674224615097, -0.004244926851242781, -0.03480267897248268, -0.027792001143097878, 0.03583376482129097, -0.002732397522777319, -0.0011592505034059286, -0.027660759165883064, 0.008767860941588879, 0.015832314267754555, 0.019439078867435455, 0.03306739777326584, -0.011594205163419247, 0.0028516279999166727, -0.0041465675458312035, 0.0017420806689187884, -0.007036707829684019, 0.01857704110443592, 0.01914789155125618, -0.02225613221526146, 0.015487697906792164, -0.00868783425539732, 0.0008960255654528737, 0.01442028395831585, 3.695723717100918e-05, 0.022937707602977753, -0.008407173678278923, 0.0260234996676445, 0.030387748032808304, -0.014321690425276756, 0.02490144968032837, -0.00838317722082138, 0.037357404828071594, -0.03521700203418732, -0.04886823147535324, 0.016057228669524193, -0.027615822851657867, -0.02473641186952591, 0.014265327714383602, 0.03921082243323326, -0.000397844152757898, 0.0025451553519815207, -0.03450524061918259, 0.023870952427387238, -0.016886631026864052, -0.010812553577125072, 0.014662454836070538, 0.04494907334446907, -0.006713378708809614, 0.027704797685146332, 0.010374105535447598, 0.032770536839962006, -0.02576620690524578, -0.010828755795955658, 0.011289242655038834, 0.019366607069969177, 0.03177530691027641, -0.02155333012342453, 0.014991411939263344, -0.030352819710969925, 0.0006598594482056797, 0.02050185576081276, 0.012714575976133347, 0.03292704373598099, -0.041665755212306976, 0.00018394773360341787, -0.015080444514751434, -0.02322751097381115, 0.025231484323740005, -0.019499851390719414, -0.0037250870373100042, 0.013436350971460342, -0.040367383509874344, 0.03061477653682232, 0.01607120782136917, 0.022691063582897186, 0.03612486273050308, -0.00023599383712280542, -0.013341277837753296, -0.018360359594225883, 0.005342379678040743, -0.006094307638704777, -0.026039069518446922, 0.009444961324334145, -0.00790367741137743, 0.005105387419462204, 0.006656202487647533, -0.016060376539826393, -0.005967478267848492, 0.007527048233896494, 0.005476756487041712, -0.0020471184980124235, -0.02353244461119175, -0.0076210531406104565, -0.029477300122380257, 0.03153114765882492, 0.042172085493803024, 0.015842320397496223, 0.019346188753843307, 0.004503306001424789, -0.027496591210365295, -0.011747587472200394, 0.010629821568727493, 0.017286574468016624, -0.024387026205658913, -0.009076339192688465, -0.0019152075983583927, 0.038288652896881104, 0.008411338552832603, -0.002871003234758973, 0.0030075476970523596, 0.009465087205171585, 0.06478102505207062, -0.028133438900113106, -0.04168584197759628, 0.009671815671026707, -0.00011125663877464831, -0.0028040942270308733, 0.007907784543931484, 0.01057881023734808, 0.01065730955451727, -0.01454804465174675, 0.0012313604820519686, -0.01631563901901245, -0.004570364952087402, 0.016617584973573685, -0.034004561603069305, -0.010471237823367119, 0.012817899696528912, -0.030864384025335312, -0.023388398811221123, -0.009816551581025124, -0.014130563475191593, 0.016919849440455437, 0.00806666910648346, 0.027896374464035034, -0.005595854949206114, -0.012696943245828152, 0.021077420562505722, -0.006598563399165869, 0.006335102021694183, -0.005771188531070948, 0.03755563870072365, -0.014868250116705894, -0.017861390486359596, 0.020164771005511284, -0.0061961812898516655, 0.005785879213362932, -0.004862338770180941, -0.008573981933295727, 0.015493250451982021, 0.017840145155787468, -0.010011639446020126, -0.012700371444225311, 0.016572730615735054, -0.02094910480082035, -0.03445485979318619, -0.015971750020980835, 0.011405827477574348, -0.0028780053835362196, 0.033100321888923645, 0.015323160216212273, -0.008978109806776047, -0.02512548491358757, 0.03808819502592087, -0.025267496705055237, -0.015487343072891235, 0.007978853769600391, 0.00234012003056705, -0.002898209262639284, -0.012423725798726082, 0.011981985531747341, -0.012444454245269299, 0.021393777802586555, -0.02337612770497799, 0.021323706954717636, 0.005808449350297451, -0.007635120768100023, -0.030416671186685562, 0.004268182907253504, -0.012949754483997822, 0.0029360076878219843, 0.0022682722192257643, 0.0030349958688020706, 0.029479002580046654, -0.005191969685256481, 0.025311773642897606, -0.017428921535611153, -0.021744875237345695, 0.0018168522510677576, 0.007383917924016714, 0.03184451907873154, -0.05348509922623634, 0.008230682462453842, -0.006954866461455822, -0.02450673282146454, 0.0047035724855959415, -0.010400931350886822, 0.0042335838079452515, 6.376854435075074e-05, -0.012636348605155945, -0.03278307616710663, 0.005677135661244392, 0.014896132051944733, -0.03780205175280571, -0.019870255142450333, -0.0020689007360488176, -0.015001856721937656, 0.0032548687886446714, -0.00017285191279370338, -0.01973102241754532, -0.028489327058196068, 0.027012012898921967, 0.015347179025411606, 0.012631113640964031, -0.028419310227036476, 0.002246337942779064, -0.017393341287970543, 0.012333303689956665, -0.008298133499920368, 0.005161084700375795, 0.01118172612041235, -0.012064434587955475, 0.05045357346534729, 0.013352238573133945, -0.02912880852818489, -0.008893787860870361, -0.021567009389400482, 0.010441948659718037, -0.03832707926630974, 0.013197212480008602, -0.03622865304350853, 0.0005121225258335471, 0.0008518125396221876, -0.008448692969977856, -0.0015565938083454967, 0.005441903602331877, 0.015735743567347527, -0.005331185646355152, -0.013224275782704353, 0.01356908492743969, 0.013954497873783112, -0.00688337255269289, 0.024425167590379715, -0.03315350413322449, -0.005879421718418598, 0.018241073936223984, 0.012985723093152046, 0.0018078909488394856, -0.0005917884409427643, 0.0007579160737805068, -0.021419217810034752, -0.0046343193389475346, 0.005443555768579245, 0.006897499784827232, 0.02539754845201969, 0.0035289376974105835, -0.014992984011769295, -0.0027378813829272985, -0.022840535268187523, 0.013569704256951809, 0.010004138574004173, -0.018167667090892792, -0.0024963030591607094, -0.03835085406899452, 0.014712563715875149, 0.04046204313635826, -0.03227841481566429, 0.0060563827864825726, -0.002288848627358675, 0.005235888063907623, 0.0013784341281279922, 0.01798032782971859, 0.005229220259934664, -0.0003275553463026881, 0.03816588595509529, 0.009005286730825901, 0.04357955977320671, -0.026677202433347702, -0.011165093630552292, -0.022360237315297127, 0.016485074535012245, 0.01660359650850296, 0.00801328755915165, -0.005061579868197441, 0.01618603803217411, 0.01086615864187479, 0.012481770478188992, 0.0327308215200901, 0.004890302196145058, -0.016575518995523453, -0.033698346465826035, 0.0002079482947010547, 0.0006951265386305749, -0.009670848958194256, -0.015486609190702438, -0.004672419745475054, -0.029794994741678238, -0.026395797729492188, 0.03534049168229103, -0.01679091900587082, -0.001305131590925157, -0.014361808076500893, -0.030743222683668137, -0.012102974578738213, 0.0004503315722104162, 0.03382943943142891, -0.017055844888091087, -0.040307026356458664, -0.004291761666536331, 0.022598765790462494, -0.011848824098706245, -0.03476199880242348, -0.02181890606880188, 0.011762927286326885, 0.0004557193606160581, -0.0028324464801698923, 0.016945671290159225, 0.013444243930280209, 0.02632562443614006, 0.014187448658049107, 0.012820153497159481, -0.02629368007183075, -0.003914481960237026, -0.006345721893012524, 0.048217225819826126, -0.027308208867907524, 0.009749522432684898, -0.000843389832880348, -0.01908053271472454, 0.0007511494331993163, 0.04224464297294617, 0.030319752171635628, 0.02641703002154827, 0.010592745617032051, -0.013650327920913696, -0.03684336692094803, -0.004413940012454987, -0.015809915959835052, -0.006445110775530338, -0.013527538627386093, -0.022363286465406418, -0.008757063187658787, -0.02252507209777832, 0.015232575125992298, -0.04174942895770073, 0.00023924164997879416, -0.02300257422029972, -0.0012981044128537178, 0.004334774799644947, -0.0412975549697876, -0.012909826822578907, -0.0207748394459486, 0.02029520459473133, -0.017834603786468506, 0.014135832898318768, -0.02017778344452381, -0.009812522679567337, -0.03726203367114067, -0.005932905711233616, 0.019776297733187675, -0.02922840230166912, 0.02911417745053768, -0.0225218515843153, -0.020619811490178108, -0.029607893899083138, -0.007695198990404606, -0.007734579499810934, -0.02329196222126484, 0.0315229594707489, -0.03411309793591499, -0.007699183654040098, 0.02427578717470169, -0.02373442053794861, -0.0015486411284655333, 0.004398397635668516, 0.02733052521944046, 0.037164367735385895, -0.0037967332173138857, -0.003258682321757078, -0.0461902990937233, 0.010960211977362633, -0.010183069854974747, 0.012304658070206642, -0.0010726966429501772, -0.01577206514775753, 0.02533186599612236, -0.010274617001414299, -0.010655304417014122, -0.004705464467406273, 0.017824770882725716, 0.014429832808673382, -0.0024712260346859694, -0.019860006868839264, -0.0002510066842660308, 0.0022259503602981567, -0.02201404608786106, 0.001340526039712131, 0.012595344334840775, -0.023962460458278656, -0.009253332391381264, 0.03579092025756836, -0.0012968629598617554, -0.009073693305253983, -0.018192263320088387, -0.010491078719496727, -0.0410909503698349, 0.008311658166348934, 0.023898888379335403, 0.013944336213171482, -0.04067066311836243, 0.014901932328939438, -0.00980483926832676, 0.021446777507662773, 0.021964414045214653, -0.023295706138014793, 0.003097950480878353, -0.013514263555407524, -0.003542006714269519, 0.03268745541572571, -3.579106123652309e-05, -0.010206630453467369, -0.027388550341129303, -0.03787171468138695, 0.016198085620999336, 0.00990413036197424, -0.028476886451244354, 0.005996836815029383, -0.05017559230327606, 0.00025943605578504503, -0.01502223964780569, 0.008640678599476814, -0.007600096520036459, 0.00882735662162304, 0.03888333961367607, 0.012459577061235905, 0.03472055494785309, -0.030886249616742134, -0.0007279285928234458, 0.004632283002138138, 0.045015234500169754, -0.004714829381555319, 0.01697782427072525, 0.005877550225704908, -0.009589629247784615, -0.02638270892202854, -0.008100870065391064, -0.020009245723485947, 0.05853470787405968, -0.018334168940782547, -0.062037143856287, 0.025391021743416786, -0.01584221050143242, 0.0151676619425416, 0.015261588618159294, -0.010247521102428436, -0.021381612867116928, 0.017028549686074257, 0.01276443712413311, -0.04137613996863365, 0.013973932713270187, 0.0009761039982549846, -0.035240717232227325, -0.0001391403202433139, 0.02645784243941307, -0.02845502831041813, 0.0033186536747962236, 0.01319733913987875, -0.0039365096017718315, -0.0010355347767472267, 0.0048551713116467, 0.014104866422712803, -0.014169370755553246, -0.022385163232684135, 0.006794883403927088, 0.01440946850925684, -0.005317187402397394, 0.0014147311449050903, 0.0019486277597025037, -0.0043029021471738815, 0.02328893356025219, 0.033843740820884705, -0.017110882326960564, -0.018049512058496475, -0.008533771149814129, 0.004623612854629755, 0.00923212245106697, -0.02254670113325119, -0.014216217212378979, 0.003003832884132862, 0.02071577124297619, 0.010219989344477654, 0.030292533338069916, 0.020197205245494843, 0.002921390812844038, -0.024892838671803474, -0.024541115388274193, -0.012243661098182201, 0.009390052407979965, 0.008737337775528431, -0.041871652007102966, -0.011410138569772243, -0.010867496021091938, -0.03317258879542351, 0.01449513342231512, 0.008131362497806549, -0.012394648976624012, 0.01362791657447815, 0.009151410311460495, 0.0015170183032751083, -0.0266879852861166, -0.04638835787773132, -0.01737823337316513, 0.010422390885651112, -0.008670968934893608, 0.0149824358522892, -0.006838984787464142, -0.013278988189995289, -0.03271058574318886, 0.011285273358225822, 0.014427503570914268, -0.002676459727808833, 0.011090894229710102, 0.016955872997641563, 0.021937351673841476, 0.02474583312869072, -0.0010650403564795852, -0.0034024652559310198, 0.02749740704894066, 0.02816741168498993, -0.027138615027070045, -0.0060816481709480286, 0.01984022743999958, -0.013309103436768055, 0.01376322191208601, -0.012181478552520275, -0.011652490124106407, -0.020779414102435112, -0.005233030300587416, -0.028818326070904732, -0.06314615160226822, 0.019334470853209496, -0.02030625194311142, -0.007441990543156862, 0.013256171718239784, -0.0008922843262553215, -0.020783375948667526, -0.005294576287269592, -0.026779117062687874, 0.009732941165566444, 0.008276225998997688, -0.0286912452429533, -0.016229765489697456, -0.0038594200741499662, 0.018581196665763855, -0.005798176396638155, -0.022566821426153183, -0.0035395089071244, 0.02066907286643982, -0.003124831710010767, -0.009255072101950645, -0.004470030311495066, 0.0007416767184622586, 0.013435865752398968, 0.005587805528193712, -0.006382547784596682, 0.00748628843575716, -0.00390613148920238, -0.022236015647649765, -0.025497227907180786, 0.014220141805708408, -0.021548453718423843, 0.018814658746123314, 0.014915169216692448, 0.0361037403345108, 0.0007415824220515788, -0.0265827476978302, 0.014931402169167995, 0.012983868829905987, 0.032958030700683594, -0.025014443323016167, 0.019766096025705338, 0.0032758747693151236, -0.00844105239957571, -0.01722066104412079, -0.02376364730298519, 0.0020612769294530153, -0.011704418808221817, -0.0369134359061718, 0.0003869427018798888, -0.0008460039389319718, 0.012860688380897045, -0.016430379822850227, -0.0001147966249845922, -0.0029208827763795853, -0.009327265433967113, -0.006565208081156015, -0.007848515175282955, 0.01822284422814846, -0.02507169358432293, 0.006398084107786417, -0.011626465246081352, 0.002819499932229519, 0.012522051110863686, 0.024286748841404915, 0.03752442076802254, -0.026547426357865334, -0.017100930213928223, 0.011428489349782467, -0.023564929142594337, -0.006763392593711615, 0.004454351030290127, -0.010983052663505077, -0.006464247591793537, 0.02492884173989296, -0.022282414138317108, -0.003077165223658085, 0.023893550038337708, -0.010255027562379837, 0.023946471512317657, 0.013117226772010326, 0.0029896891210228205, -0.00785940419882536, -0.012270325794816017, 0.024225512519478798, 0.014026153832674026, -0.011750862933695316, -0.006071619223803282, -0.002120976336300373, 0.03166044503450394, -0.010720708407461643, 0.008399372920393944, 0.036427613347768784, 0.004213287495076656, -0.006287194322794676, -0.0002875439531635493, 0.034862346947193146, -0.0361122190952301, -0.010731959715485573, -0.03953979164361954, 0.00875548180192709, -0.03412548080086708, -0.039526477456092834, -0.008133656345307827, 0.028686875477433205, -0.02478092536330223, 0.009158803150057793, 0.04682858660817146, -0.012498060241341591, -0.02219594456255436, -0.0037106163799762726, 0.023810870945453644, 0.0015200887573882937, -0.020297309383749962, 0.02688886970281601, -0.0017139283008873463, 0.0045175193808972836, 0.027659067884087563, 0.02545902691781521, -0.008655384182929993, 0.007158516440540552, 0.0033582705073058605, -0.009962077252566814, 0.011967353522777557, -0.0026752552948892117, -0.02019738219678402, -0.022146979346871376, 0.02029000222682953, -0.009812076576054096, -0.021024202927947044, 0.015397903509438038, 0.021100535988807678, 0.01919105462729931, 0.012043127790093422, 0.03334115073084831, -0.010893168859183788, 0.01166580244898796, -0.003752326825633645, 0.022836530581116676, -0.001698870211839676, -0.005337184760719538, 0.0311859380453825, 0.006633133161813021, 0.026380378752946854, 0.025729378685355186, 0.018504153937101364, -0.0030828446615487337, 0.03691456466913223, -0.006693808361887932, 0.012912834994494915, 0.022907337173819542, -0.02657080627977848, 0.02501986175775528, -0.02619200386106968, -0.001976883737370372, 0.008016782812774181, -0.006757256109267473, 0.019671397283673286, 0.019551755860447884, -0.011597479693591595, -0.014594598673284054, 0.010870452038943768, 0.0022937131579965353, 0.027772212401032448, -0.0351809486746788, -0.02521168254315853, -0.046958647668361664, -0.01953931152820587, 0.01756213791668415, 0.015432598069310188, -0.040505263954401016, -0.006260890047997236, 0.006610125303268433, -0.013745018281042576, -0.026644036173820496, 0.004249619785696268, 0.008154530078172684, 0.004371718969196081, 0.0052474807016551495, -0.006805575452744961, -0.007217169739305973, 0.025256827473640442, 0.016519205644726753, 0.027070103213191032, -0.010300691239535809, 0.01892140321433544, 0.001073411083780229, -0.026812834665179253, 0.003366952296346426, 0.02946113981306553, -0.0021624634973704815, -0.01643233932554722, 0.016771238297224045, -0.027348298579454422, 0.02369713969528675, -0.02106253243982792, -0.002619092585518956, 0.02857816219329834, 0.02464696392416954, -0.0035519746597856283, -0.023254483938217163, 0.017296768724918365, -0.03897865489125252, 0.005592959932982922, 0.014115795493125916, 0.00023655148106627166, -0.01675601303577423, 0.022616805508732796, -0.0014499068493023515, 0.026309233158826828, -0.03791855275630951, -0.019520847126841545, -0.0056269182823598385, 0.01410730555653572, 0.005851890426129103, -0.005526132881641388, -0.03236744552850723, 0.012311961501836777, -0.030962934717535973, -0.03494562581181526, 0.011442247778177261, 0.02476838231086731, 0.021617041900753975, 0.004132701549679041, -0.0014696881407871842, -0.012529950588941574, 0.0106381606310606, -0.012155178934335709, 0.027513287961483, 0.0326436385512352, -0.028059307485818863, -0.0006262405659072101, -0.02802780643105507, -0.02202407270669937, -0.02355591207742691, 0.024751044809818268, 0.001531123765744269, 0.03176012262701988, 0.02404344268143177, 0.0025546641554683447, -0.023284684866666794, 0.0176375862210989, -0.006969344336539507, 0.019043028354644775, 0.012044860050082207, -0.023485930636525154, -0.020778721198439598, 0.0007058083429001272, 0.013299633748829365, -0.03450853005051613, -0.007564529310911894, -0.04732707887887955, -0.010213320143520832, -0.0002575431717559695, 0.012625860050320625, 0.01697448454797268, 0.025944774970412254, -0.0016055104788392782, 0.027350150048732758, -0.023552246391773224, 0.0277536790817976, 0.0402427613735199, 0.006182631943374872, -0.016581453382968903, 0.03833030164241791, 0.010773991234600544, -0.02506556175649166, -0.006298047490417957, -0.016522729769349098, 0.025212449952960014, 0.014293543063104153, -0.052098628133535385, 0.008456064388155937, 0.006438085809350014, -0.0034621236845850945, 0.016986602917313576, 0.005095794331282377, 0.0228202436119318, 0.008837475441396236, -0.017489615827798843, 0.02055676095187664, 0.02993938699364662, -0.029679935425519943, -0.028554493561387062, -0.002565887523815036, 0.0018691123696044087, -0.004912360571324825, 0.001975650666281581, -0.0023479654919356108, 0.025273235514760017, 0.022181469947099686, -0.01462478470057249, 0.028008924797177315, -0.03974093869328499, 0.016370445489883423, 0.004665457643568516, 0.007896647788584232, 0.007193825673311949, 0.013183264061808586, 0.012801459059119225, 0.01940799690783024, 0.011324325576424599, 0.009924977086484432, -0.03295959532260895, 0.010324313305318356, 0.011554517783224583, -0.033595722168684006, 0.03820904716849327, -0.009905925951898098, -0.010886815376579762, -0.016275374218821526, -0.021600041538476944, 0.02613649144768715, 0.013398002833127975, -0.009416859596967697, -0.013780924491584301, 0.050142936408519745, 0.024409977719187737, -0.036070816218853, 0.03110995888710022, -0.0013104848330840468, 0.00178760034032166, 0.02431086078286171, -0.013125362806022167, -0.00798146240413189, 0.009024611674249172, 0.05849601700901985, 0.009277256205677986, -0.003244108287617564, 0.009827704168856144, 0.027599552646279335, 0.008715437725186348, 0.047050122171640396, 0.06202273443341255, -0.005174870137125254, 0.018709490075707436, 0.03537964075803757, -0.00266971904784441, 0.04085499048233032, -0.022193405777215958, 0.010085292160511017, -0.0010919470805674791, 0.03722788766026497, -0.02539246156811714, -0.0031109938863664865, 0.0038337234873324633, -0.0439605675637722, 0.00352325476706028, -0.0069518242962658405, -0.019159391522407532, -0.03993387147784233, -0.0008552940562367439, 0.002390682464465499, 0.0033208997920155525, 0.031909339129924774, -0.004907472990453243, -0.018675683066248894, -0.020868662744760513, -0.01577645353972912, 0.02638770453631878, 0.0003081198956351727, 0.004259856417775154, -0.009696019813418388, -0.007595304865390062, -0.008598473854362965, 0.008768891915678978, 0.02837587334215641, 0.0005383656825870275, 0.003208073088899255, 0.002188925165683031, -0.011780328117311, -0.009555099532008171, 0.019711649045348167, 1.9881359548890032e-05, 0.01872343383729458, 0.002456777263432741, 0.01089091319590807, 0.026576383039355278, -0.005875616334378719, 0.026467090472579002, -0.01679551973938942, -0.016356253996491432, -0.023897161707282066, -0.014075489714741707, 0.011206995695829391, -0.003778164042159915, 0.00610476965084672, -0.015528037212789059, 0.01606767624616623, 0.006961675360798836, -0.012901829555630684, 0.03599939122796059, -0.013265641406178474, -0.005273999180644751, -0.011154010891914368, 0.0016631740145385265, -0.002158774994313717, 0.013066845946013927, -0.030696168541908264, -0.014480534940958023, -0.02783524990081787, -0.018153728917241096, 0.012585597112774849, -0.00364800076931715, 0.015534546226263046, 0.018213577568531036, 0.007625380996614695, -0.015652349218726158, 0.02225925214588642, -0.0019217903027310967, 0.00029415986500680447, -0.0026917047798633575, 0.00033037905814126134, -0.021894050762057304, -0.019350310787558556, 0.023442205041646957, -0.008528056554496288, 0.013181323185563087, -0.007624040823429823, -0.0013896232703700662, 0.0016465972876176238, -0.0019176750211045146, -0.0026443698443472385, -0.007599039934575558, -0.024702774360775948, 0.005776603240519762, 0.006851757410913706, -0.024231910705566406, -0.0010821925243362784, -0.014040560461580753, 0.009388060308992863, -0.03665082901716232, -0.019536275416612625, 0.002776215085759759, 0.003281942568719387, 0.030283717438578606, 0.03727415204048157, 0.0037326423916965723, 0.012879574671387672, 0.018809784203767776, -0.06306197494268417, -0.018261976540088654, 0.013392754830420017, -0.019029343500733376, 0.029376959428191185, 0.037234533578157425, 0.003923121839761734, 0.013622554019093513, -0.02542499639093876, 0.03897275775671005, 0.018884453922510147, -0.030654093250632286, -0.03499279543757439, 0.025344884023070335, -0.04961558058857918, -0.004329446703195572, 0.028009332716464996, 0.027990035712718964, -0.01404841523617506, 0.0021599249448627234, -0.012884081341326237, -0.01975175179541111, 0.004667842760682106, -0.05959822237491608, 0.017084887251257896, 0.021038303151726723, -0.009678849019110203, -0.018737826496362686, 0.016132939606904984, 0.01788807474076748, 0.03208714723587036, -0.008414681069552898, -0.009469414129853249, 0.008950218558311462, 0.013158784247934818, 0.018009193241596222, -0.008186922408640385, 0.021621499210596085, -0.0042479089461266994, 0.02120129205286503, 0.006971606053411961, 0.01929769292473793, 0.025517726317048073, 0.004918962717056274, 0.009196039289236069, 0.01714726723730564, -0.0010379849700257182, 0.0022898910101503134, 0.0013233161298558116, 0.004016151651740074, -0.0013907029060646892, 0.02043922059237957, 0.016112476587295532, -0.022224057465791702, -0.009025360457599163, -0.0031029542442411184, 0.026093265041708946, 0.001427692361176014, 0.0160999596118927, -0.009324515238404274, 0.028305955231189728, 0.0047186436131596565, -0.002992539433762431, -0.0061784833669662476, 0.007090894505381584, -0.025925269350409508, -0.02136421948671341, -0.026933996006846428, -0.006660349667072296, -0.013469996862113476, 0.029444647952914238, -0.02300969697535038, -0.0010993672767654061, -0.00347505253739655, -0.007967422716319561, -0.029246268793940544, -0.009774868376553059, 0.019484007731080055, -0.014111850410699844, -0.028166089206933975, 0.003674748819321394, 0.0034874414559453726, -0.012098081409931183, -0.005172224249690771, 0.016126954928040504, -0.00940778199583292, 0.05321980267763138, 0.005217467434704304, -0.005534512456506491, 0.01619141548871994, -0.011627539992332458, -0.0499894805252552, -0.012703574262559414, -9.52749905991368e-05, 0.009417317807674408, 0.002493658335879445, -0.020136605948209763, 0.005125899333506823, 0.0009506961214356124, 0.011508364230394363, -0.013822774402797222, 0.01867527887225151, -0.027614250779151917, 0.011992163024842739, -0.012839405797421932, -0.046963002532720566, 0.02358032763004303, 0.007590782828629017, 0.021641278639435768, 0.005375017412006855, 0.0014983268920332193, -0.005759093444794416, 0.005000060889869928, 0.01971764862537384, 0.012910687364637852, 0.03868404030799866, -0.005643360782414675, -0.003474712371826172, -0.013984675519168377, 0.0008818507194519043, -0.004083788022398949, 0.009957706555724144, -0.02019326016306877, -0.007097611203789711, 0.010591039434075356, 0.0069672423414886, 0.03507283702492714, -0.01607566885650158, -0.008081764914095402, 0.0060669174417853355, -0.012345834635198116, -0.021857909858226776, 0.031253691762685776, 0.0018009190680459142, -0.03170842304825783, -0.008877630345523357, 0.001169054303318262, 0.027596214786171913, -0.03319305554032326, 0.004748592618852854, -0.010507962666451931, -0.0049071707762777805, 0.022826150059700012, 0.019251707941293716, 0.01909460313618183, 0.018662134185433388, -0.021977167576551437, 0.006522204726934433, -0.014060070738196373, -0.022586865350604057, 0.010481125675141811, 0.03182506188750267, 0.014506266452372074, 0.015590835362672806, 0.029996929690241814, -0.002987796673551202, 0.020230775699019432, -0.0157023798674345, -0.005813333205878735, -0.027833644300699234, 0.020094001665711403, -0.008759630843997002, 0.0025784745812416077, -0.0007707829936407506, 0.0004357439174782485]], "dtype": "float32"}, {"__nd__": [501.76226806640625, 496.6609802246094, 507.5078125], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.317169189453125, 0.8605504035949707, -1.3380041122436523, -0.9385737180709839, -0.3389393389225006, 0.5451122522354126, 0.9501990675926208, 0.3440331220626831, -0.3463037312030792, -0.6336036324501038, 0.06583534926176071, 1.3258838653564453, 0.0017066020518541336, -0.056631702929735184, 2.08893084526062, 2.302475690841675, 1.9933640956878662], [-0.32484450936317444, -0.80660080909729, 0.37009114027023315, 0.006929066497832537, 0.2746172249317169, 1.7264853715896606, 0.2787357270717621, 0.2255767583847046, -0.79665207862854, -1.0636725425720215, 0.7227184772491455, 0.616237223148346, -1.3892525434494019, 0.43553969264030457, 0.021399974822998047, 0.6518576741218567, -0.2646275758743286], [-1.038861870765686, 0.06524590402841568, 2.067952871322632, -0.5954447984695435, 1.4220621585845947, 0.9930500984191895, 1.1008473634719849, -1.2895443439483643, 1.2083218097686768, 1.427123785018921, -1.3853381872177124, 0.6620772480964661, -1.4952263832092285, -0.506271243095398, 0.06907463073730469, 0.2292892336845398, 1.0842857360839844], [0.09569163620471954, 0.2517987787723541, 1.7190712690353394, -0.2684623599052429, 0.4735374450683594, 0.1544489711523056, 0.9769867658615112, 0.9078550934791565, 0.17664366960525513, -0.9739282727241516, 0.03581823781132698, 1.1460864543914795, 0.08664457499980927, -0.8481442332267761, -1.6168957948684692, -0.11474745720624924, -0.9249219298362732], [1.0043178796768188, -0.7739457488059998, -0.8643293976783752, -1.341638445854187, -0.24567396938800812, -1.6745634078979492, -0.7804857492446899, -0.01809937320649624, -0.8846482634544373, -0.514319121837616, -1.2148184776306152, 0.3956575393676758, -0.20055562257766724, -0.0346427746117115, 0.871067225933075, 0.4991297125816345, 1.3858853578567505], [-0.7744773626327515, -0.8022589087486267, -0.49027547240257263, 0.3418215215206146, 0.42046406865119934, 0.1061314195394516, 0.3262084722518921, -0.3619583547115326, 1.4006824493408203, 1.2870149612426758, -0.16513493657112122, -0.8829021453857422, -0.2716093361377716, -1.5221381187438965, -0.7923920154571533, -1.991974949836731, -1.6561799049377441], [-1.4522769451141357, -1.292114019393921, -1.677722454071045, 0.46940675377845764, -0.892490565776825, -2.65219783782959, -0.21176402270793915, 1.4000260829925537, 0.054291147738695145, -0.9489840865135193, 0.5955437421798706, 0.19200992584228516, 0.9956074357032776, 1.7234035730361938, -1.0382837057113647, -1.5739445686340332, 0.038976553827524185], [-0.7015650272369385, -0.5083645582199097, 0.853281557559967, 0.28017449378967285, -0.7796342968940735, -2.3413169384002686, 0.7063956260681152, -0.6125086545944214, 0.04609218239784241, 0.39267051219940186, -0.5059269070625305, -1.0184054374694824, -0.41018402576446533, -1.399972677230835, 0.21347734332084656, 0.24665869772434235, -0.05852613225579262], [-2.2244913578033447, -0.7049524188041687, 0.5312972068786621, 1.5841330289840698, 0.12801556289196014, -1.7693452835083008, -0.5597162246704102, -1.398258090019226, -2.2095794677734375, 0.14755681157112122, -0.5273487567901611, -0.36800941824913025, 2.336604118347168, 0.5712183117866516, -1.3079315423965454, -1.229225993156433, 0.6044088006019592], [-0.7558814287185669, 0.6464707851409912, -1.568355679512024, -0.6181345582008362, 0.3363250494003296, -1.3693698644638062, -1.3072683811187744, 0.888725221157074, 1.7616798877716064, 0.14463680982589722, -1.0326032638549805, -0.41594183444976807, -0.8372501730918884, 0.7249686121940613, -1.1635452508926392, 1.1231921911239624, -0.6318820118904114], [1.1686694622039795, 0.053686391562223434, -0.03911540284752846, -0.7767091393470764, -1.4728094339370728, 1.40413236618042, -0.011933524161577225, 0.07240232080221176, 1.227962613105774, 1.2140518426895142, 0.4588777422904968, 0.7124572992324829, 1.0451663732528687, -0.17382271587848663, 2.0306527614593506, -0.7470283508300781, -0.11387411504983902], [-1.1024311780929565, 1.4057384729385376, 1.4468002319335938, 0.8743178248405457, -0.17496293783187866, 0.8519425392150879, -0.3387780785560608, 0.23085260391235352, 1.749767541885376, -0.8051741123199463, -0.28235653042793274, 0.29972177743911743, 0.29730066657066345, -1.9995671510696411, -0.3952905535697937, -1.4581679105758667, 2.064058780670166], [0.0968003198504448, -0.028048695996403694, -0.1503896713256836, -0.7520754933357239, -1.5320119857788086, -0.3366570472717285, 1.0513323545455933, -1.0452500581741333, -1.7139729261398315, -1.0724401473999023, 0.011328398250043392, 0.2980688214302063, -0.038161274045705795, -0.7565576434135437, 0.7910917401313782, -0.4423367381095886, 0.5413063168525696], [1.036455512046814, 0.8883954882621765, 0.5469454526901245, 0.7934762835502625, -0.966151773929596, 0.6197741627693176, 0.5692837834358215, -0.766696035861969, -1.6529544591903687, 0.10847285389900208, -0.11770223081111908, -0.7288264036178589, -0.12589086592197418, -1.0986679792404175, 0.01374642550945282, -0.4050457179546356, -2.2292563915252686], [-0.2542106509208679, 1.5933150053024292, -0.07369157671928406, 0.5233514904975891, 0.5570225119590759, 1.0219321250915527, 1.1879308223724365, -0.46438759565353394, -0.5282161831855774, 0.8431574106216431, 1.4684525728225708, -1.211330771446228, -2.2170379161834717, -1.3804033994674683, 0.7596230506896973, -0.6049430966377258, -1.5987639427185059], [0.4938841760158539, -1.009506344795227, 0.7478877305984497, -1.0455362796783447, 0.9285958409309387, -1.1006231307983398, -0.9330407977104187, -1.501915454864502, 0.033919695764780045, -0.47471553087234497, -1.4204075336456299, -1.0707964897155762, -0.0822974294424057, -1.1997241973876953, 1.8750187158584595, 0.9687926173210144, 1.88302481174469], [-0.6583167910575867, 1.8365018367767334, -0.22837047278881073, -0.3072245419025421, -0.6251316666603088, 0.38187119364738464, 1.0350611209869385, -0.10594923049211502, -1.2653625011444092, -1.5729200839996338, -0.4524543583393097, -0.2666451930999756, 0.6411681175231934, -0.5174933075904846, -1.089454174041748, 2.0687167644500732, 0.14112867414951324], [2.080881357192993, 0.047559622675180435, -1.899951457977295, 1.4021214246749878, -2.1571109294891357, 1.688855528831482, 0.10475514829158783, -1.279731035232544, 0.12635280191898346, 0.7644322514533997, 0.6794862151145935, 1.4855924844741821, -1.0930228233337402, -0.5693296790122986, 1.2395203113555908, -1.3228111267089844, -2.02536678314209], [0.08354967832565308, -0.13145731389522552, -1.554574966430664, 0.05398072302341461, 0.4313228726387024, 1.3157105445861816, 0.1591195911169052, -1.8007773160934448, 2.45150089263916, 0.7373934984207153, 0.6320602893829346, -0.916499674320221, 0.34930655360221863, -1.4146637916564941, 0.1395702362060547, 1.0135791301727295, -0.12286414206027985], [0.061339519917964935, 1.7906279563903809, -0.19567939639091492, 0.2704538106918335, -0.7657060623168945, 0.8878780007362366, 1.2014554738998413, -0.04902634024620056, -2.009582996368408, 2.5068747997283936, -0.45882463455200195, -0.7152120471000671, -2.1173095703125, -2.9466986656188965, 1.3605772256851196, -1.843955159187317, -0.4829223155975342], [-1.3935903310775757, 0.08832249045372009, -0.8491973280906677, -1.3562793731689453, 1.2546354532241821, -0.35943078994750977, 0.8679469227790833, -0.196587935090065, 0.09981081634759903, 0.10547884553670883, -0.19195766746997833, -0.0678219422698021, 0.9712682962417603, -0.10050702095031738, 0.5104792714118958, -0.672261655330658, 0.7968633770942688], [0.3670806884765625, -0.1335621178150177, -0.47843366861343384, 0.9647990465164185, 0.3691096305847168, -0.28778037428855896, -0.12645761668682098, 0.7971397638320923, 0.4485273063182831, 1.9070746898651123, -0.2998880445957184, 0.06523565948009491, 0.960951030254364, 0.08857531100511551, -1.078281044960022, -0.784676730632782, -0.2369731217622757], [1.065979242324829, 0.019772714003920555, -1.1809161901474, -0.42345699667930603, -0.5205990672111511, 1.0328102111816406, -0.39167290925979614, 0.21303987503051758, 1.998715877532959, -1.7476396560668945, -0.844495415687561, -2.00211238861084, -0.9854038953781128, -0.2415400594472885, -0.7319698929786682, 0.06485442817211151, 1.404962420463562], [1.2430905103683472, 0.6981954574584961, 0.13036639988422394, -0.23059508204460144, -0.9597656726837158, -0.713968813419342, -1.4236382246017456, 0.4495253264904022, -1.3260173797607422, 0.3294020891189575, -0.032500505447387695, -0.7089030146598816, -0.2053232043981552, -0.1407463252544403, -0.3642674684524536, -1.4735084772109985, -1.3088363409042358], [0.48759377002716064, 0.053502410650253296, 1.3705854415893555, 1.3018091917037964, -0.8743636608123779, -1.170531988143921, 0.4338270425796509, 0.5992892384529114, -0.2836059629917145, -1.3366153240203857, -1.9948654174804688, -2.295346260070801, -1.0839505195617676, -0.016370335593819618, 0.7418422102928162, 1.3369544744491577, 1.910652995109558], [-1.189614176750183, -0.4190557301044464, 0.6623672842979431, -0.027514362707734108, -1.5146265029907227, 1.1952242851257324, -1.0465493202209473, -0.8512434959411621, -0.09030716121196747, 1.6543347835540771, 0.12711849808692932, -0.025532253086566925, -0.057373352348804474, -1.1473864316940308, 1.653798222541809, 0.6354669332504272, 0.1525389701128006], [-1.0167274475097656, 0.03740149736404419, 1.1255947351455688, -1.0076757669448853, -0.700667679309845, 0.5211538672447205, 1.1458708047866821, 1.0218909978866577, -0.44023486971855164, -0.5830731391906738, -0.5422269105911255, 0.169400155544281, 0.1452820748090744, 0.7806083559989929, -0.6101288199424744, 1.1790897846221924, -0.49988842010498047], [-0.6649393439292908, 0.9279415011405945, -0.6612393260002136, -0.5738003253936768, 1.2025556564331055, 0.6972739696502686, 1.138803482055664, 1.0481010675430298, 0.28111541271209717, -0.34579998254776, 0.6525992155075073, 0.194986492395401, 0.7846181988716125, -0.678567111492157, -1.514563798904419, -0.9747622609138489, -0.6333610415458679], [-1.1627991199493408, 0.07391124218702316, -0.8651596903800964, 0.8221345543861389, -0.9703207612037659, 0.7072701454162598, -1.3698548078536987, -1.125877022743225, -0.08449365943670273, -0.2409471571445465, 0.9002652168273926, -1.9167616367340088, -0.3868580460548401, 0.3197920620441437, -0.8637822270393372, -0.14632263779640198, -0.3793792426586151], [-1.2353839874267578, 0.4155493974685669, -1.4348998069763184, 0.6250730752944946, 0.19452719390392303, 0.31948375701904297, -0.22776059806346893, 2.303485870361328, -1.7875183820724487, -0.32840481400489807, -1.1764951944351196, 0.3000815808773041, 0.4680112898349762, -0.8015022277832031, 0.4860524535179138, -0.12799249589443207, -1.1209722757339478], [0.7197238802909851, -0.3798539638519287, -2.318298578262329, 0.3321862816810608, 0.2367192655801773, -1.292683482170105, 0.2421455681324005, -0.15563561022281647, 0.025896688923239708, -0.42200013995170593, -0.78814697265625, -0.25431451201438904, -2.672126054763794, 0.6138983368873596, -2.1093060970306396, 0.3924410343170166, -0.29436197876930237], [0.4392904043197632, 0.43840473890304565, 0.21992667019367218, -1.0971391201019287, -0.25178200006484985, 1.4204580783843994, 0.9425081610679626, 1.2407734394073486, -1.34221351146698, 0.21288654208183289, 0.8729756474494934, -0.5046640038490295, -0.05011122301220894, 0.7610291838645935, -0.9945027828216553, -0.12395099550485611, 0.6644483208656311], [-1.3992005586624146, -2.5801210403442383, -0.07654288411140442, -0.4153948128223419, -0.95673006772995, 3.815796375274658, 0.6864758133888245, -0.8133391737937927, -1.9787507057189941, 0.5654048919677734, 1.0754481554031372, -0.11922868341207504, -0.18109530210494995, 1.538364291191101, 0.47117432951927185, 1.0942293405532837, -0.746694803237915], [-0.13703936338424683, -1.2235748767852783, -1.229921579360962, 0.6229252815246582, -1.2418787479400635, -0.9124882221221924, -0.7454279661178589, 0.09416457265615463, -0.8566612005233765, -0.8917370438575745, -0.41704922914505005, -0.3805272579193115, -0.30569636821746826, 0.20614445209503174, -1.1735897064208984, 1.0663470029830933, -0.3450077176094055], [-0.8309237957000732, -0.21325437724590302, 1.0370097160339355, -0.5033850073814392, 0.8463122844696045, 1.324422001838684, -0.8527967929840088, -0.11159471422433853, 0.1874319463968277, 1.2478115558624268, -0.6231913566589355, -0.930748462677002, -0.13755933940410614, 1.0247222185134888, 0.5195097327232361, -1.0277565717697144, 0.31349503993988037], [-0.3313809931278229, -1.34647798538208, 1.2360928058624268, -0.05035223066806793, 0.014772788621485233, 0.7949423789978027, -0.8240734934806824, -0.4724296033382416, 0.5336215496063232, 1.4203264713287354, -0.747682511806488, 1.6863040924072266, -1.2398158311843872, -0.3352598547935486, 0.47924259305000305, -0.7558321356773376, -0.7972536683082581], [0.6958709359169006, 0.29114586114883423, 0.303115576505661, 1.3696995973587036, -0.4218819737434387, -0.058269985020160675, 0.7829045057296753, 0.08156533539295197, -1.353886365890503, 0.19184859097003937, -0.6799576878547668, -0.7160071134567261, 0.4029969274997711, 0.621957004070282, 0.22744883596897125, 0.6559773683547974, -1.536300778388977], [1.8892285823822021, -0.05438639968633652, 0.17726701498031616, 1.4659671783447266, 0.945639967918396, -1.136038064956665, -1.2930978536605835, 1.0197696685791016, -0.2915494740009308, 0.15514522790908813, -0.24031201004981995, -0.3929840624332428, -0.3405807912349701, -0.4133128821849823, 0.29236623644828796, 1.4682719707489014, 0.1052103117108345], [-1.6704270839691162, -0.7924684286117554, -0.03314368799328804, -0.5347314476966858, 1.559413194656372, -1.9860813617706299, 0.929506778717041, -1.0943292379379272, 0.7859326004981995, -1.864522099494934, -0.1292198896408081, 0.28214141726493835, 0.2896387577056885, 0.4976310729980469, -0.6669354438781738, 1.980320692062378, -0.006989473011344671], [2.3291754722595215, 0.3362380564212799, -0.9878038763999939, -0.5619874000549316, 0.635811984539032, 0.6785195469856262, -1.2719417810440063, -0.27854540944099426, -2.1618850231170654, -1.1077975034713745, 0.26081037521362305, 1.0459579229354858, -0.8450703024864197, 1.5998114347457886, 0.5682000517845154, -1.239660620689392, -1.1481828689575195], [1.1407321691513062, -0.05133300647139549, -0.3444004952907562, 0.18034297227859497, -1.0611313581466675, 1.1059868335723877, -0.6907856464385986, -0.18024541437625885, -1.1229240894317627, -0.6167368292808533, -0.4068049490451813, 0.2863938510417938, -0.034248996526002884, 0.0032759280875325203, 0.2743740975856781, -0.339606910943985, -0.9131744503974915], [-0.18836112320423126, -0.061611566692590714, -0.23741181194782257, -0.9943918585777283, -1.4541796445846558, 0.08398906141519547, 0.9632126688957214, 0.9182432293891907, 1.228609323501587, 1.2868013381958008, 0.20851080119609833, 0.58970707654953, -0.4940679669380188, -0.09771034121513367, -0.40948808193206787, -0.9993626475334167, -0.13579918444156647], [0.44756776094436646, 0.33246174454689026, 1.8928049802780151, -0.3088259994983673, -0.9087740182876587, 2.4623301029205322, 0.5642551779747009, -0.21008440852165222, -0.22245000302791595, -0.4680418372154236, -2.001983165740967, -0.21869473159313202, -0.34760841727256775, -0.25690746307373047, 0.021469129249453545, -0.8054056763648987, 1.32144296169281], [0.3497563898563385, -0.20163239538669586, 0.520160436630249, 0.889796793460846, 1.4177261590957642, 0.3978436589241028, -0.24929086863994598, -0.4493783116340637, 1.1434319019317627, 0.28386685252189636, 0.8661239147186279, 0.364714652299881, 0.48541533946990967, 0.5164111256599426, 0.6606106162071228, -2.0956270694732666, -0.20462293922901154], [2.6753411293029785, -0.5891181230545044, 0.5198996067047119, -0.7020082473754883, 0.4755640923976898, 0.6933296918869019, -1.0806844234466553, -0.7318786978721619, 1.0467774868011475, 0.11060511320829391, 0.17966324090957642, 1.0459463596343994, 1.8292605876922607, -0.12229719758033752, -1.0524753332138062, 0.6003768444061279, -0.196607306599617], [-0.1986408680677414, -0.7127810716629028, -1.7304573059082031, 0.7486327886581421, -1.0280195474624634, 0.30526211857795715, 0.6581822633743286, -1.7680213451385498, -0.1842593550682068, 0.48830610513687134, 0.07600811123847961, 0.07962983846664429, 1.0892634391784668, -0.07629825919866562, -0.7341650128364563, -0.8001623749732971, -1.2817859649658203], [-0.5835185050964355, -0.7267569899559021, -1.2034611701965332, 0.9267337322235107, -0.6257426142692566, 0.3590784966945648, -0.10739005357027054, -0.3431975841522217, 0.48070430755615234, -1.2150249481201172, -0.6822589039802551, -0.09059174358844757, -0.49268513917922974, -0.38777410984039307, 0.6164742708206177, 0.23592600226402283, 0.3117181360721588], [-1.6473278999328613, 0.12600483000278473, 0.06327208131551743, 1.6353355646133423, -0.28436774015426636, -0.017556749284267426, -0.12846313416957855, -0.40708300471305847, -1.0277270078659058, 1.1109691858291626, 0.7149634957313538, -2.02496075630188, -0.9340012073516846, 0.9160106182098389, -0.012485446408390999, 0.3303275406360626, 0.7811312079429626], [0.5668017864227295, -0.22161823511123657, 1.4450284242630005, -0.6052245497703552, -1.450461745262146, -0.4638574719429016, 1.513593077659607, -0.3768875300884247, -0.4923010766506195, -0.025679701939225197, 0.057785216718912125, 0.17549093067646027, -0.6771479845046997, -0.015370399691164494, 1.3784068822860718, 0.3907070755958557, 0.265755295753479], [-2.0024867057800293, -0.12865185737609863, 0.38612207770347595, -0.07461496442556381, 1.7763701677322388, 0.08632524311542511, -0.8311994075775146, -0.17656461894512177, 1.6017136573791504, -1.4783400297164917, -0.34708499908447266, 0.7290996313095093, 0.33001020550727844, 0.6918131709098816, -1.800890326499939, -0.44349831342697144, 0.680424153804779], [-0.873719334602356, -0.7314339876174927, 2.082033395767212, -0.4288967549800873, 0.27133530378341675, 1.5305814743041992, 0.7358922958374023, 0.2985036075115204, 0.1321111023426056, 0.20900698006153107, -0.724996030330658, -0.29516637325286865, -0.27271199226379395, 0.8243559002876282, -0.9084363579750061, -2.5544400215148926, 0.6450877785682678], [1.2425262928009033, -1.423981785774231, -1.4527459144592285, -0.17710010707378387, -0.8100379705429077, -0.11961307376623154, 0.7732951045036316, 0.6130598783493042, 0.2150156944990158, -1.093125581741333, -0.40159571170806885, -0.7826328277587891, 0.4601866900920868, 1.1872562170028687, -0.413621723651886, 0.8811031579971313, 0.9406237006187439], [0.5727056264877319, -1.0998860597610474, -1.047072410583496, -0.3009570240974426, 0.3716185986995697, -0.6706945896148682, -0.6992965340614319, 1.7639367580413818, -0.03715825080871582, 1.6598271131515503, -0.3803604245185852, -0.43685922026634216, 0.42541199922561646, -0.04291822388768196, -1.6670492887496948, 2.3701303005218506, -0.03871012479066849], [2.8580703735351562, -1.558084487915039, 0.40505799651145935, 0.6143420338630676, 1.8875943422317505, 1.7574222087860107, -0.42717409133911133, -0.707446277141571, 0.27536505460739136, 0.3424656093120575, 0.6297687888145447, -2.851670980453491, 0.19661159813404083, 0.7690713405609131, -0.909710168838501, 0.17807503044605255, -0.3256102502346039], [-0.8980441689491272, -0.44935154914855957, -0.1953948736190796, 0.2550288438796997, 0.2370738536119461, -0.008778799325227737, 0.20835191011428833, 0.2288147211074829, 0.11461904644966125, -0.2996390759944916, 0.7060023546218872, -1.460071086883545, 1.624959945678711, 0.3460410535335541, -1.5957742929458618, -0.041442662477493286, 0.4141389727592468], [0.37646928429603577, 0.14811399579048157, 0.058664143085479736, -0.18389873206615448, -0.7962309718132019, -1.2631407976150513, 1.1396116018295288, 0.8410141468048096, 0.05766592174768448, 1.762269377708435, -0.4502025246620178, 1.9426050186157227, -0.33837461471557617, 1.7952440977096558, 1.1772164106369019, -0.9351602792739868, 0.606910228729248], [0.4204705059528351, -0.5745645761489868, -0.5595293641090393, -1.3588244915008545, 0.5249101519584656, 0.8065129518508911, 0.8859673142433167, -1.1298619508743286, 0.4749079644680023, 0.4977495074272156, 0.6724496483802795, -1.1092603206634521, -0.16790559887886047, -0.6890965700149536, -1.9436486959457397, 0.5930795073509216, -0.031883396208286285], [-0.5576266050338745, 0.06577251106500626, -0.6064497232437134, 0.495299756526947, -1.5036277770996094, 0.010057322680950165, -0.3504523038864136, -0.4651087522506714, 0.2318543642759323, 1.3482484817504883, -1.5688296556472778, 0.025142397731542587, 0.7161502242088318, 0.5077829360961914, -1.4605045318603516, 0.7457610964775085, 1.30839204788208], [-0.9513764977455139, 0.9855871796607971, 0.5923081040382385, -1.085755467414856, 0.6587986350059509, 0.9133021235466003, 0.25352048873901367, -1.5414060354232788, 0.66978919506073, 0.01353091187775135, 0.23216839134693146, -1.705869197845459, -0.8345238566398621, -0.6884881854057312, -0.843297004699707, 0.5105721950531006, 0.09771512448787689], [-1.5658878087997437, -0.9456948041915894, -2.5111324787139893, -2.5658915042877197, -1.3512277603149414, 0.34641775488853455, -1.3446813821792603, -2.2607839107513428, -0.2166692018508911, -0.16267840564250946, 0.37510719895362854, -1.6018571853637695, 0.18491707742214203, -0.9528055191040039, -1.3053838014602661, -0.17429007589817047, -0.4637347161769867], [-0.1241728886961937, 2.73032283782959, -0.046256497502326965, 0.8934923410415649, -0.2686864137649536, -1.9290213584899902, 0.8265048861503601, -0.2980363965034485, -0.03399013727903366, 0.11374547332525253, -1.7275985479354858, -0.4269482493400574, 0.3100821375846863, -0.6047803163528442, -1.8027079105377197, 1.4643770456314087, 0.3507786989212036], [-0.8601906895637512, 1.4210844039916992, -0.10250642895698547, -1.7519714832305908, -0.471929669380188, 0.3984740972518921, -0.1588362753391266, 2.4753572940826416, -0.6324064135551453, 0.9456790089607239, -0.20529809594154358, -0.4033467173576355, 0.7768810987472534, 0.07473927736282349, -0.25505444407463074, 0.6791385412216187, -0.3257186710834503], [1.0022112131118774, -1.4417270421981812, 0.16088394820690155, -0.10262004286050797, -0.46164488792419434, 1.808975100517273, 0.7031195163726807, 0.7311840653419495, -2.4852757453918457, -0.1986103355884552, 1.55682373046875, -0.8602281808853149, -0.7432529330253601, 0.5763777494430542, -0.9388671517372131, 0.6488865613937378, 0.8961567878723145], [-0.14117982983589172, -1.0896621942520142, 0.8810688257217407, 1.606823205947876, 0.7985957264900208, 0.7007265686988831, -0.8154361844062805, -0.6301217675209045, -0.6354399919509888, 0.14644096791744232, -1.1366493701934814, 1.4672247171401978, -0.04041599854826927, -1.3700975179672241, 0.787697434425354, -0.8892937302589417, 1.9727613925933838]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[-0.2776506841182709, 0.1813984215259552, -0.28204256296157837, -0.1978452354669571, -0.07144621014595032, 0.11490611732006073, 0.20029577612876892, 0.0725199431180954, -0.07299857586622238, -0.13355952501296997, 0.013877663761377335, 0.2794876992702484, 0.0003597406030166894, -0.011937594972550869, 0.4403330385684967, 0.48534688353538513, 0.42018818855285645], [-0.10617231577634811, -0.2636297345161438, 0.12096074223518372, 0.0022646989673376083, 0.08975600451231003, 0.5642852187156677, 0.09110210090875626, 0.07372760027647018, -0.2603780925273895, -0.3476511836051941, 0.23621360957622528, 0.20141123235225677, -0.45406386256217957, 0.14235197007656097, 0.006994376424700022, 0.21305343508720398, -0.08649098873138428], [-0.2257491946220398, 0.014178218320012093, 0.4493751525878906, -0.12939274311065674, 0.3090202808380127, 0.21579410135746002, 0.23921892046928406, -0.2802235782146454, 0.26257357001304626, 0.31012019515037537, -0.30104002356529236, 0.14387226104736328, -0.324919193983078, -0.11001494526863098, 0.01501021720468998, 0.04982554540038109, 0.23562000691890717], [0.028058534488081932, 0.07383199781179428, 0.504063069820404, -0.07871806621551514, 0.13884982466697693, 0.04528725519776344, 0.28647035360336304, 0.26619967818260193, 0.051795147359371185, -0.2855735421180725, 0.010502560995519161, 0.3360534608364105, 0.025405770167708397, -0.2486913651227951, -0.4741033613681793, -0.03364604711532593, -0.27120399475097656], [0.2756146788597107, -0.21239371597766876, -0.23719768226146698, -0.36818546056747437, -0.06742023676633835, -0.4595499634742737, -0.21418848633766174, -0.0049670059233903885, -0.24277377128601074, -0.14114445447921753, -0.3333823084831238, 0.10858018696308136, -0.05503842234611511, -0.009507006965577602, 0.23904673755168915, 0.13697603344917297, 0.38032811880111694], [-0.19253523647785187, -0.19944174587726593, -0.12188258767127991, 0.08497690409421921, 0.10452745854854584, 0.026384294033050537, 0.08109549432992935, -0.08998292684555054, 0.3482099771499634, 0.31995218992233276, -0.04105257987976074, -0.2194896638393402, -0.06752213835716248, -0.3784038722515106, -0.19698882102966309, -0.4952054023742676, -0.4117266833782196], [-0.28780460357666016, -0.25606435537338257, -0.3324822187423706, 0.09302455931901932, -0.17686907947063446, -0.5255986452102661, -0.041966281831264496, 0.2774498164653778, 0.010759133845567703, -0.18806467950344086, 0.11802172660827637, 0.03805151581764221, 0.19730424880981445, 0.3415350615978241, -0.2057616114616394, -0.3119160532951355, 0.007724168244749308], [-0.20078828930854797, -0.14549420773983002, 0.2442096322774887, 0.08018609136343002, -0.22313174605369568, -0.670086145401001, 0.20217080414295197, -0.17530030012130737, 0.013191606849431992, 0.11238250881433487, -0.14479655027389526, -0.29146817326545715, -0.11739488691091537, -0.40067291259765625, 0.06109732761979103, 0.07059384882450104, -0.016750210896134377], [-0.417837917804718, -0.13241492211818695, 0.0997963473200798, 0.2975560426712036, 0.024045836180448532, -0.3323454260826111, -0.10513444244861603, -0.26264217495918274, -0.4150369465351105, 0.027716372162103653, -0.09905469417572021, -0.06912514567375183, 0.43889665603637695, 0.10729493945837021, -0.24567566812038422, -0.23089198768138885, 0.11352928727865219], [-0.18374404311180115, 0.1571478694677353, -0.3812449872493744, -0.15025973320007324, 0.08175583928823471, -0.3328743577003479, -0.3177783787250519, 0.21603646874427795, 0.4282393455505371, 0.0351591520011425, -0.2510111927986145, -0.10110954940319061, -0.20352362096309662, 0.1762295663356781, -0.2828413248062134, 0.2730320692062378, -0.1536015421152115], [0.2955414652824402, 0.013576596975326538, -0.009891781955957413, -0.19641974568367004, -0.37245455384254456, 0.355087012052536, -0.003017834620550275, 0.018309615552425385, 0.3105359375476837, 0.30701807141304016, 0.11604427546262741, 0.18017128109931946, 0.2643091082572937, -0.04395752772688866, 0.5135259628295898, -0.18891385197639465, -0.028797298669815063], [-0.23785239458084106, 0.3032917380332947, 0.3121509253978729, 0.18863634765148163, -0.037748709321022034, 0.1838088184595108, -0.0730922520160675, 0.049807049334049225, 0.37751689553260803, -0.1737184077501297, -0.06091915816068649, 0.06466575711965561, 0.06414338946342468, -0.43141183257102966, -0.08528496325016022, -0.3146035075187683, 0.4453260600566864], [0.02902616560459137, -0.008410572074353695, -0.04509526118636131, -0.22551442682743073, -0.45938313007354736, -0.10094866901636124, 0.31524842977523804, -0.31342458724975586, -0.5139452219009399, -0.3215777277946472, 0.0033968891948461533, 0.0893777534365654, -0.011442890390753746, -0.22685842216014862, 0.23721368610858917, -0.1326373666524887, 0.16231399774551392], [0.27062758803367615, 0.23196782171726227, 0.14281223714351654, 0.2071835845708847, -0.2522706687450409, 0.1618284434080124, 0.14864496886730194, -0.20019102096557617, -0.4316008388996124, 0.028323208913207054, -0.030733080580830574, -0.19030293822288513, -0.03287120535969734, -0.2868717908859253, 0.0035893116146326065, -0.10576096922159195, -0.5820782780647278], [-0.0557427778840065, 0.349378764629364, -0.01615893468260765, 0.1147594153881073, 0.1221427246928215, 0.22408711910247803, 0.2604869604110718, -0.10182993113994598, -0.11582612991333008, 0.18488578498363495, 0.321999192237854, -0.26561805605888367, -0.486147403717041, -0.3026919364929199, 0.1665685474872589, -0.13265064358711243, -0.3505735993385315], [0.10789697617292404, -0.22054296731948853, 0.16338814795017242, -0.22841428220272064, 0.20286676287651062, -0.24044889211654663, -0.2038378268480301, -0.328117698431015, 0.007410305552184582, -0.10370927304029465, -0.310310959815979, -0.23393279314041138, -0.017979202792048454, -0.26209911704063416, 0.40962812304496765, 0.2116483896970749, 0.4113771915435791], [-0.16412745416164398, 0.4578652083873749, -0.056935906410217285, -0.07659531384706497, -0.15585394203662872, 0.09520575404167175, 0.2580550014972687, -0.026414602994918823, -0.3154723048210144, -0.3921506404876709, -0.11280310899019241, -0.06647831946611404, 0.1598520427942276, -0.12901821732521057, -0.2716159522533417, 0.5157595872879028, 0.0351853221654892], [0.3723089098930359, 0.008509312756359577, -0.33993712067604065, 0.25086596608161926, -0.3859478235244751, 0.3021681010723114, 0.018742671236395836, -0.22896800935268402, 0.022606898099184036, 0.1367713361978531, 0.12157289683818817, 0.26580050587654114, -0.19556239247322083, -0.1018638163805008, 0.22177354991436005, -0.23667584359645844, -0.36237630248069763], [0.01936875656247139, -0.030474860221147537, -0.36038661003112793, 0.01251398678869009, 0.09999066591262817, 0.3050122857093811, 0.03688761964440346, -0.4174620509147644, 0.5683149099349976, 0.1709449589252472, 0.1465262621641159, -0.21246592700481415, 0.08097738027572632, -0.32795193791389465, 0.03235562518239021, 0.23497119545936584, -0.028482763096690178], [0.0102463960647583, 0.29911360144615173, -0.03268706053495407, 0.045177675783634186, -0.12790657579898834, 0.14831465482711792, 0.20069588720798492, -0.008189554326236248, -0.3356887102127075, 0.4187583029270172, -0.07664388418197632, -0.11947185546159744, -0.3536837697029114, -0.4922282099723816, 0.22727620601654053, -0.30802157521247864, -0.08066925406455994], [-0.45528265833854675, 0.028854748234152794, -0.2774307429790497, -0.4430932402610779, 0.4098864197731018, -0.11742518842220306, 0.283556193113327, -0.06422480940818787, 0.03260795772075653, 0.034459687769412994, -0.06271211057901382, -0.02215726673603058, 0.31731104850769043, -0.03283540531992912, 0.16677236557006836, -0.2196262925863266, 0.2603333592414856], [0.12317768484354019, -0.04481813684105873, -0.1605433225631714, 0.3237482011318207, 0.12385851889848709, -0.09656765311956406, -0.042434148490428925, 0.26748842000961304, 0.15050792694091797, 0.639938473701477, -0.1006305068731308, 0.021890493109822273, 0.322456955909729, 0.02972235344350338, -0.3618282377719879, -0.26330631971359253, -0.07951875776052475], [0.24101945757865906, 0.004470639396458864, -0.26700687408447266, -0.09574424475431442, -0.11770820617675781, 0.23351989686489105, -0.08855781704187393, 0.04816862568259239, 0.4519125819206238, -0.39514386653900146, -0.19094164669513702, -0.4526805281639099, -0.22280126810073853, -0.05461256206035614, -0.1654994636774063, 0.014663681387901306, 0.3176640570163727], [0.35572078824043274, 0.1997944861650467, 0.03730544075369835, -0.06598671525716782, -0.27464500069618225, -0.2043081670999527, -0.40738603472709656, 0.1286354511976242, -0.3794510066509247, 0.09426117688417435, -0.0093002924695611, -0.2028585523366928, -0.05875495821237564, -0.04027574136853218, -0.10423819720745087, -0.4216568171977997, -0.3745345175266266], [0.09796156734228134, 0.010749070905148983, 0.2753618061542511, 0.2615440785884857, -0.17566679418087006, -0.2351694256067276, 0.08715939521789551, 0.12040209770202637, -0.056978754699230194, -0.26853692531585693, -0.40078476071357727, -0.4611538052558899, -0.21777451038360596, -0.0032889340072870255, 0.14904215931892395, 0.2686050534248352, 0.3838657736778259], [-0.3090530037879944, -0.1088675931096077, 0.17207813262939453, -0.007148028817027807, -0.3934887945652008, 0.31051045656204224, -0.27188581228256226, -0.22114679217338562, -0.02346113510429859, 0.42978397011756897, 0.03302444890141487, -0.006633091252297163, -0.014905174262821674, -0.29808253049850464, 0.4296445846557617, 0.16508962213993073, 0.03962849825620651], [-0.32254043221473694, 0.011865023523569107, 0.35707682371139526, -0.3196689486503601, -0.22227555513381958, 0.16532768309116364, 0.36350908875465393, 0.3241784870624542, -0.13965743780136108, -0.18497058749198914, -0.17201277613639832, 0.05373947694897652, 0.04608840122818947, 0.24763545393943787, -0.19355356693267822, 0.37404727935791016, -0.15858156979084015], [-0.19372551143169403, 0.270349383354187, -0.1926475465297699, -0.16717278957366943, 0.3503563404083252, 0.20314599573612213, 0.33178257942199707, 0.30535706877708435, 0.08190105110406876, -0.10074645280838013, 0.1901303082704544, 0.056807976216077805, 0.2285931259393692, -0.19769588112831116, -0.4412577748298645, -0.2839902937412262, -0.18452540040016174], [-0.321504145860672, 0.020435834303498268, -0.2392093539237976, 0.22731326520442963, -0.26828551292419434, 0.1955542266368866, -0.37875330448150635, -0.3112955093383789, -0.02336178347468376, -0.0666198581457138, 0.24891573190689087, -0.5299684405326843, -0.10696298629045486, 0.08841980993747711, -0.23882849514484406, -0.04045697674155235, -0.10489515960216522], [-0.30145442485809326, 0.10140102356672287, -0.3501396179199219, 0.152528315782547, 0.047467898577451706, 0.07795939594507217, -0.05557740479707718, 0.562089204788208, -0.43618446588516235, -0.08013628423213959, -0.28708454966545105, 0.07322493940591812, 0.1142026036977768, -0.19557999074459076, 0.11860495060682297, -0.0312323160469532, -0.2735360264778137], [0.15694965422153473, -0.08283446729183197, -0.5055496096611023, 0.07243961095809937, 0.051621194928884506, -0.2818945348262787, 0.052804503589868546, -0.03393934294581413, 0.005647271405905485, -0.0920252576470375, -0.171870619058609, -0.055458176881074905, -0.5827085375785828, 0.13387234508991241, -0.4599747955799103, 0.08557932078838348, -0.0641912966966629], [0.13222651183605194, 0.1319599151611328, 0.06619797646999359, -0.33023911714553833, -0.07578643411397934, 0.42755818367004395, 0.28369516134262085, 0.37347307801246643, -0.4040065407752991, 0.06407889723777771, 0.2627658545970917, -0.15190397202968597, -0.015083489008247852, 0.22906994819641113, -0.29934555292129517, -0.037309274077415466, 0.19999907910823822], [-0.2358894944190979, -0.4349794089794159, -0.012904270552098751, -0.07003089785575867, -0.16129393875598907, 0.6433004140853882, 0.1157321110367775, -0.1371198445558548, -0.3335951268672943, 0.09532091021537781, 0.18130847811698914, -0.020100615918636322, -0.03053063340485096, 0.2593509256839752, 0.07943470031023026, 0.18447476625442505, -0.12588435411453247], [-0.041369225829839706, -0.36937084794044495, -0.3712867796421051, 0.18804769217967987, -0.37489640712738037, -0.27546051144599915, -0.2250286191701889, 0.02842625230550766, -0.25860753655433655, -0.26919618248939514, -0.12589816749095917, -0.11487296968698502, -0.09228313714265823, 0.062230560928583145, -0.35428139567375183, 0.3219071626663208, -0.10415038466453552], [-0.25503575801849365, -0.06545424461364746, 0.31828975677490234, -0.15450415015220642, 0.25975894927978516, 0.4065053164958954, -0.2617492377758026, -0.034251805394887924, 0.057528555393218994, 0.3829912543296814, -0.19127635657787323, -0.28567495942115784, -0.04222113639116287, 0.31451836228370667, 0.15945331752300262, -0.31544968485832214, 0.09622114896774292], [-0.08927415311336517, -0.362741619348526, 0.33300381898880005, -0.01356490794569254, 0.003979794215410948, 0.21415774524211884, -0.2220056802034378, -0.12727269530296326, 0.14375782012939453, 0.3826364278793335, -0.2014259248971939, 0.45429089665412903, -0.33400678634643555, -0.09031911939382553, 0.12910811603069305, -0.20362143218517303, -0.21478040516376495], [0.22489319741725922, 0.0940932035446167, 0.09796160459518433, 0.44266271591186523, -0.1363448053598404, -0.018831830471754074, 0.25302091240882874, 0.02636047638952732, -0.43755215406417847, 0.06200207769870758, -0.219750314950943, -0.23140086233615875, 0.1302414983510971, 0.20100551843643188, 0.07350745052099228, 0.2120002955198288, -0.4965052902698517], [0.5139939785003662, -0.01479666493833065, 0.048228245228528976, 0.39883914589881897, 0.2572760283946991, -0.30907678604125977, -0.3518073558807373, 0.27744418382644653, -0.07932056486606598, 0.04220966994762421, -0.06538061797618866, -0.10691741853952408, -0.09266029298305511, -0.11244818568229675, 0.07954277843236923, 0.3994661867618561, 0.028624098747968674], [-0.36424243450164795, -0.17280049622058868, -0.007227096240967512, -0.11660005152225494, 0.34003546833992004, -0.43307194113731384, 0.20268218219280243, -0.2386222928762436, 0.1713753342628479, -0.40656551718711853, -0.028176845982670784, 0.06152191758155823, 0.06315673887729645, 0.10851018130779266, -0.14542758464813232, 0.4318158030509949, -0.001524078892543912], [0.48668068647384644, 0.07025686651468277, -0.20640139281749725, -0.11742714047431946, 0.13285276293754578, 0.14177650213241577, -0.2657719552516937, -0.058201998472213745, -0.45172539353370667, -0.23147404193878174, 0.054496269673109055, 0.21855266392230988, -0.17657725512981415, 0.33428022265434265, 0.11872527003288269, -0.25902679562568665, -0.23991255462169647], [0.42428863048553467, -0.01909301057457924, -0.1280977427959442, 0.06707751005887985, -0.39468157291412354, 0.4113653004169464, -0.25693365931510925, -0.06704122573137283, -0.417665034532547, -0.2293916493654251, -0.15130871534347534, 0.10652251541614532, -0.012738713063299656, 0.0012184622464701533, 0.10205183923244476, -0.12631480395793915, -0.33964988589286804], [-0.05964808911085129, -0.019510461017489433, -0.07518091052770615, -0.31489285826683044, -0.4604933261871338, 0.026596715673804283, 0.30501940846443176, 0.2907789647579193, 0.38906222581863403, 0.4074898362159729, 0.06602886319160461, 0.18674182891845703, -0.1564559042453766, -0.030941816046833992, -0.1296720951795578, -0.31646695733070374, -0.04300336539745331], [0.10506492108106613, 0.07804419845342636, 0.444329172372818, -0.07249579578638077, -0.2133314311504364, 0.5780231356620789, 0.13245686888694763, -0.04931655898690224, -0.05221933871507645, -0.10987113416194916, -0.4699583351612091, -0.05133780092000961, -0.08159982413053513, -0.060308102518320084, 0.005039800889790058, -0.189066082239151, 0.3102039694786072], [0.1040414348244667, -0.05997924134135246, 0.15473122894763947, 0.26468631625175476, 0.42172855138778687, 0.11834587156772614, -0.07415612787008286, -0.13367579877376556, 0.34013471007347107, 0.08444138616323471, 0.257644385099411, 0.10849104076623917, 0.14439566433429718, 0.15361593663692474, 0.1965107023715973, -0.623382568359375, -0.06086883321404457], [0.6344690322875977, -0.13971197605133057, 0.12329649925231934, -0.16648437082767487, 0.11278213560581207, 0.16442620754241943, -0.2562890946865082, -0.17356827855110168, 0.2482479214668274, 0.026230493560433388, 0.04260793328285217, 0.24805080890655518, 0.4338172674179077, -0.0290033221244812, -0.24959918856620789, 0.14238202571868896, -0.046626295894384384], [-0.05486014485359192, -0.1968541145324707, -0.47791340947151184, 0.20675554871559143, -0.2839159071445465, 0.08430653810501099, 0.18177515268325806, -0.4882877469062805, -0.050888292491436005, 0.13485917448997498, 0.0209917314350605, 0.021991971880197525, 0.30083006620407104, -0.021071864292025566, -0.20275987684726715, -0.22098685801029205, -0.3540004789829254], [-0.2216178923845291, -0.27601927518844604, -0.4570695161819458, 0.3519695997238159, -0.2376544326543808, 0.1363765150308609, -0.040786292403936386, -0.13034500181674957, 0.1825694888830185, -0.4614613950252533, -0.25911909341812134, -0.03440636396408081, -0.1871197521686554, -0.14727498590946198, 0.23413434624671936, 0.08960370719432831, 0.11838924884796143], [-0.4252777099609375, 0.03252967819571495, 0.016334455460309982, 0.42218172550201416, -0.07341298460960388, -0.004532488062977791, -0.03316431865096092, -0.10509341955184937, -0.26532021164894104, 0.2868101894855499, 0.18457651138305664, -0.5227681994438171, -0.2411237508058548, 0.2364792674779892, -0.0032232694793492556, 0.08527806401252747, 0.20165850222110748], [0.17545437812805176, -0.06860227137804031, 0.44731077551841736, -0.18734820187091827, -0.44899266958236694, -0.14358779788017273, 0.46853509545326233, -0.11666611582040787, -0.15239256620407104, -0.007949191145598888, 0.017887502908706665, 0.05432349070906639, -0.20961220562458038, -0.004757930990308523, 0.42668798565864563, 0.12094397842884064, 0.0822649672627449], [-0.4725213646888733, -0.03035763092339039, 0.0911121815443039, -0.017606690526008606, 0.4191652536392212, 0.020369933918118477, -0.19613587856292725, -0.04166347533464432, 0.37795203924179077, -0.3488398790359497, -0.0819007083773613, 0.17204366624355316, 0.07787161320447922, 0.1632452756166458, -0.4249512255191803, -0.10465109348297119, 0.1605578511953354], [-0.2056037038564682, -0.1721211075782776, 0.4899442791938782, -0.10092802345752716, 0.06385064870119095, 0.36017656326293945, 0.17317023873329163, 0.07024389505386353, 0.031088395044207573, 0.04918354004621506, -0.1706061214208603, -0.06945857405662537, -0.0641746073961258, 0.19398750364780426, -0.21377332508563995, -0.6011110544204712, 0.15180210769176483], [0.34694144129753113, -0.3976079225540161, -0.40563952922821045, -0.049450356513261795, -0.22618092596530914, -0.03339867666363716, 0.21592147648334503, 0.17118018865585327, 0.060037244111299515, -0.3052253723144531, -0.11213460564613342, -0.21852879226207733, 0.128494530916214, 0.3315087854862213, -0.11549253761768341, 0.24602393805980682, 0.262643426656723], [0.1328432559967041, -0.25512662529945374, -0.2428761124610901, -0.06980918347835541, 0.08619965612888336, -0.15557251870632172, -0.16220694780349731, 0.40915805101394653, -0.008619127795100212, 0.3850090503692627, -0.08822738379240036, -0.10133269429206848, 0.09867742657661438, -0.00995519571006298, -0.38668426871299744, 0.5497690439224243, -0.00897909700870514], [0.5326683521270752, -0.29038554430007935, 0.07549204677343369, 0.11449702829122543, 0.3517974317073822, 0.32753679156303406, -0.07961389422416687, -0.13184918463230133, 0.05132072791457176, 0.06382648646831512, 0.11737216264009476, -0.5314756631851196, 0.03664317727088928, 0.14333446323871613, -0.16954579949378967, 0.03318845480680466, -0.06068509817123413], [-0.291720449924469, -0.1459672451019287, -0.06347202509641647, 0.08284351229667664, 0.07701101154088974, -0.0028517029713839293, 0.06768098473548889, 0.07432811707258224, 0.03723282366991043, -0.09733468294143677, 0.2293376326560974, -0.4742891490459442, 0.5278515815734863, 0.11240789294242859, -0.5183709263801575, -0.01346222497522831, 0.13452880084514618], [0.09002456814050674, 0.03541829064488411, 0.014028274454176426, -0.04397544637322426, -0.1904015988111496, -0.3020530939102173, 0.2725137174129486, 0.20111052691936493, 0.013789570890367031, 0.421409010887146, -0.10765630006790161, 0.46453240513801575, -0.08091504871845245, 0.4292941987514496, 0.28150612115859985, -0.22362355887889862, 0.14512959122657776], [0.11919703334569931, -0.16288037598133087, -0.15861812233924866, -0.3852061927318573, 0.14880409836769104, 0.22863422334194183, 0.25115832686424255, -0.32029879093170166, 0.1346292346715927, 0.14110445976257324, 0.19062930345535278, -0.3144585192203522, -0.047598697245121, -0.19534845650196075, -0.5509949922561646, 0.1681290715932846, -0.009038460440933704], [-0.1544574648141861, 0.018218383193016052, -0.16798101365566254, 0.137193500995636, -0.4164911210536957, 0.0027857862878590822, -0.09707207977771759, -0.12883086502552032, 0.06422153860330582, 0.37345248460769653, -0.4345514476299286, 0.0069642141461372375, 0.19836705923080444, 0.14065122604370117, -0.4045463800430298, 0.20656900107860565, 0.36241260170936584], [-0.2671830952167511, 0.2767907679080963, 0.16634288430213928, -0.30492186546325684, 0.1850159764289856, 0.2564903497695923, 0.07119829952716827, -0.4328860640525818, 0.18810255825519562, 0.003800000064074993, 0.06520181149244308, -0.4790736436843872, -0.23436637222766876, -0.19335395097732544, -0.2368302196264267, 0.1433883011341095, 0.027442162856459618], [-0.2822789251804352, -0.1704781949520111, -0.452675998210907, -0.46254727244377136, -0.24358266592025757, 0.0624479204416275, -0.2424025684595108, -0.4075462222099304, -0.03905845060944557, -0.029325656592845917, 0.0676196962594986, -0.28876304626464844, 0.03333456814289093, -0.17176002264022827, -0.23531848192214966, -0.03141886368393898, -0.083596371114254], [-0.026476223021745682, 0.5821611881256104, -0.009862840175628662, 0.19051100313663483, -0.0572894886136055, -0.4113071858882904, 0.1762278974056244, -0.0635475143790245, -0.0072473990730941296, 0.024252882227301598, -0.36835968494415283, -0.0910341814160347, 0.06611591577529907, -0.12895165383815765, -0.3843745291233063, 0.31223541498184204, 0.0747932568192482], [-0.22126086056232452, 0.3655356466770172, -0.02636701427400112, -0.45064741373062134, -0.1213911846280098, 0.10249672085046768, -0.04085635021328926, 0.6367189288139343, -0.16266949474811554, 0.24325042963027954, -0.05280739814043045, -0.1037500649690628, 0.19983170926570892, 0.019224664196372032, -0.06560587882995605, 0.17469008266925812, -0.08378234505653381], [0.22407925128936768, -0.32234832644462585, 0.03597121313214302, -0.022944288328289986, -0.10321680456399918, 0.4044594466686249, 0.15720687806606293, 0.1634816825389862, -0.5556700229644775, -0.04440626502037048, 0.34808221459388733, -0.19233399629592896, -0.1661801040172577, 0.12886933982372284, -0.20991647243499756, 0.14508120715618134, 0.20036709308624268], [-0.033455245196819305, -0.2582162022590637, 0.2087860256433487, 0.3807673454284668, 0.18924245238304138, 0.1660504937171936, -0.19323311746120453, -0.1493193358182907, -0.15057960152626038, 0.03470197319984436, -0.26935070753097534, 0.3476868271827698, -0.009577339515089989, -0.324670672416687, 0.18665990233421326, -0.21073506772518158, 0.4674833416938782]], "dtype": "float32"}, {"__nd__": [4.743979454040527, 3.0595972537994385, 4.601840972900391, 3.410429000854492, 3.6439201831817627, 4.022522449493408, 5.046051979064941, 3.494053602218628, 5.3238139152526855, 4.113773822784424, 3.9543333053588867, 4.6349382400512695, 3.334933042526245, 3.829822301864624, 4.560423374176025, 4.577368259429932, 4.011009693145752, 5.589126110076904, 4.313631057739258, 5.986447811126709, 3.06093430519104, 2.980090856552124, 4.422793388366699, 3.494568109512329, 4.97739839553833, 3.8492238521575928, 3.152247905731201, 3.4323787689208984, 3.6167469024658203, 4.098079204559326, 4.585699081420898, 3.3222568035125732, 5.931593418121338, 3.312591791152954, 3.258068084716797, 3.7119479179382324, 3.094228506088257, 3.6755850315093994, 4.586030960083008, 4.785839080810547, 2.6885757446289062, 3.1578736305236816, 4.259916305541992, 3.3617029190063477, 4.21666145324707, 3.620859384536743, 2.632993698120117, 3.873534679412842, 3.2304797172546387, 4.237875461578369, 4.249530792236328, 3.581371784210205, 4.311138153076172, 5.365571975708008, 3.0784409046173096, 4.181850433349609, 3.527524948120117, 3.610227584838867, 3.5607662200927734, 5.547306537628174, 4.689977645874023, 3.88767671585083, 4.472574710845947, 4.219960689544678], "dtype": "float32"}]}}, {"a": [{"__nd__": [[4.805839853361249e-05, -0.00011568110494408756, 0.0007361683528870344, -0.00096009427215904, -0.00024389605096075684, 0.0007176270592026412, -0.0007839328027330339, 0.0003117549931630492, -0.0006988756940700114, 0.0008140230784192681, -0.000838499516248703, -0.00015823938883841038, 0.0002831672609318048, -0.0002007202128879726, 0.0007857910241000354, 0.00042485998710617423, -0.000695969385560602, 0.0006595348240807652, -0.0003559830947779119, -5.245057764113881e-05, 0.0005891548935323954, 0.00016104456153698266, -0.0004070052527822554, 0.0005021354299969971, 0.0004132252652198076, 0.00034571992000564933, -0.0009757666848599911, -0.0001478472404414788, -0.00046556771849282086, -0.0003087199293076992, -0.0003222267550881952, -0.00021872359502594918, 0.0006869893404655159, -0.0004684389568865299, 0.0006130773108452559, 0.000709907675627619, 0.0002503872092347592, 0.00034576692269183695, -0.00034559774212539196, 0.00034818725544027984, 0.0008585884934291244, 0.0005688261589966714, 0.0007586156134493649, -0.000996474176645279, 0.0005470954929478467, 0.0004817890003323555, -0.0003338126407470554, 0.0003408975899219513, 0.0007445249357260764, -0.0004863087960984558, -0.0002850546734407544, 0.0006939918384887278, -0.00032414690940640867, -0.0003859447315335274, 0.0006921809399500489, 0.0008461685501970351, -0.0002177266142098233, -0.0008135966490954161, -0.0005642938194796443, 0.000750844192225486, 0.00019280955893918872, -1.703877205727622e-05, 0.0006425154278986156, -0.00042745840619318187, -0.0004000358167104423, -0.0003062216273974627, 0.0008842964889481664, -0.0007950044819153845, 0.00013556165504269302, 0.0004675082745961845, 0.0005676798173226416, -0.00040957334567792714, 0.0003149781550746411, 0.0004101654631085694, 1.9943392544519156e-05, 0.0001670078345341608, -0.0006752820918336511, 0.00047060829820111394, -0.00044377738959155977, 0.0008376964833587408, -0.000751459738239646, -0.0008709246758371592, -0.0009830573108047247, 0.00010709571506595239, -0.0006038484862074256, -0.0009635615278966725, 0.00011404611723264679, 0.00014609155186917633, 0.0004087529086973518, -6.301310349954292e-05, 0.00012889645586255938, -0.000811487901955843, -0.0009940803283825517, -0.0005602783639915287, 0.0005845517734996974, -0.0006690680747851729, -0.0005379050271585584, -0.00045529441558755934, 0.00024140415189322084, -0.00041923209209926426, -0.00013735450920648873, -0.00024052774824667722, 0.0008720837067812681, -0.0003799928235821426, 0.00042481120908632874, -0.0005851503810845315, -0.0005539027042686939, -0.000712297041900456, 0.0004243417934048921, -0.00028252878109924495, -0.00046971975825726986, 0.00030022679129615426, 0.0007724750321358442, -0.0006735532660968602, -0.0008033128106035292, -0.0007919613854028285, 0.00088321982184425, 0.00036309679853729904, -0.000646372267510742, 0.0003243692044634372, 0.00038735390990041196, 0.0007545201806351542, -0.0002662186452653259, -0.0005470296018756926, 0.0006654395256191492, 0.00028338824631646276, -0.0004498899506870657, -0.0006062929169274867, -0.0009041832527145743, -9.020236757351086e-05, -0.0002744946104940027, 0.0004335256526246667, -0.000953294278588146, 7.428837852785364e-05, 0.0009676379268057644, -9.530482202535495e-05, -0.0005616944399662316, -0.0003215307369828224, -0.00012372473429422826, -0.0002561813744250685, -0.000749550701584667, -0.00012460276775527745, 0.0005299787735566497, 0.0005198151920922101, -0.0007619054522365332, -0.0004312977835070342, -0.0009241689112968743, -0.0005888424348086119, 0.0006338068633340299, -0.0008011838654056191, -0.0009594373987056315, -0.0005244336789473891, -0.0009727072902023792, -0.00048497060197405517, 0.0005563334561884403, -0.0006403974839486182, 0.000270335644017905, -7.659474067622796e-05, 0.0006446047336794436, 0.0008448850712738931, 0.0006124736391939223, -0.00017820684297475964, -0.0009045462938956916, 0.00047232076758518815, -0.0003574893344193697, -0.0008973170188255608, -0.0008462769910693169, -9.953107655746862e-05, -0.0006627492839470506, -0.000725180609151721, 0.0008394825854338706, -0.0006561742047779262, 0.0008372361189685762, 7.224499859148636e-05, 9.933584078680724e-05, -0.0007824749918654561, -0.000945453648455441, -0.000606162881013006, 0.00014198542339727283, 0.0004619321844074875, -0.00032099478994496167, -0.00041066977428272367, 0.0005354255554266274, 0.000934317649807781, 0.00041565505671314895, -7.258209370775148e-05, -0.0008164711180143058, -0.0007991006714291871, -0.0007147245923988521, -0.00037287716986611485, 0.00039920120616443455, 0.00016066408716142178, 0.0007536627817898989, -0.00042951421346515417, -0.0007173269405029714, -0.00035357693559490144, -0.0003586571547202766, 0.00014550518244504929, 0.00010636488877935335, -0.0003012844827026129, -0.0006516193971037865, -0.0008206134662032127, 0.00023773251450620592, -0.0005376109038479626, -0.0003451677621342242, 0.00034599698847159743, -0.0005393751780502498, -0.000380761397536844, -0.0006060858722776175, 5.628240978694521e-05, 0.00011261238978477195, 0.0003865657199639827, 0.0008644987246952951, 0.000835415325127542, 0.0009536384604871273, -0.0006447111372835934, 0.0006002344307489693, -0.0006656680488958955, 8.050278665905353e-06, 0.0005877812509424984, -0.00038024556124582887, 0.00015851284842938185, 0.0006722952821291983, -0.0006613096920773387, 2.0641797163989395e-05, -0.00044323308975435793, 0.0007341525051742792, -0.0007536121411249042, -0.00026645424077287316, -0.0007975339540280402, 0.0005650355014950037, 0.0005238544545136392, 0.0002122770674759522, -0.0004902519285678864, -0.000694918679073453, 0.0005191398086026311, -0.00020547723397612572, -0.00037469013477675617, -0.00020406905969139189, 0.0009917818242684007, -0.00011626951163634658, 0.0001920145732583478, 0.0009848782792687416, 0.0008779412019066513, -0.0009952441323548555, 0.0009199613705277443, 0.00019969446293544024, 0.0009562979103066027, 0.0006084071355871856, 0.00044257447007112205, -0.00075214181561023, -0.0006776597583666444, 0.0005226065404713154, -0.00038120339740999043, 0.0009414369706064463, -0.00022320589050650597, 0.0005212822579778731, 0.0007632395136170089, -0.0009709514561109245, -0.0009871969232335687, -0.0003507630026433617, -0.0009622325305826962, 0.0007471831631846726, 1.6799096556496806e-05, 0.0007222497370094061, 0.00016603508265689015, -9.780404070625082e-05, -0.0002886323491111398, 0.00023045485431794077, -0.0005133153754286468, 0.000962749938480556, 0.00024415727239102125, -3.977420783485286e-05, 0.0008453790214844048, 0.0008233807748183608, -0.0002194913395214826, 0.0005178682622499764, -0.0008845431148074567, 0.0004464004305191338, -2.6960100512951612e-05, -9.631001012166962e-05, 0.0005612362874671817, 0.0007762900786474347, 8.453199552604929e-05, -0.0002731341228354722, 0.0006096921279095113, 0.00021242012735456228, -0.0003192517615389079, 0.0008573246886953712, -0.00046273728366941214, 0.00038701188168488443, -8.614169200882316e-05, 0.00011433970939833671, -0.0003994923026766628, -0.0006013726815581322, -0.00022295840608421713, 0.0007868455722928047, 3.0341818728629733e-06, -0.0006430397043004632, 7.935531175462529e-05, -0.0006243176758289337, 0.0008417945355176926, 0.0009202153887599707, 1.3628913620777894e-05, 0.0009061110322363675, 0.0008676133002154529, -9.970366954803467e-05, 0.00036359470686875284, 0.0006169346161186695, -0.00034000378218479455, 0.000911699899006635, -0.0005034022033214569, -0.0009736689389683306, -0.0005104611627757549, 0.0008609014912508428, 0.0008087903843261302, -0.0003414023667573929, 0.000987568753771484, -0.0002012734767049551, -0.0008581308065913618, -0.0005295767332427204, 0.00017809949349611998, 0.0005548771005123854, -0.0006874290993437171, 0.000942830927670002, -0.0007308116764761508, -0.00040778613765724003, 0.0005324663361534476, -0.00045220827450975776, 0.0007554264739155769, 0.00010842600022442639, 0.0002535033563617617, 0.000416818744270131, 0.000112472778710071, 0.000854936835821718, 0.00038344398490153253, -0.0009227465488947928, 0.00023345985391642898, 0.0008363877423107624, 0.0009813479846343398, -0.0005519023980014026, 0.0009393349173478782, -0.000991647131741047, 0.00018067787459585816, 0.0003976497973781079, -0.0006374030490405858, -0.0008572367369197309, -0.0004808385856449604, 5.652687104884535e-05, 0.0006664394750259817, 0.0004098780918866396, 0.00029652391094714403, 3.5169555303582456e-06, -0.0006996283191256225, 0.00084826466627419, -0.0003269880835432559, -8.28311312943697e-05, -1.708919444354251e-05, -0.0009617531322874129, -0.00044189367326907814, 0.0007125725969672203, 0.0006754369242116809, 0.0009219470084644854, -0.00045019277604296803, -0.0005735732847824693, 0.00022505219385493547, -0.0006646282272413373, -0.0006654268363490701, 0.000838802894577384, 0.0009551994735375047, 1.1816540791187435e-05, 0.00021958201250527054, 0.00012591012637130916, 0.0009791604243218899, -0.0007463831570930779, -0.000981794553808868, -3.528870001900941e-05, -0.0003191347059328109, 0.00046824614400975406, 0.0003000544966198504, 0.00037463416811078787, 0.0008534086518920958, -0.00045960713759995997, -0.0005983077571727335, 0.0008341372013092041, -0.0006445266189984977, -0.0003592963039409369, 0.0009893883252516389, 0.000385884108254686, -0.0008514693472534418, -0.00033478939440101385, -0.0007415537838824093, -0.0008323461515828967, 0.0008834595791995525, -2.6776107915793546e-05, 0.0008820719667710364, 0.0007184793939813972, 0.0009528007358312607, 0.0006180019117891788, 5.031420732848346e-05, 0.0008886680006980896, 3.5653119994094595e-05, 0.00011395323235774413, -0.0006318812957033515, 5.005832645110786e-05, -0.0007315730326808989, -6.997342279646546e-05, -0.0008366034599021077, 0.0004349142836872488, -3.094517524004914e-05, -0.00017656928685028106, -0.000976824900135398, 5.7416356867179275e-05, -0.0005024077836424112, 0.00011152059596497566, 0.0007342353346757591, -0.0004077785415574908, 0.0008293314022012055, -0.0009531098767183721, -0.0003059208975173533, 0.00023934394994284958, 0.0004459120682440698, 3.0437122404691763e-05, 0.0009864772437140346, 0.0002481454284861684, 0.0004506179830059409, -0.00045774722821079195, 0.0004893812001682818, 0.0004154285416007042, 0.00043467062641866505, -0.0005684012430720031, -0.0008463711128570139, -0.0002971014764625579, -0.00035859449417330325, 0.00021495188411790878, 0.0001966066483873874, -0.000623050145804882, -0.000776417669840157, -0.0006458567804656923, -0.0004222303978167474, 0.0008308969554491341, -0.0005199008737690747, 0.0007589212036691606, -0.0008462361292913556, -0.0002728566469158977, 0.0005393246538005769, -0.00041512653115205467, -0.00034973613219335675, -0.0004954550531692803, 0.0003346652665641159, 0.00026860859361477196, -0.0002571746881585568, 0.00030370496097020805, -0.00011086389713454992, -0.0009057269198819995, -6.0781741922255605e-05, -0.0009486355702392757, 0.0005632323445752263, 0.0003757017257157713, -7.983223622431979e-05, -6.406695320038125e-05, -9.782648703549057e-05, 0.0006667806301265955, -0.00017164877499453723, 0.0004994223709218204, -5.525250526261516e-05, -0.00038320038584060967, -0.0007267881883308291, 0.0007427321397699416, 0.0009760679095052183, 0.0009555867873132229, -9.460581350140274e-05, -0.00022420802270062268, 0.0004494953609537333, -0.0004688628832809627, 0.0008006091811694205, -0.0003610353742260486, 0.0009581606136634946, -0.0001375248102704063, -0.0009671642910689116, -0.0006241469527594745, -0.00010593007027637213, 0.00086166278924793, 0.0009471591329202056, -0.0006188434781506658, 9.024250175571069e-05, -0.0009387469035573304, -0.0004317882703617215, 3.567734165699221e-05, -0.00028809421928599477, -0.0009218193590641022, 0.0006840309360995889, -0.00034210673766210675, -0.0006452074158005416, 0.0007550930022262037, -0.0009445320465601981, 0.0006589440745301545, -0.0008611283847130835, 0.0006220754003152251, -0.0006889824289828539, 0.0008679735474288464, 3.3814496418926865e-05, -0.00014551507774740458, -0.0009883188176900148, -0.00048469885950908065, 8.01941569079645e-05, 0.00017854227917268872, -0.0005672617116943002, 0.0006436888361349702, 0.0006500662420876324, 0.0007484871312044561, 0.000660235236864537, 0.00022050672851037234, -0.0005958606489002705, -3.0444844014709815e-05, 0.0007339210715144873, -0.0005715539446100593, -6.40971438770066e-06, 0.00043942502816207707, -0.0002589296782389283, -0.00014410818403121084, -0.0008432823815383017, -0.00019432739645708352, 0.0006687934510409832, -0.0004696375399362296, 0.0007896043825894594, 0.0007705456810072064, 0.00027846774901263416, -0.00012706666893791407, 0.0005408240249380469, -0.0005585005274042487, 6.680608203168958e-05, -0.0002638290752656758, 0.0009908989304676652, 0.0007296091644093394, 0.0006022147717885673, -0.0006094577256590128, 0.0004376918950583786, 0.00019150682783219963, -0.0008160851430147886, 1.711809636617545e-05, -0.0007426314405165613, -0.00040998609620146453, 0.0003353209758643061, -0.0006370332557708025, -0.00023199942370411009, -0.0001762237079674378, 0.0005791274597868323, -0.0005584445898421109, -0.00010945317626465112, 0.00036027218447998166, 0.0006384107982739806, 0.0006271228776313365, 0.0008824158576317132, 7.5285747698217165e-06, -0.0009478777647018433, 0.0005060218973085284, -0.0007822628831490874, -0.0009429435594938695, -3.086329161305912e-05, 0.0005811398732475936, 0.0002147729101125151, 6.435357499867678e-05, -0.0008135882089845836, -0.0008861825335770845, 0.00035787324304692447, -0.000852269004099071, -0.0006773927016183734, 0.0004909794661216438, -0.0006355139194056392, -0.0006574494182132185, 0.000596546393353492, -0.0004284556780476123, -0.00011653624824248254, -0.00028586440021172166, -9.659806528361514e-05, 0.00042752036824822426, -0.0006206540856510401, -0.000301547406706959, 0.00017754166037775576, 0.00046718050725758076, -0.000302988279145211, -0.0005024567944929004, -0.0005396426422521472, 9.669296559877694e-05, -0.0008157892152667046, 0.0006510139792226255, -0.0007183679263107479, 0.000436099391663447, -0.0005499037215486169, 0.00036976896808482707, 1.3554842553276103e-05, -0.0006459259311668575, 0.00011600889411056414, 3.305274003650993e-05, 0.00037079001776874065, 0.0008626101189292967, 0.0007226267480291426, 0.000617606972809881, 0.0004738123679999262, 0.00031594932079315186, -6.628367555094883e-05, 0.0002827128046192229, 0.00023194419918581843, 0.0007200821419246495, 0.0007730675861239433, -0.0006526667857542634, 0.00029825756791979074, 0.0009878335986286402, -0.0008054859936237335, 0.00035196891985833645, -0.0009636222384870052, 0.0003297724178992212, -7.1317634137813e-05, 0.0007860647747293115, 0.0009554967400617898, 0.000625628512352705, 0.0009421605500392616, 0.0005066104349680245, -0.0002298041945323348, 0.0009221514919772744, -0.0004999228403903544, -0.00017903406114783138, 0.0001354080595774576, 0.0007359873270615935, -0.0007656797533854842, -5.716667874366976e-05, -2.2299323973129503e-05, 0.000580167630687356, -0.0007893759175203741, -0.0007089234422892332, 0.0003564874641597271, -0.00036361190723255277, 0.0005289243999868631, -5.568071355810389e-05, -0.0005016409559175372, -0.0008708378300070763, 0.0007944339886307716, -0.0008394874748773873, 0.0008549978956580162, -0.0006137583404779434, 0.00014194422692526132, -0.0006513791158795357, -0.0006078614969737828, -0.00084513402543962, 0.0005718468455597758, 0.0008411904564127326, 0.00047796248691156507, 0.0008129234192892909, -0.00045172753743827343, -0.0005728938849642873, -0.0005937946261838078, -0.0008928866009227931, 0.0007165924180299044, 0.0007561659440398216, 0.0008955835946835577, 0.00019440257165115327, 0.00045307388063520193, 0.0008514845976606011, 0.0009232792071998119, 0.0008090200135484338, 0.0004985879058949649, -0.0005649448721669614, -0.0007073445594869554, 0.00044956489000469446, 0.00016125771799124777, -0.0003624132659751922, 0.0006605188827961683, 0.0004037853213958442, -0.0005778847262263298, 0.0008260776521638036, 0.0001278798299608752, -0.0008596914703957736, -0.000723698700312525, -0.0006979919271543622, 0.00042631590622477233, 0.00042413544724695385, 0.00010288184421369806, 0.0007018503383733332, 0.00012900792353320867, -0.0003610209678299725, 0.0003295860660728067, -0.00033070900826714933, 0.0008847624412737787, 0.0006301820976659656, -0.0005400758236646652, -0.000512322352733463, -0.00025281167472712696, -0.0001864082587417215, -0.00010086157271871343, -0.0004949337453581393, -0.0003088282246608287, -0.0008769943960942328, 0.0009068316430784762, -0.0006944376509636641, 0.0002016883372562006, -0.0006834793020971119, -0.0006189627456478775, -0.00037531889392994344, -0.0005904308636672795, -0.0005729328258894384, -0.0008970333146862686, 0.0007655455265194178, 0.00032998877577483654, 0.0008723121136426926, -0.0006724343402311206, -0.000973884598352015, 0.0007743766764178872, -0.000174333035829477, 0.0003329519822727889, 0.0009045701008290052, -3.284749254817143e-05, -0.00046867624041624367, 0.0006878920830786228, 0.0003955657302867621, -0.0008616008562967181, 0.0006793616339564323, 0.00039625453064218163, 0.0006590512348338962, 0.000539604399818927, 0.000807195610832423, 0.0009662036318331957, 5.8756082580657676e-05, 0.0007265745662152767, -0.0004604071145877242, 0.0003109199460595846, 0.0007087300764396787, -0.0009810245828703046, -0.000687075313180685, -0.0004958013887517154, 0.00031653238693252206, 0.0005052016349509358, -0.0008202362805604935, -0.00058641197392717, -4.1885807149810717e-05, 0.00033321691444143653, -0.0007979267393238842, -0.0007845401996746659, -0.0001921256771311164, 0.0004812927218154073, 0.0003509740927256644, 0.0007138643995858729, 0.0007659149705432355, 0.0002526642638258636, -0.0008813826716504991, 0.0009068757062777877, -0.0007070776191540062, 0.0008311887504532933, -0.0009926302591338754, -0.000982827739790082, 0.0006934695993550122, 0.0008491267217323184, 0.0003738899831660092, 0.0005174535908736289, 0.0005788964335806668, -2.82336950476747e-05, 0.00021185746300034225, -0.0007256580865941942, -0.000671441201120615, 0.0005141770234331489, -7.962140807649121e-05, -0.0008256135624833405, 0.0006215313333086669, -0.0009544836357235909, -0.0005185322952456772, -0.00011114133667433634, -0.0007012778078205884, -0.0006085739587433636, 0.00035831404966302216, 0.0004312655364628881, 0.00021679015480913222, -0.0007652270141988993, 0.00018096478015650064, 0.000650288537144661, -0.0009472197270952165, 3.042706157430075e-05, 0.00014352436119224876, -0.0007295560790225863, 0.000901645515114069, -0.0005493860808201134, 0.0007914603920653462, 0.00028472000849433243, 0.0009008009801618755, 0.0005770592833869159, 0.000972742447629571, 0.00017620799189899117, 0.000920924823731184, 0.00018085028568748385, -8.223035547416657e-05, -5.465311187435873e-05, -0.0003049394872505218, 0.0003204012173227966, 0.00018510667723603547, 0.0007826503133401275, 0.0007015766459517181, 0.0008569342899136245, 0.0005161987501196563, 0.0007165268179960549, -5.8787845773622394e-05, -0.00019786023767665029, 7.077010377543047e-05, -0.00047015619929879904, -0.0003308330778963864, -0.00021627095702569932, 0.000296287500532344, -0.0006519280141219497, -0.0006776945083402097, 0.0003501310129649937, 0.0009852228686213493, -0.0006838814006187022, -0.00012792577035725117, 5.892954504815862e-06, -0.0009246706613339484, -0.0006346682785078883, 0.0006886691553518176, -4.812547194887884e-05, 0.000834667996969074, -0.000402064120862633, -0.0006958349840715528, -1.4801132238062564e-05, -0.000734088069293648, -0.0006646604160778224, 0.0008086730958893895, 4.690218702307902e-06, 0.0003024204634130001, -6.782590207876638e-05, -0.00020167134061921388, -4.6178462071111426e-05, 0.00031960519845597446, -0.0009282203973270953, -0.0008603299502283335, 0.0009906806517392397, 0.0007408357923850417, -0.0006394297233782709, 0.00040976039599627256, 0.000450194493168965, 0.0008054935606196523, -0.00034669594606384635, -0.00022365937184076756, 0.0004965405678376555, -0.0005298014730215073, 0.0004507870471570641, 0.00027730935835279524, 0.0005972348153591156, -0.0007577428477816284, -0.0004023220972158015, -0.0009451001533307135, 0.0009004492312669754, -0.0008707106462679803, 0.0002305543312104419, -0.00036567318602465093, -0.0009215656318701804, 0.000853668840136379, -0.0009318104130215943, 0.00062864541541785, 0.0009289816371165216, 0.0006343292770907283, -0.0008799561765044928, -0.0001345344353467226, -0.00017862828099168837, -0.0007945157121866941, -0.0002833587641362101, -0.00045519781997427344, 0.0007862788042984903, 6.940509774722159e-05, 0.0009839866543188691, -0.0007793729309923947, -0.0009278777870349586, -0.0005843811086378992, 0.0004074934695381671, -0.0005593032692559063, -0.00040063023334369063, 0.0005103638395667076, 0.0002306224632775411, -0.0004441841156221926, 0.00017579337873030454, -0.0006148329703137279, -0.0002777624176815152, -0.0006723852711729705, 0.0001576985523570329, -0.0003215529432054609, 0.0004923962987959385, -0.0006818596739321947, -0.00048240923206321895, -0.0005408196011558175, 0.0006793846259824932, 0.00010636745719239116, -0.0008106063469313085, 0.0002123318990925327, 0.00038425077218562365, -0.0004669054760597646, -0.00035044862306676805, -0.0003367037861607969, 0.00034319149563089013, 4.444568548933603e-05, 0.00036749718128703535, -1.9933338535338407e-06, 0.0008451485191471875, 0.00024250640126410872, -0.0009273795294575393, -0.00022518735204357654, 0.0006390893831849098, 0.0009717034408822656, 0.0005081503768451512, -0.0004479598137550056, -0.0003534540592227131, 0.0003769673639908433, 0.0002748390252236277, -0.0004998339572921395, 8.97651698323898e-05, -0.00014786093379370868, -0.0005769220879301429, -0.00012092479300918058, -0.0006002935697324574, -0.0009219520725309849, -0.0004918057820759714, -0.000946861575357616, 0.0008524074801243842, -0.0005991102079860866, 0.0008933372446335852, -0.0007150621386244893, 0.00023827583936508745, -4.229291880619712e-05, 0.0005720118060708046, 0.0009394614025950432, 0.0005763840163126588, 0.0007564867846667767, 0.0006241472437977791, -0.0005534408846870065, 0.0005400782683864236, 0.000916890159714967, -0.000332819065079093, 7.033750648588466e-07, -0.0008993212832137942, 0.00019403075566515326, 0.0006043739267624915, -0.00032885040855035186, -0.0005540557322092354, 0.0005570820649154484, 0.0002961822028737515, -0.0009753300109878182, 0.0008428383152931929, 0.00036713574081659317, -0.0008134188828989863, -0.0005819494836032391, 0.0008588715572841465, -4.372098919702694e-05, 0.0004995230119675398, -0.0005933415959589183, 0.0009904479375109076, -0.0006274213665165007, -3.253338218200952e-05, -0.0008287134696729481, -4.487657133722678e-05, -5.710223558708094e-05, 0.0009198528714478016, 0.00040209750295616686, 0.0007132109603844583, 0.000979008269496262, -5.758909901487641e-05, 0.0002620631130412221, -0.00016653303464408964, 0.00021206993551459163, 0.0005420700763352215, -0.0005790495779365301, -0.000570893578696996, 0.00037606514524668455, 0.0002846520219463855, -0.0009169562836177647, -0.0002701489138416946, -0.00050008250400424, -0.0007854487048462033, 0.00032316974829882383, -0.00035367999225854874, 0.0007595117203891277, 0.0005262098275125027, 0.0005284872022457421, -0.0006981065962463617, 0.0006868973141536117, 0.0006795438239350915, 0.0008383967797271907, 7.939447095850483e-05, 0.0007079332717694342, 0.000185377910383977, 0.0009486799244768918, 0.0009234139579348266, 8.939480903791264e-05, -0.0006219490896910429, -0.0003533565322868526, -0.0008741187048144639, -0.0006032012752257288, -0.0004093250317964703, -0.0007284568273462355, 4.323818939155899e-05, 0.0007439895998686552, 0.0008164278697222471, 0.0007970247534103692, -0.0003428039781283587, -0.0009986208751797676, 0.0007179363165050745, -0.0004887345712631941, 0.0008329972624778748, 0.0007972396560944617, 0.0008011707686819136, 0.0001208716057590209, 0.00026213342789560556, -3.554547583917156e-05, -0.0005402396782301366, 0.0006913356482982635, 0.00010564809781499207, 0.0001509972644271329, -0.0001337891590083018, 0.0007460458436980844, 0.0009806447196751833, 0.00040108259418047965, 1.4093862773734145e-05, -0.00029986415756866336, 0.00035172011121176183, -0.0004418875032570213, -0.000306002504657954, 0.0005618427530862391, 0.00025456599541939795, 0.0009378393879160285, 0.00039600065792910755, 0.0004919202183373272, 0.00091538269771263, -0.0004894558223895729, 0.00024047220358625054, -0.0009942699689418077, 0.0003142048080917448], [0.0007976079941727221, 0.0006488697836175561, 0.0008210954256355762, 6.136011506896466e-05, 0.00024148737429641187, -0.000438796472735703, -0.00013206119183450937, 0.0006500814924947917, -0.0008878212538547814, -0.0005426639690995216, 0.0008163020829670131, 0.00031446703360415995, 0.000565068272408098, 0.0006009062053635716, -0.00027499583666212857, 0.0009529544622637331, 0.0006917478749528527, 0.00010473285510670394, -0.0006516182911582291, -0.0006588652031496167, 0.00010875108273467049, -0.0006713177426718175, -5.8598983741831034e-05, 5.783697633887641e-05, 0.0005823675310239196, 0.00010979459329973906, -0.000274596648523584, 0.0003792813222389668, 0.0006846751202829182, 0.0001376186264678836, -0.0005186168127693236, -0.0006922177271917462, -0.0006854379898868501, -0.0005906537990085781, 0.0006443116581067443, -0.00035282238968648016, 0.00039216887671500444, -0.0008680628379806876, -0.0006196683389134705, 0.0001802601182134822, -0.0006872203084640205, 0.0003088877128902823, -0.0005543709266930819, 0.0001777463621692732, -0.0006086017237976193, 0.00023423192033078521, 0.0008652624092064798, 0.0009615655289962888, 0.0009219275088980794, 0.0006085900240577757, -0.0006817258545197546, 0.00027080217842012644, 7.645360165042803e-05, -0.000752258871216327, 0.0003975015424657613, 0.0006701062666252255, -0.0009281648090109229, 0.0002753734588623047, -0.00043797792750410736, -0.0004555624327622354, 0.00030573748517781496, -0.0009247827692888677, -0.0008326389943249524, -0.0004961610538884997, 0.0005816245102323592, -4.7313944378402084e-05, 6.508778460556641e-05, 0.0004565565614029765, 0.00034548123949207366, -0.0007950927829369903, 0.0004654382064472884, 0.0007758709252811968, 0.0005030272295698524, 2.6158175387536176e-05, -9.462579328101128e-05, 0.0001182884443551302, -6.82003956171684e-05, 0.0009645227692089975, -2.030705218203366e-06, 0.00017992107314057648, 3.117226151516661e-05, -0.0004750350199174136, -0.0001088763601728715, 0.000979257165454328, 0.0008270136313512921, -0.0009499153238721192, -0.00021701582591049373, 0.0007669064798392355, -7.890439155744389e-05, -0.0009003027225844562, -0.000137643568450585, 0.00043412166996859014, -0.0005974796949885786, 0.0006473789107985795, -0.0009332616464234889, -0.0007647969760000706, 0.000679063843563199, 0.000873195065651089, 3.626143006840721e-05, 4.1047354898182675e-05, -0.00030660515767522156, -0.00067476078402251, -0.0003302505938336253, 0.0009533777483738959, -0.0008933359058573842, 0.00015133038687054068, -0.0007454364676959813, -0.0004512563464231789, 0.0006554671563208103, 0.000245492352405563, 0.00035997392842546105, -0.0007801892352290452, 0.0008111742208711803, -0.0004405866202432662, -0.0003126520605292171, -0.0003658975474536419, -0.00026466912822797894, 0.0004817921726498753, -0.00045468966709449887, -0.000764011696446687, -0.000603624852374196, -0.0004405845538713038, -0.00046686932910233736, 0.000500001129694283, 0.00047350028762593865, 0.00034341125865466893, 0.0009612541180104017, -0.0007476943428628147, -0.00035820435732603073, -0.0008000164525583386, 0.00023452125606127083, -0.0008363247034139931, -0.0005072391359135509, 0.0009195500751957297, -0.0006698177894577384, 0.00041393734863959253, 0.0008922972483560443, -0.0006662189261987805, 3.858839409076609e-05, 0.00092087168013677, 0.0007415054133161902, 0.00018057069974020123, -0.0007465306553058326, -0.00010532439046073705, 0.00022363280004356056, 0.0006863713497295976, -0.0003409050696063787, 0.0009288541623391211, -2.9003858799114823e-05, 0.0007960457005538046, -0.0004435738082975149, -0.0003697454812936485, 0.0005278519238345325, 0.0009485364425927401, -0.00019415961287450045, 5.5372809583786875e-05, 0.00030614191200584173, -0.00036091546644456685, -0.0006431099027395248, -0.00016008027887437493, -0.0002802697126753628, 0.0009487071656621993, 0.0004976564086973667, -2.2343344880937366e-06, -0.0007107499986886978, -0.00013492467405740172, -0.0009387939353473485, -0.0008182846941053867, 0.00018335018830839545, 0.0002308032853761688, -0.00025547953555360436, -0.0004281301808077842, 0.00023002538364380598, 0.0003631323925219476, -0.0002613971009850502, -8.826710836729035e-05, 6.571774429176003e-05, -0.0008423233521170914, 0.00010720462159952149, 0.00030274756136350334, 0.00043018077849410474, -0.0008046257426030934, 0.0003631868166849017, -0.0004587481380440295, 8.938107930589467e-05, 0.0008517518290318549, 0.0004520737857092172, -0.0001591250766068697, -0.0009846562752500176, -0.000483956013340503, -0.0003394932427909225, -0.0004329849034547806, 0.0009709431906230748, 0.0009907361818477511, 0.0007812186959199607, -0.0009158216416835785, -0.0001255842944374308, 0.0004990343004465103, -0.0007952636224217713, 0.0009792241035029292, -0.0005805987166240811, 0.0006407288601621985, -6.265294359764084e-05, -0.00099840946495533, 0.0009459491120651364, -0.0007332560489885509, -0.0007187305600382388, -0.000666654494125396, 0.0009342783596366644, 0.00015516785788349807, -0.0005057432572357357, 5.970036727376282e-05, 0.00013481949281413108, 0.0009285731357522309, -0.00020110733748879284, -0.0009303144761361182, -0.0002628063957672566, -0.00013238766405265778, -0.0007354939007200301, 0.0005910102627240121, -0.0007825582288205624, 0.00044788685045205057, 0.0007748876232653856, -0.0009616621537134051, 0.0008412225870415568, -6.291967292781919e-05, -0.0005878874217160046, -0.0005470769247040153, 0.0008395694894716144, -0.0005426804418675601, 0.0007590605528093874, -0.0006276326603256166, 0.00069674156839028, 0.000802589813247323, -0.00040720563265495, -0.0002774207678157836, -0.0002489698235876858, -0.00029039717628620565, 0.0002677604788914323, 3.305274003650993e-05, -0.0005466958973556757, 4.779336086357944e-05, -0.00035822801874019206, -0.0005355928442440927, 0.0005106221651658416, 0.00025692832423374057, -0.00021703111997339875, -9.417316323379055e-05, 0.0003809684421867132, -0.0003638613852672279, -0.00021307783026713878, 0.0009858205448836088, -0.0008796885958872736, -0.0007600946701131761, -0.0008197564748115838, 0.00019734844681806862, 0.0005306570674292743, 0.0007146736024878919, -0.0009957285365089774, 0.0009119991445913911, 0.000998243922367692, -0.0002409007865935564, -0.0009217955521307886, -0.0008744048536755145, -0.0006671994924545288, 0.00014861664385534823, -0.00024625228252261877, 0.0008847599383443594, 0.00046952677075751126, 3.0381705073523335e-05, 0.00017272825061809272, -0.0004237479588482529, 7.237689715111628e-05, -0.00028489198302850127, -0.000700268370565027, -0.0007226616144180298, -0.0007573929033242166, -0.00011816034384537488, -0.00038638239493593574, 1.3613094779429957e-05, 0.0006704076658934355, -0.000706033140886575, -0.0009867158951237798, 8.453593181911856e-05, -0.0008746791863813996, -0.0006012502708472311, 0.00031919204047881067, 0.0003709132724907249, 0.00046939431922510266, 0.00020610852516256273, 0.0008304438088089228, 0.0007044890080578625, -0.0006970478571020067, 6.341881089610979e-06, -0.0005176442791707814, 0.00035165974986739457, 0.0004896768368780613, 7.478270708816126e-05, 0.0003148597606923431, 0.0002779276401270181, 0.00014251067477744073, -0.0003735041245818138, -0.0007598952506668866, 0.0008770827553234994, -0.0002462906704749912, 0.0006652084412053227, -7.041349454084411e-05, -0.00049429020145908, -0.0005242311162874103, 0.00019798989524133503, 0.00029029083088971674, -0.0007662695134058595, 0.0005254716379567981, -0.00032919124350883067, -0.0009227570844814181, 8.731284469831735e-05, -0.00011036652722395957, 0.0002732413704507053, -0.0007597773219458759, 0.0005057092057541013, -0.0009553871350362897, 0.0008828152786009014, -0.0008436065400019288, 0.0007201620028354228, -0.0008175526163540781, 0.00039731545257382095, 7.129413279471919e-05, -0.000591726740822196, -0.0005424902774393559, 0.00040329800685867667, -0.00014859347720630467, 0.0003656342742033303, 7.10181484464556e-05, -3.478280996205285e-05, -0.000817697960883379, -0.00019687482563313097, -0.00046003732131794095, -0.0006893268437124789, -2.799617050186498e-06, -0.0008853693725541234, -0.00045329827116802335, -0.0008850596495904028, 0.0008839573711156845, -0.0003336023073643446, -0.00039575027767568827, -0.0006907988572493196, -0.00022520724451169372, 0.0003816062235273421, 0.00031381568987853825, 0.0009055979317054152, 0.0007227722089737654, 0.0008698404999449849, -0.0005716849118471146, -0.00018083324539475143, 0.0009482039604336023, 8.424650150118396e-05, -0.0009804468136280775, 0.0008106706663966179, -0.0003924887569155544, 3.44543150276877e-05, -0.0006917305872775614, -0.00017925709835253656, -9.451824735151604e-05, 3.5746175854001194e-05, -0.00057063321582973, -0.0002628101210575551, -0.0004000636108685285, 0.00048304046504199505, -0.0009372257045470178, 0.0006738301599398255, 0.00026484817499294877, 0.0008749740663915873, -0.00020464236149564385, -0.0001880842464743182, 0.00020960044639650732, 0.0008448372827842832, 0.0005104440497234464, -0.0008257712470367551, 3.8107762520667166e-05, -0.0006418628036044538, 0.0009223331580869853, 0.000784054456744343, -0.0001999412343138829, 0.0002510137273930013, -0.0005952569772489369, -0.00010332110105082393, 0.0005535828531719744, 2.1726180420955643e-05, -6.866546027595177e-05, -7.997488137334585e-05, -0.00020303977362345904, 0.0008879164815880358, 0.0006271573365665972, -0.0007501992513425648, 0.0007292001973837614, -0.00024869476328603923, 0.0003286955179646611, -0.0004853338759858161, -0.0007369546219706535, 0.0004008393734693527, 0.0007224705186672509, 0.0005565589526668191, -4.9927060899790376e-05, -0.0002823269460350275, 0.0006363455322571099, 0.0004300088039599359, -0.00032742254552431405, 0.0004921554354950786, -0.00021206254314165562, 0.0005512309726327658, 0.0005689039244316518, 0.000714445486664772, 0.0003436872211750597, 0.0001393069396726787, -0.0008137082331813872, -0.0003879377036355436, 7.69008111092262e-05, 0.0004805159114766866, -0.00043538532918319106, 0.0008604852482676506, -0.00014522408309858292, 0.0005653027910739183, 0.0005376831977628171, -0.00020713919366244227, 0.0005463292472995818, 0.00014764555089641362, 0.0006035194383002818, 0.0008837062632665038, 0.0005966406315565109, -0.00017096327792387456, -0.0008997560362331569, 0.00010986508277710527, 0.000723662378732115, -0.0007796940044499934, -0.00028872693656012416, 0.0008728320244699717, 0.0004832630220334977, -0.0003154169535264373, -0.0001165013964055106, 1.638573667150922e-05, -5.907724334974773e-05, -0.0009815272642299533, 0.00014230026863515377, -0.0002066783927148208, -0.0007657181704416871, -1.4758543329662643e-05, 8.619313302915543e-05, 0.0009638365590944886, -0.0004290770448278636, 0.00048780482029542327, 0.00025918075698427856, -0.00021512176317628473, -0.0008361570653505623, -4.139613884035498e-05, -0.0006194449961185455, -0.00018610694678500295, -0.0009958789451047778, -0.0005006612045690417, 0.000688849831931293, 0.0007243996951729059, -0.000907412264496088, -0.0006388673209585249, -0.00010625915456330404, -0.000736819114536047, -0.0005816400516778231, 0.0006585075752809644, -0.0008520162082277238, 0.0007074233726598322, -0.00029974625795148313, 0.0004993086913600564, 0.0005142797017470002, -0.00032443326199427247, -0.000272065051831305, 0.0008600809960626066, -4.667439134209417e-05, 0.0008553285151720047, 0.00045379664516076446, -0.00036762384115718305, -0.0005379124195314944, 0.0004516358894761652, 0.0002576342667452991, -0.0006869710050523281, -0.0002570922370068729, -0.0001758837461238727, 0.0007062226068228483, -0.0007562582031823695, -0.00044727904605679214, -0.0003722898254636675, 0.00014128962357062846, 0.0005274572176858783, 0.0007667692261748016, -0.0009119455935433507, -0.0001779058075044304, -0.0007400157628580928, -1.481280264670204e-06, 0.0009187913383357227, -0.0006786009762436152, -0.0003908248618245125, 0.0006462058518081903, 0.0009230313007719815, 0.0008688809466548264, -0.0005510695045813918, -0.0006716433563269675, 0.000941580452490598, -5.940553455729969e-05, 0.0006050246302038431, -0.0007525726687163115, 0.00021217578614596277, -0.0002945735468529165, -0.0001072142695193179, -2.441493961669039e-05, -0.00019612839969340712, -0.0006461558514274657, 0.000982615863904357, 0.0006182130891829729, -0.0005870306631550193, -0.00013231998309493065, 0.0006248832796700299, 0.0002211620594607666, -4.8005593271227553e-05, 0.0009548157686367631, -0.000607748981565237, -0.0006003978778608143, 1.5894796888460405e-05, 0.0009007456828840077, 0.0008147937478497624, 0.0005872766487300396, -0.0005114629748277366, -0.00045876740477979183, -0.0005503712454810739, -0.00024598772870376706, -0.0006539872847497463, 0.00015174711006693542, 0.000268186122411862, -6.980311445659027e-05, -6.900112930452451e-05, -0.0007796357967890799, -0.0003493685508146882, 0.00014487917360384017, -0.0006861586007289588, -0.0009658591006882489, -0.0004804607015103102, 0.00016366178169846535, -0.00027572104590944946, -0.000297014630632475, -0.0005863545811735094, 0.0005861151148565114, -0.000250080949626863, 0.0008527321624569595, 0.0006090924725867808, 0.00020081391267012805, -0.0003322552947793156, 0.000988465384580195, -0.00044216628884896636, 3.124097202089615e-05, 0.000357106706360355, -0.0008165089529938996, 0.0005136851104907691, 1.4437279787671287e-05, -0.0006503672921098769, -0.0005023444537073374, 0.0001916015025926754, 0.0006691782618872821, 0.00047991209430620074, 0.0008215292473323643, 0.0002484397264197469, -0.0003204664390068501, 0.0005935588851571083, -0.0007199174142442644, 0.0007264002342708409, -9.814286750042811e-05, -0.0007091015577316284, 0.00018810780602507293, -0.000173903041286394, -7.636247028131038e-05, 0.00043476748396642506, 0.00015133658598642796, -0.0009367301245220006, 0.00029110859031789005, 3.584608293749625e-06, -0.00030715204775333405, 3.321064286865294e-05, 0.0009348414605483413, -0.0007924003875814378, 0.0006480508018285036, 0.0007191180484369397, -0.000908399117179215, -0.0003117761807516217, 0.0002381754748057574, 0.0008778579067438841, -0.00022868452651891857, 0.0004878881445620209, -0.0005329158157110214, -0.000581605767365545, 0.000697617419064045, 0.0006491463864222169, -0.0008917024242691696, 1.6872510968823917e-05, -0.0006927168578840792, -0.0004509736900217831, 0.0007267629844136536, 0.0006825958844274282, -3.0276962206698954e-05, 0.00015106360660865903, 0.00014871843450237066, -4.499178885453148e-06, 0.00015468358469661325, -3.712783609444159e-06, -0.0008854253683239222, -0.00036465429002419114, 0.00021150264365132898, -0.00010892999853240326, 0.0004963153623975813, 0.0007410576799884439, -1.3720650713366922e-06, -0.0007492337608709931, 0.0009914812399074435, -0.0007078281487338245, -0.00031715634395368397, -0.00010002309136325493, -0.00048686302034184337, 0.0007144843111746013, -0.000987233011983335, -5.645154487865511e-06, -0.0007930524880066514, -0.0007341585587710142, -3.734487108886242e-05, 0.0004560116503853351, -0.0004855598381254822, 0.0009582078200764954, -0.0008564796298742294, -0.0005771174910478294, -0.0004669420886784792, -0.0007374241831712425, 0.0007720580324530602, 0.0009130326216109097, -0.00047642411664128304, 0.0007859201868996024, 0.00016343293827958405, -0.0003082737675867975, 4.1824827349046245e-05, 0.00014130362251307815, -0.00014239520533010364, -0.00020274095004424453, 0.0009905361803248525, -0.0005126302712596953, -0.0002663199556991458, 0.0007731526857241988, 0.00035211179056204855, -0.00043929435196332633, 0.00016873839194886386, 0.00012475179391913116, 0.0006648541893810034, 0.0008995576645247638, 0.0006641309591941535, -0.00013243452121969312, -0.0006118763121776283, -0.00047447788529098034, -0.0009237737394869328, -0.0003779165563173592, 0.0007197638042271137, -0.00030389640596695244, 0.00020244510960765183, 0.0001245472813025117, -0.0007071656873449683, -0.0009579138713888824, -0.0004292182275094092, -0.0008485918515361845, 0.0006009220378473401, 0.00039766813279129565, -4.0094757423503324e-05, 7.247902249218896e-05, 0.000332820403855294, 0.000462738040369004, -0.00017518886306788772, 0.0009691379964351654, -0.00019687297753989697, -0.0005225295899435878, 0.0007836184231564403, 0.0004960430087521672, 0.00025540596107020974, -0.000833576952572912, 0.00046187787665985525, 0.0009070524829439819, -0.000585098285228014, -0.0007214774959720671, -0.00031502192723564804, -4.077242556377314e-05, -9.79379256023094e-05, 0.0007298451382666826, 0.0009039876749739051, -0.0009803923312574625, 0.00038710495573468506, 0.00016696134116500616, 0.0009286786080338061, -0.0008825305849313736, -0.0006385135347954929, -0.0006336091901175678, -0.00014343565271701664, 0.0004623841668944806, -0.0005025765858590603, -0.0006513692205771804, -0.00046230226871557534, -0.0003137339954264462, 0.0005051387124694884, 0.0006109655369073153, -0.00030888052424415946, -0.0008342080400325358, -0.00032694381661713123, -0.00021230211132206023, -0.0001063939489540644, 0.0009307906148023903, -0.0008090108749456704, -1.861127657321049e-06, -0.00036099611315876245, 0.00026723259361460805, -0.00020385121752042323, -0.0005380083457566798, 0.0002555820974521339, -0.0009183334768749774, -9.346766455564648e-05, 0.00023148863692767918, 0.0005415815976448357, 0.0005072938511148095, 0.00022198542137630284, -0.0008703768253326416, 0.0008323764195665717, 0.00047397275920957327, -0.000756483874283731, 0.00014164973981678486, 0.00047811237163841724, -0.0007666912279091775, 0.0009265599655918777, -0.0006430663634091616, 0.0007710320642217994, 0.00015893395175226033, 0.00027857456007041037, -0.00023482181131839752, 0.0004831071419175714, -0.0007058762130327523, -0.00035485229454934597, -6.658990605501458e-05, 0.0003497202415019274, 0.0003689447185024619, -0.000862800981849432, 0.0009649308631196618, -0.0008383503300137818, 0.0007075999747030437, -0.0008383651729673147, 0.0008239121525548398, -0.00030796037754043937, 0.0005422460963018239, -0.0007643992430530488, -0.0008612480014562607, -0.0006042534951120615, 9.057694114744663e-06, 0.0007738273125141859, 0.0003074240521527827, 0.00015361368423327804, -0.00018572621047496796, -5.0309838115936145e-05, 0.0009131425176747143, 0.00022979739878792316, 0.0009162084897980094, -0.0005075393710285425, 0.00047510775038972497, -8.777041512075812e-05, 0.0005804641987197101, -7.076064503053203e-05, -0.00037213467294350266, 0.0001685598399490118, -0.0004915675963275135, -0.0007880732882767916, -0.0004763938195537776, -0.0008178848074749112, 0.0007470287382602692, -0.00098974141292274, -0.00053993338951841, 0.0008881920948624611, 0.0005124065792188048, 0.0009587498498149216, 0.0005735672311857343, -0.0008240024326369166, 0.0008437398937530816, 0.00017220107838511467, 2.2950621314521413e-06, -6.44962492515333e-05, 3.2797150197438896e-05, -0.0005582610028795898, 0.0009526379290036857, -0.0007553701871074736, -0.0003855513350572437, 0.0003783373977057636, 0.0001764940534485504, -0.0002366852859267965, -0.0006397927063517272, 0.0002279252075823024, 0.00030928198248147964, 0.00032086827559396625, 0.00024140308960340917, 0.0008303659851662815, 0.0007864300860092044, -0.0009158537723124027, -1.7459535229136236e-05, 0.0007346738129854202, 0.0004320454318076372, -0.0005048697348684072, -0.0006996533484198153, -0.0006903274334035814, -0.0004353473777882755, 7.81171620474197e-05, 0.0003051077073905617, 0.0001570852764416486, 0.0005627918872050941, 0.00041648137266747653, -0.0003549620450939983, -0.000319103361107409, 0.0008607733179815114, -0.0009334019850939512, -0.0003614280722104013, 2.0638544810935855e-05, 0.00021009570627938956, 0.0004638149111997336, 0.00043343284050934017, 2.848074109351728e-05, 0.0004387432709336281, -0.0008437116048298776, -0.00020375289022922516, -0.0009748156298883259, 0.00017972207569982857, 0.0006362036801874638, -0.00015937129501253366, -0.00024126272182911634, -0.000407372135668993, 0.0008579996065236628, -0.0002808043791446835, 0.0004488263512030244, -0.0005922215059399605, -0.0002761697687674314, 0.000656654592603445, 0.0004098032950423658, -3.481049134279601e-05, -0.0005891479668207467, 0.0009809703333303332, -0.00039384959382005036, -0.0006514308042824268, 0.0007420121692121029, 7.514305616496131e-05, 0.0006138451281003654, 0.0005774319288320839, 0.0004204700526315719, -0.0004826489312108606, -0.0008794707828201354, -0.00046102236956357956, 0.000438999937614426, -0.0005454961792565882, -0.00021591500262729824, 0.000628833775408566, -0.0008470400352962315, -6.145342194940895e-05, 0.00026758943567983806, 0.0009835538221523166, -0.000553459394723177, -0.00014506893057841808, 0.00016922998474910855, -5.494220386026427e-05, 0.000468057842226699, -0.00024609098909422755, 0.0007841680780984461, -0.0007018899777904153, -0.0007350760279223323, 0.0003866148763336241, 4.801232080353657e-06, 0.0004592463665176183, -3.110313991783187e-05, -0.0005500987754203379, -0.000309336232021451, -0.0003051544481422752, 0.0003983564965892583, 0.00045952422078698874, -0.00031367974588647485, 0.00017200947331730276, 0.0006186303216964006, 0.000921082915738225, -0.0008075556834228337, -0.0009587054955773056, -0.0006405856111086905, -0.0009461964364163578, -0.0006301890825852752, -0.00033353123581036925, 0.00011044007260352373, 0.0007851665723137558, -0.0007431792910210788, 0.0004270948702469468, -9.171369310934097e-05, -0.0004448603722266853, 0.0009251240408048034, 0.0005817859782837331, -0.0004273461818229407, -0.000909437658265233, -0.0007038851617835462, -0.0003255256451666355, 0.0008189745713025331, 0.00017831155855674297, -0.00012401775165926665, 0.00011731139238690957, 0.0001579337113071233, 0.0007429499528370798, -0.0008615978877060115, -0.0006278238724917173, 0.0008303035865537822, -3.577973984647542e-05, 0.0005878832889720798, 0.00013236052473075688, 0.00043845197069458663, 0.0005054961075074971, -0.0005735380109399557, -0.00020437063358258456, 0.000834374688565731, 0.0008663854096084833, 0.00041787183727137744, 0.0007874729344621301, 6.972574919927865e-05, -0.0001293547247769311, 0.0009966302895918489, 0.0008599490392953157, -0.0009030550718307495, -0.0003636110632214695, -0.0004296959668863565, -0.00012801302364096045, -0.0002517368411645293, 0.00032261223532259464, 0.0008122061262838542, -0.0008906561997719109, 0.00012250962026882917, -0.00013073162699583918, 0.0008923825807869434, 0.000606363348197192, 2.4289787688758224e-05, 0.00011347379768267274, -0.000555632112082094, -0.00014420965453609824, -0.0009302174439653754, 0.00038145831786096096, 0.0006723649566993117, 2.0669043806265108e-05, -0.00023955240612849593, 0.00029765625367872417, -0.0006148773245513439, -0.0006904354086145759, -0.0002322695218026638, -0.0002804304240271449, 0.00044906584662385285, 0.0007915938622318208, 0.0009359249379485846, 4.242565046297386e-05, 0.0007447181851603091, 0.0006214989116415381, -0.0008561297436244786, 0.0005823733517900109, -0.0009824507869780064, 0.000582794426009059, 0.00034140024217776954, -0.0002277322200825438, 0.00010497499897610396, 0.0008340526837855577, -0.0009473912650719285, -0.00037743683787994087, -0.00045855416101403534, -0.000641189981251955, 0.0007591552566736937, 0.0003962626797147095, -0.0002555733371991664, 0.00016396306455135345, 0.0006479144794866443, -0.0008217033464461565, -0.00039414039929397404, 3.087556251557544e-05, 0.00047435887972824275, -0.000868615519721061, 4.497101326705888e-05, 0.0007816566503606737, -0.00031745093292556703, -0.0006355812074616551, 0.0004684824962168932, 0.00018034335516858846, -0.0007092516170814633, 0.0005714063299819827, -0.00013916536408942193, 4.223555151838809e-05, 0.0002145643811672926, 0.000800100970081985, 0.0001764828630257398, -0.0009567482629790902, -0.00035005362587980926, -0.00028927376843057573, -0.00012943071487825364, -0.0008749061380513012, 2.4873035727068782e-05, -0.0002639532904140651, 7.804590131854638e-05, -0.0006292265024967492, 0.0004888711264356971, -0.0004982455284334719, -0.0003102109767496586, -0.00035669212229549885, 0.0005604299949482083, -0.0001840250042732805, -5.067340680398047e-05, -0.0005851297755725682, -0.0008483818382956088, -0.0003322142583783716, 0.00023905964917503297, -0.00022922878270037472, 0.00026696614804677665, 0.000776536762714386, -0.0008758640033192933, -0.0006847766926512122, 0.00020625791512429714, 0.00011362289660610259, 0.0002062225539702922, 0.0007494923775084317, -0.0009028983768075705, 0.00027607378433458507, 0.0008187959901988506, -0.000728978484403342, 0.00012574021820910275, 0.00029277190333232284, -0.00018168435781262815, 0.0005279784090816975]], "dtype": "float32"}, 1e-12], "e": {"__tup__": [{"__nd__": [[0.0025396165437996387, -0.006113096605986357, 0.038902364671230316, -0.05073558911681175, -0.012888536788523197, 0.037922561168670654, -0.04142644628882408, 0.01647450029850006, -0.036931656301021576, 0.043016549199819565, -0.044309988617897034, -0.008362063206732273, 0.01496379915624857, -0.010606936179101467, 0.04152464121580124, 0.02245146408677101, -0.03677807375788689, 0.034852709621191025, -0.018811706453561783, -0.0027717184275388718, 0.03113352693617344, 0.008510300889611244, -0.02150794118642807, 0.0265350379049778, 0.02183663472533226, 0.018269356340169907, -0.05156378820538521, -0.007812896743416786, -0.02460263855755329, -0.01631411537528038, -0.017027873545885086, -0.011558313854038715, 0.036303527653217316, -0.024754367768764496, 0.032397691160440445, 0.03751463443040848, 0.013231557793915272, 0.01827183924615383, -0.018262900412082672, 0.018399741500616074, 0.04537158086895943, 0.030059268698096275, 0.04008857533335686, -0.05265806242823601, 0.02891092374920845, 0.02545984275639057, -0.017640123143792152, 0.01801452226936817, 0.03934396058320999, -0.025698687881231308, -0.01506353821605444, 0.03667357191443443, -0.01712934300303459, -0.020395010709762573, 0.03657787665724754, 0.04471525549888611, -0.011505628935992718, -0.04299401491880417, -0.029819758608937263, 0.039677899330854416, 0.010188902728259563, -0.0009004034218378365, 0.03395333141088486, -0.02258877642452717, -0.02113964594900608, -0.01618209294974804, 0.046730104833841324, -0.04201152175664902, 0.007163672242313623, 0.0247051864862442, 0.029998689889907837, -0.02164365164935589, 0.016644826158881187, 0.02167494036257267, 0.0010538962669670582, 0.008825425989925861, -0.03568486496806145, 0.024869006127119064, -0.023451142013072968, 0.044267553836107254, -0.039710428565740585, -0.046023476868867874, -0.05194905772805214, 0.005659407004714012, -0.0319100022315979, -0.0509188137948513, 0.006026696879416704, 0.007720117922872305, 0.02160029485821724, -0.0033298886846750975, 0.006811453960835934, -0.04288257658481598, -0.052531562745571136, -0.029607564210891724, 0.03089027851819992, -0.035356491804122925, -0.02842525951564312, -0.024059752002358437, 0.012756853364408016, -0.022154061123728752, -0.007258414290845394, -0.012710540555417538, 0.046084724366664886, -0.020080486312508583, 0.022448886185884476, -0.03092191182076931, -0.02927064709365368, -0.037640899419784546, 0.022424081340432167, -0.0149300592020154, -0.0248220507055521, 0.015865299850702286, 0.04082096740603447, -0.03559350594878197, -0.042450569570064545, -0.04185071215033531, 0.046673208475112915, 0.019187627360224724, -0.03415714576840401, 0.01714109070599079, 0.020469479262828827, 0.03987215459346771, -0.014068160206079483, -0.028907442465424538, 0.035164739936590195, 0.014975477010011673, -0.023774156346917152, -0.03203917667269707, -0.04778100550174713, -0.00476668868213892, -0.014505498111248016, 0.022909395396709442, -0.050376247614622116, 0.003925723489373922, 0.05113422870635986, -0.005036324728280306, -0.029682395979762077, -0.01699109375476837, -0.0065381573513150215, -0.013537746854126453, -0.03960954397916794, -0.006584556307643652, 0.028006400913000107, 0.027469312772154808, -0.04026242345571518, -0.02279166504740715, -0.04883713647723198, -0.031117016449570656, 0.03349313139915466, -0.04233806952834129, -0.050700876861810684, -0.02771337516605854, -0.05140211805701256, -0.025627972558140755, 0.029399098828434944, -0.03384140878915787, 0.014285720884799957, -0.0040476019494235516, 0.03406374156475067, 0.04464743286371231, 0.032365791499614716, -0.009417231194674969, -0.04780019074678421, 0.0249595008790493, -0.01889130286872387, -0.04741816595196724, -0.04472098499536514, -0.005259658209979534, -0.03502257913351059, -0.03832172229886055, 0.044361941516399384, -0.03467512130737305, 0.0442432276904583, 0.003817742457613349, 0.005249341484159231, -0.04134940728545189, -0.049961917102336884, -0.032032303512096405, 0.007503132335841656, 0.024410521611571312, -0.016962772235274315, -0.021701591089367867, 0.028294233605265617, 0.04937344044446945, 0.021965036168694496, -0.003835556097328663, -0.04314591363072395, -0.042227983474731445, -0.03776917979121208, -0.019704464823007584, 0.02109554223716259, 0.00849019456654787, 0.03982684388756752, -0.022697413340210915, -0.0379066988825798, -0.01868455484509468, -0.01895301602780819, 0.007689131889492273, 0.0056207869201898575, -0.01592119224369526, -0.034434426575899124, -0.043364811688661575, 0.01256282813847065, -0.028409717604517937, -0.018240178003907204, 0.01828399859368801, -0.028502948582172394, -0.020121101289987564, -0.032028235495090485, 0.002974209375679493, 0.005950932390987873, 0.020427826792001724, 0.04568390175700188, 0.044147007167339325, 0.050394438207149506, -0.03406936302781105, 0.03171901777386665, -0.03517681732773781, 0.00042541202856227756, 0.03106093779206276, -0.020093843340873718, 0.008376513607800007, 0.035527028143405914, -0.0349465049803257, 0.0010908030671998858, -0.023422379046678543, 0.0387958362698555, -0.039824169129133224, -0.014080610126256943, -0.04214518889784813, 0.02985895238816738, 0.027682766318321228, 0.011217650957405567, -0.025907061994075775, -0.03672254830598831, 0.027433622628450394, -0.010858317837119102, -0.019800269976258278, -0.010783903300762177, 0.052410099655389786, -0.006144190672785044, 0.010146891698241234, 0.05204528570175171, 0.046394262462854385, -0.052593063563108444, 0.04861479252576828, 0.010552730411291122, 0.05053497478365898, 0.03215090185403824, 0.023387575522065163, -0.039746470749378204, -0.03581051155924797, 0.027616821229457855, -0.02014445886015892, 0.04974965751171112, -0.011795178055763245, 0.0275468397885561, 0.04033292084932327, -0.05130933225154877, -0.05216781422495842, -0.018535854294896126, -0.0508485846221447, 0.03948443382978439, 0.0008877379004843533, 0.038166843354701996, 0.00877402164041996, -0.005168394185602665, -0.015252598561346531, 0.01217824500054121, -0.027125835418701172, 0.05087592825293541, 0.012902340851724148, -0.0021018434781581163, 0.044673532247543335, 0.0435110479593277, -0.01159888505935669, 0.027366429567337036, -0.046743135899305344, 0.023589756339788437, -0.0014246898936107755, -0.005089443176984787, 0.029658185318112373, 0.04102257266640663, 0.004467041231691837, -0.014433604665100574, 0.0322188064455986, 0.011225210502743721, -0.016870662569999695, 0.0453047938644886, -0.024453066289424896, 0.02045140415430069, -0.004552104510366917, 0.006042211316525936, -0.02111092396080494, -0.03177917003631592, -0.01178209949284792, 0.04158036783337593, 0.0001603394775884226, -0.03398103639483452, 0.004193482454866171, -0.0329916812479496, 0.04448411241173744, 0.04862821474671364, 0.0007202115375548601, 0.04788288101553917, 0.04584848880767822, -0.005268779117614031, 0.01921393908560276, 0.032601527869701385, -0.017967291176319122, 0.04817821830511093, -0.02660197950899601, -0.05145293474197388, -0.026975005865097046, 0.04549380764365196, 0.042740028351545334, -0.018041197210550308, 0.05218746140599251, -0.01063617318868637, -0.045347392559051514, -0.02798515558242798, 0.009411557577550411, 0.02932213805615902, -0.03632676601409912, 0.04982331767678261, -0.03861929103732109, -0.021549208089709282, 0.028137855231761932, -0.02389666810631752, 0.03992004692554474, 0.005729705095291138, 0.013396228663623333, 0.022026529535651207, 0.0059435549192130566, 0.0451786108314991, 0.020262861624360085, -0.04876197129487991, 0.012337042018771172, 0.04419839382171631, 0.05185873061418533, -0.029164941981434822, 0.04963857680559158, -0.052402980625629425, 0.00954781100153923, 0.02101355791091919, -0.0336831696331501, -0.04530014842748642, -0.025409618392586708, 0.0029871277511119843, 0.03521758317947388, 0.02165975607931614, 0.015669623389840126, 0.00018585134239401668, -0.036971427500247955, 0.044826023280620575, -0.017279483377933502, -0.004377160221338272, -0.0009030679357238114, -0.05082325264811516, -0.023351598531007767, 0.037655461579561234, 0.03569304943084717, 0.04871972277760506, -0.023790160194039345, -0.030310126021504402, 0.011892744340002537, -0.035121869295835495, -0.03516406938433647, 0.044326022267341614, 0.05047692731022835, 0.0006244378164410591, 0.011603675782680511, 0.00665364321321249, 0.05174313113093376, -0.03944215923547745, -0.05188232660293579, -0.0018648096593096852, -0.0168644767254591, 0.024744179099798203, 0.015856195241212845, 0.0197973120957613, 0.04509785398840904, -0.02428765594959259, -0.03161720559000969, 0.04407946765422821, -0.034059613943099976, -0.018986791372299194, 0.05228361487388611, 0.020391808822751045, -0.044995371252298355, -0.01769173890352249, -0.039186954498291016, -0.04398481920361519, 0.046685878187417984, -0.0014149668859317899, 0.04661254957318306, 0.03796759992837906, 0.050350166857242584, 0.0326579287648201, 0.00265882327221334, 0.046961113810539246, 0.0018840671982616186, 0.006021788343787193, -0.033391378819942474, 0.0026453013997524977, -0.03865952789783478, -0.0036977024283260107, -0.0442097932100296, 0.022982778027653694, -0.0016352786915376782, -0.009330695495009422, -0.05161970853805542, 0.0030341320671141148, -0.026549430564045906, 0.005893236957490444, 0.03880021348595619, -0.021548805758357048, 0.043825507164001465, -0.05036650598049164, -0.0161662008613348, 0.012647983618080616, 0.023563949391245842, 0.0016084309900179505, 0.05212978273630142, 0.013113092631101608, 0.023812629282474518, -0.02418936975300312, 0.025861047208309174, 0.02195306494832039, 0.02296990156173706, -0.030036812648177147, -0.04472595825791359, -0.015700144693255424, -0.018949704244732857, 0.011358999647200108, 0.010389557108283043, -0.032924700528383255, -0.04102931171655655, -0.034129902720451355, -0.022312505170702934, 0.04390823841094971, -0.02747384086251259, 0.04010472446680069, -0.04471882805228233, -0.014418940991163254, 0.028500279411673546, -0.021937105804681778, -0.01848158985376358, -0.026182016357779503, 0.01768518052995205, 0.014194455929100513, -0.013590238057076931, 0.016049101948738098, -0.0058585344813764095, -0.04786258190870285, -0.003211973700672388, -0.05013006180524826, 0.029763666912913322, 0.019853726029396057, -0.004218685440719128, -0.0033855787478387356, -0.005169580224901438, 0.035235609859228134, -0.00907067395746708, 0.026391668245196342, -0.00291978451423347, -0.02024998888373375, -0.038406673818826675, 0.039249222725629807, 0.051579706370830536, 0.0504973940551281, -0.0049993861466646194, -0.01184813492000103, 0.02375330589711666, -0.024776769801974297, 0.04230769723653793, -0.0190786924213171, 0.05063340812921524, -0.0072674136608839035, -0.051109202206134796, -0.032982662320137024, -0.005597809329628944, 0.04553404077887535, 0.050052039325237274, -0.03270240128040314, 0.004768809303641319, -0.049607500433921814, -0.02281758561730385, 0.0018853471847251058, -0.015224161557853222, -0.04871297627687454, 0.03614719212055206, -0.01807842031121254, -0.03409558907151222, 0.039902426302433014, -0.04991321265697479, 0.03482149541378021, -0.04550579935312271, 0.03287319093942642, -0.03640885278582573, 0.04586752876639366, 0.0017869062721729279, -0.00768965482711792, -0.05222709849476814, -0.025613613426685333, 0.004237810615450144, 0.009434957057237625, -0.02997659519314766, 0.034015338867902756, 0.03435235098004341, 0.03955334052443504, 0.03488972410559654, 0.011652542278170586, -0.03148788958787918, -0.001608839025720954, 0.03878360614180565, -0.03020341694355011, -0.00033871742198243737, 0.023221144452691078, -0.01368297915905714, -0.007615308277308941, -0.04456273838877678, -0.010269111953675747, 0.03534197807312012, -0.024817707017064095, 0.04172615706920624, 0.040719009935855865, 0.014715456403791904, -0.006714759860187769, 0.028579512611031532, -0.029513616114854813, 0.003530326299369335, -0.013941884972155094, 0.05236344411969185, 0.03855574503540993, 0.031823668628931046, -0.032206419855356216, 0.023129558190703392, 0.010120060294866562, -0.043125517666339874, 0.0009045952465385199, -0.039243899285793304, -0.021665463224053383, 0.017719829455018044, -0.03366363048553467, -0.012259867042303085, -0.009312433190643787, 0.03060363419353962, -0.029510660097002983, -0.005783985834568739, 0.019038362428545952, 0.03373642638325691, 0.033139921724796295, 0.04663072153925896, 0.00039784290129318833, -0.050090014934539795, 0.026740415021777153, -0.041338201612234116, -0.04982927069067955, -0.001630951650440693, 0.030709978193044662, 0.011349542066454887, 0.0034007250797003508, -0.04299356788396835, -0.046829771250486374, 0.018911590799689293, -0.04503763094544411, -0.0357964001595974, 0.025945506989955902, -0.03358334302902222, -0.034742508083581924, 0.03152412548661232, -0.02264147624373436, -0.006158286239951849, -0.015106327831745148, -0.00510466517880559, 0.022592050954699516, -0.03279808163642883, -0.015935087576508522, 0.009382079355418682, 0.024687865749001503, -0.016011228784918785, -0.026552019640803337, -0.02851708233356476, 0.005109680350869894, -0.04310987889766693, 0.03440243378281593, -0.03796171024441719, 0.023045403882861137, -0.02905932255089283, 0.0195402130484581, 0.0007162973051890731, -0.034133557230234146, 0.006130418740212917, 0.001746651716530323, 0.01959417015314102, 0.04558410122990608, 0.03818676620721817, 0.032637059688568115, 0.025038322433829308, 0.016696147620677948, -0.003502720035612583, 0.014939784072339535, 0.01225694827735424, 0.03805229812860489, 0.04085227847099304, -0.03448977321386337, 0.01576123759150505, 0.05220145732164383, -0.04256541281938553, 0.018599580973386765, -0.05092202499508858, 0.017426621168851852, -0.003768736496567726, 0.041539110243320465, 0.05049263685941696, 0.0330609530210495, 0.049787893891334534, 0.026771515607833862, -0.012143861502408981, 0.048730526119470596, -0.026418114081025124, -0.009460944682359695, 0.007155555300414562, 0.038892798125743866, -0.04046187549829483, -0.0030209380201995373, -0.0011783940717577934, 0.030658600851893425, -0.04171408340334892, -0.03746262192726135, 0.018838360905647278, -0.019214848056435585, 0.02795068360865116, -0.002942413091659546, -0.026508906856179237, -0.04601888731122017, 0.041981372982263565, -0.04436219856142998, 0.04518183693289757, -0.032433681190013885, 0.007500955369323492, -0.03442172706127167, -0.03212206810712814, -0.04466058686375618, 0.03021889366209507, 0.04445219039916992, 0.02525763399899006, 0.04295843839645386, -0.02387126348912716, -0.030274223536252975, -0.03137871250510216, -0.0471840426325798, 0.03786788508296013, 0.039959125220775604, 0.04732656478881836, 0.010273084044456482, 0.023942410945892334, 0.04499617964029312, 0.04879011958837509, 0.04275216534733772, 0.0263475701212883, -0.02985416352748871, -0.0373791866004467, 0.023756979033350945, 0.008521564304828644, -0.019151505082845688, 0.03490471467375755, 0.021337786689400673, -0.030537962913513184, 0.043653566390275955, 0.00675773061811924, -0.04542986676096916, -0.03824341297149658, -0.03688495233654976, 0.022528400644659996, 0.02241317555308342, 0.005436727777123451, 0.037088848650455475, 0.006817344110459089, -0.019077930599451065, 0.01741677336394787, -0.017476113513112068, 0.04675472527742386, 0.03330158442258835, -0.028539974242448807, -0.027073359116911888, -0.0133596770465374, -0.009850629605352879, -0.005329967476427555, -0.026154469698667526, -0.016319837421178818, -0.04634422808885574, 0.04792096093297005, -0.036697130650281906, 0.010658095590770245, -0.0361180417239666, -0.032708704471588135, -0.01983349584043026, -0.0312009546905756, -0.030276281759142876, -0.04740317165851593, 0.04045478254556656, 0.017438054084777832, 0.0460967980325222, -0.03553437814116478, -0.05146433040499687, 0.04092145711183548, -0.009212521836161613, 0.01759464293718338, 0.04780144989490509, -0.0017358055338263512, -0.02476690709590912, 0.03635123372077942, 0.02090342715382576, -0.045530766248703, 0.03590044751763344, 0.02093982696533203, 0.034827157855033875, 0.028515061363577843, 0.04265575483441353, 0.05105843394994736, 0.003104928880929947, 0.038395386189222336, -0.024329930543899536, 0.01643037237226963, 0.03745240345597267, -0.05184163898229599, -0.03630807250738144, -0.026200318709015846, 0.016726959496736526, 0.0266970694065094, -0.043344881385564804, -0.030988579615950584, -0.0022134296596050262, 0.017608642578125, -0.04216594621539116, -0.04145854339003563, -0.01015276275575161, 0.0254336167126894, 0.01854700967669487, 0.03772372379899025, 0.0404743030667305, 0.013351887464523315, -0.0465761236846447, 0.047923289239406586, -0.037365082651376724, 0.04392365738749504, -0.052454933524131775, -0.05193692818284035, 0.03664597496390343, 0.044871579855680466, 0.019757986068725586, 0.027344515547156334, 0.03059142455458641, -0.0014919922687113285, 0.011195477098226547, -0.03834695369005203, -0.03548189625144005, 0.027171367779374123, -0.004207544028759003, -0.04362903907895088, 0.03284443914890289, -0.050439100712537766, -0.027401519939303398, -0.005873195361346006, -0.03705859184265137, -0.0321597158908844, 0.018934885039925575, 0.022789960727095604, 0.01145614217966795, -0.04043795168399811, 0.009562972001731396, 0.03436409682035446, -0.05005524307489395, 0.001607899321243167, 0.007584456354379654, -0.038552939891815186, 0.0476469025015831, -0.029031969606876373, 0.04182423651218414, 0.015045853331685066, 0.047602273523807526, 0.030494341626763344, 0.05140397697687149, 0.00931160245090723, 0.04866570606827736, 0.009556922130286694, -0.004345412366092205, -0.002888110000640154, -0.016114339232444763, 0.016931405290961266, 0.009781848639249802, 0.041358672082424164, 0.037074387073516846, 0.04528416320681572, 0.027278205379843712, 0.03786441683769226, -0.0031066075898706913, -0.010455802083015442, 0.0037398026324808598, -0.02484511397778988, -0.01748267002403736, -0.011428705416619778, 0.015657130628824234, -0.03445073589682579, -0.03581234812736511, 0.018502457067370415, 0.05206349492073059, -0.036139290779829025, -0.006760158576071262, 0.00031140955979935825, -0.04886365309357643, -0.03353865444660187, 0.036392297595739365, -0.0025431609246879816, 0.04410751536488533, -0.021246830001473427, -0.03677096962928772, -0.0007821567123755813, -0.038792431354522705, -0.03512357175350189, 0.04273383319377899, 0.00024785171262919903, 0.015981223434209824, -0.003584217978641391, -0.010657197795808315, -0.0024402723647654057, 0.016889339312911034, -0.04905123636126518, -0.04546360671520233, 0.05235190689563751, 0.03914901241660118, -0.03379026800394058, 0.021653534844517708, 0.023790251463651657, 0.042565811425447464, -0.018320932984352112, -0.011819141916930676, 0.02623938024044037, -0.027997031807899475, 0.02382156439125538, 0.014654242433607578, 0.0315605066716671, -0.0400424525141716, -0.021260462701320648, -0.0499432347714901, 0.0475836843252182, -0.046012166887521744, 0.01218350138515234, -0.019323773682117462, -0.04869956895709038, 0.045111604034900665, -0.049240946769714355, 0.033220380544662476, 0.049091462045907974, 0.033520739525556564, -0.04650074243545532, -0.007109389174729586, -0.009439501911401749, -0.04198569431900978, -0.014973918907344341, -0.024054648354649544, 0.041550420224666595, 0.003667669603601098, 0.05199816823005676, -0.04118548333644867, -0.04903313145041466, -0.030881259590387344, 0.021533740684390068, -0.0295560359954834, -0.02117105759680271, 0.02696986310184002, 0.012187101878225803, -0.023472635075449944, 0.009289693087339401, -0.03249046951532364, -0.014678183943033218, -0.03553178533911705, 0.008333482779562473, -0.016992267221212387, 0.02602037973701954, -0.036032453179359436, -0.02549261786043644, -0.02857927791774273, 0.03590166196227074, 0.005620922893285751, -0.04283599182963371, 0.011220548301935196, 0.020305495709180832, -0.024673331528902054, -0.018519241362810135, -0.017792904749512672, 0.018135743215680122, 0.0023487049620598555, 0.019420161843299866, -0.00010533649765420705, 0.04466135427355766, 0.012815101072192192, -0.04900680109858513, -0.011899886652827263, 0.03377228602766991, 0.05134906992316246, 0.026852892711758614, -0.023672159761190414, -0.01867806166410446, 0.019920608028769493, 0.014523698948323727, -0.026413418352603912, 0.004743584897369146, -0.007813620381057262, -0.030487092211842537, -0.006390196271240711, -0.03172214329242706, -0.048719990998506546, -0.025989172980189323, -0.050036314874887466, 0.045044947415590286, -0.03165961056947708, 0.04720785468816757, -0.03778701648116112, 0.01259153988212347, -0.002234943211078644, 0.03022761084139347, 0.049645259976387024, 0.030458658933639526, 0.039976079016923904, 0.03298267722129822, -0.029246242716908455, 0.028540102764964104, 0.04845249652862549, -0.01758761890232563, 3.716942228493281e-05, -0.04752407968044281, 0.01025343593209982, 0.03193776682019234, -0.01737789809703827, -0.02927873469889164, 0.02943865768611431, 0.01565156690776348, -0.051540713757276535, 0.04453927278518677, 0.01940106227993965, -0.04298461973667145, -0.03075276128947735, 0.04538653790950775, -0.0023104087449610233, 0.026396986097097397, -0.03135477006435394, 0.05233960971236229, -0.03315569460391998, -0.0017192065715789795, -0.04379285126924515, -0.0023714748676866293, -0.003017532406374812, 0.04860905930399895, 0.02124859392642975, 0.037689194083213806, 0.05173508822917938, -0.0030432604253292084, 0.0138485636562109, -0.008800335228443146, 0.011206705123186111, 0.028645358979701996, -0.03059951774775982, -0.030168520286679268, 0.019872931763529778, 0.0150422602891922, -0.048455990850925446, -0.014275852590799332, -0.02642655186355114, -0.04150655120611191, 0.01707770675420761, -0.018690001219511032, 0.04013592749834061, 0.02780723385512829, 0.02792758122086525, -0.036891013383865356, 0.03629866614937782, 0.035910073667764664, 0.044304560869932175, 0.0041955518536269665, 0.037410296499729156, 0.009796181693673134, 0.05013240501284599, 0.04879724234342575, 0.004724013619124889, -0.03286651521921158, -0.01867290772497654, -0.046192266047000885, -0.03187580034136772, -0.021630529314279556, -0.03849485144019127, 0.002284895395860076, 0.03931567072868347, 0.0431436263024807, 0.04211828112602234, -0.018115265294909477, -0.05277150496840477, 0.03793890401721001, -0.02582687698304653, 0.04401922598481178, 0.04212963953614235, 0.04233737662434578, 0.006387385539710522, 0.013852279633283615, -0.001878378796391189, -0.02854863367974758, 0.036533206701278687, 0.005582908634096384, 0.007979357615113258, -0.007070005871355534, 0.03942433372139931, 0.05182156711816788, 0.02119496278464794, 0.0007447815150953829, -0.015846136957406998, 0.018586432561278343, -0.02335127256810665, -0.01617051474750042, 0.02969023399055004, 0.013452382758259773, 0.049559544771909714, 0.02092641033232212, 0.025995220988988876, 0.04837283492088318, -0.025864992290735245, 0.01270760502666235, -0.052541583776474, 0.01660395972430706], [0.04386839643120766, 0.035687804222106934, 0.0451602041721344, 0.003374803112819791, 0.013281792402267456, -0.02413378283381462, -0.007263358682394028, 0.03575444594025612, -0.048830121755599976, -0.029846489429473877, 0.04489657282829285, 0.01729566976428032, 0.031078726053237915, 0.0330498106777668, -0.015124756842851639, 0.05241244658827782, 0.03804609552025795, 0.005760301370173693, -0.03583897277712822, -0.036237552762031555, 0.0059813037514686584, -0.03692243993282318, -0.00322294095531106, 0.0031810307409614325, 0.03203018382191658, 0.006038696970790625, -0.015102800913155079, 0.020860452204942703, 0.03765709698200226, 0.007569017354398966, -0.028523897752165794, -0.03807193785905838, -0.037699054926633835, -0.032485928386449814, 0.03543710708618164, -0.019405212253332138, 0.02156926691532135, -0.047743409872055054, -0.034081727266311646, 0.009914296679198742, -0.03779708221554756, 0.016988808289170265, -0.030490372329950333, 0.0097760409116745, -0.03347306326031685, 0.012882743030786514, 0.047589387744665146, 0.052886053919792175, 0.05070596560835838, 0.033472418785095215, -0.03749488666653633, 0.01489410549402237, 0.004204944241791964, -0.04137419909238815, 0.021862564608454704, 0.03685580939054489, -0.051049016416072845, 0.015145525336265564, -0.02408876270055771, -0.025055909529328346, 0.0168155450373888, -0.050863005220890045, -0.045795101672410965, -0.027288831770420074, 0.03198931738734245, -0.002602264517918229, 0.003579824697226286, 0.02511058747768402, 0.019001450389623642, -0.04373006150126457, 0.025599077343940735, 0.042672861367464066, 0.027666471898555756, 0.0014386982657015324, -0.005204413551837206, 0.006505858153104782, -0.0037510180845856667, 0.05304870009422302, -0.0001116886778618209, 0.00989564973860979, 0.0017144727753475308, -0.02612690068781376, -0.005988194141536951, 0.05385909229516983, 0.04548570513725281, -0.05224529281258583, -0.011935858987271786, 0.04217981547117233, -0.004339737351983786, -0.04951660335063934, -0.0075703891925513744, 0.023876668885350227, -0.03286135196685791, 0.03560580685734749, -0.051329340785741806, -0.042063795030117035, 0.03734847530722618, 0.04802568256855011, 0.0019943767692893744, 0.0022576022893190384, -0.016863267868757248, -0.037111807614564896, -0.018163764849305153, 0.052435725927352905, -0.04913342744112015, 0.008323163725435734, -0.04099896550178528, -0.02481907606124878, 0.03605065867304802, 0.013502066023647785, 0.01979854702949524, -0.042910367250442505, 0.04461453855037689, -0.02423224039375782, -0.017195846885442734, -0.020124346017837524, -0.01455678790807724, 0.026498544961214066, -0.025007907301187515, -0.04202060401439667, -0.033199336379766464, -0.024232126772403717, -0.025677789002656937, 0.027500035241246223, 0.02604249119758606, 0.018887601792812347, 0.05286892503499985, -0.041123148053884506, -0.019701220095157623, -0.044000864028930664, 0.012898656539618969, -0.04599781334400177, -0.02789812535047531, 0.05057520419359207, -0.03683994337916374, 0.022766532376408577, 0.049076300114393234, -0.036642007529735565, 0.0021223595831543207, 0.05064789205789566, 0.040782757103443146, 0.009931378997862339, -0.04105914756655693, -0.0057928357273340225, 0.012299791909754276, 0.03775038942694664, -0.01874976046383381, 0.05108692869544029, -0.001595210749655962, 0.04378247261047363, -0.024396536871790886, -0.020335981622338295, 0.029031828045845032, 0.05216945335268974, -0.010678768157958984, 0.0030455016531050205, 0.016837788745760918, -0.01985033228993416, -0.03537100926041603, -0.008804406970739365, -0.015414819121360779, 0.052178844809532166, 0.027371076866984367, -0.0001228882756549865, -0.039091210812330246, -0.007420849986374378, -0.05163361504673958, -0.045005615800619125, 0.010084250941872597, 0.012694168835878372, -0.014051361009478569, -0.023547137156128883, 0.012651383876800537, 0.019972262904047966, -0.014376826584339142, -0.004854686092585325, 0.003614472458139062, -0.04632773995399475, 0.005896248389035463, 0.016651099547743797, 0.023659920319914818, -0.044254373759031296, 0.01997525617480278, -0.025231122970581055, 0.004915954545140266, 0.04684630408883095, 0.024864034727215767, -0.008751871064305305, -0.054156042635440826, -0.02661755494773388, -0.01867211051285267, -0.02381414733827114, 0.053401824086904526, 0.05449043586850166, 0.04296698793768883, -0.05037014186382294, -0.006907129660248756, 0.027446860447525978, -0.04373945668339729, 0.05385727435350418, -0.03193289786577225, 0.0352400541305542, -0.003445908660069108, -0.054912466555833817, 0.052027150988578796, -0.040329042822122574, -0.03953014314174652, -0.036665961146354675, 0.05138526111841202, 0.008534223772585392, -0.02781585231423378, 0.0032835169695317745, 0.007415065076202154, 0.051071472465991974, -0.011060892604291439, -0.05116724595427513, -0.014454337768256664, -0.007281314581632614, -0.040452126413583755, 0.03250553458929062, -0.04304066300392151, 0.024633754044771194, 0.042618777602910995, -0.05289136618375778, 0.04626719653606415, -0.0034605786204338074, -0.032333776354789734, -0.030089201405644417, 0.046176277101039886, -0.029847396537661552, 0.0417482890188694, -0.03451976180076599, 0.03832074999809265, 0.04414239898324013, -0.022396288812160492, -0.015258127823472023, -0.013693327084183693, -0.015971830114722252, 0.014726812019944191, 0.0018178989412263036, -0.03006824478507042, 0.002628632355481386, -0.019702522084116936, -0.0294575784355402, 0.02808419242501259, 0.014131044037640095, -0.011936699971556664, -0.005179518833756447, 0.020953243598341942, -0.020012356340885162, -0.011719269677996635, 0.05422007665038109, -0.048382826149463654, -0.041805166751146317, -0.045086562633514404, 0.010854153893887997, 0.029186110943555832, 0.03930700942873955, -0.054765015840530396, 0.05015990510582924, 0.054903361946344376, -0.013249530456960201, -0.05069870501756668, -0.04809221997857094, -0.0366959385573864, 0.008173907175660133, -0.013543862849473953, 0.04866174980998039, 0.02582394704222679, 0.001670992118306458, 0.009500044398009777, -0.023306114599108696, 0.003980725537985563, -0.015669044107198715, -0.038514722138643265, -0.039746351540088654, -0.04165656864643097, -0.006498812697827816, -0.02125101163983345, 0.0007487194961868227, 0.03687238693237305, -0.03883178532123566, -0.054269321262836456, 0.004649471957236528, -0.04810730740427971, -0.03306873142719269, 0.017555546015501022, 0.020400211215019226, 0.02581666223704815, 0.011335957795381546, 0.045674365013837814, 0.03874685987830162, -0.038337595760822296, 0.0003488031215965748, -0.02847040817141533, 0.01934126764535904, 0.02693220041692257, 0.004113045055419207, 0.01731727086007595, 0.015286005102097988, 0.007838079705834389, -0.020542707294225693, -0.041794199496507645, 0.04823950678110123, -0.013545974157750607, 0.03658642992377281, -0.003872738452628255, -0.027185935527086258, -0.028832683339715004, 0.010889433324337006, 0.015965979546308517, -0.0421447828412056, 0.028900912031531334, -0.01810550130903721, -0.050751589238643646, 0.004802201874554157, -0.006070153322070837, 0.01502826064825058, -0.04178771376609802, 0.027813980355858803, -0.052546240389347076, 0.04855479300022125, -0.04639831557869911, 0.039608873426914215, -0.04496534913778305, 0.02185232937335968, 0.003921173512935638, -0.03254494071006775, -0.02983693592250347, 0.022181369364261627, -0.00817263312637806, 0.020109865814447403, 0.003905994351953268, -0.0019130527507513762, -0.04497334361076355, -0.010828104801476002, -0.02530202828347683, -0.03791293874382973, -0.00015397879178635776, -0.04869526997208595, -0.02493138052523136, -0.048678234219551086, 0.04861760884523392, -0.01834810897707939, -0.021766243502497673, -0.0379939004778862, -0.012386386282742023, 0.02098832279443741, 0.01725984551012516, 0.0498078390955925, 0.03975243493914604, 0.04784118011593819, -0.031442638486623764, -0.009945819154381752, 0.052151165902614594, 0.004633553326129913, -0.053924523293972015, 0.044586844742298126, -0.021586861461400986, 0.0018949855584651232, -0.03804514557123184, -0.009859130717813969, -0.005198498722165823, 0.0019660377874970436, -0.031384795904159546, -0.01445454265922308, -0.022003477439284325, 0.02656720019876957, -0.05154736340045929, 0.03706062212586403, 0.014566635712981224, 0.04812352731823921, -0.011255319230258465, -0.01034462358802557, 0.0115280132740736, 0.046466004103422165, 0.028074394911527634, -0.04541737586259842, 0.002095924923196435, -0.03530241921544075, 0.05072827637195587, 0.04312295466661453, -0.010996757075190544, 0.01380574144423008, -0.032739102840423584, -0.005682655144482851, 0.03044702857732773, 0.0011949387844651937, -0.003776596626266837, -0.004398614168167114, -0.011167176999151707, 0.04883535951375961, 0.034493621438741684, -0.04126092046499252, 0.040105972439050674, -0.013678198680281639, 0.018078235909342766, -0.02669333666563034, -0.040532466024160385, 0.022046145051717758, 0.039735838770866394, 0.030610712245106697, -0.0027459857519716024, -0.015527967363595963, 0.034998971968889236, 0.023650461807847023, -0.018008222803473473, 0.027068523690104485, -0.011663428507745266, 0.030317673459649086, 0.031289685517549515, 0.03929446265101433, 0.01890277862548828, 0.007661874406039715, -0.04475390911102295, -0.021336553618311882, 0.004229540470987558, 0.026428349316120148, -0.023946169763803482, 0.04732664301991463, -0.007987316697835922, 0.03109162300825119, 0.02957254834473133, -0.011392644606530666, 0.030048079788684845, 0.008120497688651085, 0.03319353610277176, 0.04860379919409752, 0.032815203070640564, -0.009402971714735031, -0.04948653280735016, 0.0060425736010074615, 0.03980139270424843, -0.04288312792778015, -0.01587996631860733, 0.048005715012550354, 0.026579441502690315, -0.0173479150980711, -0.00640757055953145, 0.0009012146620079875, -0.0032492452301084995, -0.05398394912481308, 0.007826507091522217, -0.011367300525307655, -0.04211445897817612, -0.0008117191027849913, 0.004740617703646421, 0.053010959178209305, -0.02359921485185623, 0.02682923898100853, 0.014254927635192871, -0.011831685900688171, -0.04598859325051308, -0.0022767854388803244, -0.03406944125890732, -0.010235872119665146, -0.05477328971028328, -0.027536340057849884, 0.03788670524954796, 0.039841946214437485, -0.0499076284468174, -0.03513766825199127, -0.005844247993081808, -0.04052501171827316, -0.0319901704788208, 0.036217883229255676, -0.04686084762215614, 0.03890824690461159, -0.01648602820932865, 0.027461951598525047, 0.028285356238484383, -0.017843812704086304, -0.014963563531637192, 0.047304410487413406, -0.0025670891627669334, 0.04704302176833153, 0.024958791211247444, -0.020219292491674423, -0.029585154727101326, 0.02483995072543621, 0.014169870875775814, -0.03778336942195892, -0.014140059240162373, -0.009673596359789371, 0.038842204958200455, -0.041594162583351135, -0.024600323289632797, -0.020475920289754868, 0.007770922034978867, 0.029010118916630745, 0.04217226803302765, -0.050156958401203156, -0.009784810245037079, -0.04070082679390907, -8.147033804561943e-05, 0.0505334734916687, -0.03732301667332649, -0.02149534597992897, 0.035541288554668427, 0.05076667293906212, 0.047788407653570175, -0.030308794230222702, -0.03694034740328789, 0.0517868734896183, -0.003267301246523857, 0.03327632322907448, -0.04139145836234093, 0.011669657193124294, -0.016201529651880264, -0.005896779242902994, -0.0013428203528746963, -0.010787051171064377, -0.03553853929042816, 0.05404382199048996, 0.03400168567895889, -0.03228665515780449, -0.007277592085301876, 0.034368548542261124, 0.012163901701569557, -0.0026403050869703293, 0.05251481756567955, -0.03342616185545921, -0.03302185237407684, 0.0008742130012251437, 0.049540966749191284, 0.0448136143386364, 0.0323001854121685, -0.028130436316132545, -0.025232182815670967, -0.030270390212535858, -0.013529311865568161, -0.035969264805316925, 0.008346082642674446, 0.0147502226755023, -0.003839167533442378, -0.0037950584664940834, -0.04287992790341377, -0.019215252250432968, 0.00796834658831358, -0.03773868829011917, -0.053122200071811676, -0.0264253132045269, 0.009001389145851135, -0.015164642594754696, -0.01633578911423683, -0.03224946931004524, 0.03223630040884018, -0.013754438608884811, 0.046900223940610886, 0.033500052988529205, 0.011044754646718502, -0.01827402412891388, 0.05436554551124573, -0.02431912161409855, 0.0017182518495246768, 0.0196408499032259, -0.04490794986486435, 0.028252653777599335, 0.00079404964344576, -0.03577016666531563, -0.02762891910970211, 0.01053807232528925, 0.036804769188165665, 0.02639514021575451, 0.04518406465649605, 0.013664172030985355, -0.017625637352466583, 0.032645706087350845, -0.03959542140364647, 0.03995197266340256, -0.005397852510213852, -0.03900054842233658, 0.01034591905772686, -0.00956465769559145, -0.004199931863695383, 0.02391218952834606, 0.008323504589498043, -0.05152010917663574, 0.016010956838726997, 0.00019715326197911054, -0.01689334586262703, 0.0018265836406499147, 0.05141622945666313, -0.04358198121190071, 0.03564275801181793, 0.03955145552754402, -0.04996190220117569, -0.017147673293948174, 0.013099638745188713, 0.048282139003276825, -0.01257763709872961, 0.026833822950720787, -0.02931034192442894, -0.03198828548192978, 0.03836892172694206, 0.03570301830768585, -0.04904358461499214, 0.0009279872174374759, -0.038099389523267746, -0.02480352856218815, 0.03997192531824112, 0.037542738020420074, -0.0016652313061058521, 0.008308490738272667, 0.008179506286978722, -0.0002474545908626169, 0.008507588878273964, -0.0002042029082076624, -0.04869834706187248, -0.020055966451764107, 0.011632634326815605, -0.005991144105792046, 0.02729731798171997, 0.04075813293457031, -7.546350389020517e-05, -0.04120781645178795, 0.054531414061784744, -0.038930509239435196, -0.01744358241558075, -0.005501264706254005, -0.026777440682053566, 0.039296600967645645, -0.05429776385426521, -0.0003104831848759204, -0.043617844581604004, -0.04037868231534958, -0.002053966047242284, 0.025080617517232895, -0.02670576609671116, 0.05270138010382652, -0.04710633307695389, -0.0317414328455925, -0.025681789964437485, -0.04055828973650932, 0.04246314987540245, 0.05021674558520317, -0.026203300803899765, 0.043225567787885666, 0.008988803252577782, -0.016955040395259857, 0.002300363266840577, 0.007771691773086786, -0.007831729017198086, -0.011150741949677467, 0.054479438811540604, -0.028194637969136238, -0.014647583477199078, 0.04252335801720619, 0.019366130232810974, -0.024161165580153465, 0.00928060244768858, 0.006861342117190361, 0.036566946655511856, 0.04947562515735626, 0.03652716800570488, -0.007283891551196575, -0.03365316614508629, -0.02609625831246376, -0.05080750584602356, -0.020785391330718994, 0.03958697244524956, -0.01671428605914116, 0.011134469881653786, 0.006850094068795443, -0.03889407590031624, -0.0526852123439312, -0.023606980219483376, -0.046672508120536804, 0.03305067867040634, 0.02187172695994377, -0.0022052095737308264, 0.003986342344433069, 0.018305104225873947, 0.02545056864619255, -0.00963537860661745, 0.05330253764986992, -0.010828003287315369, -0.028739100322127342, 0.043098971247673035, 0.027282338589429855, 0.014047314412891865, -0.045846689492464066, 0.025403259322047234, 0.04988783970475197, -0.03218037635087967, -0.03968122601509094, -0.01732618920505047, -0.0022424813359975815, -0.005386580713093281, 0.04014144465327263, 0.049719274044036865, -0.05392152816057205, 0.021290753036737442, 0.009182864800095558, 0.05107727274298668, -0.04853913560509682, -0.035118211060762405, -0.034848470240831375, -0.00788895320147276, 0.025431104004383087, -0.02764168567955494, -0.03582527115941048, -0.025426600128412247, -0.017255352810025215, 0.027782602235674858, 0.03360307216644287, -0.016988413408398628, -0.045881398022174835, -0.017981892451643944, -0.011676604859530926, -0.005851661786437035, 0.05119343474507332, -0.04449555650353432, -0.00010236192611046135, -0.019854767248034477, 0.014697778970003128, -0.01121180597692728, -0.029590429738163948, 0.014057002030313015, -0.0505082942545414, -0.005140716675668955, 0.012731863185763359, 0.029786959290504456, 0.02790113538503647, 0.01220918633043766, -0.04787068068981171, 0.04578065872192383, 0.02606847696006298, -0.04160657152533531, 0.007790728006511927, 0.026296155527234077, -0.04216797649860382, 0.05096074938774109, -0.03536861762404442, 0.04240672290325165, 0.00874135922640562, 0.015321586281061172, -0.012915187515318394, 0.026570867747068405, -0.03882315382361412, -0.019516857340931892, -0.0036624413914978504, 0.019234593957662582, 0.02029193937778473, -0.04745400696992874, 0.053071144968271255, -0.04610922560095787, 0.038917962461709976, -0.046110041439533234, 0.04531512409448624, -0.016937805339694023, 0.029823506250977516, -0.04204191640019417, -0.04736859351396561, -0.03323391079902649, 0.0004981727106496692, 0.04256046190857887, 0.016908306628465652, 0.00844874419271946, -0.010214931331574917, -0.002767038531601429, 0.050222791731357574, 0.012638844549655914, 0.05039141699671745, -0.027914637699723244, 0.026130901649594307, -0.004827368073165417, 0.0319254994392395, -0.0038918317295610905, -0.02046738751232624, 0.009270782582461834, -0.027036191895604134, -0.04334399104118347, -0.026201635599136353, -0.044983621686697006, 0.04108653962612152, -0.05443572625517845, -0.029696308076381683, 0.048850517719984055, 0.028182335197925568, 0.052731189876794815, 0.03154616802930832, -0.045320089906454086, 0.04640565067529678, 0.009471050463616848, 0.00012622828944586217, -0.003547290340065956, 0.0018038415582850575, -0.030704325065016747, 0.052395034581422806, -0.041545320302248, -0.021205302327871323, 0.020808536559343338, 0.009707164019346237, -0.013017678633332253, -0.03518856689333916, 0.012535874731838703, 0.017010493203997612, 0.01764773763716221, 0.013277157209813595, 0.04567008465528488, 0.04325361177325249, -0.05037190765142441, -0.0009602734935469925, 0.04040702059864998, 0.023762475699186325, -0.027767809107899666, -0.03848089650273323, -0.037967972457408905, -0.023944083601236343, 0.004296439699828625, 0.016780907288193703, 0.008639682084321976, 0.030953524634242058, 0.022906454280018806, -0.01952289417386055, -0.017550667747855186, 0.047342486679553986, -0.05133705958724022, -0.019878525286912918, 0.0011351188877597451, 0.011555252596735954, 0.025509795174002647, 0.02383878268301487, 0.0015664392849430442, 0.02413085661828518, -0.046404093503952026, -0.011206397786736488, -0.05361480638384819, 0.00988470483571291, 0.03499116748571396, -0.008765412494540215, -0.013269436545670033, -0.02240544557571411, 0.04718993231654167, -0.015444225631654263, 0.02468542568385601, -0.032572150230407715, -0.015189322642982006, 0.036115966737270355, 0.022539159283041954, -0.0019145752303302288, -0.03240310773253441, 0.05395331606268883, -0.021661706268787384, -0.03582866117358208, 0.04081062972545624, 0.004132864065468311, 0.03376144915819168, 0.03175872564315796, 0.02312583103775978, -0.02654566615819931, -0.0483708456158638, -0.02535620518028736, 0.0241449736058712, -0.030002260580658913, -0.011875313706696033, 0.034585826098918915, -0.04658715799450874, -0.0033799349330365658, 0.014717404730618, 0.05409540981054306, -0.030440237373113632, -0.007978783920407295, 0.009307640604674816, -0.0030218183528631926, 0.025743156671524048, -0.013534991070628166, 0.04312920197844505, -0.03860391303896904, -0.040429141372442245, 0.02126379869878292, 0.0002640675229486078, 0.02525852620601654, -0.0017106710001826286, -0.030255403369665146, -0.01701347716152668, -0.01678347773849964, 0.02190958708524704, 0.02527380734682083, -0.017252368852496147, 0.0094605116173625, 0.03402463346719742, 0.05065951123833656, -0.04441551864147186, -0.05272874981164932, -0.03523217514157295, -0.05204075574874878, -0.03466036543250084, -0.018344201147556305, 0.006074198056012392, 0.04318412020802498, -0.040874820202589035, 0.023490196093916893, -0.005044248420745134, -0.02446729689836502, 0.050881773233413696, 0.031998198479413986, -0.023504016920924187, -0.050019022077322006, -0.038713645190000534, -0.017903894186019897, 0.04504355788230896, 0.009807126596570015, -0.006820969749242067, 0.006452120374888182, 0.008686346001923084, 0.04086220636963844, -0.04738783836364746, -0.03453027829527855, 0.04566665366292, -0.001967883901670575, 0.03233354911208153, 0.0072798216715455055, 0.024114836007356644, 0.027802258729934692, -0.031544558703899384, -0.01124037429690361, 0.04589056223630905, 0.047651153057813644, 0.022982928901910782, 0.043310970067977905, 0.0038349125534296036, -0.007114503066986799, 0.05481461435556412, 0.047297149896621704, -0.04966798052191734, -0.019998589530587196, -0.023633254691958427, -0.007040709722787142, -0.013845512643456459, 0.017743656411767006, 0.04467129334807396, -0.0489860437810421, 0.006738022435456514, -0.007190232630819082, 0.049080993980169296, 0.03334995359182358, 0.001335937064141035, 0.0062410528771579266, -0.030559737235307693, -0.007931523025035858, -0.051161911338567734, 0.020980186760425568, 0.0369800366461277, 0.0011367963161319494, -0.01317536924034357, 0.01637107878923416, -0.03381821885704994, -0.03797391057014465, -0.012774811126291752, -0.015423658303916454, 0.024698598310351372, 0.04353762045502663, 0.05147582292556763, 0.0023334084544330835, 0.04095946252346039, 0.03418240696191788, -0.04708709195256233, 0.03203050419688225, -0.05403473973274231, 0.032053664326667786, 0.018776996061205864, -0.01252526044845581, 0.005773619282990694, 0.045872852206230164, -0.05210646986961365, -0.020759006962180138, -0.02522045373916626, -0.03526541590690613, 0.04175350069999695, 0.021794427186250687, -0.014056519605219364, 0.009017960168421268, 0.03563526272773743, -0.045193642377853394, -0.02167770080268383, 0.0016981542576104403, 0.026089712977409363, -0.0477738082408905, 0.0024734034668654203, 0.04299107566475868, -0.01745978370308876, -0.034956932067871094, 0.025766512379050255, 0.009918875060975552, -0.03900879994034767, 0.03142731636762619, -0.007654087617993355, 0.0023229531943798065, 0.011801029555499554, 0.044005509465932846, 0.009706548415124416, -0.05262110382318497, -0.01925293169915676, -0.015910042449831963, -0.007118682377040386, -0.048119790852069855, 0.0013680156553164124, -0.014517417177557945, 0.0042925202287733555, -0.0346074253320694, 0.026887886226177216, -0.02740347757935524, -0.017061587423086166, -0.019618047401309013, 0.030823620036244392, -0.010121365077793598, -0.0027870347257703543, -0.032182104885578156, -0.046660955995321274, -0.01827176660299301, 0.01314826775342226, -0.012607570737600327, 0.014683123677968979, 0.042709480971097946, -0.0481724739074707, -0.03766268119215965, 0.011344174854457378, 0.006249253172427416, 0.011342229321599007, 0.041222039610147476, -0.049659363925457, 0.015184043906629086, 0.045033738017082214, -0.04009377956390381, 0.006915705278515816, 0.016102438792586327, -0.00999263022094965, 0.02903878502547741]], "dtype": "float32"}, {"__nd__": [0.01892348751425743, 0.01818183623254299], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(l2_normalize, json.loads(_TESTS), 'triton', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")